In [ ]:
print("[cell] setup", flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv || true")
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), torch.cuda.device_count(), flush=True)
PIP = os.environ.get("PIP_SPEC", 'vllm==0.32.0')   # --env PIP_SPEC=... / PIP_POST=... override the built-in install
if PIP and torch.cuda.is_available():
    sh(f"pip install -q {PIP}")
    if os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'):
        sh(os.environ.get("PIP_POST", 'pip uninstall -y torchaudio torchao'))
sh("pip list 2>/dev/null | grep -i -E '^(vllm|torch|transformers|qwen-asr|pandas|openpyxl) ' || true")


In [ ]:
print("[cell] write code files", flush=True)
FILES = {
    'llm.py': '"""Greedy chat generation shared by the screening runs: vLLM (LLM_BACKEND=vllm, default) or transformers (LLM_BACKEND=hf).\n\n    m = LLM("Qwen/Qwen3-4B-Instruct-2507", max_len=16384, tp=1)\n    texts = m.gen([[{"role": "system", "content": ...}, {"role": "user", "content": ...}], ...], max_tokens=512)\n\nOne process loads one repo; trying the next candidate (fp16 -> AWQ, vllm -> hf) is multirun.py\'s job, in a new process.\nPrompts are tokenized here (chat template, enable_thinking off unless think=True) and passed as token ids, so no double BOS.\nA prompt longer than max_len - max_tokens comes back as None and is counted in m.overflow (never crashes the batch).\nText after </think> is kept when a model still thinks. m.ntok_in / m.ntok_out count tokens of every call.\n"""\nimport os\nimport time\n\n# ChatML, for tokenizers that ship no chat template (vilm/vinallama-2.7b-chat): the format its model card describes\nCHATML = ("{% for m in messages %}{{ \'<|im_start|>\' + m[\'role\'] + \'\\n\' + m[\'content\'] + \'<|im_end|>\\n\' }}{% endfor %}"\n          "{% if add_generation_prompt %}{{ \'<|im_start|>assistant\\n\' }}{% endif %}")\n\n\nclass LLM:\n    def __init__(self, repo, max_len=16384, tp=1, think=False, **kw):\n        self.repo, self.max_len, self.think = repo, max_len, think\n        self.overflow = self.ntok_in = self.ntok_out = 0\n        self.backend = os.environ.get("LLM_BACKEND", "vllm")\n        t0 = time.time()\n        if self.backend == "vllm":\n            from vllm import LLM as V\n            self.eng = V(model=repo, dtype="float16", max_model_len=max_len, tensor_parallel_size=tp, seed=0,\n                         gpu_memory_utilization=float(os.environ.get("GPU_UTIL", "0.90")), trust_remote_code=True, **kw)\n            self.tok = self.eng.get_tokenizer()\n        else:\n            import torch\n            import transformers\n            from transformers import AutoModelForCausalLM, AutoTokenizer\n            import transformers.integrations.sdpa_attention as sd\n            if hasattr(sd, "use_gqa_in_sdpa"):  # T4: enable_gqa only runs in the math kernel (heads x L x L scores, OOM): repeat K/V instead\n                sd.use_gqa_in_sdpa = lambda *a, **k: False\n            dkw = "dtype" if int(transformers.__version__.split(".")[0]) >= 5 else "torch_dtype"\n            self.tok = AutoTokenizer.from_pretrained(repo, trust_remote_code=True)\n            cuda = torch.cuda.is_available()\n            load = lambda remote: AutoModelForCausalLM.from_pretrained(repo, **{dkw: torch.float16 if cuda else torch.float32},\n                                                                       device_map="auto" if cuda else None, trust_remote_code=remote).eval()\n            try:\n                self.model = load(True)\n            except ImportError as e:  # repo code written for an older transformers (Phi-4-mini: LossKwargs); the built-in class works\n                print(f"[llm] remote code failed ({e}); loading the built-in transformers class", flush=True)\n                self.model = load(False)\n        if not getattr(self.tok, "chat_template", None):\n            self.tok.chat_template = CHATML\n            print(f"[llm] {repo}: no chat template, using ChatML", flush=True)\n        print(f"[llm] {repo} loaded with {self.backend} in {time.time() - t0:.0f}s", flush=True)\n\n    def ids(self, msgs):\n        kw = dict(tokenize=False, add_generation_prompt=True, enable_thinking=self.think)\n        try:\n            text = self.tok.apply_chat_template(msgs, **kw)\n        except Exception:  # templates without a system role: fold it into the first user turn\n            if msgs[0]["role"] != "system":\n                raise\n            text = self.tok.apply_chat_template([dict(msgs[1], content=msgs[0]["content"] + "\\n\\n" + msgs[1]["content"])] + msgs[2:], **kw)\n        if os.environ.get("STRIP_EMPTY_THINK") == "1":  # models fine-tuned by PIPE-T1-04 train.py on Qwen3.5 (no empty think block)\n            text = text.replace("<think>\\n\\n</think>\\n\\n", "")\n        return self.tok(text, add_special_tokens=False).input_ids\n\n    def gen(self, batch, max_tokens):\n        ids = [self.ids(m) for m in batch]\n        ok = [i for i, x in enumerate(ids) if len(x) <= self.max_len - max_tokens]\n        self.overflow += len(ids) - len(ok)\n        self.ntok_in += sum(len(ids[i]) for i in ok)\n        out = [None] * len(batch)\n        if self.backend == "vllm":\n            from vllm import SamplingParams\n            sp = SamplingParams(temperature=0.0, max_tokens=max_tokens, seed=0)\n            res = self.eng.generate([{"prompt_token_ids": ids[i]} for i in ok], sp, use_tqdm=False)\n            for i, r in zip(ok, res):\n                out[i] = r.outputs[0].text\n                self.ntok_out += len(r.outputs[0].token_ids)\n        else:\n            import torch\n            bs = int(os.environ.get("HF_BATCH", "4"))\n            pad = self.tok.pad_token_id if self.tok.pad_token_id is not None else self.tok.eos_token_id\n            dev = next(self.model.parameters()).device\n            order = sorted(ok, key=lambda i: len(ids[i]))\n            for k in range(0, len(order), bs):\n                part = order[k:k + bs]\n                n = max(len(ids[i]) for i in part)\n                x = torch.tensor([[pad] * (n - len(ids[i])) + ids[i] for i in part], device=dev)\n                mask = torch.tensor([[0] * (n - len(ids[i])) + [1] * len(ids[i]) for i in part], device=dev)\n                with torch.no_grad():\n                    y = self.model.generate(input_ids=x, attention_mask=mask, max_new_tokens=max_tokens, do_sample=False, pad_token_id=pad)\n                for i, row in zip(part, y[:, n:].tolist()):\n                    row = [t for t in row if t != pad]\n                    self.ntok_out += len(row)\n                    out[i] = self.tok.decode(row, skip_special_tokens=True)\n        return [o.split("</think>")[-1].strip() if o is not None else None for o in out]\n',
    'multirun.py': '"""Run one worker process per model over the GPUs of a Kaggle T4 x2 box.\n\n    ONLY=tag,tag python -u multirun.py jobs.json worker.py [-- extra worker args]      (ONLY empty or unset: every job)\n\njobs.json: [{"tag": "q3-4b", "tp": 1, "env": {"HARNESS": "4"}, "cands": [{"repo": "...", "backend": "vllm", ...}, ...], ...}, ...]\n(env: extra environment of that job\'s worker processes)\nEach candidate runs as `python -u worker.py --job \'<job + candidate as json>\' extra...` in its own process (a failed vLLM\nengine leaves a dirty process); exit code 0 ends the job, anything else tries the next candidate (fp16 -> AWQ, vllm -> hf).\ntp 2 jobs run one at a time on both GPUs first; tp 1 jobs then run two at a time, one per GPU. No GPU: one at a time.\nAfter a candidate its Hugging Face cache folder is deleted (Kaggle disk) unless the next candidate uses the same repo, or KEEP_HF=1.\nEvery worker line is echoed with a [tag] prefix; a summary of which candidate won is written to multirun_summary.json.\n"""\nimport json\nimport os\nimport queue\nimport shutil\nimport subprocess\nimport sys\nimport threading\nimport time\nfrom pathlib import Path\n\nLOCK = threading.Lock()\n\n\ndef log(msg):\n    with LOCK:\n        print(msg, flush=True)\n\n\ndef ngpu():\n    try:\n        return len(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip().splitlines())\n    except FileNotFoundError:\n        return 0\n\n\ndef run_job(job, worker, extra, gpus):\n    for k, cand in enumerate(job["cands"]):\n        spec = dict(job, **cand, cand_index=k)\n        spec.pop("cands")\n        env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpus, PYTHONUNBUFFERED="1", LLM_BACKEND=cand.get("backend", "vllm"), **job.get("env", {}))\n        t0 = time.time()\n        log(f"[{job[\'tag\']}] start candidate {k}: {cand[\'repo\']} ({env[\'LLM_BACKEND\']}) on GPU \'{gpus}\'")\n        p = subprocess.Popen([sys.executable, "-u", worker, "--job", json.dumps(spec, ensure_ascii=False)] + extra,\n                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)\n        tail = []\n        for line in p.stdout:\n            line = line.rstrip()\n            tail = (tail + [line])[-15:]\n            if "it/s]" not in line and "s/it]" not in line:  # drop tqdm bars\n                log(f"[{job[\'tag\']}] {line}")\n        rc = p.wait()\n        nxt = job["cands"][k + 1]["repo"] if k + 1 < len(job["cands"]) else None\n        if os.environ.get("KEEP_HF") != "1" and (rc == 0 or nxt != cand["repo"]):  # keep the weights for a retry of the same repo\n            hub = Path(os.environ.get("HF_HOME", Path.home() / ".cache/huggingface")) / "hub"\n            shutil.rmtree(hub / ("models--" + cand["repo"].replace("/", "--")), ignore_errors=True)\n        log(f"[{job[\'tag\']}] candidate {k} rc {rc} after {time.time() - t0:.0f}s")\n        if rc == 0:\n            return dict(tag=job["tag"], repo=cand["repo"], cand_index=k, ok=True, sec=round(time.time() - t0))\n    return dict(tag=job["tag"], ok=False, tail=tail)\n\n\ndef main():\n    argv = sys.argv[1:]\n    extra = argv[argv.index("--") + 1:] if "--" in argv else []\n    argv = argv[:argv.index("--")] if "--" in argv else argv\n    jobs = json.load(open(argv[0], encoding="utf-8"))\n    worker = argv[1]\n    if os.environ.get("ONLY"):\n        keep = os.environ["ONLY"].split(",")\n        jobs = [j for j in jobs if j["tag"] in keep]\n    n = ngpu()\n    log(f"[multirun] {len(jobs)} jobs, {n} GPU(s): {[j[\'tag\'] for j in jobs]}")\n    done = []\n    if n < 2:\n        done = [run_job(j, worker, extra, "0" if n else "") for j in jobs]\n    else:\n        done = [run_job(j, worker, extra, "0,1") for j in jobs if j.get("tp", 1) > 1]\n        q = queue.Queue()\n        for j in jobs:\n            if j.get("tp", 1) == 1:\n                q.put(j)\n\n        def drain(g):\n            while True:\n                try:\n                    j = q.get_nowait()\n                except queue.Empty:\n                    return\n                done.append(run_job(j, worker, extra, g))\n\n        ts = [threading.Thread(target=drain, args=(str(g),)) for g in range(2)]\n        [t.start() for t in ts]\n        [t.join() for t in ts]\n    json.dump(done, open("multirun_summary.json", "w"), indent=1)\n    log("[multirun] summary: " + json.dumps([(d["tag"], d.get("repo"), d["ok"]) for d in done]))\n\n\nif __name__ == "__main__":\n    main()\n',
    'issue_worker.py': '"""GS-Issue worker: one local LLM finds the issues of each claim and proposes edits (same prompt for every model).\n\n    INPUTS=layer_a.jsonl,d_edit.jsonl python -u issue_worker.py --job \'{"tag": "q3-4b", "repo": "Qwen/Qwen3-4B-Instruct-2507", "tp": 1}\'\n\nRuns inside a Kaggle notebook built by tools/kaggle/build_nb.py with llm.py and multirun.py of experiments/AG-1.\nGreedy, thinking off. Input rows: {id, claim, evidence}. Output: out/<tag>__<input stem>.jsonl, one row per input row:\n{id, raw, parsed (or null)}; out/<tag>_summary.json. Env: LIMIT=n (first n rows of each input, smoke), MAX_TOKENS (900).\nThe prompt is the D-EDIT labelling guide (benchmark/gs_eval_v2/LABELING_GUIDE.md section 2), shortened; the taxonomy\nis benchmark/gs_issue/taxonomy.py (embedded next to this file).\n"""\nimport argparse\nimport json\nimport os\nimport re\nimport time\nfrom pathlib import Path\n\nfrom taxonomy import CODES, KINDS\n\nSYSTEM = "Bạn là kiểm toán viên soát xét tuyên bố môi trường trong báo cáo của doanh nghiệp Việt Nam. Bạn chỉ trả lời bằng JSON."\nPROMPT = """Đọc TUYÊN BỐ (và BẰNG CHỨNG nếu có). Tìm các vấn đề khiến tuyên bố có thể gây hiểu nhầm, rồi đề xuất sửa.\n\nMã vấn đề (chỉ dùng các mã này):\n{codes}\n\nBốn loại sửa được phép:\n{kinds}\n\nQuy tắc:\n- Chỉ nêu vấn đề có thật trong câu; câu đã đủ chỉ số, số, kỳ, năm gốc, phạm vi, nguồn và không nói quá thì "issues": [].\n- Câu không phải tuyên bố môi trường (tiêu đề, mục lục, câu tài chính, định nghĩa) thì "is_green_claim": false và không sửa.\n- BẰNG CHỨNG chỉ để hiểu bối cảnh (ví dụ: tổng phát thải thật ra tăng, số của công ty mẹ khác số hợp nhất, mục tiêu chưa đạt).\n- Không thêm bất kỳ con số nào không có trong tuyên bố. Thông tin chưa biết thì viết chỗ trống "[bổ sung: ...]".\n- Không đổi nghĩa, không biến mục tiêu thành thành tích, không xoá thông tin đúng.\n- "span" và "target_span" chép NGUYÊN VĂN một đoạn của tuyên bố.\n\nTUYÊN BỐ: {claim}\nBẰNG CHỨNG: {evidence}\n\nTrả lời đúng một đối tượng JSON:\n{{"is_green_claim": true, "issues": [{{"code": "...", "span": "..."}}], "edits": [{{"kind": "...", "rule_code": "...", "target_span": "...", "replacement": "..."}}], "rewritten_claim": "tuyên bố sau khi áp mọi sửa"}}"""\n\n\ndef parse(text):\n    if not text:\n        return None\n    text = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.M)\n    start = text.find("{")\n    while start != -1:\n        depth, in_str, esc = 0, False, False\n        for i in range(start, len(text)):\n            ch = text[i]\n            if in_str:\n                esc = (ch == "\\\\" and not esc)\n                if ch == \'"\' and not esc:\n                    in_str = False\n                continue\n            if ch == \'"\':\n                in_str = True\n            depth += {"{": 1, "}": -1}.get(ch, 0)\n            if depth == 0:\n                try:\n                    obj = json.loads(text[start:i + 1])\n                    return obj if isinstance(obj, dict) else None\n                except json.JSONDecodeError:\n                    break\n        start = text.find("{", start + 1)\n    return None\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument("--job")\n    job = json.loads(ap.parse_args().job)\n    tag = job["tag"]\n    inputs = [p for p in os.environ.get("INPUTS", "").split(",") if p]\n    rows = []\n    for path in inputs:\n        part = [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]\n        if os.environ.get("LIMIT"):\n            part = part[: int(os.environ["LIMIT"])]\n        rows += [(Path(path).stem, r) for r in part]\n    print(f"[{tag}] {len(rows)} rows from {inputs}, repo {job[\'repo\']}", flush=True)\n    from llm import LLM\n    t0 = time.time()\n    m = LLM(job["repo"], max_len=job.get("max_len", 4096), tp=job.get("tp", 1), **job.get("kw", {}))\n    load_s = time.time() - t0\n    codes = "\\n".join(f"- {k}: {v}" for k, v in CODES.items())\n    kinds = "\\n".join(f"- {k}: {v}" for k, v in KINDS.items())\n    prompts = [[{"role": "system", "content": SYSTEM},\n                {"role": "user", "content": PROMPT.format(codes=codes, kinds=kinds, claim=r["claim"],\n                                                          evidence=(r.get("evidence") or "(không có)")[:900])}] for _, r in rows]\n    t1 = time.time()\n    raw = []\n    for i in range(0, len(prompts), 64):\n        raw += m.gen(prompts[i:i + 64], max_tokens=int(os.environ.get("MAX_TOKENS", "900")))\n        print(f"[{tag}] generated {len(raw)}/{len(prompts)} ({time.time() - t1:.0f}s)", flush=True)\n    gen_s = time.time() - t1\n    Path("out").mkdir(exist_ok=True)\n    bad, files = 0, {}\n    for (stem, r), text in zip(rows, raw):\n        obj = parse(text)\n        bad += obj is None\n        files.setdefault(stem, []).append({"id": r["id"], "raw": text, "parsed": obj})\n    for stem, out in files.items():\n        with open(f"out/{tag}__{stem}.jsonl", "w", encoding="utf-8") as fh:\n            for o in out:\n                fh.write(json.dumps(o, ensure_ascii=False) + "\\n")\n    summary = {"tag": tag, "repo": job["repo"], "rows": len(rows), "unparsed": bad, "load_s": round(load_s), "gen_s": round(gen_s),\n               "ntok_in": getattr(m, "ntok_in", None), "ntok_out": getattr(m, "ntok_out", None), "overflow": getattr(m, "overflow", None)}\n    json.dump(summary, open(f"out/{tag}_summary.json", "w"), ensure_ascii=False)\n    print(f"[{tag}] SUMMARY {json.dumps(summary, ensure_ascii=False)}", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n',
    'taxonomy.py': '"""GS-Issue taxonomy: issue codes, families, edit kinds, and the fix oracle (shared by generator, scorer and worker).\n\nThe codes are those of benchmark/gs_eval_v2/LABELING_GUIDE.md section 2 (D-EDIT), unchanged.\nA family groups codes the guide itself lets overlap (MISSING_SCOPE says "công ty mẹ hay tập đoàn",\nENTITY_SCOPE_AMBIGUOUS says the same): scoring reports strict codes and families.\n"""\nimport re\nimport unicodedata\n\nCODES = {\n    "MISSING_PERIOD": "Thiếu kỳ báo cáo / năm của số liệu",\n    "MISSING_BASELINE": "Tuyên bố thay đổi (giảm, tăng, tiết kiệm) không có năm gốc",\n    "MISSING_SCOPE": "Thiếu ranh giới (nhà máy nào, công ty mẹ hay tập đoàn, phạm vi phát thải 1/2/3)",\n    "NO_FIGURE": "Tuyên bố hiệu quả môi trường mà không có số liệu",\n    "VAGUE_TERM": "Từ mơ hồ hoặc quảng cáo: xanh, thân thiện môi trường, bền vững, không ngừng",\n    "ABSOLUTE_CLAIM": "Tuyệt đối hoá: 100 %, hoàn toàn, không phát thải, tốt nhất",\n    "OFFSET_NEUTRALITY": "Trung hòa / net zero dựa vào bù trừ, không nêu tỷ lệ bù trừ",\n    "FUTURE_AS_ACHIEVED": "Cam kết / mục tiêu diễn đạt như đã đạt",\n    "UNVERIFIED_LABEL_OR_CERT": "Nhắc nhãn, chứng nhận, giải thưởng mà không nêu tổ chức cấp, phạm vi, hiệu lực",\n    "NO_EVIDENCE_REF": "Tuyên bố định lượng không dẫn nguồn / phương pháp",\n    "HIDDEN_TRADEOFF": "Nêu một mặt tốt, che mặt xấu lớn hơn liên quan trực tiếp",\n    "IRRELEVANT_CLAIM": "Khoe điều hiển nhiên hoặc bắt buộc theo luật như một thành tích",\n    "INTENSITY_VS_ABSOLUTE": "Dùng cường độ giảm để ngụ ý tổng phát thải giảm",\n    "ENTITY_SCOPE_AMBIGUOUS": "Không rõ số của công ty mẹ hay tập đoàn",\n}\nFAMILY = {"MISSING_SCOPE": "SCOPE", "ENTITY_SCOPE_AMBIGUOUS": "SCOPE",\n          "VAGUE_TERM": "VAGUE", "NO_FIGURE": "VAGUE"}\n\n\ndef family(code: str) -> str:\n    return FAMILY.get(code, code)\n\n\nKINDS = {\n    "add_scope": "thêm kỳ, năm gốc, ranh giới, phạm vi còn thiếu, dạng chỗ trống [bổ sung: ...]",\n    "placeholder": "thay cụm mơ hồ / câu không có số bằng chỗ trống [bổ sung: chỉ số và giá trị]",\n    "evidence_ref": "thêm dẫn chiếu nguồn: [bổ sung: nguồn xác minh]",\n    "soften": "bỏ hoặc làm nhẹ lời nói quá, nói lại đúng mức",\n}\n# The kind the guide prescribes for each code (several are acceptable for some).\nKIND_OK = {\n    "MISSING_PERIOD": {"add_scope"}, "MISSING_BASELINE": {"add_scope"}, "MISSING_SCOPE": {"add_scope"},\n    "ENTITY_SCOPE_AMBIGUOUS": {"add_scope"}, "NO_FIGURE": {"placeholder"}, "VAGUE_TERM": {"placeholder", "soften"},\n    "ABSOLUTE_CLAIM": {"soften"}, "OFFSET_NEUTRALITY": {"soften"}, "FUTURE_AS_ACHIEVED": {"soften"},\n    "UNVERIFIED_LABEL_OR_CERT": {"add_scope", "evidence_ref"}, "NO_EVIDENCE_REF": {"evidence_ref"},\n    "HIDDEN_TRADEOFF": {"add_scope", "evidence_ref", "soften"}, "IRRELEVANT_CLAIM": {"soften"},\n    "INTENSITY_VS_ABSOLUTE": {"soften", "add_scope"},\n}\n\nPH = re.compile(r"[\\[(][^\\])]*(?:bổ sung|cần nêu|cần bổ sung|cần ghi)[^\\])]*[\\])]", re.I)\n# What a placeholder must mention to fill each missing slot.\nSLOT_WORDS = {\n    "period": r"kỳ|năm(?! gốc)|thời gian|giai đoạn",\n    "baseline": r"gốc|so sánh|mốc",\n    "scope": r"phạm vi|ranh giới|nhà máy|cơ sở|công ty mẹ|hợp nhất|tập đoàn|đơn vị",\n    "figure": r"số|giá trị|chỉ số|mức|định lượng",\n    "source": r"nguồn|trang|tài liệu|báo cáo|phương pháp|xác minh|kiểm toán|bảo đảm",\n    "issuer": r"tổ chức|cấp|hiệu lực|phạm vi|đơn vị",\n    "context": r"tổng|toàn bộ|diễn biến|bối cảnh|cùng kỳ|tuyệt đối|phát thải",\n    "offset": r"tỷ lệ|bù trừ|tín chỉ|sổ đăng ký",\n}\n\n\ndef fold(s: str) -> str:\n    s = unicodedata.normalize("NFC", s or "").lower()\n    return re.sub(r"\\s+", " ", s).strip()\n\n\nNUM = re.compile(r"\\d+(?:[.,]\\d+)*")\n\n\ndef numbers(s: str) -> set[str]:\n    return {re.sub(r"[.,]", "", x) for x in NUM.findall(s or "")}\n\n\ndef placeholders(text: str) -> list[str]:\n    return [fold(m.group(0)) for m in PH.finditer(text or "")]\n\n\ndef slot_filled(before: str, after: str, slot: str) -> bool:\n    """A new placeholder (not in the text before) names the slot."""\n    old = placeholders(before)\n    new = [p for p in placeholders(after) if p not in old]\n    return any(re.search(SLOT_WORDS[slot], p) for p in new)\n\n\ndef oracle(case: dict, before: str, after: str) -> dict:\n    """Did the edited text fix this case\'s planted issue, and keep everything else?\n\n    resolves: the slot is asked for (drop mutations) or the planted phrase is gone or softened (inject mutations).\n    keeps_numbers: every number of the text before is still there. new_numbers: numbers the edit invented.\n    """\n    o = case["oracle"]\n    a = fold(after)\n    if o["type"] == "all":  # several planted issues: every one must be fixed\n        parts = [oracle(dict(case, oracle=p), before, after) for p in o["parts"]]\n        return {"resolves": all(p["resolves"] for p in parts), "keeps_numbers": all(p["keeps_numbers"] for p in parts),\n                "new_numbers": sorted(set().union(*(p["new_numbers"] for p in parts)))}\n    if o["type"] == "slot":\n        res = slot_filled(before, after, o["slot"])\n    elif o["type"] == "phrase":\n        gone = fold(o["phrase"]) not in a\n        asked = any(re.search(SLOT_WORDS[o.get("slot", "context")], p) for p in placeholders(after) if p not in placeholders(before))\n        res = gone or asked\n        if o.get("forbid"):  # e.g. FUTURE_AS_ACHIEVED: "đã giảm" must not remain\n            res = res and not re.search(o["forbid"], a)\n        if o.get("require"):\n            res = res and bool(re.search(o["require"], a))\n        if o.get("alt"):  # another acceptable fix, e.g. "theo quy định" for IRRELEVANT_CLAIM\n            res = res or bool(re.search(o["alt"], a))\n    else:\n        res = False\n    before_n, after_n = numbers(before), numbers(after)\n    droppable = numbers(o.get("phrase", ""))  # softening "100 %" may drop the 100\n    return {"resolves": bool(res), "keeps_numbers": (before_n - droppable) <= after_n,\n            "new_numbers": sorted(after_n - before_n)}\n',
    'jobs_32b.json': '[\n {\n  "tag": "q3-32b-awq",\n  "tp": 2,\n  "max_len": 4096,\n  "cands": [\n   {\n    "repo": "Qwen/Qwen3-32B-AWQ"\n   }\n  ]\n }\n]',
    'gsi_layer_b3.jsonl': '{"id": "B-A-2b1808326a80a7e8", "claim": "Activities involving the hunting of wildlife or or within protected areas or sensitive areas that natural organisms take place in may affect biodiversity or threatened species.", "evidence": "Farming Areas Ï Minimizing impacts on ecosystems: Strictly comply with farming and pond operation procedures to minimize negative impacts on the surrounding environment, including both aquatic and terrestrial ecosystems. Ï Protection of marine and coastal biodiversity: Activities include supporting communities in afforestation and forest protection, prohibiting the hunting of endangered species, conserving genetic resources and native species, preventing invasive species, and ensuring that farmed shrimp do not escape into the natural environment, which could threaten other aquatic species. No occurrences 304-1 Operational sites owned, leased, managed in, or adjacent to protected areas and ar"}\n{"id": "B-A-fdd751c625fe715f", "claim": "Organic Farming Areas for Cashew and Coffee To achieve regenerative circularity and a sustainable balance among natural environmental factors, PAN’s subsidiaries, Lafooco and SHIN Coffee, have consistently complied with standards and strived to deliver verified and certified results as evidence of their sustainability commitments in agriculture in general and coffee in particular.", "evidence": "One of the key measurable outcomes is the development of Robusta coffee and cashew products certified organic under U.S. (USDA), European (EU Organic), and Japanese (JAS) standards. These products are sourced from sustainable coffee-growing areas in Đak F uti farming areas Ï 01 cashew cooperative pa rib Ï 7.1 hectares of organic coffee ck Ï 01 coffee cooperatives PACKAGED ist agi farming areas D FOODS ng"}\n{"id": "B-A-38415a90b4b1d6b2", "claim": "NO Use of chemical fertilizers NO Use of harmful pesticides or herbicides Agriculture The project “1 million hectares of high-quality, low- emission rice” We are collaborating with farmers and cooperatives to help them increase productivity, improve income, and enhance resilience by promoting regenerative agriculture practices, providing sustainable farming solutions, and building regenerative farming communities.", "evidence": "17ha restored Sustainable agriculture activities: The PAN Group actively collaborates with farmers to enhance productivity, boost income, and strengthen resilience through promoting innovative agricultural practices, providing sustainable farming solutions and fostering regenerative agricultural communities. In 2024, PAN successfully cultivated 1,035.3 hectares of high-quality, low-emission rice and developed 455 hectares of cashew-sourcing areas with USDA, EU, and JAS organic certi昀椀cations. With a vision to become a leading agribusiness, PAN has committed to achieving Net Zero by 2050. Organized training sessions on cultivation techniques and standardization of rice production processes, a"}\n{"id": "B-A-6685afd637957ddf", "claim": "Unlike traditional agriculture models that tend to deplete natural resources, regenerative agriculture focuses on restoring and regenerating soil, biodiversity, and natural ecosystems.", "evidence": "114 Chemical safety management 120 Biodiversity 122 Regenerative agriculture 124 86 87 3,00 Responsible farming Sustainable farming and aquaculture area Farmers Regenerative agriculture Water conservation Product R&D Workers Circular economy Soil regeneration"}\n{"id": "B-A-a97d9a9ed65c553c", "claim": "“Transforming Rice Value Chains for Climate Resilience and Sustainable Development in the Mekong Delta (TRVC)” with a total area of 47,520 hectares, 20 times higher than in 2024.", "evidence": "Member companies proactively restructured, innovated products, invested in technology, and advanced green transformation. The area of high-quality, low-emission rice models in the Mekong Delta was expanded to 47,520 hectares—an increase of 47 times compared to 2024. Vinaseed – a member of PAN Group – continued to expand its low-emission rice cultivation area to over 47.520 hectares (20 times higher than in 2024). This contributes to the project “Sustainable development of one million hectares of high-quality, low-emission rice cultivation associated with green growth in the Mekong"}\n{"id": "B-A-be19458a1fc2e6e1", "claim": "During the Winter– Spring crop 2024–2025, rice cultivated by 5,473 farmers within Vinaseed’s value chain achieved lower emissions compared to conventional practices.", "evidence": "Unlike traditional agriculture models that tend to deplete natural resources, regenerative agriculture focuses on restoring and regenerating soil, biodiversity, and natural ecosystems. VinaRice, a subsidiary of Vinaseed, participated in Phase 3 (Winter–Spring crop 2024–2025) of the project (Source: Vinaseed’s 2024 Financial Report) In 2024, the Group’s seed and rice business achieved a robust 20% revenue increase. However, gross pro昀椀t growth was moderated by higher input costs for food production and other raw materials, resulting in slightly lower pro昀椀t margins compared to 2023."}\n{"id": "B-A-bcf7a624dc163547", "claim": "The approach focuses on maintaining and restoring the diversity of plants, fungi, and microorganisms within a given area, enabling them to interact and form a balanced ecosystem that supports life.", "evidence": "Ï Support is given for the establishment and maintenance of legal associations and groups within the company. Unlike traditional agriculture models that tend to deplete natural resources, regenerative agriculture focuses on restoring and regenerating soil, biodiversity, and natural ecosystems. VinaRice, a subsidiary of Vinaseed, participated in Phase 3 (Winter–Spring crop 2024–2025) of the project"}\n{"id": "B-A-96763c2ed26ee86b", "claim": "Organic EU Organic JAS HECTARES of high-quality, low-emission rice cultivation area Organic USDA HECTARES of linked coffee raw material areas certified organic under USDA, EU, and JAS standards At the same time, VinaRice – Vinaseed was granted a Certificate of Trademark Usage", "evidence": "HECTARES of linked cashew raw material areas certified organic under USDA, EU, and JAS standards 127 126 Vinaseed – a member of PAN Group – continued to expand its low-emission rice cultivation area to over 47.520 hectares (20 times higher than in 2024). This contributes to the project “Sustainable development of one million hectares of high-quality, low-emission rice cultivation associated with green growth in the Mekong"}\n{"id": "B-A-73db6302af23961c", "claim": "Rights for the label “Green Low-Emission Vietnamese Rice” on April 24, 2025, by the Vietnam Rice Industry Association, for rice products sourced from the company’s project implementation areas.", "evidence": "Vinaseed – a member of PAN Group – continued to expand its low-emission rice cultivation area to over 47.520 hectares (20 times higher than in 2024). This contributes to the project “Sustainable development of one million hectares of high-quality, low-emission rice cultivation associated with green growth in the Mekong Ï Member of the High-Quality Vietnamese Goods Business Association Ï Member of the Vietnam Animal Feed Association 27 26"}\n{"id": "B-A-6ac3de5d0ed7daea", "claim": "Optimizing high-quality chemical product lines with lower toxicity toward greater safety and effectiveness.", "evidence": "United States, and Japan. Chemicals used for facility Chemical Safety Management Practices at the Plant: Ï Fully equip employees with appropriate personal protective equipment (PPE) to ensure safety and prevent incidents. 10,292 12,500 Optimizing the Use of Raw Materials Raw clams Tons 813 738 2,005 Over the years, PAN Group has progressively upgraded and newly built all of its production facilities and farming operations using modern technologies, optimizing production lines to conserve resources and protect the environment. In addition, process improvements aimed at enhancing production efficiency and product quality have also contributed to reducing raw material losses."}\n{"id": "B-A-37bafbead6b46bcd", "claim": "Reduce emissions and negative environmental impacts;", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY STRATEGY We are committed to minimizing negative environmental impacts and continuously improving products and technologies toward greater environmental friendliness. The Group has established environmental management systems in line with international standards such as ISO PAN is progressively integrating environmental and sustainability factors into its investment and production activities, aiming to minimize environmental impacts and promote green growth. Through efficient resource utilization, emissions reduction, and participation in sustainable finance initiatives, the Group contributes to generating positive indirect "}\n{"id": "B-A-2960fd42d6bb7218", "claim": "The initiative aims to promote the application of biotechnology, contributing to the development of a green, safe, and sustainable agriculture sector.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE CLOSED-LOOP VALUE CHAIN PAN focuses on building a sustainable raw material production foundation across its three key sectors—agriculture, aquaculture, and packaged foods—through investments in seed research and development, the application of biotechnology, and the implementation of internationally certified farming and cultivation models. Scientific Council, marking a significant milestone in its journey to advance research and apply science and technology to improve seed quality and promote sustainable development. Innovation and Digital Transformation Activities In 2025, the Vietnamese Government issued Resolution 68 on private sector "}\n{"id": "B-A-ea1fbc395015dc8c", "claim": "This event marks a new milestone in strengthening collaboration between government authorities and businesses to realize the goals of ecological agriculture development, emission reduction, and enhancement of the value of Vietnamese agricultural products.", "evidence": "Talents” programs; and organized a series of training and research activities in collaboration with the Vietnam National University of Agriculture and Thai Nguyen University of Agriculture and Forestry, with a total value of VND 750 million. The PAN Group operates under a corporate structure where the parent company oversees management, optimizes resources, and fosters collaboration between subsidiaries. The parent company’s investment activities primarily focus on M&A (acquiring new subsidiaries, increasing ownership in existing ones) and developing new projects to strengthen the Group’s value chain. Additionally, this investment expands Atani’s product and service offerings."}\n{"id": "B-A-2e0aa99ad7163b0e", "claim": "In the context of the agricultural and food industry transforming towards sustainability and technology application, the Group focused on optimizing resources, improving labor productivity, and creating conditions for employees to develop their long-term capabilities.", "evidence": "author groups across six companies within the Group. implemented a Kaizen reward and emulation regulation across the organization. Accordingly, the company encourages employees to proactively propose improvement initiatives in production, business, and management processes, such as improving workflows, optimizing equipment and technology, reducing costs, enhancing product quality, and improving working conditions and the working environment. ORIENTATION MARKET DEVELOPMENT ENHANCING R&D CAPACITY NEW PRODUCT DEVELOPMENT The Group focuses on collaborating with leading domestic and international research institutes and scienti昀椀c organizations to apply advanced technologies to production, improv"}\n{"id": "B-A-f968e3ef2117d8fd", "claim": "PAN places strong emphasis on ensuring stable income and a safe working environment, while enhancing training programs, skill development, and improving the material and spiritual well-being of its employees.", "evidence": "Amid ongoing economic fluctuations, PAN remained committed to a development strategy that integrates business growth with social responsibility, recognizing human capital as the foundation for sustainable growth. The Group places strong emphasis on building a safe, stable, and equitable working environment, while also creating opportunities for employees to enhance their capabilities through appropriate training programs. PAN and its subsidiaries fully comply with legal obligations regarding wages, taxes, and insurance; Ï Provision of enhanced health insurance programs. In addition to competitive compensation and benefits, PAN places strong emphasis on building a fair working environment tha"}\n{"id": "B-A-5cc1f9f48c39073a", "claim": "Recruitment programs are expanded through partnerships with universities, research institutes, and expert networks in the agriculture and food sectors, aiming to attract high-quality talent to support the Group’s transformation and sustainable development strategy.", "evidence": "To enhance the efficiency and transparency of recruitment activities, the Group promotes digital transformation in the recruitment process. Candidates can access job information and submit applications online through the company’s recruitment website, official communication channels, and reputable recruitment platforms. This approach helps broaden access to high-quality talent while ensuring that all candidates have an equal and convenient opportunity to obtain recruitment information. Ï Collaborating and conducting research with partners, institutes, and universities to develop organic and biological crop protection products."}\n{"id": "B-A-df567ee5834acb15", "claim": "At the Group level, the PAN Innovation Awards program is held every two years to seek, honor, and reward innovative initiatives and solutions with high practical value, positively contributing to the Group’s sustainable development.", "evidence": "Since 2020, the Group has also organized the “PAN Innovation Awards” to recognize innovative initiatives that deliver signi昀椀cant value to both the Group and society. The Group is actively improving its marketing operations and further strengthening its brand presence with consumers. SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT INNOVATION At member companies, the culture of innovation is implemented in a diverse and flexible manner to suit each operating model and business characteristics. At PAN Group, we consider that “Innovation is not a strategy, but the very breath in every step of the agricultural sector.”"}\n{"id": "B-A-965daf1978b8df62", "claim": "In 2025, PAN Group implemented measures to strengthen occupational health and safety, including regular safety training, enhancing employees’ ability to identify and control workplace risks, and reviewing operational procedures to minimize potential hazards.", "evidence": "strengthening the monitoring of working environment conditions at production facilities. 10,000 1.0 In 2025, PAN Group did not record any newly identified cases of occupational diseases, reflecting the effectiveness of risk control measures and safety training programs implemented across its member companies. 0.8 8,000 Ï Implementing occupational safety risk assessments for 0 0 0 In 2025, PAN implemented occupational health and safety management programs with a focus on risk prevention and raising employee awareness, including: Ï Issuing and applying safety procedures for specific job Number of injury cases requiring medical leave"}\n{"id": "B-A-2e3eeaff17a508b7", "claim": "Safety inspection programs were conducted regularly on a daily, weekly, and monthly basis at production facilities.", "evidence": "Thanks to regular workplace safety training programs and inspections, PAN recorded no workplace accidents in 2024. Additionally, professional development programs for employees are organized regularly, with a total of 412 training courses throughout the year. Key Concerns The PAN Group Actions Ï Market research and surveys. Product quality, Ï Invest in technology, equipment, and improve Ï Weekly and monthly meetings Continuous"}\n{"id": "B-A-933afaaec3aade51", "claim": "After each incident, the units conducted root cause investigations, reviewed relevant procedures, and implemented corrective measures to prevent recurrence, thereby contributing to strengthening the safety culture across the entire system.", "evidence": "In 2025, PAN Group implemented measures to strengthen occupational health and safety, including regular safety training, enhancing employees’ ability to identify and control workplace risks, and reviewing operational procedures to minimize potential hazards. Safety inspection programs were conducted regularly on a daily, weekly, and monthly basis at production facilities. During the year, the Group recorded 02 occupational accidents at the Bibica Mien Tay factory; strengthening the monitoring of working environment conditions at production facilities. 10,000 1.0 In 2025, PAN Group did not record any newly identified cases of occupational diseases, reflecting the effectiveness of risk control"}\n{"id": "B-A-0bbeec450beb636a", "claim": "GRI 403 PAN Group identifies occupational health and safety as one of the fundamental factors ensuring stable operations and sustainable development.", "evidence": "50 Attracting talent 138 Association membership 26 Sustainable development strategy 56 Occupational health and safety 140 Creating sustainable value 60 Education & Training Worker participation, consultation, and communication on occupational health and safety 120 - 121, 140 - 141 411-1 Incidents of violations involving rights of indigenous peoples 145 403-5 Worker training on occupational health and safety"}\n{"id": "B-A-3ef9caf5f608dbc4", "claim": "Building a safe working environment not only helps minimize operational risks but also enhances labor productivity and improves the quality of human resources across the entire system.", "evidence": "The PAN Group pays special attention to recruiting and training quali昀椀ed, professional and ethical personnel. At the same time, the company continuously improves its remuneration policy, ensures internal fairness, and creates a professional working environment for the staff. Thanks to its appropriate human resource risk management policies, The PAN Group has been building a team of high-quality human resources to serve the fast growing scale of the Group. Solutions to minimize risk: Solutions to minimize the risks: Build a comprehensive human resource policy committed to the basic issues: not using child labor or illegal labor; ensuring human rights, women’s rights and equal rights at work;"}\n{"id": "B-A-43a7c3ac706b9302", "claim": "The investment cost for occupational health and safety activities in 2025 was BILLION VND 2023 2024 2025 Number of serious occupational accidents involving employees", "evidence": "2025 1 Total Employees 2025 Person 10,349 Number of Serious Accidents Involving Staff and Workers Case 0 0 0 2 Turnover Rate % 3.4 3 Employee Distribution by Gender Worker participation, consultation, and communication on occupational health and safety 120 - 121, 140 - 141 411-1 Incidents of violations involving rights of indigenous peoples 145 403-5 Worker training on occupational health and safety"}\n{"id": "B-A-1f636d86d3d09dde", "claim": "10,000 1.0 In 2025, PAN Group did not record any newly identified cases of occupational diseases, reflecting the effectiveness of risk control measures and safety training programs implemented across its member companies.", "evidence": "Occupational Diseases (in 2023, 2024, 2025) cases requiring treatment for occupational diseases cases of occupational diseases 140 141 In 2025, PAN Group implemented measures to strengthen occupational health and safety, including regular safety training, enhancing employees’ ability to identify and control workplace risks, and reviewing operational procedures to minimize potential hazards. Safety inspection programs were conducted regularly on a daily, weekly, and monthly basis at production facilities. During the year, the Group recorded 02 occupational accidents at the Bibica Mien Tay factory;"}\n{"id": "B-A-345d1a53d918fb7f", "claim": "Organized training sessions on cultivation techniques and standardization of rice production processes, attracting GRI 404 With a farmer-centric strategy, Vinaseed and VFC—two entities operating in the agricultural sector—are actively providing technical advisory services and transferring sustainable farming technologies to farmers and cooperatives nationwide.", "evidence": "Reducing In 2025, Vinaseed organized 769 rice cultivation technical training courses for more than 38,850 farmers, helping them better understand standard rice farming practices, reduce input costs, and save land and water resources. water usage compared to traditional farming methods M3/YEAR 110 111 Promoting Local Economic Development The Group is currently collaborating in production linkages and offtake agreements with 50,700 farming households and 221 cooperatives nationwide through partnership models and technical transfer tailored to the specific characteristics of its member companies, such as shrimp farming, organic cashew cultivation, coffee growing, and rice farming."}\n{"id": "B-A-e3a0d55008842423", "claim": "PAN Group identifies training as a key factor in ensuring a workforce that meets development requirements, especially as the agriculture–food sector increasingly demands high standards in quality, technology, and sustainability.", "evidence": "In 2025, PAN Group maintained a stable workforce of 10,349 employees, a decrease of 5.5% compared to the same period in 2024, reflecting a challenging year for the agricultural sector as the workforce gradually shifted towards industry and services. In the context of the agricultural and food industry transforming towards sustainability and technology application, the Group focused on optimizing resources, improving labor productivity, and creating conditions for employees to develop their long-term capabilities. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 SUSTAINABILITY REPORT 2025 OVERVIEW OF PAN GROUP KEY PRODUCTS IN THE AGRICULTURE SECTOR KEY PRODUCTS IN THE AQUACULTURE SECTOR Shrimp"}\n{"id": "B-A-97df6f9e99363e41", "claim": "-6% CLAMS GRI 416 Fish sauce -6% With the mission “Born to feed the world,” safe and high- quality products are the top priority in all production and business activities of PAN Group.", "evidence": "TABLE | SUSTAINABILITY PERFORMANCE REPORT Growth of Key Product categories in 2025 Product Growth rate Pangasius fish -2% CLAMS Cakes -6% Fish sauce -6% Rice seeds -23% SHRIMP Fumigation goods -3% Plant protection -1% products OTHER NUTRITIONAL PRODUCTS CASHEW PRODUCTS DRIED FRUITS INSTANT COFFEE PROCESSED RICE Results of dedicated efforts Ï Certification of Food Safety Management System in accordance with international standards Ï Laboratory accredited to ISO 17025 standard | CONTRIBUTED TO CSR ACTIVITIES WORKING ACROSS VIETNAM ON SAFE AND RESPONSIBLE USE OF COOPERATIVES PLANT PROTECTION PRODUCTS ENGAGED IN PRODUCTION LINKAGE"}\n{"id": "B-A-1a2dc822a7f3bb63", "claim": "We prioritize the use of materials with clear origins, produced through sustainable practices that are not harmful to consumer health or the environment.", "evidence": "We have implemented measures to minimize environmental impacts and to advance sustainable agricultural practices, and we participate in reforestation initiatives aimed at protecting the environment and preserving biodiversity. The PAN Group’s commitment to sustainable development, ensuring bene昀椀ts for the community, employees, and the environment. 105 104 THE PAN GROUP 58% 50 prioritize sustainable results. innovation and customers, both internal and external feedback 52% 40 new opportunities. partners, communities, and practices. for continuous 30 the environment. improvement."}\n{"id": "B-A-36d331e9499c0043", "claim": "This is not only a commitment to transparency but also PAN’s utmost responsibility in safeguarding consumer health and safety.", "evidence": "This also serves as a guiding principle for PAN Group in affirming its reputation and responsibility toward the community and stakeholders. The Practice Principles focuses on realizing the vision of a sustainable farm-to-table value chain, while ensuring food security and nutritional safety. It also emphasizes requirements on traceability, waste SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT 2025 Key Highlights PAN Group considers water not only as a resource but also as a shared responsibility. In line with the United Nations’ “Forward Faster initiative, Water"}\n{"id": "B-A-14537ba941199e9f", "claim": "Number of violations related to food safety and hygiene.", "evidence": "No. Content Kết quả 1 Number of violations related to food safety and hygiene. 0 2 Number of product recalls due to quality non-compliance. 0 3 Total number of incidents of non-compliance with regulations and voluntary codes related to product and service information and labeling. Material Topics Key Performance Indicators (KPIs) Sustainable economic growth Revenue; profit Customer and consumer satisfaction Number of food safety violations"}\n{"id": "B-A-1abc2a1c6041219e", "claim": "With the principle of “sustainable development and community orientation,” PAN not only focuses on business operations but also actively participates in programs and projects that deliver long-term benefits to society.", "evidence": "Ï Foster a corporate culture that PAN prioritizes environmental protection in all production and business activities, aiming to minimize adverse impacts and promote a green, circular economy model. Community engagement is a key component of PAN’s sustainable development strategy. The Group not only contributes financially but also directly participates in community support programs, helping improve the lives of employees, farmers, and local residents. PAN understands its responsibility to the community and social environment. Beyond economic development, the company actively engages in activities that bring sustainable benefits to the community, fostering a fair and civilized society. Amid a"}\n{"id": "B-A-943551cbb3e4a0da", "claim": "Bonus levels are proportional to work effectiveness through initiatives that improve labor productivity, product quality, and reduce waste.", "evidence": "In bottling and packaging, automation has been enhanced to reduce human error and improve labor productivity. PAN Group and the Ministry of Agriculture and Environment have also agreed to collaborate on standardizing and modernizing shrimp seed production processes, gradually replacing imported broodstock with domestically produced varieties that offer superior productivity, disease resistance, and ecological adaptability through the application of advanced scientific and biotechnological innovations. Board of Management. Ï At Lafooco, we encourage and recognize individuals who Ï At VFC, the company’s innovation culture is people- contribute ideas and efforts in improving productivity, produ"}\n{"id": "B-A-0e5a18c48a514a63", "claim": "Waste Prevention Team to monitor and maintain the “Right and Enough” culture, helping to raise awareness and responsibility among employees in practicing savings and using resources efficiently.", "evidence": "In addition, VFC regularly conducts training sessions to raise chemical safety awareness among all employees, ensuring everyone fully understands the risks and preventive measures. a year. This requires a daily commitment to raising awareness and ensuring compliance with laws, regulations, and ethical standards. Ï Environmental Policy: Focus on pollution prevention, efficient use of resources (energy, water, land), reduction of greenhouse gas emissions, and promotion of sustainable production models. Our Code of Conduct serves as a guiding compass, helping us stay on the right path and fully comply with all applicable legal requirements."}\n{"id": "B-A-1de076d88dbdff4c", "claim": "In 2025, the company enhanced its governance culture by establishing a Sustainability – ESG Subcommittee to promote digital transformation, data governance, and the implementation of sustainability standards.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY PERFORMANCE REPORT Standardized and Transparent ESG Data Outcomes after two years of implementation Since October 2024, PAN Group has deployed an internal sustainability data collection system across all member companies. SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABILITY GOAL FRAMEWORK With a vision to become a leading agriculture and food group in ESG practices in the region by 2030, PAN Group has been systematically implementing initiatives to integrate sustainability into the strategies and governance practices of its member companies. In 2024, the Group launched the “Sustainable Business Assessment and"}\n{"id": "B-A-2afa74a6777d4ac8", "claim": "PAN’s member companies will focus on achieving sustainability goals through concrete action programs over the next five years, creating economic value alongside environmental protection, improving quality of life, and building a transparent governance system.", "evidence": "To ful昀椀ll its vision and mission, the Group not only prioritizes economic growth but also places a strong emphasis on social responsibility and environmental protection. Through concrete actions, PAN actively minimizes the environmental footprint while implementing programs that bene昀椀t the community, employees, and stakeholders. ENVIRONMENT 35.8% 2024 marked a signi昀椀cant milestone in The PAN Group’s sustainability journey. The PAN Group focuses on building a common corporate culture with shared values, close connections between member companies, while respecting the differences and traditions of each company. between the parent company and member companies through environmental activities"}\n{"id": "B-A-11b4f79ce0c2832a", "claim": "Based on the three ESG pillars (Environment – Social – Governance), PAN aims for sustainable growth and positive contributions to the development of Vietnam’s economy.", "evidence": "SUSTAINABILITY REPORT 2025 SUSTAINABILITY GOVERNANCE SUSTAINABLE DEVELOPMENT STRATEGY PAN Group has established its Sustainable Development Strategy 2030, encompassing all three pillars of sustainability: ECONOMIC SOCIAL “ The Group remains committed to its mission The PAN Group has developed a prudent business plan and a sustainable development strategy to overcome these dif昀椀culties. To achieve these objectives, PAN focuses on sustainable development, addressing environmental, social, and governance issues. The sustainable agriculture model has enabled PAN to af昀椀rm its leadership position both domestically and internationally."}\n{"id": "B-A-8048a7bc004352e3", "claim": "Environment Social Governance PAN is working towards an environmentally friendly production model that efficiently utilizes natural resources and minimizes negative impacts on ecosystems.", "evidence": "Farming Areas Ï Minimizing impacts on ecosystems: Strictly comply with farming and pond operation procedures to minimize negative impacts on the surrounding environment, including both aquatic and terrestrial ecosystems. Ï Protection of marine and coastal biodiversity: Activities include supporting communities in afforestation and forest protection, prohibiting the hunting of endangered species, conserving genetic resources and native species, preventing invasive species, and ensuring that farmed shrimp do not escape into the natural environment, which could threaten other aquatic species. Ï Water conservation Ï Safe and friendly working environment Farmers PAN Group’s Sustainable Developmen"}\n{"id": "B-A-baa84c74d2f473d5", "claim": "\uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 \uf0a1 ISO 9001 1 \uf0a1 ISO 17025 4 \uf0a1 \uf0a1 \uf0a1 \uf0a1 ISO 22000 2 \uf0a1 \uf0a1 FSSC 22000 4 \uf0a1 \uf0a1 \uf0a1 \uf0a1 Food Quality and Safety Standards HALAL", "evidence": "TABLE HACCP, BRC, IFS, HALAL | BRC, HACCP, HALAL, IFS | FSSC 22000, HACCP | FSSC 22000, ISO 22000 | BRCGS, HACCP | FSSC 22000, HALAL | FSSC 22000 TCVN 5107-2018 | QCVN 01- 188:2018/ BNNPTNT | ISO 22000 ISO 17025 | | | | | | | ISO 17025 | ISO 17025 TABLE Total Number of Units Collecting Data | 11 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 ISO 9001 | 1 | \uf0a1 | | | | | | | | | | ISO 17025 | 4 | \uf0a1 | \uf0a1 | \uf0a1 | | | | | | \uf0a1 | | ISO 22000 | 2 | | | \uf0a1 | | \uf0a1 | | | | | | FSSC 22000 | 4 | | \uf0a1 | | | \uf0a1 | | \uf0a1 | \uf0a1 | | | HALAL | 5 | | \uf0a1 | | | | | | \uf0a1 | \uf0a1 | \uf0a1 | \uf0a1 BRC | 5 | | \uf0a1 | | | | \uf0a1 | | | \uf0a1 | \uf0a1 | \uf0a1 HACCP | 5 | | \uf0a1 | | | | \uf0a1 | | | \uf0a1 | \uf0a1 | \uf0a1 IFS | 2 | | | | | | | | | \uf0a1 | | \uf0a1 Global GAP | 3 | | \uf0a1 | | \uf0a1 | | | | "}\n{"id": "B-A-acd8c04d89adf4c1", "claim": "7,881,985 6,614,893 6,735,787 Coal Ton 5,673 148 245 Reused Water m3 NA 1,516 18,032 Biomass – Rice husks and seed shells Ton 1,231 5,643 3,601 Treated Wastewater m3", "evidence": "592,200 490,000 80,427 Seawater m3 5,417,441 4,250,774 4,510,417 Total water consumption m3 7,881,985 6,614,893 6,735,787 Water use efficiency for the period 2023 – 2025 TABLE m3 | 1,361,826 | 1,412,120 m3 | 510,518 | 461,999 m3 | 592,200 | 490,000 m3 | 5,417,441 | 4,250,774 | 7,881,985 | 6,614,893 m3 | NA | 1,516 m3 | NA | 4,777,878 m3/ Revenue (VND bn) | 597 | 409 % | |"}\n{"id": "B-A-c08f4e1311d7f2ad", "claim": "Liter 165,481 605,970 527,584 Reduction Rate Compared to 2023 % -6.3% Fuel oil (FO) Liter 210,740 270,604 297,779 Reused water data and treated wastewater data from aquaculture ponds in 2022;", "evidence": "1,662,644 1,857,724 Fuel oil (FO) litres 210,740 270,604 297,779 Petrol (xăng) litres 27,146 766,679 807,887 LPG kg 78,607 468,045 TABLE KWh | 81,215,939 | 110,780,751 KWh | 2,423,711 | 3,770,675 KWh | 83,639,650 | 114,551,426 % | 2.9% | 3.3% Ton | 5,673 | 148 Ton | 1,231 | 5,643 Ton | 92 | 133 Liter | 165,481 | 605,970 Liter | 210,740 | 270,604 Ton | 78,608 | 468,046 Kg | 8,700 | 27,240 Liter | 2,050,197 | 1,056,674 Liter | 27,146 | 766,679 MJ | 398,350,028 | 538,639,081 GJ/ Revenue (VND bn) | 30.2 | 33.3 % | |"}\n{"id": "B-A-fff141881ac73f53", "claim": "38,661 Diesel (DO) Liter 2,050,197 1,056,674 1,330,140 Mobile Fuel Sources Domestic Waste kg 897,986 298,139 615,714 Petrol (gasoline)", "evidence": "TABLE kg | 22,000 | 315,127 kg | 897,986 | 298,139 kg | 15,273,599 | 24,916,780 kg | 376,780 | 764,068 kg | 16,570,365 | 26,294,114 TABLE KWh | 81,215,939 | 110,780,751 KWh | 2,423,711 | 3,770,675 KWh | 83,639,650 | 114,551,426 % | 2.9% | 3.3% Ton | 5,673 | 148 Ton | 1,231 | 5,643 Ton | 92 | 133 Liter | 165,481 | 605,970 Liter | 210,740 | 270,604 Ton | 78,608 | 468,046 Kg | 8,700 | 27,240 Liter | 2,050,197 | 1,056,674 Liter | 27,146 | 766,679 MJ | 398,350,028 | 538,639,081 GJ/ Revenue (VND bn) | 30.2 | 33.3 % | |"}\n{"id": "B-A-ad3a9a500fd54947", "claim": "Liter 27,146 766,679 807,887 Non-Hazardous Waste Recyclable Waste kg 15,273,599 24,916,780 23,043,527 Total Energy Consumption MJ 398,350,028", "evidence": "TABLE kg | 22,000 | 315,127 kg | 897,986 | 298,139 kg | 15,273,599 | 24,916,780 kg | 376,780 | 764,068 kg | 16,570,365 | 26,294,114 TABLE KWh | 81,215,939 | 110,780,751 KWh | 2,423,711 | 3,770,675 KWh | 83,639,650 | 114,551,426 % | 2.9% | 3.3% Ton | 5,673 | 148 Ton | 1,231 | 5,643 Ton | 92 | 133 Liter | 165,481 | 605,970 Liter | 210,740 | 270,604 Ton | 78,608 | 468,046 Kg | 8,700 | 27,240 Liter | 2,050,197 | 1,056,674 Liter | 27,146 | 766,679 MJ | 398,350,028 | 538,639,081 GJ/ Revenue (VND bn) | 30.2 | 33.3 % | |"}\n{"id": "B-A-ef489b4f1d749aae", "claim": "538,639,081 573,427,828 GJ/ Revenue (VND bn) Non-Recyclable Waste kg 376,780 764,068 112,845 Energy Usage Intensity 30.2 33.3 32.6 Total Waste Generated kg", "evidence": "However, through the implementation of effective energy management solutions, the Group’s energy consumption intensity per unit of revenue decreased by 2%, from 33.3 to 32.6 GJ per VND billion in 2025. Energy Consumption in 2025 Group’s energy consumption intensity per unit of revenue TABLE kg | 22,000 | 315,127 kg | 897,986 | 298,139 kg | 15,273,599 | 24,916,780 kg | 376,780 | 764,068 kg | 16,570,365 | 26,294,114"}\n{"id": "B-A-ce720ceb34a22035", "claim": "20,505.837 9,575,150 10,475,100 Scope 2 CO2 Emissions kgCO2e 53,537,547 73,026,671 77,015,181 Total CO2 Emissions kgCO2e 74,043,384 82,601,821 87,490,282 (ton CO2e/", "evidence": "TABLE kgCOe 2 | 20,505.837 | 9,575,150 kgCOe 2 | 53,537,547 | 73,026,671 kgCOe 2 | 74,043,384 | 82,601,821 (ton COe/ 2 Revenue VND bn) | 5.60 | 5.09 % | | 16,570,365 26,294,114 23,810,747 Reduction Rate Compared to 2023 % -2% In 2025, coal data from Vinaseed units will be added GREENHOUSE GAS EMISSIONS Scope 1 CO2 Emissions kgCO2e"}\n{"id": "B-A-1c3d90e903f826c5", "claim": "Revenue CO2 Emission Intensit 5.60 5.09 4.97 VND bn) Reduction Rate Compared to 2023 % -2% 162 163", "evidence": "16,570,365 26,294,114 23,810,747 Reduction Rate Compared to 2023 % -2% In 2025, coal data from Vinaseed units will be added GREENHOUSE GAS EMISSIONS Scope 1 CO2 Emissions kgCO2e 5.09 4.97 Reduced 73,027 77,015 53,538 TONS CO2/ BILLION VND 20,456 9,415 10,475 CO2 emissions intensity per revenue compared PAN Group, with its ambition to become a leading agriculture and food corporation, has committed to achieving net zero emissions by 2050."}\n{"id": "B-A-cc601265ed777d16", "claim": "05 Climate Change/Greenhouse Gas Emissions and Energy Consumption 43 SABECO\'s Sustainability Milestones 07 Water Stewardship 49 LEADERSHIP COMMUNICATION", "evidence": "ENVIRONMENT DIMENSION CLIMATE CHANGE/GREENHOUSE GAS EMISSIONS AND ENERGY CONSUMPTION [GRI 302-1; GRI 302-3; GRI 303-2; GRI 305] WATER STEWARDSHIP [GRI 303] Enviroment Social Governance \\u0017 Climate Change/ \\u0017 Talent Attraction and \\u0017 Corporate Governance Greenhouse Gas Emissions and Energy Consumption"}\n{"id": "B-A-345283e70893296d", "claim": "10 Packaging and Circular Economy 53 Chairman’s Message 10 61 SOCIAL DIMENSION General Director\'s Message 11 Occupational Health and Safety 61 ABOUT SABECO", "evidence": "Management Packaging and Circular Economy MATERIALITY ASSESSMENT PROCESS Occupational Health and Safety Corporate Governance and Ethics Occupational health and safety Management System and Services Packaging and Circular Economy 403-7 61-62 GRI 3 – Material Topics 2021 Management of material topics"}\n{"id": "B-A-b30ceedc39ebc470", "claim": "25 Sustainability Purpose and Vision 28 Sustainability Targets and KPIs 29 SUSTAINABILITY GOVERNANCE 34 STAKEHOLDER ENGAGEMENT", "evidence": "TABLE 03 SUSTAINABILITY STRATEGY AT SABECO MATERIALITY ASSESSMENT APPROACH SUSTAINABILITY, TARGETS AND KPIs SUSTAINABILITY GOVERNANCE STAKEHOLDER ENGAGEMENT | | 03 SUSTAINABILITY STRATEGY AT SABECO MATERIALITY ASSESSMENT APPROACH SUSTAINABILITY, TARGETS AND KPIs SUSTAINABILITY GOVERNANCE STAKEHOLDER ENGAGEMENT STAKEHOLDER ENGAGEMENT 03 SUSTAINABILITY STRATEGY AT SABECO MATERIALITY ASSESSMENT APPROACH SUSTAINABILITY, TARGETS AND KPIs SUSTAINABILITY GOVERNANCE"}\n{"id": "B-A-806490c45545069a", "claim": "TOP 100 TOP 10 Sustainable Enterprises in Vietnam (Manufacturing) awarded by Vietnam Chamber of Commerce and Industry (VCCI) and Vietnam Business Council for Sustainable Development (VBCSD) – CSI100 Reference for 02 consecutive years (2023-2024).", "evidence": "and the Council of Sustainable Development Enterprises of the Vietnam Chamber of Commerce and Industry (VCCI), we are committed to promoting sustainable development in Vietnam and create lasting positive impacts for both our business and the communities we serve. Collectors Aluminum Coil Manufacturer Recycled aluminum products - used in construction and automotive \\u0017 Signed a cooperation program with the Ministry of Culture, Sports and Tourism to elevate national sports and establish Vietnam’s tourism brand. 2023 2024 \\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2023 by CSI100. \\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2024 by CSI100."}\n{"id": "B-A-4bbd6ce23dc9dd92", "claim": "SABECO’s production plants are certified with ISO 22000 – Food Safety Management System.", "evidence": "ISO 22000 Food Safety All of SABECO\'s factories are certified with ISO 9001 (Quality Management System), ISO 14001 (Environmental Management System), ISO 22000 (Food Safety Management System), and HACCP (Hazard Analysis and Critical Control Points). Additionally, all of SABECO\'s factories have laboratories accredited to ISO 17025 standards, ensuring the quality of beer produced across the entire system."}\n{"id": "B-A-f0de0858ff0eb452", "claim": "40,000+ 100% 100% SMBC’s Green Deposit Framework, formulated with the support of Sustainalytics – a leading ESG ratings agency, provides funds for environmentally beneficial projects.", "evidence": "2024, as announced by Anphabe Customer Experience Initiative of the Year award - FMCG Asia Awards 2024 50,000+ Total 49,879 100% Green Deposit for Sustainability Contribution restriction text and a warning not to drink and drive. Vietinbank’s Sustainable Finance Framework aims to provide funds for projects that bring environmental and social benefits. 109 km ZERO ZERO of solar-powered streetlights, with"}\n{"id": "B-A-62883e843fa70cb8", "claim": "Vietinbank’s Sustainable Finance Framework aims to provide funds for projects that bring environmental and social benefits.", "evidence": "Green Deposit Framework and by Vietinbank under its Sustainable Finance Framework. 40,000+ 100% 100% SMBC’s Green Deposit Framework, formulated with the support of Sustainalytics – a leading ESG ratings agency, provides funds for environmentally beneficial projects. \\u0017 Focus on the development of young talents through youth empowerment programs and sponsorships for Vietnamese young entrepreneurs and sports talents, and other projects that contribute to national goals \\u0017 Projects of sustainable development \\u0017 SABECO was recognized as a strategic partner who plays a major role in community sustainable development projects"}\n{"id": "B-A-419e533f927579fb", "claim": "Solar Project with a capacity of 9 MWp at 9 breweries, initiating the use of renewable energy in the beer production process.", "evidence": "\\u0017 Partnered with SP Group (Singapore) to implement Phase 2 of the Rooftop Solar Project at 9 breweries, with the highest capacity reaching 10.44 MWp. \\u0017 Inaugurated the SABECO Beer Research and TABLE SABECO\'S SUSTAINABILITY MILESTONES 2008 2013 2013-2015 2016 \\u0017 Through equitization, Saigon Beer \\u0017 The first brewery obtained \\u0017 Developed the Environmental \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation certification under ISO 50001:2013 Management System in accordance \\u0017 Awarded the \\"Environmentally Friendly Product\\" (SABECO) was transformed into a for the Energy Management System. with ISO 14001 standards. certificate issued by the Vietnam Environment joint st"}\n{"id": "B-A-6d92db6c7264a9e2", "claim": "2023 2024 \\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2023 by CSI100.", "evidence": "TOP 100 TOP 10 Sustainable Enterprises in Vietnam (Manufacturing) awarded by Vietnam Chamber of Commerce and Industry (VCCI) and Vietnam Business Council for Sustainable Development (VBCSD) – CSI100 Reference for 02 consecutive years (2023-2024). TABLE SABECO\'S SUSTAINABILITY MILESTONES 2008 2013 2013-2015 2016 \\u0017 Through equitization, Saigon Beer \\u0017 The first brewery obtained \\u0017 Developed the Environmental \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation certification under ISO 50001:2013 Management System in accordance \\u0017 Awarded the \\"Environmentally Friendly Product\\" (SABECO) was transformed into a for the Energy Management System. with ISO 14001 standards. "}\n{"id": "B-A-7ffd656a8005b7b7", "claim": "\\u0017 Named among the Top 100 Sustainable Businesses in Vietnam 2024 by CSI100.", "evidence": "TOP 100 TOP 10 Sustainable Enterprises in Vietnam (Manufacturing) awarded by Vietnam Chamber of Commerce and Industry (VCCI) and Vietnam Business Council for Sustainable Development (VBCSD) – CSI100 Reference for 02 consecutive years (2023-2024). TABLE SABECO\'S SUSTAINABILITY MILESTONES 2008 2013 2013-2015 2016 \\u0017 Through equitization, Saigon Beer \\u0017 The first brewery obtained \\u0017 Developed the Environmental \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation certification under ISO 50001:2013 Management System in accordance \\u0017 Awarded the \\"Environmentally Friendly Product\\" (SABECO) was transformed into a for the Energy Management System. with ISO 14001 standards. "}\n{"id": "B-A-d1c7f6eb4a26debc", "claim": "\\u0017 Committed to sustainable practices by setting clear targets to reduce emissions, energy consumption, water withdrawal, and waste by 2030.", "evidence": "TABLE SABECO\'S SUSTAINABILITY MILESTONES 2008 2013 2013-2015 2016 \\u0017 Through equitization, Saigon Beer \\u0017 The first brewery obtained \\u0017 Developed the Environmental \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation certification under ISO 50001:2013 Management System in accordance \\u0017 Awarded the \\"Environmentally Friendly Product\\" (SABECO) was transformed into a for the Energy Management System. with ISO 14001 standards. certificate issued by the Vietnam Environment joint stock company. 2013 – 2015 Industry Association. \\u0017 SABECO was awarded the \\u0017 Began transitioning to biomass boilers to replace title of Labor Hero during the fossil fuel systems at selected brewer"}\n{"id": "B-A-a4e07dfba25e72bf", "claim": "\\u0017 Became a member of Packaging Recycling Organization Vietnam to promote packaging management and support the nation\'s green growth.", "evidence": "TABLE SABECO\'S SUSTAINABILITY MILESTONES 2008 2013 2013-2015 2016 \\u0017 Through equitization, Saigon Beer \\u0017 The first brewery obtained \\u0017 Developed the Environmental \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation certification under ISO 50001:2013 Management System in accordance \\u0017 Awarded the \\"Environmentally Friendly Product\\" (SABECO) was transformed into a for the Energy Management System. with ISO 14001 standards. certificate issued by the Vietnam Environment joint stock company. 2013 – 2015 Industry Association. \\u0017 SABECO was awarded the \\u0017 Began transitioning to biomass boilers to replace title of Labor Hero during the fossil fuel systems at selected brewer"}\n{"id": "B-A-8714d800c82898d7", "claim": "foster positive values and support sustainable economic, social, and community MWp at 9 breweries, initiating the development.", "evidence": "Communist Youth Union to implement social and community initiatives that foster positive values and support sustainable economic, social, and community development. Solar Project with a capacity of 9 MWp at 9 breweries, initiating the use of renewable energy in the beer production process. 50001:2018 for the Energy Management System. SOCIAL Striving to create jobs nationwide, improve living conditions, and contribute to the creation of thriving sustainable communities through 6 key areas: public health, art and culture, community development, education, sports, and environmental sustainability. SUSTAINABILITY STRATEGY Community Development: fostering local employment, improving quality of li"}\n{"id": "B-A-dade363734e0662a", "claim": "\\u0017 Became a member of Packaging Recycling", "evidence": "\\u0017 Committed to sustainable practices by setting clear targets to reduce emissions, energy consumption, water withdrawal, and waste by 2030. \\u0017 Became a member of Packaging Recycling Organization Vietnam to promote packaging management and support the nation\'s green growth. \\u0017 Member of Board of Directors Mr Le Thanh Tuan \\u0017 Member of Nomination and Renumeration Committee \\u0017 Member of Land Committee \\u0017 Member of Board of Directors"}\n{"id": "B-A-30fe150e1e0d6347", "claim": "LEADERSHIP COMMUNICATION [GRI 2-22] CHAIRMAN’S MESSAGE Dear Shareholders, This year marked an important milestone for SABECO, as we began the annual reporting of Sustainability separately for the first time in line with the Global Reporting Initiative", "evidence": "Department via: This report is prepared in accordance with the GRI Sustainability Reporting Standards 2021, established by the Global Reporting Initiative (GRI) for the period from January 1, 2024, to December 31, OF THIS REPORT [GRI 2-2; GRI 2-3] REPORTING FRAMEWORK POINT OF CONTACT This is the first Sustainability Report of SABECO. The purpose of the report is to transparently share with stakeholders the company’s strategies, ambitions, and actions in fulfilling its responsibilities towards environmental, social, and governance issues."}\n{"id": "B-A-7c46e6e877cc677f", "claim": "This demonstrates our increasing commitment to transparency and accountability in environmental, social and governance (ESG) practices.", "evidence": "As part of our ongoing commitment to responsible business practices, we have made Governance a priority, ensuring we uphold the standards of transparency, accountability and ethical behavior. We believe that strong governance is fundamental to building trust with our stakeholders and ensuring long-term value creation. Contributing to the UN SDGs: LIGHTING UP THE RURALS AND THE BORDERLINES MANAGEMENT APPROACH SABECO practices Environmental, Social, and Governance (ESG) principles to drive sustainable growth and create long-term value. In line with our commitment to sustainability, SABECO’s approach to community development includes:"}\n{"id": "B-A-541c633317766804", "claim": "As part of our ongoing commitment to responsible business practices, we have made Governance a priority, ensuring we uphold the standards of transparency, accountability and ethical behavior.", "evidence": "(GRI). This demonstrates our increasing commitment to transparency and accountability in environmental, social and governance (ESG) practices. We believe that sustainability development is integral to our long-term success, and we are proud to take this important step in furthering our journey towards a more sustainable future. communities through various social impact programs, including promoting responsible drinking. As a leading brewer in Vietnam, we recognize the critical importance of responsible waste management. We have made great progress in ensuring that our primary packaging materials are recyclable, supporting our efforts to reduce greenhouse gas emissions and minimize environmen"}\n{"id": "B-A-be2a4f73caebee2c", "claim": "Sustainability remains at the core of SABECO’s strategy, directly linked to our ability to operate responsibly and sustainably.", "evidence": "1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas OUR SUSTAINABILITY POLICY SUSTAINABILITY TARGETS AND KPIs 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas PRODUCT INNOVATION PRODUCT LABELLING We encourage consumer to enjoy our product responsibility."}\n{"id": "B-A-efc66a82eb84620e", "claim": "On the Environmental front, from greenhouse gas emission reduction, water conservation, wastewater management, energy-efficient brewing processes to the deployment of solar panels at our breweries, we are investing in initiatives that contribute to the sustainability of our operations and the communities we engaged with.", "evidence": "As a leading brewer in Vietnam, we recognize the critical importance of responsible waste management. We have made great progress in ensuring that our primary packaging materials are recyclable, supporting our efforts to reduce greenhouse gas emissions and minimize environmental impacts. Additionally, we maintain a responsible approach to water conservation across our operations. These efforts have enhanced both our energy and water efficiency. Currently, 13 of our breweries in system are equipped with solar panels, and we are aiming to expand this initiative across all our sites to reduce our environmental footprint and support sustainable operations. Good governance remains a critical comp"}\n{"id": "B-A-3cc01d70ac24fc66", "claim": "On the Social front, we prioritized the well-being and safety of our workforce while actively engaged the local Yours faithfully, Koh Poh Tiong Chairman of the Board", "evidence": "Directors (VIOD) \\u0017 Chairman of Board of Directors Mr Koh Poh Tiong \\u0017 Audit Committee Master Program, Vietnam Institute of \\u0017 Chairman of Sustainability and Risk Management Committee. TABLE Member Mr. Koh Poh Tiong | Board Position Chairman of the Board | SRMC Position Chairman Mr. Michael Chye Hin Fah | Member | Member Ms. Ngo Minh Chau | Member | Member"}\n{"id": "B-A-a0d58269660591e7", "claim": "Over the past year, we have made significant strides toward a more sustainable and efficient future.", "evidence": "(GRI). This demonstrates our increasing commitment to transparency and accountability in environmental, social and governance (ESG) practices. We believe that sustainability development is integral to our long-term success, and we are proud to take this important step in furthering our journey towards a more sustainable future. communities through various social impact programs, including promoting responsible drinking. The report focuses on presenting SABECO\'s sustainability efforts for the year 2024, addressing the key concerns raised by our stakeholders. We value and appreciate all feedback to make our future sustainability report more relevant to stakeholder’s needs. If you have any sugg"}\n{"id": "B-A-f55b14ab4cebd0bf", "claim": "These efforts have enhanced both our energy and water efficiency.", "evidence": "From installing solar-powered street lighting to building community football fields to supporting disaster relief efforts after Typhoon Yagi, we are dedicated to making a tangible, positive contribution to the communities we serve. Over the past year, we have made significant strides toward a more sustainable and efficient future. We have expanded our use of solar energy and made improvements to our brewing processes. EMISSIONS & ENERGY CONSUMPTION MANAGEMENT APPROACH SABECO is committed to raising the proportion of renewable energy usage by increasing investment in innovative low-carbon technologies that contribute to climate change mitigation and adaptation efforts, as well as promoting op"}\n{"id": "B-A-7d2fd0580e84bbeb", "claim": "Currently, 13 of our breweries in system are equipped with solar panels, and we are aiming to expand this initiative across all our sites to reduce our environmental footprint and support sustainable operations.", "evidence": "As a leading brewer in Vietnam, we recognize the critical importance of responsible waste management. We have made great progress in ensuring that our primary packaging materials are recyclable, supporting our efforts to reduce greenhouse gas emissions and minimize environmental impacts. Additionally, we maintain a responsible approach to water conservation across our operations. On the Environmental front, from greenhouse gas emission reduction, water conservation, wastewater management, energy-efficient brewing processes to the deployment of solar panels at our breweries, we are investing in initiatives that contribute to the sustainability of our operations and the communities we engaged "}\n{"id": "B-A-a8524f15587ec966", "claim": "As a leading brewer in Vietnam, we recognize the critical importance of responsible waste management.", "evidence": "The remaining breweries (64% of beer production sites) are considered as having Low-Medium as and Low water-stressed risk level. At SABECO, we are dedicated to responsible water stewardship, recognizing water as an essential ingredient in our beer, in our brewing process and community well-being. 7% Reduction of water intensity by 2030, compared to 2023 (beverages) 100% Recognize the importance of natural resources and the environment by encouraging the use of resources effectively along with developing innovation, and technologies in order to increase efficiency and reduce any environmental impact. Place importance on and care of the quality and safety of the food chain by serving consumers"}\n{"id": "B-A-66253293d4751565", "claim": "Additionally, we maintain a responsible approach to water conservation across our operations.", "evidence": "In the water-scarce regions of Vietnam where our breweries operate, conserving water is essential. We maintain our collaboration with local stakeholders to ensure rigorous adherence to wastewater regulations, manage water resources responsibly, and safeguard local water supplies for future generations. Surface water (ML) Groundwater (ML) Third-party water (ML) These efforts have enhanced both our energy and water efficiency. Currently, 13 of our breweries in system are equipped with solar panels, and we are aiming to expand this initiative across all our sites to reduce our environmental footprint and support sustainable operations. Good governance remains a critical component of our sustain"}\n{"id": "B-A-0376247ab8e19bf2", "claim": "By optimizing water usage, we ensure that our sustainability efforts extend beyond our operations to benefit the broader communities that share these vital resources.", "evidence": "I am proud to say that our efforts go beyond our operations. Through our partnership with the Ho Chi Minh Communist Youth Union, we have launched a nationwide initiative to plant approximately 20 hectares of trees annually. Our commitment to giving back to the community reaches often-overlooked sectors, including fishermen and vulnerable groups. These efforts have enhanced both our energy and water efficiency. Currently, 13 of our breweries in system are equipped with solar panels, and we are aiming to expand this initiative across all our sites to reduce our environmental footprint and support sustainable operations. Good governance remains a critical component of our sustainability strateg"}\n{"id": "B-A-a0abe28e0b267a6b", "claim": "Looking ahead, we remain committed to growing SABECO in a way that reflects our values of innovation, sustainability, and community engagement.", "evidence": "LEADERSHIP COMMUNICATION GENERAL DIRECTOR’S MESSAGE Dear Stakeholders, As we continue our commitment to sustainable growth and operational excellence, I am proud to reflect on our deep-rooted traditions that have shaped our success while embracing modernity, innovation and sustainability. These core values shape our identity and guide our strategic direction as we continue to evolve and move forward. 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas PRODUCT INNOVATION PRODUCT LABELLING We encourage consumer to enjoy our product responsibility."}\n{"id": "B-A-1a78c6b66d7f0443", "claim": "Through responsible resource management, operational excellence, and a focus on employee and community development, we’re confident that we will continue to build a company that thrives and makes a lasting, positive impact on the environment and society.", "evidence": "We continue to uphold strong corporate oversight through two board- level committees focused on governance and risk management, with a third committee is in inception. These initiatives ensure that we’re making informed, transparent decisions that support our long-term growth. Community activities hosted Positive Impacts SABECO believes that sports should be accessible to everyone, not just professional athletes. To create lasting impact, we focus on building & supporting community sports initiatives that encourage people to lead active lifestyles."}\n{"id": "B-A-fd518bca60bd6833", "claim": "3 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas ABOUT SABECO [GRI 2-1;", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas SUPPLY CHAIN MANAGEMENT [GRI 308; GRI 414]"}\n{"id": "B-A-bc3b236ac859fb3d", "claim": "Department via: This report is prepared in accordance with the GRI Sustainability Reporting Standards 2021, established by the Global Reporting Initiative (GRI) for the period from January 1, 2024, to December 31,", "evidence": "LEADERSHIP COMMUNICATION [GRI 2-22] CHAIRMAN’S MESSAGE Dear Shareholders, This year marked an important milestone for SABECO, as we began the annual reporting of Sustainability separately for the first time in line with the Global Reporting Initiative OF THIS REPORT [GRI 2-2; GRI 2-3] REPORTING FRAMEWORK POINT OF CONTACT This is the first Sustainability Report of SABECO. The purpose of the report is to transparently share with stakeholders the company’s strategies, ambitions, and actions in fulfilling its responsibilities towards environmental, social, and governance issues."}\n{"id": "B-A-772a432deb1ed602", "claim": "ir@sabeco.com.vn The sustainability data were collected covering all facilities under SABECO’s operational control, directly or indirectly owned, in which it controls more than 50% of the voting rights.", "evidence": "This includes guidelines for greenhouse gas (GHG) accounting and energy consumption for reporting sustainability data, which are standardized across all SABECO breweries. Additionally, we have organized specialized training sessions for our brewery teams to assist them in collecting and monitoring environmental data on energy usage, emissions, water use, and waste management. So far, more than 1,400 employees have completed the training. We are committed to continuously improving our practices, measuring our impact and setting aspiring goals for the future. This report is just the beginning, and we look forward to sharing our progress in the years to come. Sustainability remains at the core "}\n{"id": "B-A-48fa3517ef788df3", "claim": "Biodiversity In 2024, SABECO conducted a Materiality Assessment in alignment with the GRI Standards 2021.", "evidence": "Medium Significance Biodiversity Low Topic Prioritization Importance To Business Low High As this is SABECO’s first time conducting a Materiality Assessment, no changes from the previous reporting period are identified. Materiality Endorsement 24 | SABECO 2024 Sustainability Report 103 SUSTAINABILITY STRATEGY AT SABECO MATERIALITY ASSESSMENT APPROACH 23 23 Materiality Assessment Process 24 Materiality Matrix - Material Topic Prioritization 25 SUSTAINABILITY FRAMEWORK, TARGETS AND KPIs"}\n{"id": "B-A-1efaee20f4517e8c", "claim": "The impact on the business is based on the degree to which the ESG topic impacts the company operationally, financially, or reputationally.", "evidence": "Meanwhile, the importance to stakeholders assesses the degree of stakeholder concern about the topic, as well as the degree to which the company creates positive or negative impacts on the broader environment and society as a result of its operations. By prioritizing the topics accordingly, this ensures that SABECO’s sustainability efforts are focused on areas of greatest importance and influence. Climate Change/Greenhouse Gas Emissions and Energy Consumption Risk management is essential for the Company\'s sustainable development. Additionally, the SRMC also collaborates closely with the Audit Committee, which oversees the integrity of the group’s reporting."}\n{"id": "B-A-5f400d30a007f074", "claim": "By prioritizing the topics accordingly, this ensures that SABECO’s sustainability efforts are focused on areas of greatest importance and influence.", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas In compliance with the Environmental Protection Law on Extended 1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas 1 2 3 4 The SRMC is comprised of the following members:"}\n{"id": "B-A-91de78342d89fc3a", "claim": "Climate Change/Greenhouse Gas Emissions and Energy Consumption", "evidence": "05 Climate Change/Greenhouse Gas Emissions and Energy Consumption 43 SABECO\'s Sustainability Milestones 07 Water Stewardship 49 LEADERSHIP COMMUNICATION Enviroment Social Governance \\u0017 Climate Change/ \\u0017 Talent Attraction and \\u0017 Corporate Governance Greenhouse Gas Emissions and Energy Consumption"}\n{"id": "B-A-f4177141bad2fc5a", "claim": "Topic Identification Stakeholder Engagement Environmental Social Governance As SABECO operates within the beverage industry, we researched peers and global trends to identify an initial list of ESG", "evidence": "Corporate Governance Annual Report 2024 30-32 2-10 Stakeholder engagement Approach to stakeholder engagement 2-29 37-41 Stakeholder Engagement Chair of the highest governance body (ESG) topics that are most significant to the business, its stakeholders, and the broader societal and environmental context. Each topic is evaluated based on its importance to stakeholders and its impact on the business. The impact on the business is based on the degree to which the ESG topic impacts the company operationally, financially, or reputationally."}\n{"id": "B-A-f4d2264b8c15c9f0", "claim": "High Packaging And Circular Economy Supply Chain Management Community Engagement Importance to stakeholders", "evidence": "\\u0017 Packaging and Management laws and regulations and maintaining transparent business practices. Circular Economy \\u0017 Community Development Strengthening our corporate governance and management TABLE Packaging | and Circular Economy"}\n{"id": "B-A-4d300e426103a8f3", "claim": "Occupational Health & Safety Analysis of Responses Customer Health & Safety Climate Change/ Greenhouse Gas Emissions & Energy Consumption", "evidence": "Enviroment Social Governance \\u0017 Climate Change/ \\u0017 Talent Attraction and \\u0017 Corporate Governance Greenhouse Gas Emissions and Energy Consumption Greenhouse Gas Emission Performance – 2024 GRI 403 – Occupational Health and Safety 2018 403-1 62 305-4 GHG emissions intensity"}\n{"id": "B-A-14da65a25c95eedd", "claim": "Interviews were conducted with internal stakeholders to obtain their feedback on the importance of material ESG topics to business operations, while interviews with external stakeholders were used to obtain their feedback on the relative importance of material ESG topics to stakeholder’s concern.", "evidence": "Our internal stakeholders consist of board level and management level executives as well as employees. External stakeholders consist of customers, suppliers, and investors. The stakeholders were engaged through interviews and surveys to obtain their opinion on the priorities of material ESG topics . topics relevant to the sector and business operations. We mapped out relevant internal and external stakeholders, considering stakeholders that play an important role in the operations of SABECO as well as those who are most impacted by the business’s operations, both positively and negatively."}\n{"id": "B-A-c61069d9a8654945", "claim": "Stakeholder feedback was then weighted and ranked through a scoring system based on their selection of material ESG topics", "evidence": "Our internal stakeholders consist of board level and management level executives as well as employees. External stakeholders consist of customers, suppliers, and investors. The stakeholders were engaged through interviews and surveys to obtain their opinion on the priorities of material ESG topics . (ESG) topics that are most significant to the business, its stakeholders, and the broader societal and environmental context. Each topic is evaluated based on its importance to stakeholders and its impact on the business. The impact on the business is based on the degree to which the ESG topic impacts the company operationally, financially, or reputationally."}\n{"id": "B-A-9f78951e6395251c", "claim": "The ESG topics were prioritized into high significance, medium significance and foundation topics based on the results above.", "evidence": "(ESG) topics that are most significant to the business, its stakeholders, and the broader societal and environmental context. Each topic is evaluated based on its importance to stakeholders and its impact on the business. The impact on the business is based on the degree to which the ESG topic impacts the company operationally, financially, or reputationally. Medium Significance Biodiversity Low Topic Prioritization Importance To Business Low High As this is SABECO’s first time conducting a Materiality Assessment, no changes from the previous reporting period are identified. Materiality Endorsement 24 | SABECO 2024 Sustainability Report"}\n{"id": "B-A-1f6d3d570249b30e", "claim": "Medium Significance Biodiversity Low Topic Prioritization Importance To Business Low High As this is SABECO’s first time conducting a Materiality Assessment, no changes from the previous reporting period are identified.", "evidence": "1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas MATERIALITY ASSESSMENT APPROACH MATERIALITY MATRIX – MATERIAL TOPIC PRIORITIZATION The ESG topics were prioritized into high significance, medium significance and foundation topics based on the results above. Material topics with different priority levels are recommended to have different implications for the company’s approach to its management. The materiality matrix is then presented to"}\n{"id": "B-A-7e6f48f2aac3c65f", "claim": "Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions", "evidence": "TABLE Pillar ENVIRONMENT | Our Commitment Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions | Our Strategy Continuously seeking innovative solutions and initiatives to minimize environmental impact while optimizing the use of resources, raw materials, packaging, energy, and water. SOCIAL | Human Capital Development: maintaining safe working conditions, taking care of physical health, mental health & welfare of employees, focusing on employee development and talent acquisition as well as respecting diversity in the workplace | WARM workforce strategy: \\u0017 Willing – Attracting and onboarding passionate talent \\u0017 Able – D"}\n{"id": "B-A-19d4a9f3453db60a", "claim": "\\u0017 ENVIRONMENT Improve the quality of life by providing high quality, safe and healthy beverages.", "evidence": "Positive Impacts Improving quality of life – Providing safer, well-lit environments for communities in rural and border areas Enhancing socio-economic infrastructure – Strengthening connectivity and mobility, fostering local development [GRI 403] SABECO’s “Corporate Occupational Health and Safety Policy” provides guidance for best practices on healthy and safe working environment, ensuring alignment with high OHS standards for protecting employees’ health and safety."}\n{"id": "B-A-fe163a292c50a6b0", "claim": "CORE VALUE \\u0017 \\u0017 Heritage Brand WARM workforce strategy: Satisfy and meet the needs of beverages under the international food safety standards “Food hygiene and safety and environmental protection”.", "evidence": "Vietnam and expand internationally, SABECO prioritizes the development of a “WARM” workforce. Executive Senior Officer Through a professional, dynamic, and fair working environment, we strengthen our competitive advantage while enhancing employee engagement and satisfaction. WARM workforce strategy: Departments register training needs Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations. SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement. SABEC"}\n{"id": "B-A-32898f5a79a3eff1", "claim": "\\u0017 Human Capital Development: maintaining safe working conditions, taking care of physical health, mental health &", "evidence": "TABLE Pillar ENVIRONMENT | Our Commitment Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions | Our Strategy Continuously seeking innovative solutions and initiatives to minimize environmental impact while optimizing the use of resources, raw materials, packaging, energy, and water. SOCIAL | Human Capital Development: maintaining safe working conditions, taking care of physical health, mental health & welfare of employees, focusing on employee development and talent acquisition as well as respecting diversity in the workplace | WARM workforce strategy: \\u0017 Willing – Attracting and onboarding passionate talent \\u0017 Able – D"}\n{"id": "B-A-f7a1b0513d3e05af", "claim": "By embedding these principles into our operations, we proactively identify key issues that most significantly impact our stakeholders, ensuring sustainable growth while creating lasting value for society and the environment.", "evidence": "Our sustainability targets and KPIs based on the 3 pillars Environment, Social and Governance are embedded into our business development strategy, promoting competitives advantages to achieve long-term, inclusive growth and sutainability. Keep the balance of economy, society and environment, including stakeholders who are working towards the implementation of sustainable growth of the business. Embedding DEI into the organizational culture and hiring practices; measuring DEI outcomes and fostering an inclusive workplace environment. Digital Transformation and Technology 5 By investing in our people and strengthening our organizational capabilities, we aim to drive sustainable growth and posi"}\n{"id": "B-A-7b89f5984401d7d7", "claim": "Enviroment Social Governance \\u0017 Climate Change/ \\u0017 Talent Attraction and \\u0017 Corporate Governance Greenhouse Gas Emissions and Energy Consumption", "evidence": "Occupational Health & Safety Analysis of Responses Customer Health & Safety Climate Change/ Greenhouse Gas Emissions & Energy Consumption 104 Climate Change/ Greenhouse Gas Emission and Energy Consumption 306-4 Waste diverted from disposal Appendix B – ESG Data 104 GRI 3 – Material Topics 2021 Management Approach"}\n{"id": "B-A-cc572852cf4e6a45", "claim": "Development and Ethics \\u0017 Occupational Health \\u0017 Supply Chain and Safety Management \\u0017 Water Stewardship \\u0017 Customer Relationship \\u0017 Complying with current", "evidence": "Consumer Health and Safety Water Stewardship Customer Relationship Management Talent Attraction and Development The process to determine material topics is broken down into 5 key steps: SOCIAL DIMENSION OCCUPATIONAL HEALTH AND SAFETY [GRI 403] TALENT ATTRACTION AND DEVELOPMENT [GRI 2-7, GRI 2-8, GRI 405-1] CUSTOMER RELATIONSHIP MANAGEMENT"}\n{"id": "B-A-170152e6e7aaa637", "claim": "Circular Economy \\u0017 Community Development Strengthening our corporate governance and management", "evidence": "Management Packaging and Circular Economy MATERIALITY ASSESSMENT PROCESS Occupational Health and Safety Corporate Governance and Ethics They also provided information on SABECO’s strengths, challenges, and areas for improvement. High Packaging And Circular Economy Supply Chain Management Community Engagement Importance to stakeholders"}\n{"id": "B-A-92334e6b47c9594c", "claim": "\\u0017 Consumer Health \\u0017 Improving risk and Safety management capcity.", "evidence": "– 2024 3-3 Management of material topics 61 47 305-2 Occupational health and safety management system Occupational Health and Safety Management System and Services 17 Consumer Health and Safety 83 91 GOVERNANCE DIMENSION Corporate Governance and Business Ethics 91 ABOUT THIS SUSTAINABILITY REPORT Supply Chain Management"}\n{"id": "B-A-a72f98887915cfcb", "claim": "| Striving to create jobs nationwide, improve living conditions, and contribute to the creation of thriving sustainable communities through 6 key areas: public health, art and culture, community development, education, sports, and environmental sustainability.", "evidence": "SOCIAL Striving to create jobs nationwide, improve living conditions, and contribute to the creation of thriving sustainable communities through 6 key areas: public health, art and culture, community development, education, sports, and environmental sustainability. SUSTAINABILITY STRATEGY Community Development: fostering local employment, improving quality of life for community prosperity. Encouraging renewable energy adoption – Promoting alternative power sources and afforestation to support sustainable growth. In this light, the Company’s ESG initiatives are developed around six key areas, namely Public Health, Art and Culture, Community Development, Education, Sports, and Environmental Su"}\n{"id": "B-A-07c0fc7034fb069f", "claim": "Our sustainability targets and KPIs based on the 3 pillars Environment, Social and Governance are embedded into our business development strategy, promoting competitives advantages to achieve long-term, inclusive growth and sutainability.", "evidence": "\\u0017 Continuous Improvement \\u0017 Ensure the development towards international integration. \\u0017 \\u0017 Ready – Providing a professional and healthy work environment OUR SUSTAINABILITY STRATEGY We believe that long-term success is built on collaboration, strong governance, inclusive stakeholder engagement, and a strategic focus on sustainability. To achieve this, our approach is built on three core pillars: Environment, Social, and Governance. By embedding these principles into our operations, we proactively identify key issues that most significantly impact our stakeholders, ensuring sustainable growth while creating lasting value for society and the environment. \\u0017 Motivated – Ensuring competitive compens"}\n{"id": "B-A-23aaf849041db2a7", "claim": "Reduce the impacts from the implementation of the Company\'s value chain to align with the growth of the business by covering dimensions of society, environment, and occupational health and safety.", "evidence": "Our sustainability targets and KPIs based on the 3 pillars Environment, Social and Governance are embedded into our business development strategy, promoting competitives advantages to achieve long-term, inclusive growth and sutainability. Keep the balance of economy, society and environment, including stakeholders who are working towards the implementation of sustainable growth of the business. 7% Reduction of water intensity by 2030, compared to 2023 (beverages) 100% Recognize the importance of natural resources and the environment by encouraging the use of resources effectively along with developing innovation, and technologies in order to increase efficiency and reduce any environmental i"}\n{"id": "B-A-20eeecc266b79ad6", "claim": "Pillars Targets and KPIs The Sustainability Policy sets out clear intentions and guidelines as follows:", "evidence": "1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas OUR SUSTAINABILITY POLICY SUSTAINABILITY TARGETS AND KPIs The policy sets out clear intentions and guidelines to minimize water impact from our production activities, by assessing water quality and quantity, applying international water resource management tools and standards, using new technologies, incorporating water sustainability management into business strategy, decision making and being responsible for water consumption. SABECO has implemented water stewardship practices for the long-term, sustainable use of water for our own operations and local communit"}\n{"id": "B-A-6487e7c1a3bec543", "claim": "42% GHG reduction in Scope 1 & 2 by 2030 (compared to 2023 baseline) Environment 50% Renewable energy by 2030 Adhere to the good conduct of its business and risk management by covering all dimensions that have an impact on an appropriate manner in business operation and compliance with applicable laws and regulations.", "evidence": "Contributing to the UN SDGs 2030 Target 2050 Target Net Zero emission for Scope 1, 2 and 3 by 2050. 50% 42% GHG reduction in Scope 1 and 2 by 2030 (compared to 2023 baseline) Renewable energy by 2030 HIGHLIGHTED PERFORMANCE Saving 5,833,142 KWh 40.54% Renewable energy replacing fossil energy in beer operation. of electricity in beer production in 2024 compared to 2023, which reduce 8.13% of electricity consumption"}\n{"id": "B-A-d822e9698d3fad8e", "claim": "Commit and conduct sustainable performance transparently for continuous development.", "evidence": "Developing employees’ competencies through training and development. Providing a professional and healthy work environment. Ensuring competitive compensation and benefits, and fair performance evaluations. Percentage of employees receiving regular performance & career development reviews in 2024, by Gender Percentage of employees receiving regular performance and career development reviews in 2024, by Employee Category Officer Male employees Employees are encouraged to actively share their insights, feedback, and ideas for continuous improvement in daily work."}\n{"id": "B-A-ed1fbb6b00e002c2", "claim": "Place importance on and care of the quality and safety of the food chain by serving consumers healthy and products, with responsibility to society and environment.", "evidence": "CORE VALUE \\u0017 \\u0017 Heritage Brand WARM workforce strategy: Satisfy and meet the needs of beverages under the international food safety standards “Food hygiene and safety and environmental protection”. \\u0017 \\u0017 Social Responsibility \\u0017 Willing – Attracting Reduce the impacts from the implementation of the Company\'s value chain to align with the growth of the business by covering dimensions of society, environment, and occupational health and safety. Pillars Targets and KPIs The Sustainability Policy sets out clear intentions and guidelines as follows:"}\n{"id": "B-A-ab3707934af65835", "claim": "of all beverage packaging to be made of reusable and, recyclable material by 2040 Zero Social Fatality Zero Lost Time Injury Frequency Rate (LTIFR)", "evidence": "Contributing to the UN SDGs: Zero fatality Zero lost time injury frequencyrate (LTIFR) Target MANAGEMENT APPROACH OHS MANAGEMENT SYSTEM AND SERVICE GRI 301-2; GRI 306-1, GRI 306-2, GRI 306-4] A CLOSED-LOOP GLASS BOTTLE RECYCLE MODEL Contributing to the UN SDGs Breweries 100% of all beverage packaging to be made of reusable and, recyclable material by 2040."}\n{"id": "B-A-7407a353e198311a", "claim": "To strengthen practices and ensure a unified approach, sustainable development has been integrated into its corporate strategy and risk management processes.", "evidence": "Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations. SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement. SABECO has established an OHS management system to ensure safety and hygiene standards are integrated into daily operations. Contributing to the UN SDGs: LIGHTING UP THE RURALS AND THE BORDERLINES MANAGEMENT APPROACH SABECO practices Environmental, Social, and Governance (ESG) principles to drive sustainable growth and create long-term valu"}\n{"id": "B-A-2ebc10a0388e167b", "claim": "Sales Administration & Operations (1) Technical Accounting CEO Office Land Management Brand Marketing Corporate Communication Sustainability Management", "evidence": "Technology Trade Marketing (1) Cu Chi Brewery Corporate Office Management System Sales Capability & Sales Audit (1) Investment Planning Assurance (1) \\u0017 Continue to improve \\u0017 Via phone awards and recognitions for its corporate brand and products and aspirations of employees to the Communist Party and Labor Union corporate governance and emails \\u0017 Enhance the overall \\u0017 Survey Management capability and corporate value operational efficiency"}\n{"id": "B-A-6e8eb41c90989b36", "claim": "Roles of Sustainability and Risk Management Committee: To report to the Board and provide appropriate advice and recommendations on material risk issues and a risk management system for the timely identification, mitigation and management of key risks that may have a material impacts on SABECO.", "evidence": "(“ESG factors”) identified as material to the business, monitoring and managing ESG factors, and overseeing standards, management processes, and strategies to achieve sustainability practices. To report to the Board and provide appropriate updates and recommendations on sustainability issues. RISK GOVERNANCE AND MANAGEMENT By identifying and addressing potential risks, we ensure continuous production and business activities, thereby balancing economic efficiency, environmental protection, and social responsibility aligned with our Sustainability Strategy. The responsibility for identifying, evaluating and monitoring risks and material ESG factors lies with the Management. They also report re"}\n{"id": "B-A-93c96927ca44c982", "claim": "(“ESG factors”) identified as material to the business, monitoring and managing ESG factors, and overseeing standards, management processes, and strategies to achieve sustainability practices.", "evidence": "RISK GOVERNANCE AND MANAGEMENT By identifying and addressing potential risks, we ensure continuous production and business activities, thereby balancing economic efficiency, environmental protection, and social responsibility aligned with our Sustainability Strategy. The responsibility for identifying, evaluating and monitoring risks and material ESG factors lies with the Management. They also report regularly to the SRMC and Board the status and effectiveness of risk mitigation measures, sustainability practices and strategy implementation. The primary function of the SRMC is to provide guidance to the Management team to ensure the effective implementation of ESG objectives and initiatives."}\n{"id": "B-A-5b127a5681060b79", "claim": "Risk management is essential for the Company\'s sustainable development.", "evidence": "Support the development of innovation and technologies to sustain operations, growth, and shape the future for the Company\'s. Commit and conduct sustainable performance transparently for continuous development. 100% Water replenishment by 2040 (water used in finished goods) The Company prioritizes aligning operations with its sustainability framework to deliver value across the value chain. To strengthen practices and ensure a unified approach, sustainable development has been integrated into its corporate strategy and risk management processes. for overseeing the corporation’s impacts and sustainability agenda."}\n{"id": "B-A-0fdf212c0237c9e5", "claim": "This includes reviewing the quarterly report on internal controls and enterprise risk management to help safeguard the assets of the Company, while ensuring compliance with legal and regulatory requirements.", "evidence": "Supplier selection is based on a robust and systematic supplier qualification in place which includes both economic and ESG aspects, ensuring compliance with legal and confidentiality requirements. Our supply chain management strategy ensures sustainability, regulatory compliance, and long-term supplier partnerships while maintaining transparency and efficiency in operations. This system includes: Safety regulations and internal controls: Employee health and insurance: Safety governance: Key elements of SABECO’s OHS management approach include:"}\n{"id": "B-A-365cc11e270fe892", "claim": "4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas STAKEHOLDER ENGAGEMENT [GRI 2-29] SABECO is strengthening its engagement with stakeholders through a variety of formal and informal communication channels and activities, fostering meaningful interactions through collaboration.", "evidence": "SABECO is strengthening its engagement with stakeholders through a variety of formal and informal communication channels and activities, fostering meaningful interactions through collaboration. Stakeholders were identified and prioritized based on those affected by the Company’s operations and decisions. SABECO has identified the following stakeholder groups for engagement: Customers/ 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas TRAINING AND ENGAGEMENT SABECO fosters active engagement with its workforce in OHS management through comprehensive training and communication programs, specifically:"}\n{"id": "B-A-7730ccf81b19e058", "claim": "This feedback is useful in determining the company’s approach to managing its stakeholder engagement and determining the strategy, direction, plans and guidelines for future business sustainability development and for the prevention of negative impacts from business operations and activities.", "evidence": "1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas STAKEHOLDER ENGAGEMENT [GRI 2-29] SABECO is strengthening its engagement with stakeholders through a variety of formal and informal communication channels and activities, fostering meaningful interactions through collaboration. The Company prioritizes aligning operations with its sustainability framework to deliver value across the value chain. To strengthen practices and ensure a unified approach, sustainable development has been integrated into its corporate strategy and risk management processes. for overseeing the corporation’s impacts and sustainability agen"}\n{"id": "B-A-92d8caedc0224365", "claim": "\\u0017 SABECO \\u0017 Comply with production website and social networks Product Safety standards and regulations and Quality \\u0017 Via phone and emails \\u0017 Market visits \\u0017 Launch new products", "evidence": "TABLE \\u0017 Apply advanced science and technology to improve product quality and designs \\u0017 Comply with production standards and regulations | \\u0017 Beer freshness was ensured in the market \\u0017 SABECO products won various international awards, a testament towards beer quality and continuous improvement \\u0017 Launch new products \\u0017 Expand the distribution system \\u0017 Launch various attractive consumer promotions \\u0017 Collect feedback from consumers \\u0017 Conduct research to meet the dynamic market demands | \\u0017 SABECO received various positive feedback from consumers \\u0017 SABECO products were easy to find \\u0017 Brand awareness and brand equity were improved \\u0017 Apply motivated incentive schemes | \\u0017 Distributors were encouraged t"}\n{"id": "B-A-0b3d0dfad317a32d", "claim": "\\u0017 The Corporation’s \\u0017 Ensure a professional, \\u0017 Safety and wellness leading position was reaffirmed \\u0003collaborative, safe, and \\u0003clean working environment of employees were ensured", "evidence": "TABLE \\u0017 Apply a performance- based salary structure to ensure the competitiveness and fairness \\u0017 Recognize and evaluate contributions and achievements of employees \\u0017 Organize training courses to nd improve the professionalism s for employees \\u0017 Communicate viewpoints and aspirations of employees to the Communist Party and Labor Union | \\u0017 Employees’ security and motivation were maintained \\u0017 Employees were supported in developing their career paths \\u0017 Work efficiency was comparable to salary and bonus \\u0017 Ensure a professional, collaborative, safe, and clean working environment \\u0017 Organize regular discussions and talks between the Management and employees \\u0017 Apply open-door policy \\u0017 Issue policies r"}\n{"id": "B-A-a3405342164b3fbd", "claim": "\\u0003Corporate Occupational \\u0003Health Safety, Human \\u0003Rights, and \\u0003Anti-Corruption \\u0017 Involve legal consulting \\u0017 SABECO was \\u0017 Suppliers in all contracts considered a reliable partner for various suppliers in the market network within", "evidence": "TABLE \\u0017 Involve legal consulting in all contracts \\u0017 Ensure data confidentiality for all suppliers \\u0017 Select suppliers based on their qualifications, experience, and reputation \\u0017 Resolve conflicts on the basis of goodwill and respect \\u0017 Maintain and strengthen the long-term partnership between SABECO and suppliers \\u0017 Information sharing for better work efficiency | \\u0017 SABECO was considered a reliable partner for various suppliers in the market \\u0017 Close and mutually beneficial relationships were maintained and developed \\u0017 Stable and high- quality sources of raw materials were ensured \\u0017 Pricing was competitive and transparent | \\u0017 Suppliers network within SABECO group of companies \\u0017 Negotiation meeti"}\n{"id": "B-A-5a40740a741732f6", "claim": "\\u0017 Ensure a professional, collaborative, safe, and clean working environment", "evidence": "\\u0017 The Corporation’s \\u0017 Ensure a professional, \\u0017 Safety and wellness leading position was reaffirmed \\u0003collaborative, safe, and \\u0003clean working environment of employees were ensured Specifically, the headquarters office was refurbished and modernized, while offices at several subsidiaries were also upgraded, ensuring a professional and comfortable working environment for our staff. The office renovation included major components such as working areas, meeting rooms, common areas, multi-functional meeting rooms, the Boardroom, and the executive lounge. This renovation modernized the office environment to align with international standards, enhanced the corporate image, and strengthened the compan"}\n{"id": "B-A-4275767c08596c68", "claim": "\\u0017 Focus on the development of young talents through youth empowerment programs and sponsorships for Vietnamese young entrepreneurs and sports talents, and other projects that contribute to national goals \\u0017 Projects of sustainable development \\u0017 SABECO was recognized as a strategic partner who plays a major role in community sustainable development projects", "evidence": "TABLE \\u0017 Launch various large- scale CSR activities \\u0017 Focus on the development of young talents through youth empowerment programs and sponsorships for Vietnamese young entrepreneurs and sports talents, and other projects that contribute to national goals | \\u0017 The reputation and image of SABECO and Bia Saigon brand were strengthened within local communities \\u0017 SABECO was recognized as a strategic partner who plays a major role in community sustainable development projects \\u0017 Young talents had more opportunities to improve their capability, contributing to their hometown’s development \\u0017 Comply with regulations on environmental protection \\u0017 Conduct various initiatives to cut down on raw materials,"}\n{"id": "B-A-8ba2c6e05fb89aa4", "claim": "Our actions are guided by our Environmental Policy, which commits us to strong environmental standards on energy, emissions, water, waste and implement the sustainable development of packaging by using returnable glass bottles, recycled cullet and alumininum can and eco-friendly packaging design, reducing materials consumption from natural resources.", "evidence": "\\u0017 Committed to sustainable practices by setting clear targets to reduce emissions, energy consumption, water withdrawal, and waste by 2030. \\u0017 Became a member of Packaging Recycling Organization Vietnam to promote packaging management and support the nation\'s green growth. GRI 2-28] Pillar Our Commitment Our Strategy Continuously seeking innovative solutions and initiatives to minimize environmental impact while optimizing the use of resources, raw materials, packaging, energy, and water. Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions"}\n{"id": "B-A-c11a5f3afaad31f1", "claim": "Contributing to the UN SDGs 2030 Target 2050 Target Net Zero emission for Scope 1, 2 and 3 by 2050.", "evidence": "Contributing to the UN SDGs: Zero fatality Zero lost time injury frequencyrate (LTIFR) Target MANAGEMENT APPROACH OHS MANAGEMENT SYSTEM AND SERVICE Contributing to the UN SDGs SUPPLY CHAIN MANAGEMENT SYSTEM TARGET 100% 100% of Active Suppliers comply with SABECO’s Supplier and Business Partner Code of Practice by 2030."}\n{"id": "B-A-a6808d7d94bddb81", "claim": "50% 42% GHG reduction in Scope 1 and 2 by 2030 (compared to", "evidence": "42% GHG reduction in Scope 1 & 2 by 2030 (compared to 2023 baseline) Environment 50% Renewable energy by 2030 Adhere to the good conduct of its business and risk management by covering all dimensions that have an impact on an appropriate manner in business operation and compliance with applicable laws and regulations. Best Corporate Governance (Vietnam Listed Company Awards 2024). Reduce 9.3% Reduction in water intensity compared to 2023. 7.3% 0.92% Overall GHG emission reduction in comparison with the base year 2023."}\n{"id": "B-A-4e7974c2485daacc", "claim": "At SABECO, we acknowledge that climate change is a critical global challenge affecting ecosystems and communities.", "evidence": "EMISSIONS & ENERGY CONSUMPTION MANAGEMENT APPROACH SABECO is committed to raising the proportion of renewable energy usage by increasing investment in innovative low-carbon technologies that contribute to climate change mitigation and adaptation efforts, as well as promoting optimization of overall resource use for maximum efficiency. This approach extends to raising awareness among suppliers and partners about climate change. 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas ENVIRONMENT DIMENSION CLIMATE CHANGE/GREENHOUSE GAS"}\n{"id": "B-A-fab6325c787ef3b4", "claim": "We are focused on reducing energy use, improving energy efficiency throughout our operations, increasing our reliance on renewable energy, and adopting high-efficiency technologies.", "evidence": "EMISSIONS & ENERGY CONSUMPTION MANAGEMENT APPROACH SABECO is committed to raising the proportion of renewable energy usage by increasing investment in innovative low-carbon technologies that contribute to climate change mitigation and adaptation efforts, as well as promoting optimization of overall resource use for maximum efficiency. This approach extends to raising awareness among suppliers and partners about climate change. Encouraging renewable energy adoption – Promoting alternative power sources and afforestation to support sustainable growth. In this light, the Company’s ESG initiatives are developed around six key areas, namely Public Health, Art and Culture, Community Development, E"}\n{"id": "B-A-d90c5486275afb11", "claim": "Reducing To reach our net zero target, we are committed to equipping our teams with the best tools, training, and guidance available.", "evidence": "Contributing to the UN SDGs 2030 Target 2050 Target Net Zero emission for Scope 1, 2 and 3 by 2050. 50% 42% GHG reduction in Scope 1 and 2 by 2030 (compared to 7% Medium-High (20-40%) Low (<10%) Low-Medium (10%-20%) We commit to reducing water consumption through innovative practices, protecting water quality by adhering to regulatory standards, and engaging with local communities to promote equitable access to clean water. We are committed to fulfilling our long-term Corporate"}\n{"id": "B-A-acd727541fde9dd4", "claim": "This includes guidelines for greenhouse gas (GHG) accounting and energy consumption for reporting sustainability data, which are standardized across all SABECO breweries.", "evidence": "Enviroment Social Governance \\u0017 Climate Change/ \\u0017 Talent Attraction and \\u0017 Corporate Governance Greenhouse Gas Emissions and Energy Consumption 05 Climate Change/Greenhouse Gas Emissions and Energy Consumption 43 SABECO\'s Sustainability Milestones 07 Water Stewardship 49 LEADERSHIP COMMUNICATION"}\n{"id": "B-A-9020f270c9df8a6d", "claim": "Additionally, we have organized specialized training sessions for our brewery teams to assist them in collecting and monitoring environmental data on energy usage, emissions, water use, and waste management.", "evidence": "As a leading brewer in Vietnam, we recognize the critical importance of responsible waste management. We have made great progress in ensuring that our primary packaging materials are recyclable, supporting our efforts to reduce greenhouse gas emissions and minimize environmental impacts. Additionally, we maintain a responsible approach to water conservation across our operations. We also have specific storage areas for waste prior to its disposal. For hazardous waste, we maintain contracts with specialized companies that handle its transportation and treatment. Our 2024 progress & Waste Management Perfomance"}\n{"id": "B-A-26c30e6bf5f633c8", "claim": "In 2024, we have invested in and implemented over 50 solutions and continuous improvement initiatives to save energy, reduce greenhouse gas (GHG) emissions and achieve a green transition in energy use.", "evidence": "This includes guidelines for greenhouse gas (GHG) accounting and energy consumption for reporting sustainability data, which are standardized across all SABECO breweries. Additionally, we have organized specialized training sessions for our brewery teams to assist them in collecting and monitoring environmental data on energy usage, emissions, water use, and waste management. So far, more than 1,400 employees have completed the training. 305-1 47 GRI 3 – Material Topics 2021 Occupational Health and Safety – Management Approach Energy indirect (Scope 2) GHG emissions (CO2e) Greenhouse Gas Emission Performance"}\n{"id": "B-A-8334fc78671baa62", "claim": "GHG Emissions Reduction Projects: These initiatives involve transitioning from diesel-powered forklifts to eco-friendly liquefied gas forklifts, phasing out the fossil-fuel boiler and transitioning to use steam from biomass boiler.", "evidence": "GHG Scope 1 32,664 7.55 6.50 28,818 Using Renewable Energy from Biomass Boiler Projects GHG Emission Reduction Projects 26,715 100% SABECO’s breweries have adopted carbon dioxide (CO2) recovery practices in their operations. KEY PROJECTS equivalent in 2023 and 90,791 tonnes CO2 equivalent in 2022). This was a reduction of 21.92% against 2022 and 9.3% compared to 2023 baseline due to the increase in adoption of renewable electricity, electricity saving measures, and transitioning to biomass boiler. RENEWABLE ENERGY PROJECTS Our 2024 Progress"}\n{"id": "B-A-614c0d8b845a2a61", "claim": "These improvements not only optimize energy efficiency but also significantly reduce environmental impact, with an estimated reduction of 7,405 tons of CO2e per year.", "evidence": "Production factory 75% -80% Start This innovation, developed in collaboration with SABECO’s suppliers, not only reduces costs for both parties but also demonstrates a commitment to Scope 3 GHG emissions reduction annually while maintaining product quality. Empty bottles SABECO also applies a circular economy model across the product lifecycle starting from the product design stage to promote resource efficiency and sustainable production. RO filtration system. By collecting and treating this water for reuse in applications such as equipment cleaning, supplying water for production processes that do not require high quality, and cooling, our brewery aims to reduce overall water consumption an"}\n{"id": "B-A-c0eb83692e26551e", "claim": "% Renewable Energy Consumption in 2024 Energy Intensity At SABECO, we consumed electricity, steam and fuel covering Diesel oil (DO), Fuel Oil (FO), Gasoline, Bituminous HCV and Liquefied Petroleum Gas (LPG) in our beer production activities.", "evidence": "Beer business in 2024 Total energy consumption at SABECO’s breweries in 2024 has been presented below: The Intensity of Energy Consumption at SABECO’s breweries Energy consumption in beer production in 2022-2024 The fossil fuel DO, FO, Bituminous HCV is used for two primary purposes: to run the generator during a power outage and a small amount to charge the fire pump. kWh 65,905,462 The amount of electricity EVN used in 2024 was % Electricity Consumption Norm"}\n{"id": "B-A-e7ce64bfce4f9af8", "claim": "Reduction in 2024 compared to 2020 Solar Power systems supply Renewable Energy-Solar power kWh 7,842,763 The amount of solar power used 46.00% 10.63% in 2024 was of the company\'s electricity needs.", "evidence": "The fossil fuel DO, FO, Bituminous HCV is used for two primary purposes: to run the generator during a power outage and a small amount to charge the fire pump. kWh 65,905,462 The amount of electricity EVN used in 2024 was % Electricity Consumption Norm 2023 baseline) Renewable energy by 2030 HIGHLIGHTED PERFORMANCE Saving 5,833,142 KWh 40.54% Renewable energy replacing fossil energy in beer operation. of electricity in beer production in 2024 compared to 2023, which reduce 8.13% of electricity consumption"}\n{"id": "B-A-2cb4c3970e147005", "claim": "KEY PROJECTS equivalent in 2023 and 90,791 tonnes CO2 equivalent in 2022).", "evidence": "We started working on internal carbon footprint accounting since 2019 and have consistently improved the scope and quality of data. In 2024, our total scope 1 and scope 2 is 70,888 tonnes CO2 equivalent (compared to 78,161 tonnes CO2 70,800 78,161 90,791 Total weight of waste disposed to landfill Tonnes 306-5 Waste directed 2,240 2,450 2,688 Total biogenic GHG emissions (CO2e) Tonnes"}\n{"id": "B-A-092c4a4002bbfccc", "claim": "We started working on internal carbon footprint accounting since 2019 and have consistently improved the scope and quality of data.", "evidence": "This includes guidelines for greenhouse gas (GHG) accounting and energy consumption for reporting sustainability data, which are standardized across all SABECO breweries. Additionally, we have organized specialized training sessions for our brewery teams to assist them in collecting and monitoring environmental data on energy usage, emissions, water use, and waste management. So far, more than 1,400 employees have completed the training. With a system of breweries and distribution nationwide, SABECO has always been aware of its crucial duty to ensure the consistency of product quality in order to bring the best products to consumers We continuously update production processes, train staff in"}\n{"id": "B-A-620d44942995de34", "claim": "In 2024, our total scope 1 and scope 2 is 70,888 tonnes CO2 equivalent (compared to 78,161 tonnes CO2", "evidence": "(kg CO2e/hL) Total GHG Emissions (Scope1 + Scope 2) 90,791 9.00 78,161 70,888 8.50 8.00 GHG Scope 2 58,127 49,343 7.50 8.87 44,173 8.47 7.00 70,800 78,161 90,791 Total weight of waste disposed to landfill Tonnes 306-5 Waste directed 2,240 2,450 2,688 Total biogenic GHG emissions (CO2e) Tonnes"}\n{"id": "B-A-a3c23edf1faf30ac", "claim": "VND per year and lowered GHG emissions by 5,170 tonnes CO2e in 2024.", "evidence": "(MJ) of thermal energy, leading to a biogenic GHG emissions of 539 tonnes CO2e per year. GHG Emission Reduction GHG Intensity Biogenic Emission kg CO2e/hL 306-4 Waste diverted Tonnes 1,657 851.4 818.6 Gross direct (Scope 1) GHG emissions (CO2e) 305-1- Direct (Scope 1) GHG emissions Tonnes"}\n{"id": "B-A-ab82f9362f83d1b8", "claim": "GHG Scope 1 32,664 7.55 6.50 28,818 Using Renewable Energy from Biomass Boiler Projects GHG Emission Reduction Projects 26,715 100% SABECO’s breweries have adopted carbon dioxide (CO2) recovery practices in their operations.", "evidence": "26,715 28,818 32,664 Recycling (non-hazardous waste) 306-4 Waste diverted Tonnes 1,185 1,564 2,180.5 Gross location-based indirect (Scope 2) GHG emissions (CO2e) 16,414 15,815 18,105 Total weight of waste incinerated with energy recover 306-5 Waste directed Tonnes 188 263 426 305-4 GHG Emission intensity GHG Emission Intensity kgCO2e/hL 26,715 28,818"}\n{"id": "B-A-96d209fede5751b4", "claim": "This involves capturing the CO2 generated from the fermentation process in the production of alcoholic beverages.", "evidence": "Biodiversity In 2024, SABECO conducted a Materiality Assessment in alignment with the GRI Standards 2021. This process involves identifying and prioritizing environmental, social, and governance Product quality assurance Continuing research on the application of barley malt in the production process to diversify raw material sources and maintain stable production. Upgrading the inner burner of the wort kettle and the operation process of the wort kettle to enhance the efficiency of bitter substance extraction, stabilize the bitterness of the finished beer, and reduce the amount of hop flowers used. ingredients are selected from a vetted list of suppliers to maintain quality."}\n{"id": "B-A-1144742b9b7f3668", "claim": "The recovered CO2 is then used for the brewing process and for carbonating beverages, such as soda and beer, thus contributing to both sustainability and operational efficiency.", "evidence": "We are working hard to minimize our water-related impact by making all our breweries as efficient as possible. In 2024, the total water withdrawal for breweries of SABECO is 2,734.42 ML, representing a saving of 5.7% of water needs for brewing process compared to 2023 baseline. 50 | SABECO 2024 Sustainability Report 49 The remaining breweries (64% of beer production sites) are considered as having Low-Medium as and Low water-stressed risk level. At SABECO, we are dedicated to responsible water stewardship, recognizing water as an essential ingredient in our beer, in our brewing process and community well-being."}\n{"id": "B-A-5a208125e481aea6", "claim": "20,000 40,000 60,000 80,000 100,000 2022 2023 2024 In 2024, SABECO used renewable energy generated from biomass boilers with 306,576,972 Megajoules", "evidence": "2022 2023 2024 82% of SABECO\'s breweries have been certified with ISO 50001 – Energy Management System 200,000 400,000 600,000 800,000 1,000,000 1,200,000 46 | SABECO 2024 Sustainability Report This involves capturing the CO2 generated from the fermentation process in the production of alcoholic beverages. The recovered CO2 is then used for the brewing process and for carbonating beverages, such as soda and beer, thus contributing to both sustainability and operational efficiency. 2022 2023 2024 64% operation sites of SABECO are currently using renewable energy (steams) generated from biomass boilers."}\n{"id": "B-A-609f3b53e2b19d86", "claim": "(MJ) of thermal energy, leading to a biogenic GHG emissions of 539 tonnes CO2e per year.", "evidence": "70,800 78,161 90,791 Total weight of waste disposed to landfill Tonnes 306-5 Waste directed 2,240 2,450 2,688 Total biogenic GHG emissions (CO2e) Tonnes VND per year and lowered GHG emissions by 5,170 tonnes CO2e in 2024. GHG Emission (tCO2e) - Beer production in 2022-2024 Scope 1+2 GHG intensity (Beer Production)"}\n{"id": "B-A-1238a0a584ff5838", "claim": "Water-stress Level 9% [GRI 303] 36% Contributing to the UN SDGs In order to implement our Water Sustainability Program, we have assessed water risks across our breweries since 2023.", "evidence": "2,734.42 ML 2,120.74 ML RECLAIMING AND REUSING WATER [GRI 303-3] WATER INTENSITY reduction in water intensity in 2024 compared to 2023. 7.3% At SABECO, we have implemented water saving initiatives to recover reject water from the brewery\'s These efforts have enhanced both our energy and water efficiency. Currently, 13 of our breweries in system are equipped with solar panels, and we are aiming to expand this initiative across all our sites to reduce our environmental footprint and support sustainable operations. Good governance remains a critical component of our sustainability strategy."}\n{"id": "B-A-d67a051ba3760908", "claim": "At SABECO, we are dedicated to responsible water stewardship, recognizing water as an essential ingredient in our beer, in our brewing process and community well-being.", "evidence": "(Compared to 2023) Contributing to the UN SDGs: 1 Malt + Barley 3.26% At SABECO, customer health and safety are our top priorities. We are dedicated to making high-quality beers with safe, responsibly sourced ingredients, following strict safety standards in our brewing process. We are working hard to minimize our water-related impact by making all our breweries as efficient as possible. In 2024, the total water withdrawal for breweries of SABECO is 2,734.42 ML, representing a saving of 5.7% of water needs for brewing process compared to 2023 baseline. 50 | SABECO 2024 Sustainability Report 49"}\n{"id": "B-A-aa082138012a52e6", "claim": "The policy sets out clear intentions and guidelines to minimize water impact from our production activities, by assessing water quality and quantity, applying international water resource management tools and standards, using new technologies, incorporating water sustainability management into business strategy, decision making and being responsible for water consumption.", "evidence": "Reduce the impacts from the implementation of the Company\'s value chain to align with the growth of the business by covering dimensions of society, environment, and occupational health and safety. Pillars Targets and KPIs The Sustainability Policy sets out clear intentions and guidelines as follows: We are working hard to minimize our water-related impact by making all our breweries as efficient as possible. In 2024, the total water withdrawal for breweries of SABECO is 2,734.42 ML, representing a saving of 5.7% of water needs for brewing process compared to 2023 baseline. 50 | SABECO 2024 Sustainability Report 49"}\n{"id": "B-A-d2c6f46be68a1e5a", "claim": "SABECO has implemented water stewardship practices for the long-term, sustainable use of water for our own operations and local communities.", "evidence": "In the water-scarce regions of Vietnam where our breweries operate, conserving water is essential. We maintain our collaboration with local stakeholders to ensure rigorous adherence to wastewater regulations, manage water resources responsibly, and safeguard local water supplies for future generations. Surface water (ML) Groundwater (ML) Third-party water (ML) 7% Medium-High (20-40%) Low (<10%) Low-Medium (10%-20%) We commit to reducing water consumption through innovative practices, protecting water quality by adhering to regulatory standards, and engaging with local communities to promote equitable access to clean water. We are committed to fulfilling our long-term Corporate"}\n{"id": "B-A-0960a039ef71a24d", "claim": "We maintain our collaboration with local stakeholders to ensure rigorous adherence to wastewater regulations, manage water resources responsibly, and safeguard local water supplies for future generations.", "evidence": "Department, ensuring compliance with regulations regarding quality and safety. Additionally, we conduct risk management in our supply chain, including regular supplier audits to ensure adherence to processes and standards. Adherence to product quality and safety standard RO filtration system. By collecting and treating this water for reuse in applications such as equipment cleaning, supplying water for production processes that do not require high quality, and cooling, our brewery aims to reduce overall water consumption and lower wastewater treatment costs. This approach not only eases the strain on natural water resources but also improves production efficiency, reinforcing the brewery\'s c"}\n{"id": "B-A-44ddca630ab811c6", "claim": "EFFLUENT of wastewater is treated to meet discharge standards in accordance with QCVN 40:2011 (Type A or Type B) before being released into the receiving environment.", "evidence": "All wastewater generated within SABECO’s brewing system is treated either at the brewery\'s wastewater treatment system or connected to the centralized wastewater treatment system of the industrial zone. After treatment, all wastewater meets discharge standards in accordance with QCVN 40:2011 (Type A or Type B) before being released into the receiving environment. Training and Engagement GRI 303 – Water and Effluent 2018 303-1 Understanding our Water-risk 50 Management of water discharge related impacts Effluent 51 303-2 Worker training on occupational healthand safety"}\n{"id": "B-A-c172fbbdfa2158a1", "claim": "Effective water management in 2024 resulted in the reduction, reuse, and recycling of water, compared to the total volume of water withdrawal, as follows:", "evidence": "GRI 303-5] INITIATIVES RELATED TO REDUCTION IN WATER USAGE, WATER REUSE Rainwater Harvesting System in Vietnam Surface Water (river water and rainwater) 2,734.42 ML 2,120.74 ML RECLAIMING AND REUSING WATER [GRI 303-3] WATER INTENSITY reduction in water intensity in 2024 compared to 2023. 7.3% At SABECO, we have implemented water saving initiatives to recover reject water from the brewery\'s"}\n{"id": "B-A-118964dece32e6f9", "claim": "These upgrades have ensured efficient operation and ongoing water savings.", "evidence": "warehouse optimizing operational efficiency. These upgrades played a crucial role in ensuring the health and safety of the workforce, improving job satisfaction, and boosting employee morale. Topics Target Employees Managers, staff directly working with chemical belonging to three groups as defined by law 1 2,734.42 ML 2,120.74 ML RECLAIMING AND REUSING WATER [GRI 303-3] WATER INTENSITY reduction in water intensity in 2024 compared to 2023. 7.3% At SABECO, we have implemented water saving initiatives to recover reject water from the brewery\'s"}\n{"id": "B-A-e966333a6f5eac86", "claim": "SABECO is committed to responsibly managing wastewater throughout the beer production process.", "evidence": "2024 PERFORMANCE of our primary packaging includes ingredient information to inform consumer choice. 100% In addition to upholding stringent quality control throughout our production process, SABECO officially launched the Beer Research and All wastewater generated within SABECO’s brewing system is treated either at the brewery\'s wastewater treatment system or connected to the centralized wastewater treatment system of the industrial zone. After treatment, all wastewater meets discharge standards in accordance with QCVN 40:2011 (Type A or Type B) before being released into the receiving environment."}\n{"id": "B-A-7711740b648d2ed5", "claim": "All wastewater generated within SABECO’s brewing system is treated either at the brewery\'s wastewater treatment system or connected to the centralized wastewater treatment system of the industrial zone.", "evidence": "Treatment System At SABECO, we are commited to sustainability and minimizing our environmental footprint. As a part of our ongoing effort on waste management in sustainability way, we have collected sluges from the wastewater treatment system at all our breweries. Sludge, which consists of organic matter, can be used and transform into a valuable resource. The primary purpose of the harvested rainwater is to support the brewery’s cooling and cleaning processes. The system is expected to reduce the brewery’s water usage by 4% during the six-month rainy season in 2024, representing a substantial cost saving of 148 million VND. SABECO is committed to responsibly managing wastewater throughout t"}\n{"id": "B-A-f8acc414ba8cb7e0", "claim": "After treatment, all wastewater meets discharge standards in accordance with QCVN 40:2011 (Type A or Type B) before being released into the receiving environment.", "evidence": "SABECO by 2030. EFFLUENT of wastewater is treated to meet discharge standards in accordance with QCVN 40:2011 (Type A or Type B) before being released into the receiving environment. Effective water management in 2024 resulted in the reduction, reuse, and recycling of water, compared to the total volume of water withdrawal, as follows: Treatment System At SABECO, we are commited to sustainability and minimizing our environmental footprint. As a part of our ongoing effort on waste management in sustainability way, we have collected sluges from the wastewater treatment system at all our breweries. Sludge, which consists of organic matter, can be used and transform into a valuable resource."}\n{"id": "B-A-0be7f9459a55bcdd", "claim": "Production factory 75% -80% Start This innovation, developed in collaboration with SABECO’s suppliers, not only reduces costs for both parties but also demonstrates a commitment to Scope 3 GHG emissions reduction annually while maintaining product quality.", "evidence": "RO filtration system. By collecting and treating this water for reuse in applications such as equipment cleaning, supplying water for production processes that do not require high quality, and cooling, our brewery aims to reduce overall water consumption and lower wastewater treatment costs. This approach not only eases the strain on natural water resources but also improves production efficiency, reinforcing the brewery\'s commitment to water stewardship. AVERAGE WATER USAGE EFFICIENCY1 IN THE PRODUCTION PROCESS VND per year and lowered GHG emissions by 5,170 tonnes CO2e in 2024. GHG Emission (tCO2e) - Beer production in 2022-2024 Scope 1+2 GHG intensity (Beer Production)"}\n{"id": "B-A-6ae2a97a0238dc0e", "claim": "Empty bottles SABECO also applies a circular economy model across the product lifecycle starting from the product design stage to promote resource efficiency and sustainable production.", "evidence": "TABLE Packaging | and Circular Economy Vietnamese suppliers on sustainable supply chain management, green innovation, ESG integration, circular economy, resource efficiency, waste reduction, etc. SABECO has established comprehensive procurement policies and procedures, including:"}\n{"id": "B-A-fb3afb5c6cc1a3f4", "claim": "For the recovered bottles that cannot be reused and the broken bottles during production, we collaborate with partners to recycle them.", "evidence": "return to The rescue factory Route to market Glass bottle Manufacturing plant 2024 PERFORMANCE Glass bottles 100% returnable glass bottles and reused in 2024. GRI 301-2; GRI 306-1, GRI 306-2, GRI 306-4] A CLOSED-LOOP GLASS BOTTLE RECYCLE MODEL Contributing to the UN SDGs Breweries 100% of all beverage packaging to be made of reusable and, recyclable material by 2040."}\n{"id": "B-A-c9995de774b5f0c7", "claim": "5% Consumer Plastic crates 100% of plastic crates are reused or recycled 10-15% Since 2020 100% Glass bottle Recycling plant of new plastic crates are recycled from used plastic crates.", "evidence": "Landfill Aluminium cans and cardboard boxes RECYCLING AND GLASS BOTTLE MANUFACTURING COMPANY 100% are reused and recycled. 54 return to The rescue factory Route to market Glass bottle Manufacturing plant 2024 PERFORMANCE Glass bottles 100% returnable glass bottles and reused in 2024."}\n{"id": "B-A-e78fd6a2ac84e9e8", "claim": "2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas In compliance with the Environmental Protection Law on Extended", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas CUSTOMER RELATIONSHIP"}\n{"id": "B-A-1266d4f0de45ac00", "claim": "PARTNERSHIP With a goal of achieving 100% of all beverage packaging to be made of reusable, recyclable material by 2040, notably, since 2019, SABECO has implemented improvements in product packaging design, reducing packaging weight, which has led to significant cost and material savings.", "evidence": "Be aware of the compliance with the sustainability policy throughout the Company\'s value chain. of all beverage packaging to be made of reusable and, recyclable material by 2040 Zero Social Fatality Zero Lost Time Injury Frequency Rate (LTIFR) GRI 301-2; GRI 306-1, GRI 306-2, GRI 306-4] A CLOSED-LOOP GLASS BOTTLE RECYCLE MODEL Contributing to the UN SDGs Breweries 100% of all beverage packaging to be made of reusable and, recyclable material by 2040."}\n{"id": "B-A-13936c0a0cf3a830", "claim": "Collectors Aluminum Coil Manufacturer Recycled aluminum products - used in construction and automotive", "evidence": "PARTNERSHIP With a goal of achieving 100% of all beverage packaging to be made of reusable, recyclable material by 2040, notably, since 2019, SABECO has implemented improvements in product packaging design, reducing packaging weight, which has led to significant cost and material savings. Imported Aluminum coil Aluminum can Manufacturers For the recovered bottles that cannot be reused and the broken bottles during production, we collaborate with partners to recycle them. 5% Consumer Plastic crates 100% of plastic crates are reused or recycled 10-15% Since 2020 100% Glass bottle Recycling plant of new plastic crates are recycled from used plastic crates."}\n{"id": "B-A-a43bf7ae6186fc16", "claim": "Reusing By-Products in Brewing Reusing waste from our own and subsidiaries production sites can contribute to more circular and efficient production with lower environmental impacts.", "evidence": "2024, with the disclosed information referenced in the GRI content index table. ir@sabeco.com.vn The sustainability data were collected covering all facilities under SABECO’s operational control, directly or indirectly owned, in which it controls more than 50% of the voting rights. Environmental data cover 11 brewery production sites, while social data cover 11 brewery production sites, 01 liquor production site, 01 engineering company, 01 non-alcohol beverage sites, On the Environmental front, from greenhouse gas emission reduction, water conservation, wastewater management, energy-efficient brewing processes to the deployment of solar panels at our breweries, we are investing in initiative"}\n{"id": "B-A-17119ceb32a98e1d", "claim": "The main by-products from our brewing processes are spent grains and waste yeast.", "evidence": "Prevent: Avoid waste generation by using fewer materials and minimizing resource consumption 1 At SABECO, in 2024, we collect 100% of the spent grains and yeast after the beer production process at all our breweries. These types of waste are used as valuable inputs from farmers. On the Environmental front, from greenhouse gas emission reduction, water conservation, wastewater management, energy-efficient brewing processes to the deployment of solar panels at our breweries, we are investing in initiatives that contribute to the sustainability of our operations and the communities we engaged with. On the Social front, we prioritized the well-being and safety of our workforce while actively eng"}\n{"id": "B-A-ba2f425d441b11ef", "claim": "Prevent: Avoid waste generation by using fewer materials and minimizing resource consumption 1 At SABECO, in 2024, we collect 100% of the spent grains and yeast after the beer production process at all our breweries.", "evidence": "KEY PROJECT MANAGEMENT APPROACH SABECO also applied waste management hierarchy approach, covering six stages on Prevent, Reduce, Reuse, Recycle, Recover and Dispose. Reusing By-Products in Brewing Reusing waste from our own and subsidiaries production sites can contribute to more circular and efficient production with lower environmental impacts. The main by-products from our brewing processes are spent grains and waste yeast. Beer business in 2024 Total energy consumption at SABECO’s breweries in 2024 has been presented below: The Intensity of Energy Consumption at SABECO’s breweries Energy consumption in beer production in 2022-2024"}\n{"id": "B-A-358a74f8e59b5194", "claim": "These types of waste are used as valuable inputs from farmers.", "evidence": "Treatment System At SABECO, we are commited to sustainability and minimizing our environmental footprint. As a part of our ongoing effort on waste management in sustainability way, we have collected sluges from the wastewater treatment system at all our breweries. Sludge, which consists of organic matter, can be used and transform into a valuable resource. In 2024, we have been partnering with Sai Gon Xanh Company Limited and DASACO Company to collect and convert 1,816 tons of sludge into fertilizer The end product is a nutrient-rich organic fertilizer that can be used by farmers and gardeners, promoting sustainable agricultural practices. Recover: Recover waste or unused materials into valu"}\n{"id": "B-A-fc11663b0a73bf96", "claim": "Reduce: Cut down waste along the value chain through smart design and process improvementst 2 In 2024 100% was used as animal fodder Reuse: Effectively use items or materials again to lessen waste and save resources", "evidence": "7% Reduction of water intensity by 2030, compared to 2023 (beverages) 100% Recognize the importance of natural resources and the environment by encouraging the use of resources effectively along with developing innovation, and technologies in order to increase efficiency and reduce any environmental impact. Place importance on and care of the quality and safety of the food chain by serving consumers healthy and products, with responsibility to society and environment. Prevent: Avoid waste generation by using fewer materials and minimizing resource consumption 1 At SABECO, in 2024, we collect 100% of the spent grains and yeast after the beer production process at all our breweries. These type"}\n{"id": "B-A-3f230354163712b6", "claim": "3 Recycle: Collect and process unused materials to create new products 4 Recycling Sludge from Wastewater", "evidence": "For the recovered bottles that cannot be reused and the broken bottles during production, we collaborate with partners to recycle them. 5% Consumer Plastic crates 100% of plastic crates are reused or recycled 10-15% Since 2020 100% Glass bottle Recycling plant of new plastic crates are recycled from used plastic crates. In 2024, we have been partnering with Sai Gon Xanh Company Limited and DASACO Company to collect and convert 1,816 tons of sludge into fertilizer The end product is a nutrient-rich organic fertilizer that can be used by farmers and gardeners, promoting sustainable agricultural practices. Recover: Recover waste or unused materials into valuable energy sources such as electrici"}\n{"id": "B-A-e5ddd44bad39be6f", "claim": "Treatment System At SABECO, we are commited to sustainability and minimizing our environmental footprint.", "evidence": "These efforts have enhanced both our energy and water efficiency. Currently, 13 of our breweries in system are equipped with solar panels, and we are aiming to expand this initiative across all our sites to reduce our environmental footprint and support sustainable operations. Good governance remains a critical component of our sustainability strategy. At SABECO, we acknowledge that climate change is a critical global challenge affecting ecosystems and communities. We are committed to minimizing our environmental impact and fostering a sustainable future. ENABLING THE ORGANIZATION PERFORMANCE Our goal is to achieve net-zero Greenhouse gases (GHG) emissions by 2050."}\n{"id": "B-A-a42ce0c5d7f8bcf6", "claim": "As a part of our ongoing effort on waste management in sustainability way, we have collected sluges from the wastewater treatment system at all our breweries.", "evidence": "All wastewater generated within SABECO’s brewing system is treated either at the brewery\'s wastewater treatment system or connected to the centralized wastewater treatment system of the industrial zone. After treatment, all wastewater meets discharge standards in accordance with QCVN 40:2011 (Type A or Type B) before being released into the receiving environment. Prevent: Avoid waste generation by using fewer materials and minimizing resource consumption 1 At SABECO, in 2024, we collect 100% of the spent grains and yeast after the beer production process at all our breweries. These types of waste are used as valuable inputs from farmers."}\n{"id": "B-A-5778babafb14bfd2", "claim": "In 2024, we have been partnering with Sai Gon Xanh Company Limited and DASACO Company to collect and convert 1,816 tons of sludge into fertilizer The end product is a nutrient-rich organic fertilizer that can be used by farmers and gardeners, promoting sustainable agricultural practices.", "evidence": "Sai Gon Beer Trading Company Limited (Satraco) Sai Gon Beer Packaging Joint Stock Company Subsidiaries & Associates 30 | SABECO 2024 Sustainability Report Treatment System At SABECO, we are commited to sustainability and minimizing our environmental footprint. As a part of our ongoing effort on waste management in sustainability way, we have collected sluges from the wastewater treatment system at all our breweries. Sludge, which consists of organic matter, can be used and transform into a valuable resource."}\n{"id": "B-A-277af55c25734624", "claim": "Recover: Recover waste or unused materials into valuable energy sources such as electricity", "evidence": "3-3 Management of material topics 44 Waste directed to disposal 306-5 Appendix B – ESG Data 104 Energy consumption within the organization GRI 302 – Energy 2016 45 Materials used by weight or volume Prevent: Avoid waste generation by using fewer materials and minimizing resource consumption 1 At SABECO, in 2024, we collect 100% of the spent grains and yeast after the beer production process at all our breweries. These types of waste are used as valuable inputs from farmers."}\n{"id": "B-A-9f1787fb6ee70f17", "claim": "6 All factories are equipped with silos for storing spent grain and designated areas for packaging waste.", "evidence": "All of SABECO\'s factories are certified with ISO 9001 (Quality Management System), ISO 14001 (Environmental Management System), ISO 22000 (Food Safety Management System), and HACCP (Hazard Analysis and Critical Control Points). Additionally, all of SABECO\'s factories have laboratories accredited to ISO 17025 standards, ensuring the quality of beer produced across the entire system. PARTNERSHIP With a goal of achieving 100% of all beverage packaging to be made of reusable, recyclable material by 2040, notably, since 2019, SABECO has implemented improvements in product packaging design, reducing packaging weight, which has led to significant cost and material savings. Imported Aluminum coil Al"}\n{"id": "B-A-ac6c3aa6128e5c9c", "claim": "Household waste is collected in covered bins to ensure cleanliness.", "evidence": "Treatment System At SABECO, we are commited to sustainability and minimizing our environmental footprint. As a part of our ongoing effort on waste management in sustainability way, we have collected sluges from the wastewater treatment system at all our breweries. Sludge, which consists of organic matter, can be used and transform into a valuable resource. SUSTAIN- ABILITY REPORT TOPIC DISCLOSURE NUMBER Waste generation and significant waste-related impacts GRI 306 – Effluent and Waste 2020 Waste Management"}\n{"id": "B-A-b4fda6a0518a1e6d", "claim": "We also have specific storage areas for waste prior to its disposal.", "evidence": "OVERVIEW 2024 SUSTAINABILITY HIGHLIGHTS GOVERNANCE ENVIRONMENT 40.54% Renewable energy consumption within breweries production. 64.22% Waste diverted from disposal. Launched the SABECO Beer Research and Development Center. 5 Dispose: Manage unwanted materials responsibly to ensure proper elimination. 6 All factories are equipped with silos for storing spent grain and designated areas for packaging waste. Household waste is collected in covered bins to ensure cleanliness."}\n{"id": "B-A-025afb997ee79061", "claim": "For hazardous waste, we maintain contracts with specialized companies that handle its transportation and treatment.", "evidence": "11 trading companies and 01 transportation company. (+84) 28 3829 4081 Additional information regarding SABECO environmental and social can be found in the Countries Supply Chain Company Regional & Nationalwide trading companies Transportation Company As of December 31, 2024. In 2016, SABECO was officially listed on"}\n{"id": "B-A-d3319f84519cf22f", "claim": "Our 2024 progress & Waste Management Perfomance", "evidence": "NUMBER Management of significant waste- related impacts 306-2 Waste Management 57-58 ENVIRONMENT DIMENSION Waste generated 306-3 Appendix B – ESG Data As a leading brewer in Vietnam, we recognize the critical importance of responsible waste management. We have made great progress in ensuring that our primary packaging materials are recyclable, supporting our efforts to reduce greenhouse gas emissions and minimize environmental impacts. Additionally, we maintain a responsible approach to water conservation across our operations."}\n{"id": "B-A-655f585c92583a0a", "claim": "[GRI 403] SABECO’s “Corporate Occupational Health and Safety Policy” provides guidance for best practices on healthy and safe working environment, ensuring alignment with high OHS standards for protecting employees’ health and safety.", "evidence": "Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations. SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement. SABECO has established an OHS management system to ensure safety and hygiene standards are integrated into daily operations. SOCIAL DIMENSION OCCUPATIONAL HEALTH AND SAFETY [GRI 403] TALENT ATTRACTION AND DEVELOPMENT [GRI 2-7, GRI 2-8, GRI 405-1] CUSTOMER RELATIONSHIP MANAGEMENT"}\n{"id": "B-A-f473163a819e2df3", "claim": "The policy commitment is reinforced through strict adherence to Vietnamese laws and international standards to maintain a safe and healthy workplace;", "evidence": "[GRI 403] SABECO’s “Corporate Occupational Health and Safety Policy” provides guidance for best practices on healthy and safe working environment, ensuring alignment with high OHS standards for protecting employees’ health and safety. AND HEALTHY WORK ENVIRONMENT A safe, ethical, and inclusive workplace is fundamental to SABECO’s success. The Company upholds a Human Rights Policy in compliance with Vietnamese regulations and global frameworks such as the UN Universal Declaration of Human Rights and the ILO Declaration on Fundamental Principles and"}\n{"id": "B-A-8806ed448de5f021", "claim": "a Whistleblowing Policy that encourages employees to report any hazards, misconduct, or safety concerns, with guarantees of protection against discrimination, retribution, or harassment;", "evidence": "Policy that encourages transparency by providing employees with a mechanism to report concerns without fear of detriment, retribution or harassment. Other key Human resource regulations cover recruitment, compensation, training, performance evaluation, and workplace democracy. Total full-time employees by age group Parental leave in 2024 In all our facilities, safety signage and excerpts from the Internal Labor Regulations are prominently displayed in visible and accessible areas to guide employees in identifying hazards, assessing risks, and preventing work-related accidents, injuries, and illnesses. A culture of open communication is encouraged, supported by whistleblowing mechanisms and r"}\n{"id": "B-A-95627810269fbc01", "claim": "Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations.", "evidence": "[GRI 403] SABECO’s “Corporate Occupational Health and Safety Policy” provides guidance for best practices on healthy and safe working environment, ensuring alignment with high OHS standards for protecting employees’ health and safety. – 2024 3-3 Management of material topics 61 47 305-2 Occupational health and safety management system Occupational Health and Safety Management System and Services"}\n{"id": "B-A-f58a03a987d1ffa5", "claim": "SABECO has established an OHS management system to ensure safety and hygiene standards are integrated into daily operations.", "evidence": "In 2024, SABECO established a Council of Labor Safety and Hygiene comprising ten members, responsible for organizing, implementing and supervising activities to comply with the Vietnamese laws and regulations on occupational safety and hygiene. A robust set of safety and hygiene principles incorporated as a separate chapter in the Company’s The Company prioritizes aligning operations with its sustainability framework to deliver value across the value chain. To strengthen practices and ensure a unified approach, sustainable development has been integrated into its corporate strategy and risk management processes. for overseeing the corporation’s impacts and sustainability agenda."}\n{"id": "B-A-0f0088b548bc5e73", "claim": "This system includes: Safety regulations and internal controls: Employee health and insurance: Safety governance: Key elements of SABECO’s OHS management approach include:", "evidence": "Management Approach 3-3 53, 57 Workers covered by an occupation health and safety management system Occupational health and safety Management System and Services 403-8 61-62 Performance in 2024 – Appendix B This includes reviewing the quarterly report on internal controls and enterprise risk management to help safeguard the assets of the Company, while ensuring compliance with legal and regulatory requirements. 32 | SABECO 2024 Sustainability Report 31"}\n{"id": "B-A-13e64c0f3d64fd43", "claim": "In 2024, SABECO established a Council of Labor Safety and Hygiene comprising ten members, responsible for organizing, implementing and supervising activities to comply with the Vietnamese laws and regulations on occupational safety and hygiene.", "evidence": "Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations. SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement. SABECO has established an OHS management system to ensure safety and hygiene standards are integrated into daily operations. Occupational health and safety training in 2024 Regular training Communication channels Continuous improvement and updates All employees, including those in office departments and production facilities, undergo period"}\n{"id": "B-A-163f63751564b242", "claim": "A robust set of safety and hygiene principles incorporated as a separate chapter in the Company’s", "evidence": "Occupational health and safety training in 2024 Regular training Communication channels Continuous improvement and updates All employees, including those in office departments and production facilities, undergo periodic training on occupational safety, hygiene, and food safety as required by law. AND HEALTHY WORK ENVIRONMENT A safe, ethical, and inclusive workplace is fundamental to SABECO’s success. The Company upholds a Human Rights Policy in compliance with Vietnamese regulations and global frameworks such as the UN Universal Declaration of Human Rights and the ILO Declaration on Fundamental Principles and"}\n{"id": "B-A-e4f138458e62c8bd", "claim": "Policy framework: Management system: Worker engagement: Policies on OHS, human rights, and whistleblowing that provide clear guidance for maintaining a safe, healthy, and ethical work environment.", "evidence": "Division Operations Division SABECO has established a comprehensive set of policies and regulations to guide workforce management. Beyond legal compliance, SABECO has developed a Business Ethics and Code of Conduct that defines ethical expectations for the Board of Directors, the executives, and all employees, a Human Rights Policy that upholds fundamental rights and freedoms in the workplace, and a Whistleblowing AND HEALTHY WORK ENVIRONMENT A safe, ethical, and inclusive workplace is fundamental to SABECO’s success. The Company upholds a Human Rights Policy in compliance with Vietnamese regulations and global frameworks such as the UN Universal Declaration of Human Rights and the ILO Decla"}\n{"id": "B-A-fe720cb68a3ef505", "claim": "Active participation of employees in training programs and feedback mechanisms to foster a strong safety culture.", "evidence": "In all our facilities, safety signage and excerpts from the Internal Labor Regulations are prominently displayed in visible and accessible areas to guide employees in identifying hazards, assessing risks, and preventing work-related accidents, injuries, and illnesses. A culture of open communication is encouraged, supported by whistleblowing mechanisms and regular safety meetings where employees can provide feedback, report hazards, and propose improvements. Training programs are regularly updated to ensure alignment with the latest regulations and evolving work processes. SABECO’s Onboarding Program for Newcomers helps new employees integrate into the company by familiarizing them with work"}\n{"id": "B-A-871b7062325afea6", "claim": "Personal Protective Equipment (PPE): Fire prevention: Implementation of fire prevention systems and regular drills to enhance workplace safety.", "evidence": "At the same time, the security and order and fire prevention of the SABECO office were also significantly improved, equipped with a high-end CCTV system, security doors, and an officially approved fire prevention and fighting system. In addition, substantial investments were made to enhance the SATRACO Provision of necessary safety gear and strict enforcement of PPE usage in all high-risk work environments. 2024 PERFORMANCE SABECO continuously monitors and evaluates OHS performance to ensure a safe and incident-free workplace. In 2024, SABECO recorded Zero work-related injuries for employees, reflecting SABECO’s strong commitment to workplace safety and accident prevention."}\n{"id": "B-A-35544201a3ed4432", "claim": "Provision of necessary safety gear and strict enforcement of PPE usage in all high-risk work environments.", "evidence": "SABECO has established regulations for regular evaluation and quality control of suppliers for each type of materials and goods. Accordingly, supplier evaluations are conducted periodically, including performance reviews, supply history assessments, and site visits if deemed necessary to ensure compliance with food safety, environmental, social and regulatory standards. SABECO upholds the highest standards for product quality by continuously improving procurement processes and production efficiency. In all our facilities, safety signage and excerpts from the Internal Labor Regulations are prominently displayed in visible and accessible areas to guide employees in identifying hazards, assessi"}\n{"id": "B-A-970b28712376e748", "claim": "In 2024, SABECO recorded Zero work-related injuries for employees, reflecting SABECO’s strong commitment to workplace safety and accident prevention.", "evidence": "Number of recordable work-related injuries Case 0 0 0 405-1 Diversity of governance bodies and employees Others Employees 1 1 1 403-9 Work-related injuries Singapore Employees 9 7 8 OCCUPATIONAL HEALTH AND SAFETY INDICATOR 405-1 Diversity of governance bodies and employees Malaysia 2 Employees 403-9 Work-related injuries"}\n{"id": "B-A-e3414c2ecff9bd6c", "claim": "Notably, in 2024, SABECO expanded our commitment to occupational health and safety by achieving ISO 45001:2018 certification for occupational health and safety management systems for the brewery in Lam Dong Province.", "evidence": "Management Approach 3-3 53, 57 Workers covered by an occupation health and safety management system Occupational health and safety Management System and Services 403-8 61-62 Performance in 2024 – Appendix B – 2024 3-3 Management of material topics 61 47 305-2 Occupational health and safety management system Occupational Health and Safety Management System and Services"}\n{"id": "B-A-8fafa9144800342f", "claim": "Occupational health and safety training in 2024 Regular training Communication channels Continuous improvement and updates All employees, including those in office departments and production facilities, undergo periodic training on occupational safety, hygiene, and food safety as required by law.", "evidence": "Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations. SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement. SABECO has established an OHS management system to ensure safety and hygiene standards are integrated into daily operations. TABLE | Managers, staff directly working with chemical belonging to three groups as defined by law1 | 299 | Fire Protection and Rescue | Fire and Rescue Team | 583 | 24 Occupational Safety and Hygiene | All employees,"}\n{"id": "B-A-de6f2b925713ca6b", "claim": "In all our facilities, safety signage and excerpts from the Internal Labor Regulations are prominently displayed in visible and accessible areas to guide employees in identifying hazards, assessing risks, and preventing work-related accidents, injuries, and illnesses.", "evidence": "The policy commitment is reinforced through strict adherence to Vietnamese laws and international standards to maintain a safe and healthy workplace; a Whistleblowing Policy that encourages employees to report any hazards, misconduct, or safety concerns, with guarantees of protection against discrimination, retribution, or harassment; and continuous efforts to identify, assess, and mitigate workplace hazards to prevent accidents, injuries, and occupational illnesses. To promote transparency, all company policies and regulations are accessible digitally through the Company’s internal system. SABECO strictly forbids forced, prison, and bonded labor. Employees have the right to resign in compli"}\n{"id": "B-A-bf33cd4e99966708", "claim": "A culture of open communication is encouraged, supported by whistleblowing mechanisms and regular safety meetings where employees can provide feedback, report hazards, and propose improvements.", "evidence": "Active participation of employees in training programs and feedback mechanisms to foster a strong safety culture. Personal Protective Equipment (PPE): Fire prevention: Implementation of fire prevention systems and regular drills to enhance workplace safety. In 2024 At SABECO, we recognize that constructive feedback and open dialogue are essential to enhancing employee performance and engagement. Instead of burdening employees with excessive meetings, we focus on cultivating an environment that encourages proactive feedback and meaningful conversations, ensuring clear and timely communication. Our commitment is to create a workplace where every employee feels heard, valued, and empowered to g"}\n{"id": "B-A-690e497357c3108f", "claim": "Occupational health and safety training in 2024 Number of Participants Average Duration(hours)", "evidence": "Occupational health and safety training in 2024 Regular training Communication channels Continuous improvement and updates All employees, including those in office departments and production facilities, undergo periodic training on occupational safety, hygiene, and food safety as required by law. 63 Training and Engagement 403-5 51 303-3 Water Balance 2024 Water Withdrawal Training and Engagement 2024 Key Projects (Occupational Health and Safety)"}\n{"id": "B-A-4de5c9d8d694d3fd", "claim": "These upgrades played a crucial role in ensuring the health and safety of the workforce, improving job satisfaction, and boosting employee morale.", "evidence": "Our occupational health and safety (OHS) strategy prioritizes compliance with Vietnamese laws and international best practices, proactive risk management, and the integration of health and safety into daily operations. SABECO is committed to ensuring the well-being of all employees and fostering a culture of safety, prevention, and continuous improvement. SABECO has established an OHS management system to ensure safety and hygiene standards are integrated into daily operations. TABLE Pillar ENVIRONMENT | Our Commitment Fully complying with environmental protection laws and actively addressing climate change by reducing greenhouse gas emissions | Our Strategy Continuously seeking innovative s"}\n{"id": "B-A-b9826948bcb8c4f4", "claim": "Chemical Safety 299 8 Fire Protection and Rescue Fire and Rescue Team 583 24 Occupational Safety and Hygiene 4,675 8 All employees, belonging to six groups as defined by law 2", "evidence": "TABLE | Managers, staff directly working with chemical belonging to three groups as defined by law1 | 299 | Fire Protection and Rescue | Fire and Rescue Team | 583 | 24 Occupational Safety and Hygiene | All employees, belonging to six groups as defined by law2 | 4,675 | 8 Radiation Safety | Managers, staff directly managing & operating equipment with radioactive sources | 68 | 24 Food Safety | Production employees | 517 | 3 First Aid | First aid team | 970 | 4 Forklift Operation Safety Training - Beer Grade Lifting | Forklift Drivers | 122 | 2 Training on HSE Procedures, Regulations & Safety Supervision in Warehouse Operation | Warehouse staff | 187 | 1.5 Training to update regulations on fo"}\n{"id": "B-A-769b32b754fef9ed", "claim": "Radiation Safety 68 24 Managers, staff directly managing & operating equipment with radioactive sources Food Safety 517 3 Production employees First Aid", "evidence": "TABLE | Managers, staff directly working with chemical belonging to three groups as defined by law1 | 299 | Fire Protection and Rescue | Fire and Rescue Team | 583 | 24 Occupational Safety and Hygiene | All employees, belonging to six groups as defined by law2 | 4,675 | 8 Radiation Safety | Managers, staff directly managing & operating equipment with radioactive sources | 68 | 24 Food Safety | Production employees | 517 | 3 First Aid | First aid team | 970 | 4 Forklift Operation Safety Training - Beer Grade Lifting | Forklift Drivers | 122 | 2 Training on HSE Procedures, Regulations & Safety Supervision in Warehouse Operation | Warehouse staff | 187 | 1.5 Training to update regulations on fo"}\n{"id": "B-A-8c41376818d37bda", "claim": "970 4 First aid team Forklift Operation Safety Training - Beer Grade Lifting 122 2 Forklift Drivers Training on HSE Procedures, Regulations & Safety Supervision in Warehouse Operation", "evidence": "TABLE | Managers, staff directly working with chemical belonging to three groups as defined by law1 | 299 | Fire Protection and Rescue | Fire and Rescue Team | 583 | 24 Occupational Safety and Hygiene | All employees, belonging to six groups as defined by law2 | 4,675 | 8 Radiation Safety | Managers, staff directly managing & operating equipment with radioactive sources | 68 | 24 Food Safety | Production employees | 517 | 3 First Aid | First aid team | 970 | 4 Forklift Operation Safety Training - Beer Grade Lifting | Forklift Drivers | 122 | 2 Training on HSE Procedures, Regulations & Safety Supervision in Warehouse Operation | Warehouse staff | 187 | 1.5 Training to update regulations on fo"}\n{"id": "B-A-7d153ed516dc4851", "claim": "Digital Transformation and Technology 5 By investing in our people and strengthening our organizational capabilities, we aim to drive sustainable growth and position", "evidence": "As we step into 2025 and beyond, SABECO is committed to building a resilient, engaged, and future-ready workforce by prioritizing employee well-being, talent development, diversity, and digital transformation. In 2024, SABECO employees actively participated in key human capital development programs organized by ThaiBev. THAIBEV TRANSFORMATION PROGRAM The program was designed to accelerate leadership growth through high-impact transformation projects and mentorship from senior executives. Our approach is driven by our commitment to create value not just for our shareholders, but also for the communities we serve, the environment we impact, and the people who trust our brand. On behalf of the "}\n{"id": "B-A-ffb3ce1cc65e7b79", "claim": "2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas CUSTOMER RELATIONSHIP", "evidence": "1. 3 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas INPUTS – RESOURCE WE USE ABOUT SABECO’S VALUE CHAIN 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY"}\n{"id": "B-A-824a27c58f38d9de", "claim": "2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY", "evidence": "1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas SUSTAINABILITY, TARGETS AND KPIs [GRI 2-24; 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas SUPPLY CHAIN MANAGEMENT [GRI 308; GRI 414]"}\n{"id": "B-A-d1dfcea1296dd8f9", "claim": "In line with our commitment to sustainability, SABECO’s approach to community development includes:", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT Positive Impacts “63 REUNION – 1 TET TOGETHER”"}\n{"id": "B-A-6968bb293a636f40", "claim": "This initiative contributed to a safer and more sustainable living environment in rural areas, reflecting our dedication to both Environmental Sustainability and Community Development.", "evidence": "SOCIAL Striving to create jobs nationwide, improve living conditions, and contribute to the creation of thriving sustainable communities through 6 key areas: public health, art and culture, community development, education, sports, and environmental sustainability. SUSTAINABILITY STRATEGY Community Development: fostering local employment, improving quality of life for community prosperity. 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY"}\n{"id": "B-A-c79edf0d1f6a20bc", "claim": "Positive Impacts Improving quality of life – Providing safer, well-lit environments for communities in rural and border areas Enhancing socio-economic infrastructure – Strengthening connectivity and mobility, fostering local development", "evidence": "SOCIAL Striving to create jobs nationwide, improve living conditions, and contribute to the creation of thriving sustainable communities through 6 key areas: public health, art and culture, community development, education, sports, and environmental sustainability. SUSTAINABILITY STRATEGY Community Development: fostering local employment, improving quality of life for community prosperity. Promoting healthy lifestyles Enhancing SABECO’s community presence Providing resources for youth THE ‘FOSTERING SPORTS’ PROJECT (2022 – 2024) engagement, social cohesion, and positive social impact Strengthening brand engagement at the local level"}\n{"id": "B-A-5a5f9865215a9c76", "claim": "Encouraging renewable energy adoption – Promoting alternative power sources and afforestation to support sustainable growth.", "evidence": "\\u0017 Committed to sustainable practices by setting clear targets to reduce emissions, energy consumption, water withdrawal, and waste by 2030. \\u0017 Became a member of Packaging Recycling Organization Vietnam to promote packaging management and support the nation\'s green growth. Communist Youth Union to implement social and community initiatives that foster positive values and support sustainable economic, social, and community development. Solar Project with a capacity of 9 MWp at 9 breweries, initiating the use of renewable energy in the beer production process. 50001:2018 for the Energy Management System."}\n{"id": "B-A-5f934c8440b7d3e2", "claim": "2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT Positive Impacts “63 REUNION – 1 TET TOGETHER”", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas COMMUNITY DEVELOPMENT [GRI 413] ENVIRONMENTAL SUSTAINABILITY 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas CUSTOMER RELATIONSHIP"}\n{"id": "B-A-34c0a6685ce9bdec", "claim": "This is why SABECO goes beyond sponsoring sports events - we are gradually shaping a sustainable sports ecosystem where: Every individual has the opportunity to train and participate, young talents receive the support they need to grow,", "evidence": "\\u0017 Focus on the development of young talents through youth empowerment programs and sponsorships for Vietnamese young entrepreneurs and sports talents, and other projects that contribute to national goals \\u0017 Projects of sustainable development \\u0017 SABECO was recognized as a strategic partner who plays a major role in community sustainable development projects This initiative supports Sports while also contributing to Community Development and Public Health. SUPPORTING NATIONAL MARATHON EVENTS SABECO also actively supports major sports events that promote fitness and well-being on a national scale such as the Tien Phong Marathon & and VnExpress Marathon: the marathon has aimed These marathons are"}\n{"id": "B-A-0a68da418b7bffcd", "claim": "Encouraging physical activity and well-being through sports SABECO has built 30 community sports fields across multiple provinces, providing safe and accessible training spaces for over 50,000 young people.", "evidence": "1,280 + Encouraging active lifestyles: Providing accessible spaces for sports and recreation to improve physical and mental well-being. Community sports playgrounds upgraded across 30 provinces Strengthening community bonds: Creating opportunities for social interaction & teamwork through sports. Trading Companies and 4 breweries) Provincial government authority representatives engaged Investing in programs and activities that identify, nurture, and support Vietnam’s emerging sports talent. Developing accessible sports facilities in rural areas to promote physical activity and improve community well-being."}\n{"id": "B-A-96a190b3e2d5caa9", "claim": "(Compared to 2023) Contributing to the UN SDGs: 1 Malt + Barley 3.26% At SABECO, customer health and safety are our top priorities.", "evidence": "MANAGEMENT [GRI 418] PROTECTING CUSTOMER PRIVACY At SABECO, customer privacy and data security are top priorities. Since 2020, we have enforced a strict Information Security and Confidentiality Policy, which applies not only to our company but also to our subsidiaries and business partners. Contributing to the UN SDGs 2030 Target 2050 Target Net Zero emission for Scope 1, 2 and 3 by 2050. 50% 42% GHG reduction in Scope 1 and 2 by 2030 (compared to"}\n{"id": "B-A-7cc5f226fe2b0a60", "claim": "We are dedicated to making high-quality beers with safe, responsibly sourced ingredients, following strict safety standards in our brewing process.", "evidence": "Normung (DIN). We ensure high-quality ingredients- such as barley malt, hops, and packaging materials- are sourced directly from approved suppliers, guaranteeing food safety and hygiene. The remaining breweries (64% of beer production sites) are considered as having Low-Medium as and Low water-stressed risk level. At SABECO, we are dedicated to responsible water stewardship, recognizing water as an essential ingredient in our beer, in our brewing process and community well-being."}\n{"id": "B-A-baad185aa0443d1b", "claim": "Quality control of raw materials input at SABECO [GRI 301] Material management performance SABECO is committed to providing products that are safe for consumer health.", "evidence": "All input materials require a “Qualified” certification from the Quality Control Department before use. Supplier Development POLICIES / STANDARDS / With a system of breweries and distribution nationwide, SABECO has always been aware of its crucial duty to ensure the consistency of product quality in order to bring the best products to consumers We continuously update production processes, train staff in quality control, and evaluate employee effectiveness to enhance our quality management system. We collaborate closely with suppliers to ensure a stable supply of quality raw materials."}\n{"id": "B-A-7946a6667e844a7e", "claim": "The decrease in packaging material waste is attributed to the improved awareness of employees through the standardization of equipment maintenance processes, thereby optimizing production efficiency and conserving materials.", "evidence": "Material used in the company’s operation [GRI 301-1] The total amount used in beer production and packaging in 2024 are below: Material Categories # Materials Increase/Decrease (%) RO filtration system. By collecting and treating this water for reuse in applications such as equipment cleaning, supplying water for production processes that do not require high quality, and cooling, our brewery aims to reduce overall water consumption and lower wastewater treatment costs. This approach not only eases the strain on natural water resources but also improves production efficiency, reinforcing the brewery\'s commitment to water stewardship. AVERAGE WATER USAGE EFFICIENCY1 IN THE PRODUCTION PROCESS"}\n{"id": "B-A-70d7dea0f6228b4b", "claim": "2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas Initiatives for saving and efficient use of materials", "evidence": "1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas CUSTOMER RELATIONSHIP 1. 3 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas INPUTS – RESOURCE WE USE ABOUT SABECO’S VALUE CHAIN"}\n{"id": "B-A-fd19611ccac5e876", "claim": "We ensure high-quality ingredients- such as barley malt, hops, and packaging materials- are sourced directly from approved suppliers, guaranteeing food safety and hygiene.", "evidence": "Approved suppliers undergo a rigorous evaluation before inclusion into our database. Essential ingredients like barley malt, hops, cans, and bottles are sourced from leading and reputable suppliers in the market to maintain quality and safety. Auxiliary ingredients that do not impact beer quality and safety are procured from an approved supplier list. (Compared to 2023) Contributing to the UN SDGs: 1 Malt + Barley 3.26% At SABECO, customer health and safety are our top priorities. We are dedicated to making high-quality beers with safe, responsibly sourced ingredients, following strict safety standards in our brewing process."}\n{"id": "B-A-4279055130894f25", "claim": "Department, ensuring compliance with regulations regarding quality and safety.", "evidence": "SABECO has established regulations for regular evaluation and quality control of suppliers for each type of materials and goods. Accordingly, supplier evaluations are conducted periodically, including performance reviews, supply history assessments, and site visits if deemed necessary to ensure compliance with food safety, environmental, social and regulatory standards. SABECO upholds the highest standards for product quality by continuously improving procurement processes and production efficiency. Supplier selection is based on a robust and systematic supplier qualification in place which includes both economic and ESG aspects, ensuring compliance with legal and confidentiality requirement"}\n{"id": "B-A-37864c0d15463b0f", "claim": "All of SABECO\'s factories are certified with ISO 9001 (Quality Management System), ISO 14001 (Environmental Management System), ISO 22000 (Food Safety Management System), and HACCP (Hazard Analysis and Critical Control Points).", "evidence": "Management System HACCP Hazard Analysis and Practice 100% BRCS British Retail Consortium Global Standar - Food Safety FSSC 2200 Food Safety System Certification ISO 9001 ISO 14001 Environment"}\n{"id": "B-A-9c86a2f22e4c49a0", "claim": "Practice 100% BRCS British Retail Consortium Global Standar - Food Safety FSSC 2200 Food Safety System Certification ISO 9001 ISO 14001 Environment", "evidence": "TABLE ISO 14001 certification | TABLE ISO 22000:2018 certification and ot | her food safety-related certifications (HACCP, GMP, FSSC 22000, BRC)"}\n{"id": "B-A-f64cba3e34c0c9be", "claim": "All SABECO’sproducts Are meet food safety requirements in accordance with the National Technical Regulation on alcoholic beverage (QCVN-6-3:2010/BYT).", "evidence": "\\u0017 Developed the Environmental Management System in accordance with ISO 14001 standards. 2013 – 2015 \\u0017 Listed on the Ho Chi Minh Stock Exchange (HOSE). – Alcohol – Beverage Corporation (SABECO) was transformed into a joint stock company. Occupational health and safety training in 2024 Regular training Communication channels Continuous improvement and updates All employees, including those in office departments and production facilities, undergo periodic training on occupational safety, hygiene, and food safety as required by law."}\n{"id": "B-A-5cbab071d37e6019", "claim": "100% Incidents of non-compliance with regulations and/or voluntary codes concerning the health and safety impacts of products and services.", "evidence": "Management of material topics 83 3-3 71 404-3 GRI 416 – Customer Health and Safety 2016 Incident of non-compliance concerning the health and safety impacts of products and services 417-1 88 Appendix B - ESG Data Table 107 405-2 GRI 417 – Marketing and Labelling 2016 Incidents of non-compliance concerning product and service information and labelling"}\n{"id": "B-A-29f62dee6cf4cbc9", "claim": "2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas GOVERNANCE DIMENSION The Board’s roles, responsibilities, and accountabilities are defined in the publicly accessible Charter,", "evidence": "1 3. 4 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas SUSTAINABILITY GOVERNANCE SABECO’s corporate governance is vital to its sustainability efforts. 1 3 4. 2 Sustainability at SABECO About this sustainability report Sustainability strategy at SABECO Sustainability material areas SOCIAL DIMENSION POLICY COMMITMENT OCCUPATIONAL HEALTH AND SAFETY"}\n{"id": "B-A-e8df7267ad166aab", "claim": "Directors (VIOD) \\u0017 Chairman of Board of Directors Mr Koh Poh Tiong \\u0017 Audit Committee Master Program, Vietnam Institute of \\u0017 Chairman of Sustainability and Risk Management Committee.", "evidence": "Mr Nguyen Tien Vy Directors (VIOD) \\u0017 Member of Nomination and Remuneration Committee \\u0017 Remuneration for Board Program, Vietnam Institute of \\u0017 Chairman of Land Committee Directors (VIOD) \\u0017 Independent Director Mr Pramoad Phornprapha \\u0017 Director Certificate Program (DCP3), Vietnam Institute of Mrs Tran Kim Nga Directors (VIOD) \\u0017 Chairman of Audit Committee"}\n{"id": "B-A-4c224739911450c9", "claim": "Ms Ngo Minh Chau \\u0017 Member of Sustainability and Risk Management Committee Directors (VIOD) \\u0017 Member of Land Committee Person in charge of corporate governance", "evidence": "Ms Ngo Minh Chau \\u0017 Member of Sustainability and Risk Management Committee 92 | SABECO 2024 Sustainability Report 91 \\u0017 Member of Board of Directors Mr Le Thanh Tuan \\u0017 Member of Nomination and Renumeration Committee \\u0017 Member of Land Committee \\u0017 Member of Board of Directors"}\n{"id": "B-A-0c972b1ab31949bb", "claim": "The Code underscores our commitment to operational transparency, safeguarding shareholder interests, and treating employees, customers, and business partners ethically.", "evidence": "Our approach is driven by our commitment to create value not just for our shareholders, but also for the communities we serve, the environment we impact, and the people who trust our brand. On behalf of the Board of Directors, I would like to extend my gratitude to our Board of Management, employees, customers, partners, and communities for their continued trust and support as we work together to build a sustainable and resilient future. As part of our ongoing commitment to responsible business practices, we have made Governance a priority, ensuring we uphold the standards of transparency, accountability and ethical behavior. We believe that strong governance is fundamental to building trust"}\n{"id": "B-A-9d4a8b03d88dcf68", "claim": "Essential ingredients like barley malt, hops, cans, and bottles are sourced from leading and reputable suppliers in the market to maintain quality and safety.", "evidence": "Normung (DIN). We ensure high-quality ingredients- such as barley malt, hops, and packaging materials- are sourced directly from approved suppliers, guaranteeing food safety and hygiene. Product quality assurance Continuing research on the application of barley malt in the production process to diversify raw material sources and maintain stable production. Upgrading the inner burner of the wort kettle and the operation process of the wort kettle to enhance the efficiency of bitter substance extraction, stabilize the bitterness of the finished beer, and reduce the amount of hop flowers used. ingredients are selected from a vetted list of suppliers to maintain quality."}\n{"id": "B-A-0354fdd4457cf457", "claim": "Auxiliary ingredients that do not impact beer quality and safety are procured from an approved supplier list.", "evidence": "Normung (DIN). We ensure high-quality ingredients- such as barley malt, hops, and packaging materials- are sourced directly from approved suppliers, guaranteeing food safety and hygiene. Auxiliary Specialized laboratories equipped with advanced technology, such as Skalar malt analyzers and HPLC, assess input materials. All materials receive “Qualified” certification from our Quality Control"}\n{"id": "B-A-5794909e07bd2ea6", "claim": "Vietnamese suppliers on sustainable supply chain management, green innovation, ESG integration, circular economy, resource efficiency, waste reduction, etc.", "evidence": "Production factory 75% -80% Start This innovation, developed in collaboration with SABECO’s suppliers, not only reduces costs for both parties but also demonstrates a commitment to Scope 3 GHG emissions reduction annually while maintaining product quality. Empty bottles SABECO also applies a circular economy model across the product lifecycle starting from the product design stage to promote resource efficiency and sustainable production. They also provided information on SABECO’s strengths, challenges, and areas for improvement. High Packaging And Circular Economy Supply Chain Management Community Engagement Importance to stakeholders"}\n{"id": "B-A-473348db987a2cbf", "claim": "Accordingly, supplier evaluations are conducted periodically, including performance reviews, supply history assessments, and site visits if deemed necessary to ensure compliance with food safety, environmental, social and regulatory standards.", "evidence": "Department, ensuring compliance with regulations regarding quality and safety. Additionally, we conduct risk management in our supply chain, including regular supplier audits to ensure adherence to processes and standards. Adherence to product quality and safety standard Supplier selection is based on a robust and systematic supplier qualification in place which includes both economic and ESG aspects, ensuring compliance with legal and confidentiality requirements. Our supply chain management strategy ensures sustainability, regulatory compliance, and long-term supplier partnerships while maintaining transparency and efficiency in operations."}\n{"id": "B-A-40c6fce9135804e5", "claim": "Supplier selection is based on a robust and systematic supplier qualification in place which includes both economic and ESG aspects, ensuring compliance with legal and confidentiality requirements.", "evidence": "of Strategic Suppliers* implementing their own Supplier Code of Conduct by 2035. Supplier Screening SABECO’s Supplier Evaluation and Selection Council advises on supplier approval based on competency and compliance. This includes reviewing the quarterly report on internal controls and enterprise risk management to help safeguard the assets of the Company, while ensuring compliance with legal and regulatory requirements. 32 | SABECO 2024 Sustainability Report 31"}\n{"id": "B-A-268003a4984ddc04", "claim": "\\u0017 Virtual interactions We adhere to stringent technical and ESG requirements for raw materials, semi-finished goods, and finished goods, based on Vietnam’s applicable laws, regulations, standards, international practices, and reference information from specialized organizations such as the European Brewery Convention (EBC), Mitteleuropaische Brautechnische Analysenkommision", "evidence": "SABECO adheres to Vietnam\'s laws, standards, and international best practices from organizations such as the European Brewery Convention (EBC), Mitteleuropäische Brautechnische Analysenkommision (MEBAK), American Society of Brewing Chemists (ASBC), and Deutsches Institutfür Supplier Assessment Quality control measures ensure that raw materials, semi-finished, and finished goods meet stringent technical requirements. Our specialized laboratories, equipped with advanced analyzers such as Skalar malt analyzer, HPLC, GC, and Trumac nitrogen analyzer, conduct rigorous testing."}\n{"id": "B-A-e005648b413b6ff3", "claim": "Committed to a sustainable and resilient supply chain, SABECO continuously works toward improving supplier partnerships, reinforcing quality control measures, and aligning with international best practices to maintain product excellence and long-term business success.", "evidence": "Supplier selection is based on a robust and systematic supplier qualification in place which includes both economic and ESG aspects, ensuring compliance with legal and confidentiality requirements. Our supply chain management strategy ensures sustainability, regulatory compliance, and long-term supplier partnerships while maintaining transparency and efficiency in operations. With a system of breweries and distribution nationwide, SABECO has always been aware of its crucial duty to ensure the consistency of product quality in order to bring the best products to consumers We continuously update production processes, train staff in quality control, and evaluate employee effectiveness to enhanc"}\n{"id": "B-A-2a2961bfa9fa3ecd", "claim": "20 2-3 Boundary and Scope of this Report Presented across the material sustainability topics of three main pillars Statement on sustainable development strategy", "evidence": "Strategy, policies and practices There is not any restatement of information given in previous reports 10,11 2-22 N/A 2-4 Restatements of information Presented across the material sustainability topics of three main pillars [GRI 1] 02 ABOUT THIS SUSTAINABILITY REPORT BOUNDARY AND SCOPE"}\n{"id": "B-A-9b4eb227aec3759d", "claim": "Strategy, policies and practices There is not any restatement of information given in previous reports 10,11 2-22 N/A 2-4 Restatements of information Presented across the material sustainability topics of three main pillars", "evidence": "20 2-3 Boundary and Scope of this Report Presented across the material sustainability topics of three main pillars Statement on sustainable development strategy 2-5 Presented across the material sustainability topics of three main pillars Embedding policy commitments N/A 2-24 Activities, value chain and other business relationships Activities and workers 17-18 2-6 About SABECO"}\n{"id": "B-A-48f96f1647f584ac", "claim": "Corporate Governance Annual Report 2024 2-13 30-32 3-2 Materiality Matrix 27 List of material topics Role of the highest governance body in sustainability reporting", "evidence": "Corporate Governance Annual Report 2024 29-30 2-11 TOPIC STANDARDS Role of the highest governance body in overseeing the management of impacts GRI 3: Material Topics 2021 Corporate Governance Corporate Governance Annual Report 2024 91-92 2-17 Report of the Board of Directors Annual Report 2024 Evaluation of the performance of the highest governance body"}\n{"id": "B-A-bf31c3be23be5e47", "claim": "SOCIAL DIMENSION Greenhouse Gas Emission Inventory – 2024 Performance Direct (Scope 1) GHG emissions (CO2e) GRI 305 – Emission 2016 Occupational Health and Safety", "evidence": "305-1 47 GRI 3 – Material Topics 2021 Occupational Health and Safety – Management Approach Energy indirect (Scope 2) GHG emissions (CO2e) Greenhouse Gas Emission Performance Greenhouse Gas Emission Performance – 2024 GRI 403 – Occupational Health and Safety 2018 403-1 62 305-4 GHG emissions intensity"}\n{"id": "B-A-3ed14fde228a7672", "claim": "2021 Percentage of employees receiving regular performance and career development reviews Food Quality and Safety – Management Approach Fair Performance Evaluation and Recognition", "evidence": "Percentage of employees receiving regular performance & career development reviews in 2024, by Gender Percentage of employees receiving regular performance and career development reviews in 2024, by Employee Category Officer Male employees Employees are encouraged to actively share their insights, feedback, and ideas for continuous improvement in daily work. Measurable, Achievable, Relevant, & Time-bound) KPIs aligned with company goals. Performance is assessed at both individual and team levels, with business unit heads calibrating evaluations and rewarding team success. 100% of eligible employees received regular performance and career development reviews."}\n{"id": "B-A-25d74479bc6b9d76", "claim": "2024 2023 2022 Non – hazardous waste Tonnes 7,093 7,041 8,153 Total number of sites within the scope of environment data collection 11 Number of sites 11 11 Total weight of waste generated", "evidence": "NEW EMPLOYEE HIRES Total number of sites within the scope of environment data collection 11 Number of sites 11 11 Total New Employee Hires by Gender Data Coverage % 100 100 100 401-1 New Employee APPENDIX B ESG PERFORMANCE DATA INDICATORS UNIT 2024 2023 2022 GRI INDEX ENVIRONMENT DATA Hazardous waste Tonnes 46 45 57 The scope of data collection"}\n{"id": "B-A-9e17cf68c7cdef22", "claim": "32,664 Total weight of waste otherwise disposed Tonnes 3.15 0.48 155 Renewable electricity sourced in our manufacturing sites at year end Percentage of recyclable plastic packaging (as a % of the total weight of all plastic packaging)", "evidence": "70,800 78,161 90,791 Total weight of waste disposed to landfill Tonnes 306-5 Waste directed 2,240 2,450 2,688 Total biogenic GHG emissions (CO2e) Tonnes 2024 2023 2022 Non – hazardous waste Tonnes 7,093 7,041 8,153 Total number of sites within the scope of environment data collection 11 Number of sites 11 11 Total weight of waste generated"}\n{"id": "B-A-b4cf7a0d67f6ac39", "claim": "ML 1,402.43 Total net freshwater consumption in water-stressed areas 328.82 348.78 ML 362.31 Percentage of ISO 22000:2018 and other food-safety related certification verified sites 100% 100% 100% % Total net freshwater consumption in non-water-stressed areas", "evidence": "Total number of ISO 2200:2018 and other food-safety related certification verified sites Number of sites 11 11 11 Total water discharge to all areas 303-4 Water withdrawal 1,543.07 1,735.59 Percentage of ISO 9001 certified sites % 100% 100% 100% Total water withdrawal from all areas with water stress ML 785.78 823.33 942.99 303-3 Water withdrawal"}\n{"id": "B-A-d23eca7e3a6690a6", "claim": "Thông tin bao gồm chiến lược phát triển bền vững của PNJ trong dài hạn, trung hạn và ngắn hạn, hoạt động cụ thể và định hướng mục tiêu trong tương lai cùng chương trình hánh động cụ thể.", "evidence": "Tất cả được xác định dựa trên đánh giá kết hợp giữa mức độ quan tâm của các bên liên quan và tầm quan trọng của các yếu tố đó đối với PNJ. Thông tin bao gồm chiến lược phát triển bền vững của PNJ trong dài hạn, trung hạn và ngắn hạn, hoạt động cụ thể và định hướng mục tiêu trong tương lai cùng chương trình hành động cụ thể. THỊ TRƯỜNG VỐN XANH ი Kỳ báo cáo: 2022 ი Chu kỳ báo cáo: 1 năm 1 lần Cụ thể: CHIẾN LƯỢC HOẠT ĐỘNG Thực hiện tốt việc tham mưu cho HĐQT về việc xây dựng định hướng chiến lược cho giai đoạn 2022 - 2027 hướng tới năm 2030. Đồng thời tiểu ban cũng luôn trao đổi và góp ý trong chiến lược và triển khai với BĐH. Các định hướng chiến lược trọng tâm đã bước đầu được các đơn vị đưa"}\n{"id": "B-A-d52b09fba3e8d0de", "claim": "Sự thành lập và tiên phong của Tiểu ban Môi trường - Xã hội - Quản trị (ESG) với sự thường trực của thành viên trong HĐQT được kì vọng sẽ nâng tầm chiến lược dài hạn và mở ra tương lai cho sự phát triển bền vững của Tập đoàn trong tầm nhìn 2022 - 2027, hướng đến 2030.", "evidence": "- Quản trị (ESG), và điều chỉnh tên Tiểu ban Chiến lược thành Tiểu ban Chiến lược - Tài chính trực thuộc HĐQT, giao nhiệm vụ cụ thể dựa trên chiến lược dài hạn của công ty nhằm hướng tới tầm nhìn chiến lược phát triển bền vững lâu dài và vươn tầm quốc tế. VỀ VIỆC TỔ CHỨC CÁC CUỘC HỌP HĐQT NGUYÊN TẮC 1.3 BẢO ĐẢM VAI TRÒ LÃNH ĐẠO HIỆU QUẢ VÀ TÍNH ĐỘC LẬP CỦA HĐQT của doanh nghiệp cũng như định hướng trong giai đoạn 2022 – 2027, hướng đến 2030. TIỂU BAN MÔI TRƯỜNG - XÃ HỘI - QUẢN TRỊ Xác định nhiệm vụ trọng tâm của ESG trong 2020 - 2023 là chuẩn hóa và đo lường."}\n{"id": "B-A-cc1875b8de2712d0", "claim": "Vì thế Chúng tôi tích cực triển khai nhiều chương trình hành động cụ thể thông qua các cuộc tái tạo làm giàu và phát huy sức mạnh của văn hóa PNJ, thực hiện tiến trình F5 Fresh đổi mới quản trị doanh nghiệp, tiến hành chuẩn hóa hoạt động phát triển bền vững theo tiêu chuẩn quốc tế.", "evidence": "PNJ nỗ lực từng ngày để duy trì và phát triển một cách sáng tạo, linh hoạt các hành động cụ thể trong chiến lược phát triển bền vững dài hạn trên nền tảng văn hóa doanh nghiệp không ngừng được tái tạo, là một trong những ngọn cờ đầu trong công cuộc hình thành đặc trưng tiêu biểu của văn hóa kinh doanh tại Việt Nam. VĂN HÓA VĂN HÓA HĐQT HĐQT TÍCH CỰC DẪN DẮT XÂY DỰNG VĂN HOÁ THỰC HÀNH TỐT QTCT VƯỢT TRÊN TUÂN THỦ MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA PNJ Tuyên bố về ESG: “PNJ hành động có trách nhiệm để tôn vinh vẻ đẹp của con người và cuộc sống một cách bền vững”. THẤP DIỄN GIẢI TẠI PNJ Xuất phát từ triết lý phát triển bền vững đặt lợi ích của khách hàng và xã hội vào lợi ích của doanh nghiệp, PNJ"}\n{"id": "B-A-041851db33ff6405", "claim": "Sẽ không thể có nền kinh tế tăng trưởng bền vững nếu thiếu cộng đồng doanh nghiệp bền vững.", "evidence": "LÀ NỀN TẢNG PHÁT TRIỂN BỀN VỮNG 2 NGUYÊN TẮC 2.3 HĐQT XÂY DỰNG CHIẾN LƯỢC KINH DOANH VÀ PHƯƠNG PHÁP QUẢN TRỊ DỰA TRÊN NỀN TẢNG LÀ VĂN HÓA DOANH NGHIỆP HƯỚNG ĐẾN PHÁT TRIỂN BỀN VỮNG PNJ chủ động tham dự báo cáo tại nhiều diễn đàn, hội thảo về ESG trong năm nhằm chia sẻ kinh nghiệm với cộng đồng doanh nghiệp. Chúng tôi tin tưởng rằng sự đồng lòng thực hiện sứ mệnh PNJ của toàn bộ cán bộ nhân viên, cùng với chiến lược phát triển đúng đắn và những nền tảng chúng ta cùng xây dựng trong suốt 35 năm qua sẽ tiếp tục giúp PNJ vươn tới những những tầm cao mới trong các công tác kinh tế, xã hội, môi trường, tiếp tục gặt hái nhiều kỳ tích và không ngừng tăng trưởng mạnh mẽ, vững chãi trong tương lai."}\n{"id": "B-A-5705a410a7e9353f", "claim": "Năm thứ 6 liên tiếp PNJ lọt Top 10 trong 100 doanh nghiệp phát triển bền vững.", "evidence": "NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN CƠ CẤU QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG PNJ liên tục 6 năm liền lọt top 10 trong 100 doanh nghiệp phát triển bền vững, bao quát ở 3 lĩnh vực chính: quản trị, môi trường và xã hội. 100 doanh nghiệp PTBV VN Ra đời Cửa hàng Kinh doanh Vàng Bạc Phú Nhuận 2016 2004 Tham gia Hội đồng doanh nghiệp vì sự phát triển bền vững Việt Nam VBCSD - VCCI. Lập báo cáo PTBV và lọt vào top 100 doanh nghiệp"}\n{"id": "B-A-fb1e47b67236272c", "claim": "Từ khi mới thành lập cho đến nay, trải qua 33 năm hình thành và phát triển, PNJ đã không ngừng cập nhật, cải tiến quy trình sản xuất, doanh nghiệp đã có cái nhìn sâu sắc để để tạo ra một mô hình kinh doanh bền vững, kết hợp tối ưu hóa chuỗi cung ứng, nâng cao ý thức bảo vệ môi trường.", "evidence": "Các chính sách kinh doanh đúng đắn được ban hành kịp thời, những nền tảng mới liên tục được nâng cấp và cập nhật, sức tư duy & sáng tạo của nhân sự được khai phóng đã giúp mang đến những kỷ lục mới trong suốt 33 năm hình thành và phát triển. Chính vì những nỗ lực trên, PNJ đã được vinh danh trong Top 10 Doanh nghiệp phát triển bền vững Hội đồng doanh nghiệp vì sự phát triển Việt Nam bình chọn, đồng thời cũng được tôn vinh là Nhà sản xuất, chế tác trang sức xuất sắc nhất năm 2021 bởi Giải thưởng Jewellery World Awards của ngành kim hoàn thế giới. TABLE 183 TĂNG TỐC TÁI TẠO BỆ PHÓNG VỮNG VÀNG HÀNH TRÌNH BAY BỀN VỮNG VUN ĐẮP VỐN XÃ HỘI SẴN SÀNG TÂM THẾ ĐẠT ĐẾN TẦM CAO MỚI TỪ QUẢN TRỊ HIỆU QUẢ B"}\n{"id": "B-A-d7e8bfc65b862080", "claim": "- Đảm bảo xuyên suốt quả chất thải nguy hại từ quy trình sản xuất.", "evidence": "Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật. PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật. Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP. Xử lý chất thải một cách an toàn: Việc sản xuất trang sức cũng dẫn đến việc tạo ra rất nhiều chất thải, bao gồm cả chất thải nguy hại như hóa chất và kim loại nặng. GIÁM SÁT VÀ QUẢN LÝ RỦI RO NHẬN DIỆN RỦI RO Tăng cường kiểm soát chất lượng sản phẩm, đảm bảo sản phẩm đáp ứng các tiêu chuẩn chất lượng và độ tin cậy cao."}\n{"id": "B-A-bdc320483991007e", "claim": "PEOPLE Con người PARTNERSHIP PEACE Hòa bình Đối tác Tuy nhiên PNJ vẫn tỉnh thức về tác động của đại dịch, chủ động “biến nguy thành cơ”, chuyển mình tái cấu trúc, kiên định bám sát mục tiêu kinh doanh đồng thời đẩy mạnh mũi nhọn phát triển vốn xã hội giữa mùa COVID theo mô hình 5 trụ cột của Phát triển Bền vững.", "evidence": "kết hợp chặt chẽ, hợp lý và hài hòa giữa phát triển kinh tế với phát triển xã hội và bảo vệ tài nguyên và môi trường, chủ động ứng phó với biến đổi khí hậu. Tại PNJ, xuất phát từ triết lý phát triển bền vững: “Đặt lợi ích khách hàng, xã hội vào lợi ích doanh nghiệp”, đến nay chiến lược phát triển bền vững được thực hiện đồng bộ cả trong và ngoài doanh nghiệp với các mục tiêu bám sát quan điểm chỉ đạo của Chính phủ qua Nghị quyết 136/ NQ-CP về phát triển bền vững ban hành ngày 25 tháng 09 năm 2020, đó cũng là nội dung cụ thể của 17 Mục tiêu Phát triển Bền vững (SDGs) của Liên Hiệp Quốc. TABLE NỀN TẢNG CỦA XÁC ÐỊNH CÁC LĨNH VỰC TRỌNG YẾU QUY TRÌNH TẠO LẬP GIÁ TRỊ TỪ MỤC TIÊU PTBV NHỮNG KỲ Căn "}\n{"id": "B-A-23fff3503c36a612", "claim": "- Hiệu quả sử dụng vốn của doanh nghiệp.", "evidence": "Mục tiêu sử dụng nguyên vật liệu tiết kiệm và hiệu quả đã được lồng ghép hài hoà trong hoạt động hàng ngày của doanh nghiệp, củng cố sự hài lòng của nhân viên đối với các sáng kiến, đáp ứng yêu cầu phát triển bền vững của doanh nghiệp. Đối với hoạt động tại văn phòng và cửa hàng, PNJ đã triển khai thực hiện các sáng kiến để tăng hiệu suất công việc và sử dụng hiệu quả nguyên vật liệu như sau: 57 58 Khi xây dựng một doanh nghiệp, quan điểm của những người sáng lập rất quan trọng, phản ánh “cái chất” của người lãnh đạo. Ở PNJ, quan điểm của những người sáng lập doanh nghiệp là theo đuổi triết lý phát triển bền vững Đặt lợi ích khách hàng và lợi ích xã hội vào lợi ích doanh nghiệp. Do đó, PNJ l"}\n{"id": "B-A-e734bdac27f7160a", "claim": "Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được tích hợp cùng các mục tiêu phát triển bền vững.", "evidence": "TABLE NỀN TẢNG CỦA XÁC ÐỊNH CÁC LĨNH VỰC TRỌNG YẾU QUY TRÌNH TẠO LẬP GIÁ TRỊ TỪ MỤC TIÊU PTBV NHỮNG KỲ Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, TÍCH chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc tích hợp cùng các mục tiêu phát triển bền vững. phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. VƯỢT Các bước tiến về kinh tế "}\n{"id": "B-A-8d1ad64b9f19cf5a", "claim": "Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích hợp vào định hướng và các mục tiêu phát triển bền vững của công ty.", "evidence": "TABLE NỀN TẢNG CỦA XÁC ÐỊNH CÁC LĨNH VỰC TRỌNG YẾU QUY TRÌNH TẠO LẬP GIÁ TRỊ TỪ MỤC TIÊU PTBV NHỮNG KỲ Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, TÍCH chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc tích hợp cùng các mục tiêu phát triển bền vững. phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. VƯỢT Các bước tiến về kinh tế "}\n{"id": "B-A-47436b7a393af31f", "claim": "1 Hiểu về Phát Triển Bền Vững - Đào tạo, nâng cao nhận thức - Cập nhật điểm mới Cao Sức khỏe & An toàn khách hàng 2 Xây dựng kế hoạch bền vững hàng năm GRI 416", "evidence": "TABLE NỀN TẢNG CỦA XÁC ÐỊNH CÁC LĨNH VỰC TRỌNG YẾU QUY TRÌNH TẠO LẬP GIÁ TRỊ TỪ MỤC TIÊU PTBV NHỮNG KỲ Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, TÍCH chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc tích hợp cùng các mục tiêu phát triển bền vững. phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. VƯỢT Các bước tiến về kinh tế "}\n{"id": "B-A-1730530f54d5bed5", "claim": "Tuân thủ kinh tế và xã hội Năng lượng 4 Giám sát & Đánh giá GRI 419 GRI 302 Đa dạng sinh học Quyền bảo mật thông tin khách hàng GRI 304 - Triển khai GRI 418 - Định lượng, vốn hóa các mục tiêu", "evidence": "TABLE NỀN TẢNG CỦA XÁC ÐỊNH CÁC LĨNH VỰC TRỌNG YẾU QUY TRÌNH TẠO LẬP GIÁ TRỊ TỪ MỤC TIÊU PTBV NHỮNG KỲ Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, TÍCH chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc tích hợp cùng các mục tiêu phát triển bền vững. phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. VƯỢT Các bước tiến về kinh tế "}\n{"id": "B-A-bd82f6f23882d380", "claim": "KỲ Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát,", "evidence": "Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được tích hợp cùng các mục tiêu phát triển bền vững. Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. ỦY BAN KIỂM TOÁN Thực hiện các hoạt động QTRR để đạt được các mục tiêu phát triển một cách PNJ thường xuyên rà soát để đạt được các mục ti"}\n{"id": "B-A-8479d50d15295a8b", "claim": "TÍCH chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được hợp vào định hướng và các mục tiêu phát triển bền vững của công ty.", "evidence": "Quy trình được xây dựng nhằm thực hiện một cách có hệ thống và được giám sát, đánh giá hiệu quả một cách chặt chẽ các hoạt động mang lại lợi ích kinh tế được tích hợp cùng các mục tiêu phát triển bền vững. Căn cứ nội dung 17 mục tiêu phát triển bền vững của Liên Hợp Quốc, PNJ đã nghiên cứu, lựa chọn trên cở sở đặc thù lĩnh vực hoạt động, bối cảnh kinh tế, xã hội Việt Nam, đồng thời tích hợp vào định hướng và các mục tiêu phát triển bền vững của công ty. Tại PNJ, công cuộc phát triển bền vững được tiến hành đồng thời cả bên trong và bên ngoài doanh nghiệp. MỤC TIÊU PHÁT TRIỂN BỀN VỮNG CỦA PNJ Tuyên bố về ESG: “PNJ hành động có trách nhiệm để tôn vinh vẻ đẹp của con người và cuộc sống một cách"}\n{"id": "B-A-38279f1926012810", "claim": "1 trưởng bền vững và cùng tạo bối cảnh thuận lợi cho sự phát triển con người, hình thành các", "evidence": "Các bước tiến về kinh tế gắn liền với hoạt động phát triển xã hội, cải thiện môi trường sống, tăng cường giáo dục, hoàn thiện hệ thống an sinh xã hội, liên kết các hiệp hội để cùng tăng trưởng bền vững và cùng tạo bối cảnh thuận lợi cho sự phát triển con người, hình thành các đơn vị kinh tế mới, kích thích tạo lập các giá trị cộng đồng. Tất cả được hoạch định chiến lược lâu dài, có kế hoạch hành động cụ thể, đồng bộ và được đánh giá hiệu quả định kỳ để có thay đổi phù hợp với những thay đổi về bối cảnh và tốc độ phát triển từng thời kỳ. TABLE VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN"}\n{"id": "B-A-2e1466c9382ccf19", "claim": "Việc khởi động quá trình nhấn nút tái tạo từ năm 2018 - 2019 để giúp cho công ty có sự linh hoạt, kiên định, sẵn sàng đối diện với những thách thức, rất quen thuộc với sự thay đổi liên tục với tốc độ cao.", "evidence": "TABLE Tiếp tục các cải tiến “Nhấn nút F5xF1 – Tăng tốc tái tạo, bứt phá vươn cao”, sự chủ động đón đầu trong hoạt động sản xuất kinh doanh và các sáng kiến chuyển đổi số, cho thấy HĐQT cùng BĐH của Công ty PNJ đã quyết tâm thực hiện những chiến lược dài hạn đề ra và nỗ lực rất cao trong việc tạo ra hiệu quả trong hoạt động chuỗi cung ứng, bán lẻ, sản xuất giúp cho PNJ đạt được hiệu quả kinh tế, giảm chi phí bán hàng & quản lý hiệu quả và phát triển tăng tốc trong năm 2022. Ngoài ra chuỗi hoạt động Siêu thị 0 đồng tiếp tục được phát huy trong năm 2022 cũng đã góp phần giúp nâng cao giá trị vốn xã hội của doanh nghiệp. Năm 2022, HĐQT đã tiên phong đổi mới trong việc thay đổi cơ cấu tổ chức HĐQ"}\n{"id": "B-A-c81a2d517b4cbdda", "claim": "Năm 2021 đã minh chứng và làm vững thêm cho quan điểm của chúng tôi là phát triển bền vững không có nghĩa là hướng đến cái \\"Tĩnh\\" của sự ổn định, tĩnh tại, bề dày của lịch sử, mà chính là tìm cái \\"Động\\" để hướng đến sự bền vững thông qua sự thích nghi, tự tái tạo và biến đổi chính mình liên tục dùng sự sáng tạo và năng động để có sự bền vững.", "evidence": "CHỦ TỊCH HỘI ĐôNG QUẢN TRỊ Điểm mới năm nay trong hoạt động Hội đồng quản trị là việc khẳng định vai trò lãnh đạo của Hội đồng quản trị đối với sự phát triển bền vững của doanh nghiệp, thúc đẩy hoạt động này không chỉ bằng sự quan tâm mà bằng hành động cụ thể. Với bề dày truyền thống luôn tiên phong trong hoạt động phát triển bền vững và trách nhiệm xã hội, PNJ chính thức nâng cao sự bài bản của hệ thống quản trị phát triển bền vững và là doanh nghiệp đầu tiên trên sàn chứng khoán thành lập Tiểu ban Môi trường - Xã hội - Quản trị (ESG). KỲ TÍCH CỦA NIỀM TIN - DẤU ẤN 2021 ỨNG BIẾN ỨNG BIẾN ỨNG BIẾN TẠO GIÁ TRỊ Mỗi quả ngọt PNJ hái được đều là kết tinh của những nỗ lực và tâm huyết mà công ty "}\n{"id": "B-A-19f5273be396fda1", "claim": "Hơn thế nữa, giá trị thương hiệu PNJ sẽ tiếp tục tăng lên thông qua việc liên kết và cộng hưởng giữa các thương hiệu trong hệ sinh thái PNJ,", "evidence": "và triển khai các chương trình xây dựng đội ngũ kế thừa. quản lý khách hàng. ◦Tiếp tục làm giàu tài sản Thương hiệu PNJ – nhóm các thương hiệu PNJ thông qua việc tạo mối liên kết và sức cộng hưởng giữa các thương hiệu của hệ sinh thái PNJ, gia tăng tính tích hợp các chương trình, đẩy mạnh hiệu quả các chương trình CSR. BÁO CÁO THƯỜNG NIÊN 2022 BÁO CÁO THƯỜNG NIÊN 2022 Hơn nữa, PNJ còn kiến tạo nền móng cho tương lai bằng chương trình Gắn kết PNJ và Sinh viên các trường đại học trên cả nước. Nhờ đó mà những giá trị về sự kết nối, quan tâm cùng phát triển, thông điệp niềm tin - nền tảng cho mọi sự khởi đầu,..."}\n{"id": "B-A-d620f89b80a7d3bf", "claim": "Chính vì những nỗ lực trên, PNJ đã được vinh danh trong Top 10 Doanh nghiệp phát triển bền vững Hội đồng doanh nghiệp vì sự phát triển Việt Nam bình chọn", "evidence": "NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN CƠ CẤU QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG PNJ liên tục 6 năm liền lọt top 10 trong 100 doanh nghiệp phát triển bền vững, bao quát ở 3 lĩnh vực chính: quản trị, môi trường và xã hội. 100 doanh nghiệp PTBV VN Ra đời Cửa hàng Kinh doanh Vàng Bạc Phú Nhuận 2016 2004 Tham gia Hội đồng doanh nghiệp vì sự phát triển bền vững Việt Nam VBCSD - VCCI. Lập báo cáo PTBV và lọt vào top 100 doanh nghiệp"}\n{"id": "B-A-1b7075c6dc7679b3", "claim": "Trong năm 2021, mặc dù có những giai đoạn chuỗi cung ứng bị đứt gãy do đại dịch COVID-19 tác động, thế nhưng khi tình hình dịch bệnh được kiểm soát, PNJ tiếp tục khẳng định doanh nghiệp đang sở hữu một chuỗi kết nối bền vững mà trong đó từng mắt xích đều giữ những vai trò quan trọng trong việc đưa đến tay người tiêu dùng những sản phẩm có chất lượng hàng đầu.", "evidence": "Năm 2021 trải qua với nhiều khó khăn, thách thức đối với nền kinh tế do dại dịch Covid-19 tác động, nhưng cũng là một năm thử thách sự bền bỉ của PNJ, là năm khẳng định các giá trị bền vững mà PNJ theo đuổi. Đó cũng là năm thử thách sự quyết tâm và ý chí trong việc nắm chặt tay lái, nắm vững tay chèo để vượt qua những sóng gió. Với định hướng phát triển bền vững mà PNJ xây đắp trong thời gian qua, một lần nữa thể hiện sức mạnh và kim chỉ nam dẫn hướng cho công ty đi xuyên qua những con sóng lớn, hóa giải những rủi ro của giai đoạn dịch bệnh và trong giai đoạn hồi phục sau dịch bệnh. Đặc biệt, 2021 là một năm chưa từng có trong lịch sử bởi những biến động do đại dịch COVID-19 gây ra, tác động"}\n{"id": "B-A-1e021697b61936f2", "claim": "Phát triển bền vững là sự phát triển hài hoà cả về 3 mặt: Kinh tế - Xã hội - Môi trường.", "evidence": "kết hợp chặt chẽ, hợp lý và hài hòa giữa phát triển kinh tế với phát triển xã hội và bảo vệ tài nguyên và môi trường, chủ động ứng phó với biến đổi khí hậu. Tại PNJ, xuất phát từ triết lý phát triển bền vững: “Đặt lợi ích khách hàng, xã hội vào lợi ích doanh nghiệp”, đến nay chiến lược phát triển bền vững được thực hiện đồng bộ cả trong và ngoài doanh nghiệp với các mục tiêu bám sát quan điểm chỉ đạo của Chính phủ qua Nghị quyết 136/ NQ-CP về phát triển bền vững ban hành ngày 25 tháng 09 năm 2020, đó cũng là nội dung cụ thể của 17 Mục tiêu Phát triển Bền vững (SDGs) của Liên Hiệp Quốc. Kiên định trách nhiệm Phát triển Nguồn nhân lực Tôn vinh Giá trị Xã hội Lan tỏa yêu thương đến Phát triển b"}\n{"id": "B-A-42f36bb3f302c832", "claim": "Năm 2021, PNJ tiếp tục kiên định thực hiện các biện pháp sử dụng tiết kiệm, hiệu quả nguồn nguyên vật liệu, tiết giảm tiêu hao năng lượng, bảo vệ nguồn nước, hạn chế tỷ lệ khí thải, chất thải, phế phẩm...", "evidence": "Để đạt được tỉ lệ như vậy, đối với sản xuất, nguồn nước luôn được tiết kiệm do PNJ sử dụng những máy móc và công nghệ hàng đầu của Ý, do đó, nhu cầu cũng như tiêu hao khi sử dụng nước thấp. Đối với nước sinh hoạt, PNJ luôn thực hiện tuyên truyền vai trò của nguồn nước và sử dụng nước tiết kiệm cho CBCNV toàn công ty, đồng thời, có những khẩu hiệu tiết kiệm nước ở các điểm sử dụng nước và các biện pháp để nhắc nhở. NGUỒN PHÁT SINH NƯỚC THẢI Nước mưa chảy tràn Vì vậy việc giảm tỷ lệ hao hụt nguyên liệu sản xuất tăng dần qua các năm, thể hiện hiệu quả của việc áp dụng các công nghệ và trang thiết bị kỹ thuật hiện đại vào sản xuất và công tác đào tạo, tuyên truyền của PNJ. Trong bối cảnh năm 202"}\n{"id": "B-A-63344ffbded938f1", "claim": "Mục tiêu sử dụng nguyên vật liệu tiết kiệm và hiệu quả đã được lồng ghép hài hoà trong hoạt động hàng ngày của doanh nghiệp, củng cố sự hài lòng của nhân viên đối với các sáng kiến, đáp ứng yêu cầu phát triển bền vững của doanh nghiệp.", "evidence": "kết hợp chặt chẽ, hợp lý và hài hòa giữa phát triển kinh tế với phát triển xã hội và bảo vệ tài nguyên và môi trường, chủ động ứng phó với biến đổi khí hậu. Tại PNJ, xuất phát từ triết lý phát triển bền vững: “Đặt lợi ích khách hàng, xã hội vào lợi ích doanh nghiệp”, đến nay chiến lược phát triển bền vững được thực hiện đồng bộ cả trong và ngoài doanh nghiệp với các mục tiêu bám sát quan điểm chỉ đạo của Chính phủ qua Nghị quyết 136/ NQ-CP về phát triển bền vững ban hành ngày 25 tháng 09 năm 2020, đó cũng là nội dung cụ thể của 17 Mục tiêu Phát triển Bền vững (SDGs) của Liên Hiệp Quốc. NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NG"}\n{"id": "B-A-dc0ae171a74865ca", "claim": "Chính vì vậy, PNJ luôn chú trọng tối đa hóa hiệu năng sử dụng và tiết kiệm năng lượng trong các hoạt động sản xuất, kinh doanh, chung tay bảo vệ môi trường.", "evidence": "CÁC RỦI RO TRỌNG YẾU Tiêu thụ năng lượng và khí thải: Việc sản xuất, vận chuyển và bán lẻ trang sức có thể gây ra lượng khí thải và tiêu thụ năng lượng cao, gây ảnh hưởng đến môi trường. TRONG NĂM 2022 Tối ưu hóa quá trình sản xuất và vận chuyển để tiết kiệm năng lượng và tài nguyên. Nâng cao nhận thức: Đẩy mạnh tuyên truyền nâng cao ý thức, nhận thức và góp phần nâng cao kiến thức quản lý và sử dụng năng lượng tiết kiệm cho toàn bộ nhân viên, góp phần tiết kiệm chi phí sản xuất và nâng cao hiệu quả kinh doanh cho doanh nghiệp Thông tin sáng kiến: Dù hoạt động sản xuất kinh doanh thuận lợi hay khó khăn thì ý thức tiết kiệm vẫn luôn là trách nhiệm của tất cả thành viên gia đình PNJ. Hưởng ứng"}\n{"id": "B-A-eebc9b5c4b5bad53", "claim": "Đơn vị đã thực hiện đăng ký hỗ trợ tiết kiệm năng lượng tại Trung tâm Tiết kiệm năng lượng TP.", "evidence": "Đơn vị đã thực hiện đăng ký hỗ trợ tiết kiệm năng lượng tại Trung tâm Tiết kiệm năng lượng TP.HCM. Để bảo đảm các hoạt động luôn được vận hành, PNJP còn sở hữu máy phát điện dự phòng với công suất 870KVA, được sử dụng trong trường hợp mất điện. Nhiên liệu dùng cho máy phát điện là 120 lít dầu DO/1 giờ mất điện. Nâng cao nhận thức: Đẩy mạnh tuyên truyền nâng cao ý thức, nhận thức và góp phần nâng cao kiến thức quản lý và sử dụng năng lượng tiết kiệm cho toàn bộ nhân viên, góp phần tiết kiệm chi phí sản xuất và nâng cao hiệu quả kinh doanh cho doanh nghiệp Thông tin sáng kiến: Dù hoạt động sản xuất kinh doanh thuận lợi hay khó khăn thì ý thức tiết kiệm vẫn luôn là trách nhiệm của tất cả thành "}\n{"id": "B-A-80317f1eb9f515ce", "claim": "Do đó, PNJ có các chính sách tiết kiệm năng lượng về cả ý thức và hành động, về cả mặt kỹ thuật và hành chính.", "evidence": "Nâng cao nhận thức: Đẩy mạnh tuyên truyền nâng cao ý thức, nhận thức và góp phần nâng cao kiến thức quản lý và sử dụng năng lượng tiết kiệm cho toàn bộ nhân viên, góp phần tiết kiệm chi phí sản xuất và nâng cao hiệu quả kinh doanh cho doanh nghiệp Thông tin sáng kiến: Dù hoạt động sản xuất kinh doanh thuận lợi hay khó khăn thì ý thức tiết kiệm vẫn luôn là trách nhiệm của tất cả thành viên gia đình PNJ. Hưởng ứng chiến dịch tiết kiệm này, các đơn vị phòng ban đã thực hiện rất tốt trong việc tiết kiệm điện, tiết kiệm nước, tiết kiệm văn phòng phẩm, chấp hành các quy định về giờ giấc làm việc. PNJP. Nguồn điện cấp cho PNJP được lấy từ mạng lưới điện của thành phố. Đơn vị đã thực hiện đăng ký hỗ"}\n{"id": "B-A-e2ebe64ba8e59d1d", "claim": "- Sử dụng các máy móc, thiết bị hiện đại, tiết kiệm năng lượng trong sản xuất và quản lý.", "evidence": "Nâng cao nhận thức: Đẩy mạnh tuyên truyền nâng cao ý thức, nhận thức và góp phần nâng cao kiến thức quản lý và sử dụng năng lượng tiết kiệm cho toàn bộ nhân viên, góp phần tiết kiệm chi phí sản xuất và nâng cao hiệu quả kinh doanh cho doanh nghiệp Thông tin sáng kiến: Dù hoạt động sản xuất kinh doanh thuận lợi hay khó khăn thì ý thức tiết kiệm vẫn luôn là trách nhiệm của tất cả thành viên gia đình PNJ. Hưởng ứng chiến dịch tiết kiệm này, các đơn vị phòng ban đã thực hiện rất tốt trong việc tiết kiệm điện, tiết kiệm nước, tiết kiệm văn phòng phẩm, chấp hành các quy định về giờ giấc làm việc. Tuy nhiên, là một doanh nghiệp luôn coi trọng yếu tố môi trường trong sản xuất, PNJ vẫn cố gắng xử lý "}\n{"id": "B-A-3483d8986a4f5ebf", "claim": "- Bố trí nơi làm việc và sản xuất thoáng đãng, gọn gàng nhằm tiết kiệm năng lượng của hệ thống điều hòa, thông gió và chiếu sáng.", "evidence": "Nâng cao nhận thức: Đẩy mạnh tuyên truyền nâng cao ý thức, nhận thức và góp phần nâng cao kiến thức quản lý và sử dụng năng lượng tiết kiệm cho toàn bộ nhân viên, góp phần tiết kiệm chi phí sản xuất và nâng cao hiệu quả kinh doanh cho doanh nghiệp Thông tin sáng kiến: Dù hoạt động sản xuất kinh doanh thuận lợi hay khó khăn thì ý thức tiết kiệm vẫn luôn là trách nhiệm của tất cả thành viên gia đình PNJ. Hưởng ứng chiến dịch tiết kiệm này, các đơn vị phòng ban đã thực hiện rất tốt trong việc tiết kiệm điện, tiết kiệm nước, tiết kiệm văn phòng phẩm, chấp hành các quy định về giờ giấc làm việc. CÁC RỦI RO TRỌNG YẾU Tiêu thụ năng lượng và khí thải: Việc sản xuất, vận chuyển và bán lẻ trang sức có"}\n{"id": "B-A-f5d0d2b69e90cd45", "claim": "- Chuyển đổi công nghệ hàn hấp từ sử dụng gas, xăng sang NH3, giảm lượng tiêu thụ năng lượng và phát thải CO2.", "evidence": "3.735 3.375 5.172 Năm 2023, PNJ đặt mục tiêu giảm năng lượng theo lộ trình giảm phát thải ròng bằng “Không” theo mục tiêu quốc gia (mỗi năm giảm 1%), không ngừng chuẩn hóa thiết bị đo đạc để xác định rõ dòng Gas (kg) Bình quân (kg/1000 sp) năng lượng tiêu thụ. Các thiết bị kiểm soát thói quen tiêu thụ năng lượng sẽ được áp dụng nhằm hỗ trợ theo dõi dữ liệu, điều tiết nhu cầu sử dụng năng lượng và giảm thiểu tự động, tránh các lỗi MỨC TIÊU THỤ NĂNG LƯỢNG Gas Xăng dầu Điện Bên cạnh nguồn nguyên vật liệu, việc sử dụng năng lượng tiết kiệm và hiệu quả cũng là một trong những nhân tố then chốt trong chiến lược phát triển bền vững. Đặc biệt là trong bối cảnh phục hồi sản xuất công nghiệp sau đại d"}\n{"id": "B-A-3b299d6530f9ef6c", "claim": "Thông tin sáng kiến: Máy tách Hydro (Giải pháp kỹ thuật) - Ứng dụng máy tách hydro từ nhiên liệu hoá thạch là xăng - Ưu điểm: 100% năng lượng bền vững không phát thải khí nhà kính, dễ lưu trữ và đa năng để thực hiện sản xuất một cách bền vững và ít tác động đến môi trường", "evidence": "Bạc Đội ngũ nhà máy PNJ được tập dượt nghiêm túc trong việc tổng hợp dữ liệu tiêu thụ năng lượng, đánh giá khách quan mức độ, cường độ tiêu thụ năng lượng trong hoạt động sản xuất kinh doanh tạo cơ sở tin cậy để cán bộ nhà máy xây dựng lộ trình cắt giảm khí thải, giảm nhu cầu tiêu thụ năng lượng ở nhiều điểm cơ hội tiềm năng. KHÍ NHÀ KÍNH 2% cho 02 nhà máy PNJ tích cực tái sử dụng các loại nguyên liệu này từ hoạt động thu đổi hướng đến kinh tế tuần hoàn. NGUYÊN LIỆU 654,69 Kg 313,60 695,73 Hợp kim thau Thông tin sáng kiến: PNJP phát động chương trình “Phát triển bền vững GRI-ESG tại PNJP” nhằm giảm lượng phát thải nhà kính trong toàn nhà máy với các hoạt động thiết thực như: “Phát triển bền "}\n{"id": "B-A-52f00b7e3510d48f", "claim": "Hưởng ứng chiến dịch tiết kiệm này, các đơn vị phòng ban đã thực hiện rất tốt trong việc tiết kiệm điện, tiết kiệm nước, tiết kiệm văn phòng phẩm, chấp hành các quy định về giờ giấc làm việc.", "evidence": "Đơn vị đã thực hiện đăng ký hỗ trợ tiết kiệm năng lượng tại Trung tâm Tiết kiệm năng lượng TP.HCM. Để bảo đảm các hoạt động luôn được vận hành, PNJP còn sở hữu máy phát điện dự phòng với công suất 870KVA, được sử dụng trong trường hợp mất điện. Nhiên liệu dùng cho máy phát điện là 120 lít dầu DO/1 giờ mất điện. PNJP. Nguồn điện cấp cho PNJP được lấy từ mạng lưới điện của thành phố. Đơn vị đã thực hiện đăng ký hỗ trợ tiết kiệm năng lượng tại Trung tâm Tiết kiệm năng lượng TP. HCM. Ngoài việc tiêu thụ năng lượng điện phục vụ cho các hoạt động của mình, PNJP còn có một máy phát điện dự phòng với công suất"}\n{"id": "B-A-88c84fd9b01e7e08", "claim": "Đặc biệt trong năm 2021, nhà máy Long Hậu mới đi vào hoạt động, hệ thống sử dụng và xử lý nước ở nhà máy này vẫn chưa hoàn toàn hiệu quả so với nhà máy Gò Vấp, làm bình quân lượng nước sử dụng trên sản phẩm và lượng nước tái chế chưa được tối ưu như 5 năm gần đây.", "evidence": "12% 25% 25% Tỷ lệ nước tuần hoàn để tái sử dụng so với năm trước đó Hiện tại, doanh nghiệp vẫn đang trong quá trình trang bị cho nhà máy mới để nâng cao hiệu quả sử dụng nguồn nước, công suất xử lý nước thải và đồng thời đưa ra những biện pháp sử dụng/tái chế hiệu quả lượng nước ở nhà máy này. PNJ tin rằng trong những năm kế tiếp, các chỉ số về hiệu quả sử dụng nước, tỉ lệ nước tái chế, tái sử dụng và xử lý nước sẽ ngày càng được cải thiện và hướng đến mức tối ưu. 84.499 42.205 28.391 KG phương pháp xử lý Khác 348 756 398 KG Năm 2021, lượng bùn thải phát sinh tăng do sản lượng năm 2021 tăng hơn so với năm 2020. Ngoài ra, PNJ xây dựng thêm nhà máy ở Long Hậu và xây dựng thêm hệ thống xử lý nư"}\n{"id": "B-A-4787a7835d4f3ce4", "claim": "PNJ tin rằng trong những năm kế tiếp, các chỉ số về hiệu quả sử dụng nước, tỉ lệ nước tái chế, tái sử dụng và xử lý nước sẽ ngày càng được cải thiện và hướng đến mức tối ưu.", "evidence": "Tuy nhiên, là một doanh nghiệp luôn coi trọng yếu tố môi trường trong sản xuất, PNJ vẫn cố gắng xử lý tối ưu nguồn nước đầu ra, đảm bảo tỉ lệ xử lý nước trước khi thải ra môi trường ở mức 84%, thậm chí còn cao hơn hẳn những năm vừa qua. Truyền thông, hướng dẫn và cụ thể hóa bằng nội quy sử dụng các máy móc và thiết bị sử dụng năng lượng tại các khu vực sản xuất, đảm bảo duy trì và phát huy tinh thần tiết kiệm điện trở thành nề nếp, ổn định và lâu dài. Đồng thời, PNJP không ngừng nhắc nhở, cảnh cáo và xử lý các cá nhân có hành vi lãng phí khi sử dụng năng lượng. Là doanh nghiệp luôn coi trọng yếu tố môi trường trong sản xuất, PNJ vẫn cố gắng xử lý tối ưu nguồn nước đầu ra, đảm bảo tỷ lệ xử lý"}\n{"id": "B-A-7d58d9b41e492e87", "claim": "Đối với nước sinh hoạt, PNJ luôn thực hiện tuyên truyền vai trò của nguồn nước và sử dụng nước tiết kiệm cho CBCNV toàn công ty, đồng thời, có những khẩu hiệu tiết kiệm nước ở các điểm sử dụng nước và các biện pháp để nhắc nhở.", "evidence": "Xây dựng danh sách mua sản phẩm mới với các tiêu chí tiết kiệm điện 3.805.239 12% 7.347 61.228 16,3 2022 n ă m g n 2 ợ 0 ư 2 67% 2 L Đối với nước sinh hoạt, PNJ luôn thực hiện tuyên truyền vai trò của nguồn nước và sử dụng nước tiết kiệm cho cán bộ nhân viên toàn công ty, đồng thời, có những khẩu hiệu tiết kiệm nước ở các điểm sử dụng nước và các biện pháp để nhắc nhở. không ngừng đưa ra sáng kiến và biện pháp tiết kiệm nước, nghiên cứu và ứng dụng các giải pháp tái sử dụng nguồn nước an toàn và hiệu quả, đồng thời xử lý nguồn nước thải một cách hợp lý để bảo vệ môi trường. Nước đầu vào được PNJP (Gò Vấp và Long Hậu) sử dụng cho mục đích sinh hoạt và sản xuất. nước cấp thủy cục (m3) /1.000 s"}\n{"id": "B-A-973537b4a1be1428", "claim": "- Giảm phát thải ra môi trường 7800m3 nước thải /năm.", "evidence": "SẴN SÀNG TÂM THẾ VƯỢT THỬ THÁCH VƯƠN TẦM BẢO VỆ MÔI TRƯỜNG BẢO VỆ MÔI TRƯỜNG Nguồn phát sinh NƯỚC THẢI Sơ đồ công nghệ xử lý nước thải công ty PNJ Vì vậy, nước thải này cần được đảm bảo xử lý theo tiêu chuẩn môi trường trước khi thải ra nguồn tiếp nhận thông qua hệ thống xử lý nước thải không ngừng cải tiến. P13 CẶN BÙN MÁY ÉP BÙN BỂ BÙN TG ĐƯỜNG NƯỚC THẢI Nước rửa P15 BƠM BÙN ĐƯỜNG CẶN BÙN lượng hoá chất sử dụng trong quá trình xử lý nước thải, góp phần vào kế hoạch tiết giảm chi phí chung của bộ phận An toàn - Môi trường và nhà máy PNJP."}\n{"id": "B-A-f41a341100cdcfe1", "claim": "Chính vì thế, dù là ở đâu, thời điểm nào thì công tác quản lý chất thải rắn cũng được PNJ chú trọng nhằm giảm tác động tiêu cực đến môi trường.", "evidence": "Tại PNJ, chất thải rắn chủ yếu phát sinh từ hoạt động của PNJP, bao gồm: Chất thải rắn được thu gom để xử lý triệt để nên hầu như không gây ô nhiễm cho môi trường. CHẤT THẢI RẮN SINH HOẠT QUY TRÌNH THU GOM VÀ XỬ LÝ CHẤT THẢI RẮN SINH HOẠT CỦA PNJP Nhằm góp phần giảm lượng rác thải cần xử lý, tăng lượng rác thải có thể tái chế, PNJ đã tăng cuờng truyền thông hoạt động phân loại rác thải đúng cách để góp phần bảo vệ môi trường. Trong đó mấu chốt vẫn là khâu phân loại rác thải tại nguồn, nếu giải quyết tốt phân loại rác thì quá trình thu gom cũng như xử lý sẽ được thực hiện triệt để, nhanh chóng và dễ dàng. Rác thải là nguồn tài nguyên tái chế để có thể phục vụ cho cuộc sống."}\n{"id": "B-A-8fe7d9fdecf6c559", "claim": "Chất thải rắn sinh hoạt chủ yếu chứa nhiều chất hữu cơ dễ phân hủy, nếu không được thu gom và xử lý đúng quy định sẽ sinh ra mùi hôi thối gây ảnh hưởng đến sức khỏe người lao động, gây ô nhiễm môi trường không khí xung quanh.", "evidence": "Tại PNJ, chất thải rắn chủ yếu phát sinh từ hoạt động của PNJP, bao gồm: Chất thải rắn được thu gom để xử lý triệt để nên hầu như không gây ô nhiễm cho môi trường. CHẤT THẢI RẮN SINH HOẠT QUY TRÌNH THU GOM VÀ XỬ LÝ các khí này có khả năng gây mùi nên có thể sẽ gây ảnh hưởng đến các khu vực trong công ty. Tuy nhiên, lượng khí này phát sinh không nhiều. Mặt khác, hệ thống thoát nước và xử lý nước thải được thiết kế kín nên khả năng ảnh hưởng đến môi trường là không đáng kể."}\n{"id": "B-A-24f40a93ed9b5a47", "claim": "Nếu chất thải nguy hại không được thải bỏ đúng cách sẽ gây nguy hại tới môi trường và gây ra những mối nguy hại tiềm ẩn đến sức khỏe cộng đồng.", "evidence": "Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật. PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật. Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP. RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại. Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện điện tử, bùn hóa DANGER chất..., các bước thực hiện như sau: chất thải nguy hại được phân loại và cho vào túi dán nhãn và cho vào thùng rác, khi thùng đầy ch"}\n{"id": "B-A-3bc2c78a546a1f9e", "claim": "Tại PNJ, chất thải rắn chủ yếu phát sinh từ hoạt động của PNJP, bao gồm: Chất thải rắn được thu gom để xử lý triệt để nên hầu như không gây ô nhiễm cho môi trường.", "evidence": "Chất thải rắn sinh hoạt chủ yếu chứa nhiều chất hữu cơ dễ phân hủy, nếu không được thu gom và xử lý đúng quy định sẽ sinh ra mùi hôi thối gây ảnh hưởng đến sức khỏe người lao động, gây ô nhiễm môi trường không khí xung quanh. Rác thải nguy hại là rác thải có chứa yếu tố độc hại, phóng xạ, lây nhiễm, dễ cháy, dễ nổ, gây ăn mòn, gây ngộ độc hoặc có đặc tính nguy hại khác. Nếu chất thải nguy hại không được thải bỏ đúng cách sẽ gây nguy hại tới môi trường và gây ra những mối nguy hại tiềm ẩn đến sức khỏe cộng đồng. CHẤT THẢI RẮN SINH HOẠT CỦA PNJP Nhằm góp phần giảm lượng rác thải cần xử lý, tăng lượng rác thải có thể tái chế, PNJ đã tăng cuờng truyền thông hoạt động phân loại rác thải đúng cách"}\n{"id": "B-A-6856cc9c34b3d46e", "claim": "Rác thải là nguồn tài nguyên tái chế để có thể phục vụ cho cuộc sống.", "evidence": "thùng này chúng ta sẽ bỏ rác hữu cơ dễ phân hủy GREEN và rác loại khác. Với rác hữu cơ dễ phân hủy chúng ta sẽ cho vào túi rác và cho vào thùng. Chất thải có khả năng tái sử dụng: - Thùng màu cam: chứa rác thải tái chế, chúng ta sẽ bỏ các vỏ chai nước uống, giấy loại, bán lại cho các đơn vị tái chế chất thải có chức năng trong thành phố. Năng lượng Phạm vi 1 Nguồn đốt cố định Nguồn đốt di động Nguồn môi chất lạnh Hệ thống xử lý nước thải Phòng cháy chữa cháy Đốt rác thải (phân kim) cụ tính toán"}\n{"id": "B-A-ce0d045a29195a3a", "claim": "Công ty thực hiện lưu trữ và phân loại chất thải rắn sinh hoạt vào các thùng chứa rác sinh hoạt chuyên dụng bằng nhựa màu xanh có nắp đậy kín và được đặt tại một khu vực riêng biệt tránh gây mùi cho xung quanh.", "evidence": "RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại. Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện điện tử, bùn hóa DANGER chất..., các bước thực hiện như sau: chất thải nguy hại được phân loại và cho vào túi dán nhãn và cho vào thùng rác, khi thùng đầy chúng ta đem xuống kho rác thải nguy hại và bàn giao cho Bộ phận An toàn Môi trường. THU GOM, PHÂN LOẠI TẬP KẾT, LƯU TRỮ Chất thải rắn sinh hoạt Phát sinh chủ yếu từ khu vực ăn uống, văn phòng, bao gồm: Rác thải hữu cơ dễ phân hủy từ hoạt động ăn uống, các cành cây, lá cây trong khuôn viên công ty, rác thải tái chế như các lon nước uống, các bao bì nhựa hay kim "}\n{"id": "B-A-0e195188f4ed0e0f", "claim": "Việc phân loại rác thải tại PNJ rất được chú trọng, có 3 thùng loại thùng rác với màu sắc riêng biệt để CBCNV phân loại rác đúng cách: - Thùng màu xanh: chứa rác sinh hoạt.", "evidence": "thùng này chúng ta sẽ bỏ rác hữu cơ dễ phân hủy GREEN và rác loại khác. Với rác hữu cơ dễ phân hủy chúng ta sẽ cho vào túi rác và cho vào thùng. Chất thải có khả năng tái sử dụng: - Thùng màu cam: chứa rác thải tái chế, chúng ta sẽ bỏ các vỏ chai nước uống, giấy loại, bán lại cho các đơn vị tái chế chất thải có chức năng trong thành phố. CHẤT THẢI RẮN SINH HOẠT CỦA PNJP Nhằm góp phần giảm lượng rác thải cần xử lý, tăng lượng rác thải có thể tái chế, PNJ đã tăng cuờng truyền thông hoạt động phân loại rác thải đúng cách để góp phần bảo vệ môi trường. Trong đó mấu chốt vẫn là khâu phân loại rác thải tại nguồn, nếu giải quyết tốt phân loại rác thì quá trình thu gom cũng như xử lý sẽ được thực hi"}\n{"id": "B-A-ea373403682c7fb9", "claim": "Chất thải có khả năng tái sử dụng: - Thùng màu cam: chứa rác thải tái chế, chúng ta sẽ bỏ các vỏ chai nước uống, giấy loại, bán lại cho các đơn vị tái chế chất thải có chức năng trong thành phố.", "evidence": "THU GOM, PHÂN LOẠI TẬP KẾT, LƯU TRỮ Chất thải rắn sinh hoạt Phát sinh chủ yếu từ khu vực ăn uống, văn phòng, bao gồm: Rác thải hữu cơ dễ phân hủy từ hoạt động ăn uống, các cành cây, lá cây trong khuôn viên công ty, rác thải tái chế như các lon nước uống, các bao bì nhựa hay kim loại không chưa thành phần nguy hại, giấy loại, các thùng carton, ... BÀN GIAO ĐƠN VỊ CHỨC NĂNG THU GOM & XỬ LÝ RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại. Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện điện tử, bùn hóa DANGER chất..., các bước thực hiện như sau: chất thải nguy hại được phân loại và cho vào túi dán nhãn và cho và"}\n{"id": "B-A-fd4cf2efe6191457", "claim": "RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại.", "evidence": "Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật. PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật. Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP. THU GOM, PHÂN LOẠI TẬP KẾT, LƯU TRỮ Chất thải rắn sinh hoạt Phát sinh chủ yếu từ khu vực ăn uống, văn phòng, bao gồm: Rác thải hữu cơ dễ phân hủy từ hoạt động ăn uống, các cành cây, lá cây trong khuôn viên công ty, rác thải tái chế như các lon nước uống, các bao bì nhựa hay kim loại không chưa thành phần nguy hại, giấy loại, các thùng c"}\n{"id": "B-A-18224a708fd84986", "claim": "Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện điện tử, bùn hóa DANGER chất..., các bước thực hiện như sau: chất thải nguy hại được phân loại và cho vào túi dán nhãn và cho vào thùng rác, khi thùng đầy chúng ta đem xuống kho rác thải nguy hại và bàn giao cho Bộ phận An toàn Môi trường.", "evidence": "thùng này chúng ta sẽ bỏ rác hữu cơ dễ phân hủy GREEN và rác loại khác. Với rác hữu cơ dễ phân hủy chúng ta sẽ cho vào túi rác và cho vào thùng. Chất thải có khả năng tái sử dụng: - Thùng màu cam: chứa rác thải tái chế, chúng ta sẽ bỏ các vỏ chai nước uống, giấy loại, bán lại cho các đơn vị tái chế chất thải có chức năng trong thành phố. Công ty thực hiện lưu trữ và phân loại chất thải rắn sinh hoạt vào các thùng chứa rác sinh hoạt chuyên dụng bằng nhựa màu xanh có nắp đậy kín và được đặt tại một khu vực riêng biệt tránh gây mùi cho xung quanh. Việc phân loại rác thải tại PNJ rất được chú trọng, có 3 thùng loại thùng rác với màu sắc riêng biệt để CBCNV phân loại rác đúng cách: - Thùng màu xa"}\n{"id": "B-A-669c3100651f1cdf", "claim": "Bùn thải từ bể tự hoại và hệ thống xử lý nước thải tập trung Do hoạt động trong lĩnh vực sản xuất, gia công vàng, bạc, PNJP phát sinh một số chất thải nguy hại, chủ yếu bao gồm: Bùn thải từ hệ thống xử lý nước thải tập trung bao gồm bùn sinh học từ bể lắng sinh học và bùn từ bể lắng hóa lý.", "evidence": "- Các loại hóa chất thải: hóa chất phòng thí nghiệm thải, xăng dầu, dầu động cơ hộp số bôi trơn thải, ... - Bùn thải từ hệ thống xử lý nước thải. Bùn sinh học được tuần hoàn lại các bể hiếu khí, kị khí để tiếp tục xử lý nước và một phần bùn dư được chứa tại bể chứa bùn sinh học sau đó được đơn vị có chức năng thu gom và xử lý chất thải nguy hại theo đúng luật định. Bùn thải từ cụm xử lý hóa lý hàng ngày được thu gom bằng dàn lọc bùn sau đó được đưa vào kho chứa chất thải nguy hại, định kỳ hàng tháng sẽ bàn giao cho đơn vị chức năng thu gom và xử lý chất thải nguy hại theo đúng quy định. Trong quá trình hoạt động sản xuất của PNJP Bụi và khí thải phát sinh chủ yếu từ: .Tổ nấu .Phòng nấu Từ ph"}\n{"id": "B-A-f9ecc675d7fcfbbd", "claim": "Bùn thải từ cụm xử lý hóa lý hàng ngày được thu gom bằng dàn lọc bùn sau đó được đưa vào kho chứa chất thải nguy hại, định kỳ hàng tháng sẽ bàn giao cho đơn vị chức năng thu gom và xử lý chất thải nguy hại theo đúng quy định.", "evidence": "Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật. PNJP đã hợp đồng với các đơn vị có chức năng để thu gom, vận chuyển và xử lý các chất thải nguy hại trên theo đúng quy định của luật. Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP. RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại. Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện điện tử, bùn hóa DANGER chất..., các bước thực hiện như sau: chất thải nguy hại được phân loại và cho vào túi dán nhãn và cho vào thùng rác, khi thùng đầy ch"}\n{"id": "B-A-76d3cb5fd1d8e9ac", "claim": "Chất thải nguy hại được phân loại, thu gom và lưu trữ vào kho chứa chất thải nguy hại đáp ứng đầy đủ yêu cầu của luật.", "evidence": "RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại. Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện điện tử, bùn hóa DANGER chất..., các bước thực hiện như sau: chất thải nguy hại được phân loại và cho vào túi dán nhãn và cho vào thùng rác, khi thùng đầy chúng ta đem xuống kho rác thải nguy hại và bàn giao cho Bộ phận An toàn Môi trường. - Các loại hóa chất thải: hóa chất phòng thí nghiệm thải, xăng dầu, dầu động cơ hộp số bôi trơn thải, ... - Bùn thải từ hệ thống xử lý nước thải. Bùn sinh học được tuần hoàn lại các bể hiếu khí, kị khí để tiếp tục xử lý nước và một phần bùn dư được chứa tại bể chứa bùn sinh học "}\n{"id": "B-A-ed95e93907945d6a", "claim": "Ngoài ra, PNJP đã tiến hành đăng ký Sổ Chủ nguồn thải chất thải nguy hại với Sở Tài nguyên và Môi trường TP.", "evidence": "Chất thải rắn sinh hoạt chủ yếu chứa nhiều chất hữu cơ dễ phân hủy, nếu không được thu gom và xử lý đúng quy định sẽ sinh ra mùi hôi thối gây ảnh hưởng đến sức khỏe người lao động, gây ô nhiễm môi trường không khí xung quanh. Rác thải nguy hại là rác thải có chứa yếu tố độc hại, phóng xạ, lây nhiễm, dễ cháy, dễ nổ, gây ăn mòn, gây ngộ độc hoặc có đặc tính nguy hại khác. Nếu chất thải nguy hại không được thải bỏ đúng cách sẽ gây nguy hại tới môi trường và gây ra những mối nguy hại tiềm ẩn đến sức khỏe cộng đồng. RECYCLE các chất thải bằng nhựa, kim loại không chứa thành phần nguy hại. Chất thải không thể tận dụng được: - Thùng màu vàng: chỉ chứa rác nguy hại như sáp thải, ắc quy, linh kiện đi"}\n{"id": "B-A-ceff7d63dbed390a", "claim": "Khi hoạt động, các phương tiện vận tải với nhiên liệu tiêu thụ chủ yếu là xăng và dầu Diesel sẽ thải ra môi trường một lượng khói thải chứa các chất ô nhiễm không khí như: NOx, CxHy, CO, CO2...", "evidence": "Chất thải rắn sinh hoạt chủ yếu chứa nhiều chất hữu cơ dễ phân hủy, nếu không được thu gom và xử lý đúng quy định sẽ sinh ra mùi hôi thối gây ảnh hưởng đến sức khỏe người lao động, gây ô nhiễm môi trường không khí xung quanh. Rác thải nguy hại là rác thải có chứa yếu tố độc hại, phóng xạ, lây nhiễm, dễ cháy, dễ nổ, gây ăn mòn, gây ngộ độc hoặc có đặc tính nguy hại khác. Nếu chất thải nguy hại không được thải bỏ đúng cách sẽ gây nguy hại tới môi trường và gây ra những mối nguy hại tiềm ẩn đến sức khỏe cộng đồng. Phương tiện giao thông hoạt động ra vào khu vực công ty bao gồm các loại xe (xe đạp, xe hai bánh, gắn máy, xe bốn bánh). Tuy nhiên, lượng xe ra vào chiếm đa số là xe gắn máy, xe ôtô c"}\n{"id": "B-A-e93f57b58cc34260", "claim": "Thường xuyên vệ sinh hành lang, văn phòng, xưởng sản xuất và thu gom các chất thải rắn để hạn chế tối đa bụi phát tán vào không khí.", "evidence": "Tại PNJ, chất thải rắn chủ yếu phát sinh từ hoạt động của PNJP, bao gồm: Chất thải rắn được thu gom để xử lý triệt để nên hầu như không gây ô nhiễm cho môi trường. CHẤT THẢI RẮN SINH HOẠT QUY TRÌNH THU GOM VÀ XỬ LÝ THU GOM, PHÂN LOẠI TẬP KẾT, LƯU TRỮ Chất thải rắn sinh hoạt Phát sinh chủ yếu từ khu vực ăn uống, văn phòng, bao gồm: Rác thải hữu cơ dễ phân hủy từ hoạt động ăn uống, các cành cây, lá cây trong khuôn viên công ty, rác thải tái chế như các lon nước uống, các bao bì nhựa hay kim loại không chưa thành phần nguy hại, giấy loại, các thùng carton, ... BÀN GIAO ĐƠN VỊ CHỨC NĂNG THU GOM & XỬ LÝ"}\n{"id": "B-A-788568983960a402", "claim": "Ý thức được tác hại của các vật dụng dùng một lần đối với môi trường, từ đầu năm 2021, PNJ đã phát động phong trào \\"Nói không với vật dụng nhựa dùng một lần\\".", "evidence": "PHONG TRÀO “NÓI KHÔNG VỚI VẬT DỤNG NHỰA DÙNG MỘT LẦN\\" Tổng 87.201 48.267 33.024 KG Tổng trọng lượng chất thải nguy hại phân chia theo các Tái chế 2.354 5.306 4.235 KG Đốt - chôn lấp CBCNV, vừa đảm bảo vệ sinh, vừa nâng cao ý thức bảo vệ môi trường thông qua tiết chế sử dụng các vật dụng nhựa dùng một lần. Tại các phòng họp và các không gian chung, PNJ thay thế các chai nước uống tiện dụng bằng nhựa bằng các bình thủy tinh, cốc thủy tinh và cây lọc nước. Phong trào đã nhận được sự hưởng ứng tích cực từ toàn thể CBCNV PNJ và hình thành được nét đẹp trong văn hóa của"}\n{"id": "B-A-9296c9a4b6a18054", "claim": "Công ty đã đề ra quy định hạn chế CBCNV mang các loại túi bóng, dụng cụ đựng thực phẩm, nước uống dùng một lần như chai, đũa, bát nhựa ...", "evidence": "CBCNV, vừa đảm bảo vệ sinh, vừa nâng cao ý thức bảo vệ môi trường thông qua tiết chế sử dụng các vật dụng nhựa dùng một lần. Tại các phòng họp và các không gian chung, PNJ thay thế các chai nước uống tiện dụng bằng nhựa bằng các bình thủy tinh, cốc thủy tinh và cây lọc nước. Phong trào đã nhận được sự hưởng ứng tích cực từ toàn thể CBCNV PNJ và hình thành được nét đẹp trong văn hóa của THU GOM, PHÂN LOẠI TẬP KẾT, LƯU TRỮ Chất thải rắn sinh hoạt Phát sinh chủ yếu từ khu vực ăn uống, văn phòng, bao gồm: Rác thải hữu cơ dễ phân hủy từ hoạt động ăn uống, các cành cây, lá cây trong khuôn viên công ty, rác thải tái chế như các lon nước uống, các bao bì nhựa hay kim loại không chưa thành phần nguy "}\n{"id": "B-A-e33f2c559ffe1627", "claim": "Những trường hợp vi phạm ăn uống tại nơi làm việc hoặc mang các vật dụng nhựa dùng một lần vào công ty sẽ bị nhắc nhở.", "evidence": "ăn uống tại nơi làm việc; CBCNV có thể đăng ký bữa ăn tại canteen của Công ty, tự mang cơm và ăn tại can- teen hoặc ăn ngoài, vừa tạo không gian làm việc sạch sẽ, vệ sinh, vừa đảm bảo việc thu gom, xử lý rác thải từ hoạt động ăn uống tập trung. 200 200 200 200 QCVN 19:2009/BTNMT (Cột B) Ngoài ra, đơn vị còn trồng cây xanh trong khuôn viên để tạo cảnh quan và hạn chế bụi và khí thải phát tán ra môi trường không khí xung quanh. Định kỳ, Bộ phận An Toàn - Môi Trường tiến hành phối hợp với các đơn vị chức năng để tiến hành đo đạc cũng như phân tích khí thải khách quan hàng năm. Ý thức được tác hại của các vật dụng dùng một lần đối với môi trường, từ đầu năm 2021, PNJ đã phát động phong trào \\"Nói"}\n{"id": "B-A-ff9fbdd8f09b87e4", "claim": "CBCNV, vừa đảm bảo vệ sinh, vừa nâng cao ý thức bảo vệ môi trường thông qua tiết chế sử dụng các vật dụng nhựa dùng một lần.", "evidence": "Công ty đã đề ra quy định hạn chế CBCNV mang các loại túi bóng, dụng cụ đựng thực phẩm, nước uống dùng một lần như chai, đũa, bát nhựa ... vào Công ty. Những trường hợp vi phạm ăn uống tại nơi làm việc hoặc mang các vật dụng nhựa dùng một lần vào công ty sẽ bị nhắc nhở. Chỉ tiêu Nguồn thải Lưu lượng P(m3/h) Bụi SẴN SÀNG TÂM THẾ VƯỢT THỬ THÁCH VƯƠN TẦM BẢO VỆ MÔI TRƯỜNG BẢO VỆ MÔI TRƯỜNG GRI 301 Sử dụng TIẾT KIỆM VÀ HIỆU QUẢ NGUYÊN VẬT LIỆU"}\n{"id": "B-A-07d35a40bfcc9d69", "claim": "Nhờ đó, nguồn lực trẻ này được kỳ vọng sẽ viết tiếp hành trình phát triển bền vững cho PNJ.", "evidence": "NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN KHÔNG SỬ DỤNG LAO ÐỘNG TRẺ EM ÐẦU TƯ VÀO ÐÀO TẠO NGUỒN NHÂN LỰC NỀN TẢNG CỦA NHỮNG KỲ TÍCH VƯỢT RANH GIỚI TỪ SỨC MẠNH NIỀM TIN VÌ MỘT MÔI TRƯỜNG PHÁT TRIỂN BỀN VỮNG CON NGƯỜI LÀM TRUNG TÂM CHO SỰ PHÁT TRIỂN KIẾN LẬP KỲ TÍCH HÀNH TRÌNH PHÁT TRIỂN BỀN VỮNG"}\n{"id": "B-A-33724c32fe55a97e", "claim": "GRI 102-2 23-24 Vai trò của cấp quản lý cao nhất trong lập báo cáo phát triển bền vững GRI 102-32 15-16 Địa điểm của trụ sở chính GRI 102-3 Các địa điểm hoạt động GRI 102-4", "evidence": "GRI 102-26 31-32 Vai trò của cấp quản lý cao nhất trong việc thiết lập các mục đích, giá trị và chiến lược GRI 201: Hiệu quả hoạt động kinh tế Kiến thức tổng hợp của cấp quản lý cao nhất GRI 102-27 27-30 Đánh giá hiệu quả hoạt động của cấp quản lý cao nhất GRI 102-45 25-26 Tuyên bố của người ra quyết định cấp cao GRI 102-14 15-16, 41-42 Các tác động, rủi ro và cơ hội chính GRI 102-15 35-36 Danh sách các chủ đề trọng yếu"}\n{"id": "B-A-296fbafb9d4ddf05", "claim": "Từ khi thành lập đến nay, Tập đoàn đã liên tục mở rộng hệ sinh thái thép từ thượng nguồn đến hạ nguồn, nâng cao năng lực sản xuất, không ngừng tập trung nghiên cứu nâng cao chất lượng sản phẩm, cho ra các mác sản phẩm mới ngày càng chuyên sâu, phức tạp, đáp ứng rộng hơn", "evidence": "Nhiều đối tác đã đồng hành cùng Tập đoàn ngay từ những ngày đầu thành lập. Không ngừng nghiên cứu đổi mới, phát triển sản phẩm và nâng cao chất lượng liên tục, đáp ứng các tiêu chuẩn quốc tế khắt khe. suất sản phẩm, nghiên cứu và mở rộng dải sản phẩm, nâng tỷ trọng các sản phẩm phức tạp và có hàm lượng công nghệ cao để đáp ứng nhu cầu đa dạng của khách hàng, đồng thời giảm thiểu tác động môi trường."}\n{"id": "B-A-7c093fae00aa5ff8", "claim": "Chúng tôi nhận thức về trách nhiệm của mình trong việc tham gia lộ trình giảm phát thải quốc gia và đang tích cực thực hành các giải pháp phù hợp", "evidence": "103-5 Giảm tiêu thụ năng lượng Hành động giảm thiểu và thích ứng với biến đổi khí hậu 102-4 Mục tiêu và tiến độ giảm phát thải khí nhà kính Đây là báo cáo năm đầu tiên của chúng tôi, lộ trình giảm phát thải chi tiết vẫn đang được BỀN VỮNG CỦA LIÊN HỢP QUỐC (UN SDGS): BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Chúng tôi hướng tới giảm phát thải khí nhà kính trong các quy trình sản xuất, đóng góp vào mục tiêu của Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát"}\n{"id": "B-A-792e3e12d3c80130", "claim": "với điều kiện kinh doanh, đặc biệt trong lĩnh vực thép, để đồng hành cùng Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050.", "evidence": "doanh. Chúng tôi cam kết nỗ lực đồng hành cùng Chính phủ Việt Nam hướng tới mục tiêu Net Zero 2050. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Tuân thủ quy định pháp luật: Tất cả các công ty thành viên của Tập đoàn thuộc ứng với Biến đổi Khí hậu” là ưu tiên phát triển bền vững quan trọng. Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm phát thải khí nhà kính phù hợp với điều kiện kinh"}\n{"id": "B-A-e3a34be81eb7933a", "claim": "Tập đoàn cũng đang triển khai ứng dụng các nguồn năng lượng tái tạo, điện mặt trời áp mái để", "evidence": "để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. vực khắt khe về tiêu chuẩn môi trường. Quản lý môi trường TRIỂN KHAI ĐIỆN MẶT TRỜI ÁP MÁI TẠI CÁC NHÀ MÁY 38"}\n{"id": "B-A-c36a157ad6e2648d", "claim": "ESG Hòa Phát tích hợp các mục tiêu phát triển bền vững (SDGs) vào chiến lược kinh doanh, phù", "evidence": "CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT 4.3 CÁC CHỦ ĐỀ TRỌNG YẾU CHỦ ĐỀ TRỌNG YẾU MỤC TIÊU CHIẾN LƯỢC ĐÓNG GÓP CHO CÁC MỤC TIÊU PHÁT TRIỂN"}\n{"id": "B-A-621e4a149da39001", "claim": "hợp với định hướng phát triển xanh của Chính phủ Việt Nam và cam kết trung hòa carbon đến năm 2050.", "evidence": "doanh. Chúng tôi cam kết nỗ lực đồng hành cùng Chính phủ Việt Nam hướng tới mục tiêu Net Zero 2050. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Tuân thủ quy định pháp luật: Tất cả các công ty thành viên của Tập đoàn thuộc với điều kiện kinh doanh, đặc biệt trong lĩnh vực thép, để đồng hành cùng Chính phủ Việt Nam hiện thực hóa cam kết đạt mức phát thải ròng bằng không (Net Zero) vào năm 2050. Tập đoàn hướng tới từng bước chuyển đổi mô hình sản xuất và quản trị, gắn liền với cải"}\n{"id": "B-A-c7cbbae54786f4b4", "claim": "tại Khu liên hợp Gang thép Hòa Phát Dung Quất Tái chế, tái sử dụng chất thải rắn MÔI TRƯỜNG nhóm thép và nông nghiệp đạt trên 99% Điểm nổi bật về", "evidence": "TABLE | 3.1 ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025 Hoàn thành kiểm kê khí nhà kính theo tiêu 90% Tự chủ nhu cầu điện chuẩn ISO 14064-1:2018 cho toàn Tập tại Khu liên hợp Gang thép Hòa Phát đoàn và được BSI thẩm định số liệu . Dung Quất 98% Tái sử dụng nước Tái chế, tái sử dụng chất thải rắn MÔI TRƯỜNG tại Khu liên hợp Gang thép Hòa Phát 99% nhóm thép và nông nghiệp đạt trên Dung Quất Số giờ đào tạo trung bình cho mỗi nhân 13.000 tỷ đồng Đóng góp gần vào 38,52 giờ ngân sách Nhà nước viên là mỗi năm 120 tỷ đồng Đóng góp hơn cho các chương trình phúc lợi xã hội. XÃ HỘI 22% thành viên Tỷ lệ phụ nữ ở các cấp quản lý trong toàn Hội đồng Quản trị là thành viên độc lập. 35,85% Tập đoàn đạt vào năm"}\n{"id": "B-A-c26f442173cc01e7", "claim": "phát triển bền vững năm 2025 Đóng góp gần 13.000 tỷ đồng vào ngân sách Nhà nước Số giờ đào tạo trung bình cho mỗi nhân Giải thưởng và vinh danh năm 2025 viên là 38,52 giờ mỗi năm", "evidence": "TABLE | 3.1 ĐIỂM NỔI BẬT VỀ PHÁT TRIỂN BỀN VỮNG 2025 Hoàn thành kiểm kê khí nhà kính theo tiêu 90% Tự chủ nhu cầu điện chuẩn ISO 14064-1:2018 cho toàn Tập tại Khu liên hợp Gang thép Hòa Phát đoàn và được BSI thẩm định số liệu . Dung Quất 98% Tái sử dụng nước Tái chế, tái sử dụng chất thải rắn MÔI TRƯỜNG tại Khu liên hợp Gang thép Hòa Phát 99% nhóm thép và nông nghiệp đạt trên Dung Quất Số giờ đào tạo trung bình cho mỗi nhân 13.000 tỷ đồng Đóng góp gần vào 38,52 giờ ngân sách Nhà nước viên là mỗi năm 120 tỷ đồng Đóng góp hơn cho các chương trình phúc lợi xã hội. XÃ HỘI 22% thành viên Tỷ lệ phụ nữ ở các cấp quản lý trong toàn Hội đồng Quản trị là thành viên độc lập. 35,85% Tập đoàn đạt vào năm"}\n{"id": "B-A-c76483a6a85fc41a", "claim": "Điểm nổi bật về phát triển bền vững năm 2025 năm 2025).", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12"}\n{"id": "B-A-8e401b5e10aa0a93", "claim": "Điểm nổi bật về phát triển bền vững năm 2025 1992 1995 1996", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10"}\n{"id": "B-A-24a58d552e08b7ed", "claim": "Điểm nổi bật về phát triển bền vững năm 2025 2015 2016 2017 2019 2020 2021 2022 2023 2024", "evidence": "2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 dự kiến 2024 65 2020 2021 2022 2023 2024 2014 2015 2016 2017 2018 2019 2020 2021 2022 2023 2024 Kế hoạch sản xuất kinh doanh Vốn chủ sở hữu Lợi nhuận / vốn chủ sở hữu (ROE)"}\n{"id": "B-A-14cd1d6a9edba673", "claim": "Phát Hưng Yên, đánh dấu bước phát triển mới tối ưu hệ sinh thái thép Hòa Quất, triển khai Khu Liên nóng/năm, tổng vốn đầu tư dự kiến 85.000", "evidence": "QUÝ 1/2022 Khởi công dự án Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 2, công suất 5,6 triệu tấn thép cuộn cán nóng/năm, tổng vốn đầu tư dự kiến 85.000 đầu khu vực Đông Nam Á. Năm 2022, Hòa Phát tiếp tục triển khai dự án Khu liên hợp gang thép Hòa Phát Dung Quất 2 (nằm cạnh dự án Dung Quất 1) có quy mô trên 280 ha, tổng vốn đầu tư là 85.000 tỷ đồng, công suất thiết kế của dự án là 5,6 triệu tấn thép cuộn cán"}\n{"id": "B-A-481738e2baf587e9", "claim": "2 6 Điểm nổi bật về phát triển bền vững năm 2025 qua cách chúng tôi lựa chọn nguồn cung, tổ chức sản xuất, hợp tác cùng đối tác, chăm lo cho người", "evidence": "CÙNG CÁC CÔNG TY THÀNH VIÊN Điểm nổi bật về phát triển bền vững năm 2025 THÉP NÔNG NGHIỆP BẤT ĐỘNG SẢN ĐIỆN MÁY GIA DỤNG Giải thưởng 3 6 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT PHÁT TRIỂN CON NGƯỜI 6.1 | Sức khoẻ và an toàn lao động 3.1 | Điểm nổi bật về phát triển bền vững năm 2025 6.2 | Quản trị nguồn nhân lực 3.2 | Giải thưởng và vinh danh năm 2025"}\n{"id": "B-A-ca416809d376ffc6", "claim": "Chúng tôi nỗ lực hoạt động hiệu quả, tiết kiệm và có trách nhiệm, góp phần ứng phó biến đổi khí hậu, thúc đẩy kinh tế tuần hoàn và bảo vệ tài", "evidence": "đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ. Trọng tâm chiến lược tập trung vào ba trụ cột ESG chính: sử dụng tài nguyên có trách nhiệm; thúc đẩy kinh tế tuần hoàn và bảo vệ hệ sinh thái, cụ thể bằng cách: Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh"}\n{"id": "B-A-a22a05f0a313c153", "claim": "nguyên chung, Biến chất thải sản xuất thành các nguyên liệu có giá trị cho các ngành khác, góp phần giảm khai thác tài nguyên thô.", "evidence": "phế, quặng sắt đã hoàn nguyên – những nguyên liệu có giá thành cao và nguồn cung hạn chế, phù hợp với các quốc gia phát triển, nơi ngành thép đã đi qua nhiều vòng đời tái chế. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Hiện nay, các biện pháp giảm phát thải trong sản xuất thép từ quặng Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế."}\n{"id": "B-A-47ff3bb452dd8a1b", "claim": "Xây dựng và hạ tầng 52% Phương tiện vận tải khác Điểm nổi bật về phát triển bền vững năm 2025 Công cụ máy móc Xương sống của hạ tầng: Là vật liệu cốt lõi xây dựng cầu đường, cảng", "evidence": "Điểm nổi bật về phát triển bền vững năm 2025 GIÁ TRỊ CỐT LÕI Giá trị cốt lõi của Tập đoàn Hòa Phát là triết lý Hòa hợp cùng phát triển. Điều này thể hiện trong mối quan hệ giữa các cán bộ công CÙNG CÁC CÔNG TY THÀNH VIÊN Điểm nổi bật về phát triển bền vững năm 2025 THÉP NÔNG NGHIỆP BẤT ĐỘNG SẢN ĐIỆN MÁY GIA DỤNG Giải thưởng"}\n{"id": "B-A-729ece7e324b5295", "claim": "10% 2% Thiết bị cơ khí 16% Chuyển đổi năng lượng: là nguyên liệu quan trọng trong quá trình chuyển đổi năng lượng tái tạo (tua-bin gió, điện mặt trời), điện khí hóa, giao thông công cộng và nền kinh tế hydro.", "evidence": "nơi có diện tích bề mặt lớn và khả năng tiếp nhận bức xạ nhiệt dồi dào. Các tấm pin này đóng vai trò hấp thụ năng lượng từ ánh sáng mặt trời, chuyển hóa thành dòng điện một chiều thông qua hiệu ứng quang điện, sau đó được các bộ biến để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy."}\n{"id": "B-A-784b6a36deb94666", "claim": "gang lỏng, sau đó tiếp tục được tinh luyện trong lò thổi oxy nhằm loại bỏ tạp chất và điều chỉnh hàm lượng carbon để trở thành thép thành phẩm.", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 Ngoài ra, bộ phận R&D của chúng tôi tại các nhà máy thép liên tục thực hiện nghiên cứu để nâng cao chất lượng thép và mở rộng dải sản phẩm bằng cách cải thiện độ bền, tính chất cơ học và khả năng chống ăn mòn - đặc biệt là trong việc phát triển các loại thép chịu được thời tiết, các loại thép chất thép hàm lượng carbon thấp sản xuất các sản phẩm như vỏ đồ hộp, đồ gia dụng, kết cấu thép, HRC cho sản xuất vỏ container cường độ cao, kháng thời tiết... Khi Dự án Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 2 hoàn thành vào"}\n{"id": "B-A-73a79787d602a0a8", "claim": "Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép.", "evidence": "HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên đoạn nào của vòng đời. Phát thải carbon phát sinh nhiều nhất ở giai đoạn đầu tiên khi khử quặng sắt để tạo ra thép nguyên chất và giảm đi đáng kể ở giai đoạn tái chế phế liệu thành thép để tạo vòng đời tiếp theo. Việc lựa chọn sản xuất thép theo công nghệ nào"}\n{"id": "B-A-bc929d96a5aa37ce", "claim": "Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có", "evidence": "như quặng sắt, than đá và đá vôi để khởi động quy trình sản xuất vật liệu mới. Giai Khác với nhiều loại vật liệu khác, thép là loại vật liệu có khả năng tái chế 100% mà không đoạn này thường gắn với công nghệ sản xuất bằng lò cao (BOF). Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai"}\n{"id": "B-A-dfb91a6eaac5f661", "claim": "Điểm nổi bật về phát triển bền vững năm 2025 một khả năng đặc biệt là tái chế 100% mà không làm mất đi đặc tính ban đầu.", "evidence": "TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 10 TABLE BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 3 PHẦN TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Điểm nổi bật về phát triển bền vững năm 2025 Giải thưởng và vinh danh năm 2025 Thành viên của các Hiệp hội ngành nghề Hoạt động kinh doanh tại Tập đoàn 12"}\n{"id": "B-A-be1f56c521c20020", "claim": "trung bình (Scrap-EAF): 0,70 tấn CO2 / tấn thép thô Điểm nổi bật về phát triển bền vững năm 2025 Cường độ phát thải CO2 Cường độ phát thải CO2", "evidence": "trung bình (BF-BOF): 2,32 tấn CO2 / tấn thép thô trung bình (DRI-EAF): 1,43 tấn CO2 / tấn thép thô Tại Hòa Phát, phần lớn thép thô sản xuất từ công nghệ lò cao (KLH Gang thép Hòa Phát Tỷ trọng thép EAF trong sản lượng toàn cầu BÁO CÁO PHÁT TRIỂN BỀN VỮNG 70,4% 29,1% 2025 PHẦN 3 TỔNG QUAN VỀ TẬP ĐOÀN HÒA PHÁT Cường độ phát thải CO2"}\n{"id": "B-A-13fd05789b86f5aa", "claim": "Hướng tới mục tiêu phát triển bền vững và giảm dấu chân carbon, Hòa Thành viên của các Hiệp hội ngành nghề Phát đang chủ động gia tăng dần tỷ lệ sử dụng thép phế liệu trong cả hai quy trình sản", "evidence": "chú trọng hai nhóm ngành lớn nhất: thép và nông nghiệp. Hòa Phát hướng tới duy trì năng lực cạnh tranh bền vững trong quá trình chuyển đổi hướng tới trung hòa carbon, đồng thời tiếp tục khẳng định cam kết về chất lượng sản phẩm dẫn & TỐI ƯU NĂNG LƯỢNG Quản lý môi trường TĂNG SỬ DỤNG THÉP PHẾ LIỆU Tập đoàn đang tăng cường tỷ lệ sử dụng thép phế và HBI trong lò thổi. Biện pháp này giúp giảm mạnh nhu cầu hoàn nguyên quặng, giảm tiêu"}\n{"id": "B-A-3acac025c023887d", "claim": "các công ty thành viên cùng các ban chức năng như ban Tài chính, Pháp chế, Nhân sự, Truyền thông của Tập đoàn nhằm điều phối các hoạt động ESG và tích hợp các mục tiêu phát triển bền vững vào hoạt động vận hành hàng ngày của doanh", "evidence": "nghiệp. CÁC BAN CHỨC NĂNG CỦA TẬP ĐOÀN: ĐIỀU PHỐI Đầu mối tổng hợp báo cáo Phát triển bền vững Điều phối chung các hoạt động Phát triển bền vững tài chính, pháp chế, ... cho các Công ty thành viên. Quan hệ Cổ đông Việc áp dụng công nghệ vào hoạt động quản trị nhân sự cũng được chú trọng. Hệ thống quản trị nhân sự đã chính thức đi vào vận hành trên toàn Tập đoàn từ tháng"}\n{"id": "B-A-5daf9c7caaabe29c", "claim": "Nhóm nghiên cứu đã sắp xếp các mối quan tâm ESG lại thành 12 chủ đề liên quan đến hoạt động của Tập đoàn để đưa vào bước ưu tiên tiếp theo thông qua tham vấn các", "evidence": "Mức độ tác động đến doanh nghiệp được xác định dựa trên ảnh hưởng của các vấn đề ESG đối với hoạt động kinh doanh, tài chính và danh tiếng của Tập đoàn. Tầm quan trọng này được đánh giá thông qua mức độ quan tâm của các bên liên quan để xác định mức độ ưu tiên. Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định. Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc"}\n{"id": "B-A-0261e5667ddfc81a", "claim": "ít carbon, sử dụng tài nguyên hiệu quả và vận hành Xây dựng môi trường làm việc an toàn, Tăng cường quản trị minh bạch, Mục tiêu công bằng và thúc đẩy sự phát triển", "evidence": "LÀNH MẠNH VÀ HÒA NHẬP Hòa Phát xây dựng môi trường làm việc an toàn, lành mạnh, tôn trọng quyền con người, thực hành lao động công bằng và chú trọng đến sự phát môi trường làm việc và tăng tính minh bạch trong đánh giá hiệu quả công việc. TẦM NHÌN & ĐỊNH HƯỚNG CHIẾN LƯỢC CHUYỂN ĐỔI SỐ Tập đoàn định hướng xây dựng mô hình doanh nghiệp sản"}\n{"id": "B-A-2ce25237fd6949fc", "claim": "an toàn lao động Quản lý nguồn nhân lực Quản trị doanh nghiệp Quản lý chất lượng Chủ đề trọng yếu hàng đầu & Thích ứng BĐKH Quản trị phát triển bền vững", "evidence": "HÒA PHÁT TRỌNG YẾU VỚI CÁC BÊN LIÊN QUAN 4 3. Quản trị doanh nghiệp 8 4. Quản lý môi trường 10 5 5. Quản lý chất lượng 6 6. Quản trị nguồn nhân lực QUẢN TRỊ VÀ ĐỔI MỚI 7.1 | Quản trị Doanh nghiệp và Quản lý rủi ro 4.1 | Quản trị phát triển bền vững 7.2 | Quản lý chất lượng 4.2 | Nhận diện các vấn đề trọng yếu 7.3 | Phát triển đổi mới sáng tạo"}\n{"id": "B-A-fb86ff5b9312be76", "claim": "Các chủ đề quan trọng Phát triển đổi Quản lý chuỗi Đa dạng sinh học Nhân quyền Phát triển cộng đồng Quản lý Nhận diện các vấn đề trọng yếu mới sáng tạo", "evidence": "9 NHÓM CHỦ ĐỀ ƯU TIÊN TRUNG BÌNH 12 11 7. Nhân quyền 8. Quản trị rủi ro Quản trị phát triển bền vững 9. Phát triển đổi mới sáng tạo Nhận diện các vấn đề trọng yếu NHÓM CHỦ ĐỀ ƯU TIÊN THẤP Chiến lược phát triển bền vững tại Tập đoàn Hòa Phát 10. Quản lý chuỗi cung ứng 11. Phát triển cộng đồng 12. Đa dạng sinh học"}\n{"id": "B-A-7485a6298fc6fc35", "claim": "dựng cơ cấu quản trị công bằng, minh bạch, tăng cường hệ thống tuân thủ, và Trong năm 2025, Hòa Phát đã xây dựng Chiến lược Phát triển Bền vững để định hướng hoạt động kinh doanh theo các xu thế dài hạn toàn cầu về ứng phó biến", "evidence": "Tăng khả năng thích ứng với biến động thị trường Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ cột ESG: QUẢN TRỊ: Hệ thống số hóa giúp tăng cường minh bạch thông QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG Định hướng chiến lược về Phát triển bền vững. Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững"}\n{"id": "B-A-6474e01b07b27e46", "claim": "đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ.", "evidence": "mua sắm xanh, phát triển thép carbon thấp, đồng thời nâng cao năng lực cạnh tranh trên thị trường trong nước và quốc tế. Hành động giảm thiểu và thích ứng với biến đổi khí hậu Chúng tôi nỗ lực hoạt động hiệu quả, tiết kiệm và có trách nhiệm, góp phần ứng phó biến đổi khí hậu, thúc đẩy kinh tế tuần hoàn và bảo vệ tài"}\n{"id": "B-A-24a214f4999ff78e", "claim": "thải ròng bằng không (Net Zero) vào năm 2050 thông qua: tăng tỷ trọng nguyên vật liệu phát thải thấp, giám sát chặt chẽ tiêu hao nhiên liệu hóa thạch, nâng cao hiệu quả sử dụng năng lượng, thúc đẩy chuyển dịch năng lượng và phát triển các", "evidence": "thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành;"}\n{"id": "B-A-24b31f96cda2bb90", "claim": "thúc đẩy kinh tế tuần hoàn và bảo vệ hệ sinh thái, cụ thể bằng cách: Tối đa hóa tái chế/tái sử dụngchất thải phát sinh thành nguyên vật liệu đưa trở lại quá trình sản xuất hoặc chế biến, xử lý tạo ra các sản phẩm phụ có giá trị kinh", "evidence": "Mô hình kinh tế tuần hoàn được áp dụng để tối ưu hiệu quả, biến chất thải thành nguồn tài nguyên đưa trở lại chu trình sản xuất hoặc xử lý thành sản phẩm phụ phục vụ cho các ngành khác, một mặt giải quyết vấn đề môi trường và đồng thời đem lại hiệu quả kinh tế. QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh."}\n{"id": "B-A-9c5ccc9c1b8f79e0", "claim": "động, củng cố quan hệ hợp tác và xác định các mục tiêu phát triển bền vững phù hợp trong từng giai đoạn.", "evidence": "QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG 4.1 BỘ MÁY QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG HỘI ĐỒNG QUẢN TRỊ: ĐỊNH HƯỚNG Định hướng chiến lược về Phát triển bền vững. Phê duyệt Chính sách, mục tiêu và các tuyên bố về Phát triển bền vững để xác định mức độ ưu tiên. Thông qua phân tích bối cảnh hoạt động của Tập đoàn và so sánh với thực hành ESG Gắn kết với các bên liên quan BƯỚC 4 tại 15 doanh nghiệp lớn cùng ngành trên thế giới, các mối quan tâm về phát triển bền vững đã được xác định. Quá trình này tham chiếu 05 bộ khung phát triển bền vững quốc"}\n{"id": "B-A-7c09d6a04cd7bb4c", "claim": "cao, đạt tiêu chuẩn chất lượng An toàn và sức khỏe người tiêu dùng (đặc biệt với sản phẩm nông nghiệp, thực phẩm) Giá bán hợp lý Bao bì thân thiện môi trường", "evidence": "PHƯƠNG THỨC TƯƠNG TÁC Sản phẩm và dịch vụ chất lượng cao, đạt tiêu chuẩn chất lượng Đổi mới sản phẩm Danh mục sản phẩm đa dạng về chủng loại và giá cả. Nông nghiệp: sản phẩm sạch và đảm bảo dinh dưỡng với nguyên tắc cốt lõi là bảo vệ sức khỏe và an toàn của người tiêu dùng;"}\n{"id": "B-A-98eb177acbd712d5", "claim": "Điện máy gia dụng: Các tính năng bảo vệ sức khỏe, tiết kiệm năng lượng, hậu mãi chu đáo Giá cả: Giá bán các sản phẩm của Tập đoàn hợp lý theo cung cầu thị trường, tạo khả năng tiếp cận đến đông đảo người tiêu dùng", "evidence": "LĨNH VỰC ĐIỆN MÁY GIA DỤNG Các sản phẩm điện lạnh và điện máy gia dụng được thực hiện đánh giá tác động đến sức khỏe người tiêu dùng trong suốt vòng đời sản phẩm. Trong lĩnh vực điện lạnh, các sản phẩm được thiết kế đảm bảo an toàn về điện, cơ khí và cách nhiệt, cao, đạt tiêu chuẩn chất lượng An toàn và sức khỏe người tiêu dùng (đặc biệt với sản phẩm nông nghiệp, thực phẩm) Giá bán hợp lý Bao bì thân thiện môi trường"}\n{"id": "B-A-90c39dfce9447549", "claim": "quốc gia về giảm phát thải khí nhà kính Duy trì cơ chế kê khai và nộp thuế minh bạch và tuân thủ pháp luật.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KIỂM KÊ KHÍ NHÀ KÍNH Từ năm 2023, Tập đoàn đã duy trì thực hiện kiểm kê khí nhà kính trên phần lớn các công ty thành viên, tuân thủ các tiêu cuẩn quốc tế ISO 14064 và ISO 14607. định kỳ, đồng thời điều chỉnh kịp thời để thích ứng với tình hình thực tế. Tập đoàn luôn tuân thủ nghiêm ngặt các quy định của pháp luật về thuế, thực hiện kê khai, nộp thuế đầy đủ, minh bạch và đúng hạn trên toàn hệ thống, khẳng"}\n{"id": "B-A-c21a7eaa52f33654", "claim": "Tuân thủ nghiêm túc các quy định pháp luật trong nước Thực hiện kiểm kê khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 và báo cáo đầy đủ theo quy định Hợp tác và phối hợp với chính quyền", "evidence": "cực về kiểm kê khí nhà kính theo tiêu chuẩn quốc tế ISO 14064-1:2018”, đại diện BSI cho biết. Hòa Phát hoàn thành kiểm kê khí nhà kính toàn tập đoàn Sáng kiến cải tiến kỹ thuật góp có thẩm quyền từ tháng 3/2025. Đến tháng 01/2025, BSI đã trao giấy kiểm tra, xác nhận, công bố khí nhà kính theo tiêu chuẩn ISO 14064-1:2018 cho các công ty thành viên của Tập đoàn Hòa Phát tham gia kiểm kê. Việc hoàn thành kiểm kê khí nhà kính là bước đi quan trọng"}\n{"id": "B-A-9ce78a199d51e44b", "claim": "Hoạt động kiểm kê KNK được thực hiện với 3 mục tiêu chính: [GRI 302-1, 302-2, 305-1, 305-2, 305-4] Nhận thức rõ Biến đổi Khí hậu là thách thức toàn cầu, và hưởng ứng cam kết Net Zero 2050 của Việt Nam tại COP26, Tập đoàn Hòa Phát xác định “Giảm thiểu và Thích", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-387a34be88af18eb", "claim": "Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm phát thải khí nhà kính phù hợp với điều kiện kinh", "evidence": "Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao hiệu quả vận hành và giảm phát thải. nội bộ 2024 Hòa Phát hoàn thành kiểm kê khí nhà kính toàn tập đoàn Sáng kiến cải tiến kỹ thuật góp phần tiết kiệm hàng trăm tỷ đồng Tự chủ 90% điện sản xuất thép Phát triển sản phẩm mới"}\n{"id": "B-A-7164eabad456a0e7", "claim": "trung vào các lĩnh vực có cường độ phát thải cao nhất — đặc biệt là ngành gang thép, chiếm hơn 99% tổng lượng phát thải KNK của Tập đoàn.", "evidence": "Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng yếu đến từ việc tiêu thụ điện năng, và hơi bão hòa mua ngoài. Quản lý môi trường Xét theo lĩnh vực hoạt động, ngành sản xuất thép là lĩnh vực phát thải lớn nhất, đóng góp chủ yếu vào tổng phát thải của toàn tập đoàn."}\n{"id": "B-A-ed73f8f32d4cb0e8", "claim": "94% tổng công suất thép thô toàn Tập đoàn — là đơn vị trọng điểm trong công tác quản lý phát thải, nghiên cứu và triển khai các sáng kiến giảm phát thải.", "evidence": "Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu PHÒNG CHỨC NĂNG TẠI CÔNG TY THÀNH VIÊN: THỰC THI Nghiên cứu và triển khai thực hành sáng kiến bền vững trong sản xuất, ứng với Biến đổi Khí hậu” là ưu tiên phát triển bền vững quan trọng. Tập đoàn đang triển khai đồng bộ các sáng kiến và hành động, đặc biệt trong lĩnh vực sản xuất thép, nhằm tiết kiệm năng lượng và giảm phát thải khí nhà kính phù hợp với điều kiện kinh"}\n{"id": "B-A-a52c4596fe156b50", "claim": "thành viên không nằm trong danh sách bắt buộc phải kiểm kê KNK của Nghị định 06/2022/NĐ-CP cũng được BSI đào tạo để chủ động thực hiện kiểm kê, phục vụ cho nhu cầu quản lý phát thải chung của cả Tập đoàn.", "evidence": "danh mục phải kiểm kê phát thải theo Nghị định 06/2022/NĐ-CP đã tiến hành kiểm kê KNK theo chuẩn ISO 14064 và được BSI chứng nhận kết quả kiểm kê, hoàn thành và nộp báo cáo kiểm kê KNK đầy đủ cho các cơ quan chức năng từ trong Nghị định 06/2022/ND-CP, Nghị định 119/2024/ND-CP. Đối với số liệu kiểm kê KNK của Tập đoàn, phương pháp hợp nhất số liệu được áp dụng theo hướng tiếp cận “Vốn chủ sở hữu”."}\n{"id": "B-A-04735e1ed7f66c4a", "claim": "động thực hiện kiểm kê dấu chân carbon cho các sản phẩm, thực hiện theo chuẩn ISO 14067 và đã được kiểm định bởi bên thứ ba – Tổ chức CARE (Cơ quan chứng nhận sản phẩm của Vương quốc Anh), phục vụ công tác kê khai theo Cơ chế điều", "evidence": "KIỂM KÊ KHÍ NHÀ KÍNH VÀ RỦI RO TỪ CƠ CHẾ ĐIỀU CHỈNH BIÊN GIỚI CARBON (CBAM) BIỆN PHÁP BÁO CÁO THƯỜNG NIÊN - 2024 Tập đoàn Hòa Phát đã phối hợp với BSI – Tổ chức chứng nhận quốc tế hàng đầu của Vương quốc Anh để đào tạo kiểm kê khí nhà kính cho các công ty thành viên phí cho mỗi tấn hàng hóa, đồng thời nâng cao hiệu quả hoạt động vận tải. Tháng 04/2024, BSI - Tổ chức chứng nhận quốc tế hàng đầu của Vương quốc Anh đã trao giấy công bố kiểm tra, xác nhận kiểm kê khí nhà kính theo tiêu chuẩn ISO"}\n{"id": "B-A-168ce8c96eaa87b5", "claim": "Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám sát hiệu quả sử dụng năng lượng, theo dõi kiểm kê phát thải và thúc đẩy các sáng kiến giảm phát thải.", "evidence": "Chuẩn Anh (BSI). Để quản lý hiệu quả năng lượng tiêu thụ và phát thải khí nhà kính, tập đoàn Hòa Phát đã áp dụng các phương pháp quản lý như sau: THIẾT LẬP BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn"}\n{"id": "B-A-8d743a19c166f31c", "claim": "Ban họp định kỳ hàng quý nhằm đánh giá tiến độ thực hiện mục tiêu, chia sẻ thông tin về các dự án tiết kiệm năng lượng, truyền thông các chủ đề trọng yếu, đồng thời tham mưu cho Ban Giám đốc trong việc xây dựng kế hoạch, đề xuất, kiến nghị giải pháp tối ưu", "evidence": "Gắn kết với các bên liên quan HỘI ĐỒNG QUẢN TRỊ BAN TỔNG GIÁM ĐỐC BAN QUẢN LÝ NĂNG LƯỢNG VÀ KHÍ NHÀ KÍNH Tham mưu, đề xuất, triển khai thực hiện các công việc liên quan đến để tối ưu sản xuất, tiết kiệm năng lượng, giảm phát thải khí nhà kính tại tất cả các khu liên hợp sản xuất gang thép. Hòa Phát đã tính toán kỹ lưỡng và dành tới 30% vốn đầu tư cho các hạng mục môi trường. Hàng loạt các giải pháp và công nghệ"}\n{"id": "B-A-d79be1994c9e3f03", "claim": "Hằng năm, Ban xây dựng kế hoạch giảm phát thải, theo dõi tiến độ triển khai và đề xuất các giải pháp chiến lược nhằm nâng cao hiệu quả năng lượng và giảm thiểu tác động", "evidence": "Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao hiệu quả vận hành và giảm phát thải. Ban họp định kỳ hàng quý nhằm đánh giá tiến độ thực hiện mục tiêu, chia sẻ thông tin về các dự án tiết kiệm năng lượng, truyền thông các chủ đề trọng yếu, đồng thời tham mưu cho Ban Giám đốc trong việc xây dựng kế hoạch, đề xuất, kiến nghị giải pháp tối ưu"}\n{"id": "B-A-38792389ab485482", "claim": "Báo cáo được thiết lập dựa trên các yêu cầu của tiêu chuẩn ISO 14064-1, theo các hướng dẫn của Ban liên chính phủ về biến đổi khí hậu (IPCC Guidelines) và đáp ứng các yêu cầu về giảm phát thải khí nhà kính của Cơ quan quản lý được đề cập", "evidence": "ISO 14064-1:2018 là tiêu chuẩn liên quan đến lĩnh vực môi trường và hướng tới chương trình chống biến đổi khí hậu cũng như các mục tiêu phát triển bền vững toàn cầu. Tiêu chuẩn này là một khuôn khổ chắc chắn để thực hiện định lượng, tính HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 KIỂM KÊ KHÍ NHÀ KÍNH Từ năm 2023, Tập đoàn đã duy trì thực hiện kiểm kê khí nhà kính trên phần lớn các công ty thành viên, tuân thủ các tiêu cuẩn quốc tế ISO 14064 và ISO 14607."}\n{"id": "B-A-e7cc94e14da8914f", "claim": "cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp).", "evidence": "động này cũng góp phần phản ánh một cách hiệu quả các kết quả mà Tập đoàn đã đạt được và các giải pháp mà Tập đoàn sẽ triển khai trong tương lai. Thu hồi nhiệt và khí thải từ các lò luyện cốc, lò cao và lò chuyển để phát điện và Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài (Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt"}\n{"id": "B-A-6ddde31462aa8d06", "claim": "Bên cạnh đó, Các nguồn phát thải xuất phát từ việc sản xuất năng lượng từ bên ngoài (Nguồn phát thải gián tiếp từ năng lượng nhập khẩu) để phục vụ quá trình sản xuất của Tập đoàn cũng được xem xét và trình bày trong Báo cáo này do hoạt", "evidence": "gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ cũng như trình bày một cách minh bạch các kết quả đã đạt được và những giải pháp ứng phó trong tương lai, Báo cáo phát thải KNK sẽ tập trung vào các nguồn phát thải mà Tập đoàn có sự kiểm soát/sở hữu (Nguồn phát thải trực tiếp)."}\n{"id": "B-A-b36d3478ee0d3988", "claim": "giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm phát thải CO2 trên mỗi tấn thép.", "evidence": "Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn bộ chuỗi sản xuất. Trong năm 2025, tỷ lệ phối phế cùng gang lỏng tại Khu liên hợp Gang lò với mục tiêu giảm lượng than cốc tiêu hao trên mỗi tấn gang. Qua đó, lượng CO2 sinh ra trong quá trình cháy và hoàn nguyên cũng giảm tương ứng. Đây là biện pháp cốt lõi trong giảm phát thải trực tiếp tại khu vực lò cao và nâng cao hiệu suất luyện gang."}\n{"id": "B-A-b6546ef1e9f80411", "claim": "Các khía cạnh quản lý phát thải KNK bao gồm: Năng lượng, Nguyên vật liệu và Công nghệ Quản lý môi trường Năm 2025, Công ty Cổ phần Thép Hòa Phát Dung Quất đã đạt chứng nhận ISO 50001:2018 về Hệ thống Quản lý Năng lượng.", "evidence": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng trong công tác quản lý năng lượng. Bên cạnh đó, Ban còn chịu trách nhiệm quản lý phát thải khí nhà kính và định lượng dấu chân carbon của sản phẩm, bao gồm thu thập, kiểm chứng và báo cáo dữ liệu theo quy định của Bộ Công Thương, Bộ"}\n{"id": "B-A-1ba12e2e5e9374e1", "claim": "thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải.", "evidence": "Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám sát hiệu quả sử dụng năng lượng, theo dõi kiểm kê phát thải và thúc đẩy các sáng kiến giảm phát thải. Các Khu liên hợp sản xuất gang thép của Hòa Phát đã triển khai các giải pháp chủ chốt về tiết kiệm năng lượng trong toàn bộ quy trình sản xuất thép nhằm nâng cao hiệu quả vận hành và giảm phát thải."}\n{"id": "B-A-0f3a814f08b11af2", "claim": "bộ dây chuyền sản xuất, đồng thời thực hiện nhiều biện pháp nâng cao hiệu suất năng lượng như thu hồi nhiệt dư để phát điện, tự chủ nguồn điện nội bộ, và nghiên cứu đầu tư điện mặt trời áp mái tại các cơ sở sản xuất.", "evidence": "để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo. Bên cạnh đó. Tập đoàn Hòa Phát nghiên cứu triển khai lắp đặt hệ thống điện mặt trời trời áp mái nhằm cung cấp một phần nhu cầu điện năng của các nhà máy. HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THU HỒI NHIỆT ĐỂ PHÁT ĐIỆN Từ năm 2025, Tập đoàn Hòa Phát chính thức đẩy mạnh lộ trình “xanh hóa” sản xuất thông qua việc triển khai hệ thống điện mặt trời áp mái tại các khu liên hợp"}\n{"id": "B-A-26b6b8324f6a112f", "claim": "Bên cạnh đó, Công ty Cổ phần Thép Hòa Phát Dung Quất đã ban hành Sổ tay Quản lý Năng lượng, quy định rõ các chính sách, mục tiêu và cam kết về tiết kiệm năng lượng, giảm phát thải carbon và cải tiến liên tục và thiết lập hệ thống giám sát năng lượng", "evidence": "Phần lớn các công ty thành viên đã thành lập Ban Quản lý Năng lượng và Khí nhà kính, có chức năng giám sát hiệu quả sử dụng năng lượng, theo dõi kiểm kê phát thải và thúc đẩy các sáng kiến giảm phát thải. Các khía cạnh quản lý phát thải KNK bao gồm: Năng lượng, Nguyên vật liệu và Công nghệ Quản lý môi trường Năm 2025, Công ty Cổ phần Thép Hòa Phát Dung Quất đã đạt chứng nhận ISO 50001:2018 về Hệ thống Quản lý Năng lượng."}\n{"id": "B-A-bafe2fcdea723631", "claim": "để tối ưu tiêu thụ, giảm cường độ năng lượng và hướng tới chuyển dịch sang các nguồn năng lượng tái tạo.", "evidence": "thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn lượng 2025 103-3 Tiêu thụ năng lượng thượng nguồn và hạ nguồn Chưa thu thập được đầy đủ thông tin từ các bên liên quan 103-4 Cường độ năng lượng Kết quả tiêu thụ năng lượng năm 2025"}\n{"id": "B-A-c65a345980b54b86", "claim": "Quy trình luyện thép bằng EAF có cường độ phát thải CO2 thấp hơn nhiều lần so với lò cao truyền thống, đặc biệt khi kết hợp với các nguồn điện sạch như điện mặt trời", "evidence": "(BOF) (World Bank). Công nghệ này giúp sản xuất thép từ quặng sắt nguyên khai với độ tinh khiết và chất lượng cao, song cũng phát thải CO2 nhiều hơn so với lò điện hồ quang (EAF). Ngược lại, EAF có mức phát thải thấp hơn, nhưng phụ thuộc vào nguồn nguyên liệu như thép Đặc biệt, các Khu liên hợp sản xuất gang thép ứng dụng đồng thời nhiều hệ thống lọc bụi phù hợp với từng loại đối tượng bụi, được lắp đặt tại từng công đoạn như lò thiêu kết, lò cao, lò tinh luyện, nồi hơi nhiệt dư và dây chuyền cán thép giúp hiệu suất xử lý đạt trên 99% ngay tại nguồn phát sinh và không phát tán ra không khí và tận dụng bụi"}\n{"id": "B-A-8a07fb243c68a94c", "claim": "phế, quặng sắt đã hoàn nguyên – những nguyên liệu có giá thành cao và nguồn cung hạn chế, phù hợp với các quốc gia phát triển, nơi ngành thép đã đi qua nhiều vòng đời tái chế.", "evidence": "Trong các lò cao làm suy giảm chất lượng, đảm bảo giá trị tài nguyên tối đa thông qua tái sử dụng, tái chế, BÁO CÁO PHÁT TRIỂN BỀN VỮNG công nghiệp, quặng sắt được hoàn nguyên và nấu chảy ở nhiệt độ cực cao để tạo ra 2025 tái sản xuất. Vòng đời của thép đi qua hai giai đoạn: HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên"}\n{"id": "B-A-c1eee59796b7b518", "claim": "(vòng đời sơ cấp – sản xuất thép mới) trên thế giới chủ yếu tập trung vào việc sử dụng các loại nguyên liệu đầu vào có hàm lượng carbon thấp làm chất khử quặng sắt như khí tự nhiên hoặc Hydrogen xanh (sản xuất", "evidence": "nằm ở sự khan hiếm nguồn cung các loại quặng sắt có hàm lượng sắt cao %Fe>65% cũng như nguồn khí tự nhiên và hydrogen sạch. Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp – HOẠT ĐỘNG KINH DOANH TẠI TẬP ĐOÀN HÒA PHÁT 3.4 VÒNG ĐỜI CỦA THÉP Giai đoạn Sản xuất Sơ cấp (Hoàn nguyên quặng sắt) - Từ quặng đến thép Vòng đời sơ cấp của thép bắt đầu từ việc khai thác các nguồn tài nguyên thiên nhiên"}\n{"id": "B-A-aee6b56f2e2bf99f", "claim": "Biện pháp thứ hai là sử dụng thép phế liệu để tái chế thành thép (vòng đời thứ cấp –", "evidence": "Giai đoạn Tái chế (Sản xuất từ phế liệu) - Từ thép đến thép Vòng đời thứ cấp (tái chế) thể hiện khả năng phục hồi vĩnh cửu của thép. Khác với Khai thác nguyên liệu thô nhiều vật liệu khác bị suy giảm đặc tính sau mỗi lần tái chế (như nhựa hay giấy), thép có tái chế thép). Biện pháp này giúp tái tạo lại thép từ phế liệu đã qua sử dụng, do không cần hoàn nguyên nên giảm đáng kể lượng tiêu thụ than và giảm phát thải. Theo nghiên cứu của Hiệp hội Thép Thế giới (2021),"}\n{"id": "B-A-a9cdc9af6804a921", "claim": "hao than cốc, tận dụng nhiệt lượng và giảm phát thải khí nhà kính trong toàn bộ chuỗi sản xuất, đồng thời gia tăng năng lực sản xuất thép.", "evidence": "Việc sử dụng nhiều hơn thép phế giúp tăng sản lượng, giảm tỷ trọng nguyên liệu phát thải cao, giảm tiêu hao than cốc và các nhiên liệu trên một tấn thép, đồng thời tiết kiệm năng lượng trong toàn bộ chuỗi sản xuất. Trong năm 2025, tỷ lệ phối phế cùng gang lỏng tại Khu liên hợp Gang vào dây chuyền cán. giúp giảm nhu cầu gia nhiệt, tiết kiệm năng lượng và giảm phát thải CO2 trên mỗi tấn thép. Tận dụng các nguồn hơi nước từ hệ thống làm mát lò thổi, lò nung phôi để tận dụng phát điện. BÁO CÁO PHÁT TRIỂN BỀN VỮNG"}\n{"id": "B-A-c8fe1a51af9f6487", "claim": "coke và quặng sắt truyền thống sang các nguyên liệu phát thải thấp như sắt hoàn nguyên trực tiếp (DRI), sắt hoàn nguyên dạng nén (HBI) và trong dài hạn là hydrogen trong lò cao.", "evidence": "từ nguồn điện sạch) để tạo ra sắt hoàn nguyên trực tiếp (DRI và HBI), thay thế cho việc hoàn nguyên bằng nhiên liệu hóa thạch truyền thống như than. Hạn chế khiến cho biện pháp này chưa thể áp dụng rộng rãi bụi và bùn chứa kẽm phát sinh tại Khu liên hợp sản xuất gang thép Hòa Phát Dung Quất 1 và 2. Công nghệ Lò đáy quay giúp tách bụi lò cao thành quặng sắt hoàn nguyên trực tiếp (DRI), là nguồn quặng đầu vào quay trở lại sản xuất thép"}\n{"id": "B-A-28c32e59188242e6", "claim": "đổi công nghệ sản xuất thép xanh trong tương lai, tuy nhiên phụ thuộc rất lớn vào nguồn cung cũng như sự hỗ trợ của Chính phủ và các tổ chức trong việc xây dựng hệ sinh thái nguyên vật liệu và năng lượng sạch.", "evidence": "hoặc điện gió. Tuy nhiên, việc chuyển dịch công nghệ phụ thuộc cao vào giải quyết bài toán nguyên vật liệu. Do sự khan hiếm trong nguồn cung nguyên liệu sạch, hiện nay, hơn 70% sản lượng thép toàn cầu vẫn đang được sản xuất theo công nghệ lò cao truyền thống lượng nguyên vật liệu đầu vào của Tập đoàn như than, quặng,... phụ thuộc khá nhiều vào nguồn cung cấp hàng từ nước ngoài. Giá trị nhập khẩu nhiều hơn xuất khẩu, vì vậy tỷ giá ảnh hưởng rất lớn tới chi"}\n{"id": "B-A-89f95b047b68be4b", "claim": "Đây là bước đi then chốt nhằm giảm dấu chân carbon trên mỗi đơn vị sản phẩm, trực tiếp nâng cao năng lực cạnh tranh cho thép và các sản phẩm công nghiệp của Tập đoàn trên thị trường quốc tế, đặc biệt là tại các khu", "evidence": "cạnh tranh của sản phẩm trên thị trường toàn cầu. Nhằm đáp ứng yêu cầu của thị trường trong nước và quốc tế, Tập đoàn Hòa Phát đã đạt được các chứng nhận sản phẩm bắt buộc tại Việt Nam cũng như mua sắm xanh, phát triển thép carbon thấp, đồng thời nâng cao năng lực cạnh tranh trên thị trường trong nước và quốc tế. Hành động giảm thiểu và thích ứng với biến đổi khí hậu"}\n{"id": "B-A-93ceba0f5ecbfb87", "claim": "toàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp bền vững và thích ứng khí hậu.", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT VÀ THÍCH ỨNG BIẾN ĐỔI KHÍ HẬU Chúng tôi cam kết tuân thủ quy định pháp luật quốc gia về môi trường;"}\n{"id": "B-A-15f075e25a917777", "claim": "này chủ yếu xuất phát từ hoạt động chăn nuôi gia súc, gia cầm, sử dụng phân bón hóa học, quản lý chất thải và vận hành thiết bị phục vụ sản xuất nông nghiệp.", "evidence": "Nông nghiệp: chăn nuôi lợn, bò, gia cầm; hoạt động dịch vụ chăn nuôi; chế biến, bảo quản thịt và các sản phẩm từ thịt; sản xuất thức ăn gia súc, gia cầm Sản xuất thức ăn gia súc, gia cầm và thủy sản; Sản xuất phân bón và hợp chất ni tơ; Buôn bán máy móc, thiết bị và phụ tùng máy nông nghiệp; Vận tải hàng hóa ven biển, đường thủy nội địa và viễn dương;"}\n{"id": "B-A-c9fb43a7781bf603", "claim": "tỷ trọng phát thải từ nông nghiệp và chăn nuôi chỉ chiếm một phần nhỏ, không đáng kể so với lĩnh vực sản xuất thép chủ lực.", "evidence": "thải. Từ tháng 9 năm 2025, Hòa Phát Dung Quất 2 đi vào vận hành, Tổng sản lượng thép thô tăng 24%, tương ứng với mức tăng phát thải của toàn tập đoàn. Lĩnh vực nông nghiệp và chăn nuôi của Tập đoàn Hòa Phát ghi nhận mức phát Mạng lưới phân phối trứng gà Hòa Phát hiện có mặt tại hơn 100 siêu thị khu vực phía Bắc và rộng khắp ba miền. Trong lĩnh vực chăn nuôi heo, sản lượng heo thành phẩm vượt kế hoạch đề ra. Lĩnh vực chăn nuôi bò Úc có chuyển biến tích cực so với năm 2023 khi sản lượng chăn nuôi cao"}\n{"id": "B-A-d3b64bdeaa9c43f3", "claim": "Tuy nhiên, để tiến tới phát triển bền vững, việc kiểm soát phát thải ở lĩnh vực này vẫn cần được chú trọng thông qua", "evidence": "MỤC LỤC BÁO CÁO PHÁT TRIỂN BỀN VỮNG NĂM 2025 TỔNG QUAN VỀ PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT CÁC LĨNH VỰC TRỌNG YẾU VỀ PHÁT TRIỂN BỀN VỮNG QUẢN TRỊ DOANH NGHIỆP Quản trị phát triển bền vững Nhận diện các vấn đề trọng yếu Chúng tôi không ngừng theo đuổi chất lượng vượt trội gắn liền với đổi mới bền vững, thông qua việc cải tiến công nghệ và quy trình sản xuất để nâng cao hiệu"}\n{"id": "B-A-980ede9decc42d31", "claim": "các biện pháp giảm phát thải khí metan, tối ưu hóa quy trình sản xuất và thúc đẩy ứng dụng công nghệ xanh.", "evidence": "XANH HÓA SẢN XUẤT QUẢN TRỊ VÀ ĐỔI MỚI Trụ cột Xanh hóa Sản xuất hướng tới hài hòa giữa tăng trưởng công nghiệp và bảo vệ môi trường, giảm thiểu biến đổi khí hậu. Chúng tôi nhận thức về trách nhiệm của mình trong việc tham gia lộ trình giảm phát thải quốc gia và đang tích cực thực hành các giải pháp phù hợp không khí nhằm giảm thiểu tác động môi trường, tuân thủ quy định pháp luật và hướng tới tăng trưởng xanh. BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 5 XANH HÓA SẢN XUẤT TUÂN THỦ PHÁP LUẬT VỀ MÔI TRƯỜNG"}\n{"id": "B-A-9c4cec554a9e720f", "claim": "Hành động giảm thiểu và thích ứng với biến đổi khí hậu Trong năm 2025, tổng lượng phát thải khí nhà kính của Tập đoàn Hòa Phát là 23.474.480 tCO2e, bao gồm các nguồn phát thải trực tiếp và nguồn phát thải", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 thải khí nhà kính ở mức tương đối thấp so với tổng lượng phát thải toàn tập đoàn (phát thải 90,846 tCO2e, chiếm 0,39%). Các nguồn phát thải chính trong lĩnh vực HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-9d687967979f1c92", "claim": "Bên cạnh đó, các nguồn phát thải trực tiếp từ quá trình đốt cháy nhiên liệu, vận hành lò cao, lò chuyển và hệ thống thiết bị phụ trợ cũng góp phần làm tăng tổng lượng phát", "evidence": "gián tiếp từ năng lượng nhập khẩu. Cụ thể, các nguồn phát thải trực tiếp như quá trình sản xuất công nghiệp, chăn nuôi, vận hành phương tiện vận tải và thiết bị nội bộ chiếm tỷ trọng lớn nhất. Bên cạnh đó, các nguồn phát thải gián tiếp chủ HÒA PHÁT trò nền tảng để cung cấp nguồn thép chất lượng cao cho các công trình hạ tầng và máy móc, tuy nhiên nó tiêu tốn một lượng lớn năng lượng và giải phóng nhiều khí thải CO2 do quá trình đốt cháy nhiên liệu hóa thạch."}\n{"id": "B-A-446a12b7ab224081", "claim": "Công tác quản lý môi trường của Tập đoàn tập trung vào các khía cạnh trọng yếu gồm quản lý và sử dụng nước, quản lý chất thải và nước thải, kiểm soát chất lượng", "evidence": "Nước thải sản xuất IWW Nước thải sinh hoạt Quản lý môi trường Nước thải sinh hoạt sau xử lý Bùn thải Tưới cây QUẢN LÝ NƯỚC theo cả khía cạnh tổng lượng nước sử dụng và lượng nước tuần hoàn. QUẢN LÝ MÔI TRƯỜNG 5.2 TIÊU THỤ NƯỚC TOÀN TẬP ĐOÀN 2025 KIỂM SOÁT CHẤT LƯỢNG NƯỚC HOÀN TRẢ LẠI MÔI TRƯỜNG Là một phần trong chiến lược phát triển bền vững của Tập đoàn Hòa Phát, tất cả"}\n{"id": "B-A-b8f6a09ead2eb8cb", "claim": "không khí nhằm giảm thiểu tác động môi trường, tuân thủ quy định pháp luật và hướng tới tăng trưởng xanh.", "evidence": "Tập đoàn Hòa Phát cam kết tuân thủ đầy đủ các quy định của pháp luật về môi trường như Luật bảo vệ môi trường năm 2020, các Nghị định và Thông tư hướng dẫn có liên quan, thông qua việc triển khai hệ thống quản lý môi trường hiệu quả sản phẩm thép ít carbon. HÀNH ĐỘNG GIẢM THIỂU PHẦN 4 PHÁT TRIỂN BỀN VỮNG TẠI TẬP ĐOÀN HÒA PHÁT VÀ THÍCH ỨNG BIẾN ĐỔI KHÍ HẬU Chúng tôi cam kết tuân thủ quy định pháp luật quốc gia về môi trường;"}\n{"id": "B-A-283403d721ec463b", "claim": "hợp và nhà máy, trang trại cho các hạng mục về bảo vệ môi trường, xử lý khí thải, nước thải, chất thải rắn, tích hợp trong hệ thống máy móc thiết bị, dây chuyền sản xuất.", "evidence": "CÁC LOẠI Bụi thải từ các hệ thống xử lý bụi, khí thải Bùn thải từ các trạm xử lý nước thải Phế liệu Xỉ thép Xỉ hạt lò cao CHẤT THẢI RẮN chất thải nguy hại, Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý"}\n{"id": "B-A-9014b372863fb49f", "claim": "kiệm năng lượng, thúc đẩy tái chế và tái sử dụng thông qua việc áp dụng công nghệ sạch, thân thiện môi trường trong toàn bộ hoạt động sản xuất.", "evidence": "QUẢN LÝ MÔI TRƯỜNG 5.2 TÁI CHẾ - TÁI SỬ DỤNG CHẤT THẢI RẮN Tập đoàn Hòa Phát cam kết giảm thiểu phát sinh chất thải và thúc đẩy việc sử dụng tài nguyên tuần hoàn trong toàn bộ hoạt động sản xuất kinh doanh. thép, nhằm chuẩn hóa công tác quản lý năng lượng, sử dụng hiệu quả tài nguyên và giảm phát thải. NĂNG LƯỢNG: TỐI ƯU HÓA SỬ DỤNG NĂNG LƯỢNG, CHUYỂN DỊCH DẦN SANG NĂNG LƯỢNG SẠCH Tập đoàn quản lý chặt chẽ việc tiêu hao than và sử dụng năng lượng trong toàn"}\n{"id": "B-A-a5dcca0f3661cfdd", "claim": "Quá trình này giúp giảm đáng kể tải lượng ô nhiễm và loại bỏ mùi hôi của chất thải trước khi xả ra môi", "evidence": "PHẦN 5 XANH HÓA SẢN XUẤT Các giải pháp quản lý chất thải tổng hợp này giúp Hòa Phát hoàn thiện vòng tuần hoàn tài nguyên, giảm khối lượng chất thải chôn lấp, đồng thời tạo ra giá trị kinh tế và môi trường từ những vật liệu trước đây được coi là phế thải. mua ngoài hiện tại của Tập đoàn. Điểm ưu việt của mô hình này nằm ở việc năng lượng được sản xuất và tiêu thụ tại chỗ, loại bỏ hoàn toàn hao hụt trong quá trình truyền tải xa và tạo ra một lớp “lá chắn nhiệt” tự nhiên giúp hạ nhiệt độ mái tôn"}\n{"id": "B-A-8ac0a371092eceb2", "claim": "các công ty thành viên đều được yêu cầu triển khai các sáng kiến tiết kiệm nước, bao gồm thu gom và sử dụng nước mưa, cũng như tận dụng nguồn nước tuần hoàn nội bộ và tái sử dụng từ bên ngoài trong quá trình sản xuất.", "evidence": "các công ty thành viên đều được yêu cầu triển khai các sáng kiến tiết kiệm nước, bao gồm thu gom và sử dụng nước mưa, cũng như tận dụng nguồn nước tuần hoàn nội bộ và tái sử dụng từ bên ngoài trong quá trình sản xuất. hiệu quả sử dụng nước và gia tăng tỷ lệ sử dụng nước tuần hoàn đã giúp giảm đáng kể nhu cầu dùng nước ngọt, trong đó Khu liên hợp Gang Thép Hòa Phát Dung Quất đạt tỷ lệ tái sử dụng nước đến 98%, tương đương với tỷ lệ tuần hoàn"}\n{"id": "B-A-a7a90e42c7e1dd15", "claim": "quan đến nước thải, thể hiện cam kết mạnh mẽ của Hòa Phát trong việc bảo vệ môi trường nước và duy trì hoạt động sản xuất bền vững, hiệu quả.", "evidence": "toàn sinh học lâu dài, thể hiện cam kết của Hòa Phát trong phát triển nông nghiệp bền vững và thích ứng khí hậu. Quản lý môi trường 2. Phát thải theo tổng công ty năm 2025 PHÁT THẢI TOÀN TẬP ĐOÀN Nước thải sản xuất IWW Nước thải sinh hoạt Quản lý môi trường Nước thải sinh hoạt sau xử lý Bùn thải Tưới cây QUẢN LÝ NƯỚC theo cả khía cạnh tổng lượng nước sử dụng và lượng nước tuần hoàn."}\n{"id": "B-A-49281c99442f5588", "claim": "Công tác quản lý chất thải là một nội dung trọng yếu trong Chính sách Môi trường, Sức khỏe và An toàn (EHS) của các đơn vị thành viên, bảo đảm tuân thủ quy định pháp luật quốc gia và phù hợp với các nguyên tắc của nền kinh tế tuần hoàn.", "evidence": "Thực hiện cuộc kiểm tra tuân thủ tại các phòng ban, bộ phận tại đơn vị thành viên đảm bảo việc chấp hành quy định pháp luật hiện hành, điều lệ Công ty, quy chế hoạt động và các nghĩa vụ bắt buộc về chính sách thuế, bảo hiểm xã hội... Kiểm tra, giám sát việc tuân thủ các chính sách, quy định, quy trình đã xây dựng tại đơn vị thành viên. Quản trị doanh nghiệp tốt Tuân thủ các quy định pháp luật và minh bạch về Thuế, Môi trường, Lao động, An toàn và Sức khỏe nghề nghiệp Đóng góp cho lộ trình và cam kết"}\n{"id": "B-A-fd5e7bcee8f0715f", "claim": "Bùn thải Chất thải khác Bụi thải Bùn thải Chất thải khác Hành động giảm thiểu và thích ứng với biến đổi khí hậu Quản lý môi trường Chất thải khác (Chất thải sinh hoạt,", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024 SẢN XUẤT Bay hơi Các nhà máy trong KLH Nước tái sử dụng Bùn thải Làm mát trực tiếp Hành động giảm thiểu và thích ứng với biến đổi khí hậu Nước sinh hoạt PW"}\n{"id": "B-A-30aa807281ec8fcc", "claim": "Các chất thải nguy hại (mỡ, giẻ lau dính dầu, hóa chất) đều được lưu giữ an toàn, dán nhãn đầy đủ và chuyển giao cho đơn vị có giấy phép xử lý theo đúng quy định pháp luật.", "evidence": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại) Sức khoẻ và an toàn lao động Quản trị nguồn nhân lực Mọi báo cáo đều được giữ bí mật, điều tra kịp thời và xử lý công bằng theo quy trình nội bộ và quy định pháp luật. Các khóa đào tạo nhận thức về Nội quy Lao"}\n{"id": "B-A-f7ec4d4df302194f", "claim": "Hành động giảm thiểu và thích ứng với biến đổi khí hậu Chuyển đổi các chất thải có khối lượng lớn thành sản phẩm phụ Các phụ phẩm từ hoạt động luyện kim được xử lý một cách hiệu quả để tối đa", "evidence": "HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 Với mục đích phản ánh một cách chính xác và khách quan nhất những tác động biến đổi khí hậu xuất phát từ các hoạt động trong phạm vi kinh doanh của mình HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-27e4d194c6eb5926", "claim": "7,085 Tổng khối lượng được sử dụng cho các mục đích khác thay vì bị vứt bỏ/tiêu hủy 6,982 Tổng khối lượng chất thải được xử lý thải bỏ 103 50", "evidence": "Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào quy trình xử lý Tổng khối lượng chất thải NGUY HẠI được xử lý thải bỏ Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý"}\n{"id": "B-A-f2f0936214e42e84", "claim": "mua sắm xanh, phát triển thép carbon thấp, đồng thời nâng cao năng lực cạnh tranh trên thị trường trong nước và quốc tế.", "evidence": "phát triển bền vững. Đây là bước đi then chốt nhằm giảm dấu chân carbon trên mỗi đơn vị sản phẩm, trực tiếp nâng cao năng lực cạnh tranh cho thép và các sản phẩm công nghiệp của Tập đoàn trên thị trường quốc tế, đặc biệt là tại các khu cạnh tranh của sản phẩm trên thị trường toàn cầu. Nhằm đáp ứng yêu cầu của thị trường trong nước và quốc tế, Tập đoàn Hòa Phát đã đạt được các chứng nhận sản phẩm bắt buộc tại Việt Nam cũng như"}\n{"id": "B-A-c5d617788d6e9458", "claim": "Điện máy gia dụng 104,6 57,8 55% Cộng 7.085 6.982 99% Năm 2025, tổng lượng chất thải rắn phát sinh của Tập đoàn là 7.085 nghìn tấn, trong đó chất thải nguy hại chiếm 3%, chất thải không nguy hại chiếm 97%.", "evidence": "(Nghìn tấn) Tổng khối lượng chất thải rắn phát sinh 7,085 Chất thải không nguy hại 6,838 Chất thải nguy hại 247 LƯỢNG CHẤT THẢI THEO CÁC HƯỚNG XỬ LÝ NĂM 2025 Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác"}\n{"id": "B-A-b645c95e603c2cc8", "claim": "đã thực sự trở thành nguồn tài nguyên đầu vào quan trọng, giúp Tập đoàn tối ưu hóa chi phí và giảm thiểu tác động tiêu cực đến hệ sinh thái.", "evidence": "Chu trình khép kín này giúp chuyển hóa toàn bộ phụ phẩm phát sinh thành điện năng sạch, phục vụ trực tiếp cho các hoạt động sản xuất tại nhà máy. Việc tái sử dụng tài nguyên không chỉ giúp Tập đoàn tự chủ nguồn điện, tối ưu hóa chi phí vận hành mà bao gồm: Hành động giảm thiểu và thích ứng với biến đổi khí hậu HỆ THỐNG: Đầu tư công nghệ hiện đại Hòa Phát ứng dụng công nghệ tiên tiến để giảm thiểu tối đa các tác động bất lợi đến môi trường. Tập đoàn dành tỷ trọng lớn trong chi phí đầu tư tại các khu liên"}\n{"id": "B-A-86ced468645dc9af", "claim": "Những thành quả này phản ánh nỗ lực không ngừng của Tập đoàn trong việc mở rộng cơ hội lãnh đạo cho phụ nữ, đồng thời đặt nền móng cho sự tiến bộ bền vững hơn trong các lĩnh", "evidence": "Điển hình như trao thưởng cho các tập thể, cá nhân có thành tích xuất sắc hoặc có sáng kiến cải tiến, áp dụng thành công trong sản xuất. Chính sách này đã tạo động lực cho CBCNV tận tâm với công việc, đồng thời không ngừng phát huy năng lực, Chỉ số ROE tăng phản ánh những nỗ lực trong quản lý và vận hành mang lại sự phục hồi trong tăng trưởng lợi nhuận ròng của Tập đoàn. Cùng với đó, chỉ số ROA của Tập đoàn cũng tăng đạt 5,4% trong khi hệ số này ở cùng kỳ năm 2023 là 3,6%."}\n{"id": "B-A-daeeba55dde8d018", "claim": "Đào tạo lập Báo cáo kiểm kê khí nhà kính", "evidence": "KIỂM KÊ KHÍ NHÀ KÍNH VÀ RỦI RO TỪ CƠ CHẾ ĐIỀU CHỈNH BIÊN GIỚI CARBON (CBAM) BIỆN PHÁP BÁO CÁO THƯỜNG NIÊN - 2024 Tập đoàn Hòa Phát đã phối hợp với BSI – Tổ chức chứng nhận quốc tế hàng đầu của Vương quốc Anh để đào tạo kiểm kê khí nhà kính cho các công ty thành viên trong Tập đoàn. Tháng 8/2024, Thủ tướng Chính phủ đã chính thức ký Quyết định số 13/2024/ QĐ-TTg ban hành danh mục lĩnh vực, cơ sở phát thải khí nhà kính phải thực hiện kiểm kê khí nhà kính và nộp báo cáo kiểm kê khí nhà kính cấp cơ sở về cơ quan"}\n{"id": "B-A-fc67b5c47453de0e", "claim": "Phát triển cộng đồng Hành trình số hóa quản trị nhân sự cũng sẽ được đẩy mạnh, với việc mở rộng thêm các phân hệ mới, hướng tới mục tiêu hình thành hệ sinh thái quản lý nhân sự số", "evidence": "QUẢN TRỊ NGUỒN NHÂN LỰC 6.2 DỰ ÁN NỔI BẬT VỀ NHÂN SỰ NĂM 2025 Năm 2025, Ban Dự án Số hóa Quản trị Nhân sự chính thức khởi động lộ trình nâng cấp hệ thống với mục tiêu tối ưu trải nghiệm người dùng: Giao diện thân thiện, tài chính, pháp chế, ... cho các Công ty thành viên. Quan hệ Cổ đông Việc áp dụng công nghệ vào hoạt động quản trị nhân sự cũng được chú trọng. Hệ thống quản trị nhân sự đã chính thức đi vào vận hành trên toàn Tập đoàn từ tháng"}\n{"id": "B-A-d7a9fbc340ed2a35", "claim": "Là một trong những tập đoàn công nghiệp hàng đầu Việt Nam, Hòa Phát coi quản trị tốt và đạo đức kinh doanh không chỉ là nền tảng của sự phát triển bền vững, mà còn là yếu tố cốt lõi để củng cố niềm tin", "evidence": "của cổ đông, khách hàng, đối tác và xã hội. Tập đoàn coi đạo đức kinh doanh là nền tảng của quản trị bền vững, đóng vai trò then chốt trong việc kiểm soát rủi ro, bảo vệ uy tín doanh nghiệp và bảo đảm tuân thủ các tiêu chuẩn ESG trong bối cảnh ngành đạo đức kinh doanh có liên quan. Tính liêm chính, minh bạch và công bằng là những nguyên tắc cốt lõi định hướng toàn bộ hoạt động sản xuất kinh doanh của Tập đoàn. BÁO CÁO PHÁT TRIỂN BỀN VỮNG"}\n{"id": "B-A-7d5864e8f2370a61", "claim": "Cuối cùng, các trang trại vận hành mô hình khép kín với hệ thống xử lý chất thải đồng bộ và truy xuất nguồn gốc", "evidence": "kín, đảm bảo vệ sinh và có hệ thống xử lý chất thải đồng bộ. Đội ngũ R&D xây dựng chương trình dinh dưỡng và thú y chuyên biệt, cùng với việc thực hiện truy xuất nguồn gốc đầy đủ, nhằm cung cấp sản phẩm thịt heo an toàn, chất lượng Với ưu thế chủ động được phôi nguyên liệu đầu vào do chính Tập đoàn sản xuất, Hòa Phát thực hiện khép kín chuỗi sản xuất, kiểm soát chất lượng tối đa và truy xuất được nguồn gốc sản phẩm. Dự kiến, Công ty sẽ hoàn thành lắp đặt các dây chuyền mới và ra sản phẩm vào quý 3/2025."}\n{"id": "B-A-76cd8dd9ac313791", "claim": "2025 Trong bối cảnh ngành công nghiệp sản xuất chịu tác động mạnh mẽ từ biến động kinh tế toàn cầu, yêu cầu giảm phát thải, tiêu chuẩn ESG ngày càng khắt khe và áp lực cạnh tranh", "evidence": "đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ. Trọng tâm chiến lược tập trung vào ba trụ cột ESG chính: tiến công nghệ, kỹ thuật, nâng cao chất lượng, mở rộng các dải mác sản phẩm nhằm đáp ứng nhu cầu ngày càng đa dạng và khắt khe của khách hàng trong nước và quốc tế. ESG Hòa Phát tích hợp các mục tiêu phát triển bền vững (SDGs) vào chiến lược kinh doanh, phù"}\n{"id": "B-A-fcfd71b1917994fd", "claim": "Chuyển đổi số được triển khai với mục tiêu: Chuẩn hóa và minh bạch hóa quản trị Tối ưu hiệu suất vận hành Nâng cao trải nghiệm khách hàng Giảm tiêu hao tài nguyên và phát thải gián tiếp", "evidence": "ưu tiêu hao năng lượng, nguyên vật liệu và tài nguyên đầu vào; giảm sử dụng giấy tờ và hồ sơ vật lý; tăng khả năng theo dõi phát thải gián tiếp thông qua dữ liệu vận hành; lò với mục tiêu giảm lượng than cốc tiêu hao trên mỗi tấn gang. Qua đó, lượng CO2 sinh ra trong quá trình cháy và hoàn nguyên cũng giảm tương ứng. Đây là biện pháp cốt lõi trong giảm phát thải trực tiếp tại khu vực lò cao và nâng cao hiệu suất luyện gang."}\n{"id": "B-A-4fd2e002f6e26079", "claim": "Tăng khả năng thích ứng với biến động thị trường Chiến lược này đồng thời đóng góp trực tiếp vào 3 trụ cột ESG: QUẢN TRỊ: Hệ thống số hóa giúp tăng cường minh bạch thông", "evidence": "đổi khí hậu, sản xuất có trách nhiệm và kinh tế tuần hoàn, đồng thời nâng cao năng lực cạnh tranh và khả năng thích ứng của Tập đoàn trong ngành thép đang biến đổi mạnh mẽ. Trọng tâm chiến lược tập trung vào ba trụ cột ESG chính: dựng cơ cấu quản trị công bằng, minh bạch, tăng cường hệ thống tuân thủ, và Trong năm 2025, Hòa Phát đã xây dựng Chiến lược Phát triển Bền vững để định hướng hoạt động kinh doanh theo các xu thế dài hạn toàn cầu về ứng phó biến"}\n{"id": "B-A-1d7b1fa7a08769ab", "claim": "thời gian và giới hạn báo cáo 2-4 Trình bày lại thông tin Đây là báo cáo công bố lần đầu tiên 2-5 Đảm bảo số liệu từ Đơn vị bên ngoài Số liệu về phát thải KNK của toàn tập đoàn đã được xác nhận bởi BSI.", "evidence": "2-2 Các Đơn vị được bao gồm trong Báo cáo Phát triển bền vững Phạm vi. thời gian và giới hạn báo cáo 2-3 Kỳ báo cáo. tần suất và thông tin liên hệ chỉnh biên giới carbon của EU (CBAM), chính thức áp dụng từ ngày 1/10/2023. Năm 2025 là năm đầu tiên Tập Đoàn Hòa Phát lập báo cáo phát thải KNK toàn tập đoàn. Báo cáo này được thẩm định và chứng nhận kết quả bởi Tổ chức Tiêu"}\n{"id": "B-A-c9d1caba90ba1291", "claim": "Quản trị Phát triển bền vững hoạt động quản lý các tác động của Công ty 2-13 Sự phân chia trách nhiệm về việc quản lý các tác động của Công ty Quản trị Phát triển bền vững 2-14 Vai trò của cơ quan quản trị cao nhất trong Báo cáo Phát triển", "evidence": "QUẢN LÝ CHẤT LƯỢNG 7.2 BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 PHẦN 7 QUẢN TRỊ VÀ ĐỔI MỚI Quản trị Doanh nghiệp và Quản lý rủi ro Quản lý chất lượng Phát triển đổi mới sáng tạo Tập đoàn cam kết xây dựng một khung quản trị bền vững vững chắc nhằm đảm bảo tăng trưởng dài hạn, minh bạch và trách nhiệm đối với các bên liên quan. Cơ cấu quản trị bền vững đã được thiết lập, trong đó Hội đồng Quản trị chịu trách"}\n{"id": "B-A-3a785947ec715b9a", "claim": "nghiên cứu xây dựng 102-5 Phát thải KNK Phạm vi 1 Kết quả kiểm kê phát thải khí nhà kính năm 2025 GRI 102 – Biến đổi khí hậu 2025 102-6 Phát thải KNK Phạm vi 1", "evidence": "TỔNG PHÁT THẢI KNK TOÀN TẬP ĐOÀN 2025 (Theo phạm vi) BÁO CÁO PHÁT TRIỂN BỀN VỮNG 2025 Phạm vi 1 PHẦN 5 XANH HÓA SẢN XUẤT Phạm vi 2 Như vậy, việc kiểm soát và giảm thiểu phát thải khí nhà kính cần tập trung vào HÀNH ĐỘNG GIẢM THIỂU VÀ THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU 5.1 THÍCH ỨNG VỚI BIẾN ĐỔI KHÍ HẬU TRONG LĨNH VỰC NÔNG NGHIỆP KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2025 KIỂM KÊ PHÁT THẢI KHÍ NHÀ KÍNH NĂM 2024"}\n{"id": "B-A-6c82e13e97482ca4", "claim": "Kết quả kiểm kê phát thải khí nhà kính năm 2025 102-8 Cường độ phát thải Do các nhà máy sử dụng công nghệ sản xuất thép khác nhau và có nhiều loại sản phẩm khác nhau, chúng tôi đang sắp xếp nguồn lực phù hợp để thu thập và tính toán chỉ tiêu này", "evidence": "Từ góc độ vòng đời, thép có thể được xem là loại vật liệu có tác động môi trường rất thấp so với nhiều vật liệu khác do khả năng sử dụng và tái chế vô hạn. Lượng phát thải carbon khi sản xuất thép khác nhau đáng kể tùy thuộc vào quy trình sản xuất thép đó nằm ở giai Lĩnh vực sản xuất gang thép và sản phẩm thép của Tập đoàn Hòa Phát tiếp tục là nguồn phát thải khí nhà kính lớn nhất, chiếm tỷ trọng vượt trội trên 99% tổng lượng phát thải toàn tập đoàn. Đặc thù ngành nghề đòi hỏi tiêu thụ nhiều năng"}\n{"id": "B-A-41f82976dcf22e6f", "claim": "Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý", "evidence": "Tấn 131.639 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải nguy hại) Tấn 23.539 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào quy trình xử lý Tổng khối lượng chất thải NGUY HẠI được xử lý thải bỏ"}\n{"id": "B-A-5e14b7f7cf7978d8", "claim": "GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo tại các", "evidence": "TABLE DỮ LIỆU VỀ MÔI TRƯỜNG TIÊU CHUẨN CHUNG ĐƠN VỊ TÍNH 2025 CHỈ SỐ THEO TIÊU CHUẨN GRI BIẾN ĐỔI KHÍ HẬU/ PHÁT THẢI KHÍ NHÀ KÍNH & NĂNG LƯỢNG Tổng phát thải khí nhà kính trực tiếp (Phạm vi 1) (CO2e) Tấn 22.540.603 305-1- Phát thải khí nhà kính (GHG) trực tiếp (Phạm vi 1) Tổng phát khí thải nhà kính gián tiếp (Phạm vi 2) (CO2e) Tấn 933.876 305- 2 Phát thải khí nhà kính (GHG) gián tiếp (tương đương CO2) từ năng lượng (Phạm vi 2) Tổng phát khí thải nhà kính (Phạm vi 1+2) (CO2e Tấn 23.474.479 Mức tiêu thụ năng lượng tại các cơ sở GJ 193.403.521 302-1 Tiêu thụ năng lượng trong tổ chức Năng lượng tiêu thụ được cung cấp từ lưới điện tại các địa điểm % 4,5% Năng lượng tiêu thụ là năng lượng tái tạo"}\n{"id": "B-A-3d27f42daf4e1451", "claim": "Triệu lít N/A Chất thải nguy hại Tấn 246.867 306-3 Chất thải phát sinh Chất thải không nguy hại Tấn 6.838.336 306-3 Chất thải phát sinh Chất thải khác", "evidence": "Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát sinh Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.981.878 306-4 Chất thải được chuyển giao khỏi quy trình xử lý (Nghìn tấn) Tổng khối lượng chất thải rắn phát sinh 7,085 Chất thải không nguy hại 6,838 Chất thải nguy hại 247 LƯỢNG CHẤT THẢI THEO CÁC HƯỚNG XỬ LÝ NĂM 2025"}\n{"id": "B-A-bda44f82ac13bf75", "claim": "Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát sinh Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.981.878 306-4 Chất thải được chuyển giao khỏi quy trình xử lý", "evidence": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại) Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý"}\n{"id": "B-A-f55d1e37f4f4fe86", "claim": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại)", "evidence": "Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát sinh Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.981.878 306-4 Chất thải được chuyển giao khỏi quy trình xử lý"}\n{"id": "B-A-088d9fbf80b0a727", "claim": "Tấn 131.639 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải nguy hại) Tấn 23.539 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý", "evidence": "306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác (bao gồm Ủ phân hữu cơ) Tấn 257.790 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142"}\n{"id": "B-A-c589a13a306facb0", "claim": "Tổng lượng chất thải không nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.768.958 306-4 Chất thải được chuyển giao khỏi quy trình xử lý", "evidence": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại) Tấn N/A Tổng lượng chất thải phát sinh Tấn 7.085.203 306-3 Chất thải phát sinh Tổng khối lượng chất thải được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 6.981.878 306-4 Chất thải được chuyển giao khỏi quy trình xử lý"}\n{"id": "B-A-14e68fb049a24e56", "claim": "Chất thải tái sử dụng (chất thải không nguy hại) Tấn 1.636.026 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải không nguy hại) Tấn 4.875.142", "evidence": "Tổng lượng chất thải nguy hại được sử dụng cho các mục đích khác thay vì bị vứt/tiêu hủy Tấn 212.920 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Chất thải tái sử dụng (chất thải nguy hại) Tấn 131.639 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Tái chế (chất thải nguy hại) Tấn 23.539 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Các hoạt động phục hồi khác Tấn 57.741 306-4 Chất thải được chuyển giao khỏi quy trình xử lý"}\n{"id": "B-A-d7ab4285f56f6249", "claim": "Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào quy trình xử lý Tổng khối lượng chất thải NGUY HẠI được xử lý thải bỏ", "evidence": "Tấn 33.948 306-5 Chất thải được đưa vào quy trình xử lý Tổng khối lượng chất thải KHÔNG NGUY HẠI được xử lý thải bỏ (Đốt không thu hồi năng lượng & chôn lấp) Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý"}\n{"id": "B-A-c30f3ff61180293e", "claim": "Tấn 69.377 306-5 Chất thải được đưa vào quy trình xử lý", "evidence": "Tổng khối lượng chất thải được xử lý thải bỏ Tấn 103.325 306-5 Chất thải được đưa vào quy trình xử lý Tổng khối lượng chất thải NGUY HẠI được xử lý thải bỏ Quản lý chất thải rắn 306-3 Lượng chất thải tạo ra Quản lý chất thải rắn 306-4 Chất thải được chuyển giao khỏi quy trình xử lý Quản lý chất thải rắn 306-5 Chất thải được đưa vào quy trình xử lý"}\n{"id": "B-A-45de5e3c59ad910d", "claim": "vague sustainability, SAF and offset/reforestation claims", "evidence": "TABLE Trường | Giá trị Phạm vi | Netherlands - Aviation Dạng case | vague sustainability, SAF and offset/reforestation claims Trạng thái pháp lý | DUTCH_COURT_FOUND_15_OF_19_STATEMENTS_MISLEADING Nhãn test kỳ vọng | CONTRADICTED Độ ưu tiên rủi ro | CRITICAL Split | evaluation Điều kiện dùng train | silver_until_exact_19_statements_archived Nguồn cần tải và kiểm tra Source ID Loại / vai trò URL hoặc trạng thái KLM_JUDGMENT_2024 court_judgment / adjudication_evidence Trang: n/a https://uitspraken.rechtspraak.nl/details?id=ECLI%3ANL%3ARBAMS%3A2024 KLM_ANNUAL_REPORT_2020 annual_report_and_financial_statements / corporate_context Trang: 7 https://img.static-kl.com/m/334e4d894aae270e/original/KLM-"}\n{"id": "B-A-01796b115af6995e", "claim": "were vague and general and that the presentation of sustainable aviation fuel and reforestation painted", "evidence": "TABLE Trường | Giá trị Phạm vi | Netherlands - Aviation Dạng case | vague sustainability, SAF and offset/reforestation claims Trạng thái pháp lý | DUTCH_COURT_FOUND_15_OF_19_STATEMENTS_MISLEADING Nhãn test kỳ vọng | CONTRADICTED Độ ưu tiên rủi ro | CRITICAL Split | evaluation Điều kiện dùng train | silver_until_exact_19_statements_archived Nguồn cần tải và kiểm tra Source ID Loại / vai trò URL hoặc trạng thái KLM_JUDGMENT_2024 court_judgment / adjudication_evidence Trang: n/a https://uitspraken.rechtspraak.nl/details?id=ECLI%3ANL%3ARBAMS%3A2024 KLM_ANNUAL_REPORT_2020 annual_report_and_financial_statements / corporate_context Trang: 7 https://img.static-kl.com/m/334e4d894aae270e/original/KLM-"}\n{"id": "B-A-69cde2b9ad361bfd", "claim": "Classify vague superiority/future language separately from quantified emissions claims.", "evidence": "TABLE Trường | Giá trị Phạm vi | Netherlands - Aviation Dạng case | vague sustainability, SAF and offset/reforestation claims Trạng thái pháp lý | DUTCH_COURT_FOUND_15_OF_19_STATEMENTS_MISLEADING Nhãn test kỳ vọng | CONTRADICTED Độ ưu tiên rủi ro | CRITICAL Split | evaluation Điều kiện dùng train | silver_until_exact_19_statements_archived Nguồn cần tải và kiểm tra Source ID Loại / vai trò URL hoặc trạng thái KLM_JUDGMENT_2024 court_judgment / adjudication_evidence Trang: n/a https://uitspraken.rechtspraak.nl/details?id=ECLI%3ANL%3ARBAMS%3A2024 KLM_ANNUAL_REPORT_2020 annual_report_and_financial_statements / corporate_context Trang: 7 https://img.static-kl.com/m/334e4d894aae270e/original/KLM-"}\n{"id": "B-A-c5fc94240af0e49b", "claim": "Do not treat SAF or offsets as equivalent to zero-impact flying.", "evidence": "TABLE Trường | Giá trị Phạm vi | Netherlands - Aviation Dạng case | vague sustainability, SAF and offset/reforestation claims Trạng thái pháp lý | DUTCH_COURT_FOUND_15_OF_19_STATEMENTS_MISLEADING Nhãn test kỳ vọng | CONTRADICTED Độ ưu tiên rủi ro | CRITICAL Split | evaluation Điều kiện dùng train | silver_until_exact_19_statements_archived Nguồn cần tải và kiểm tra Source ID Loại / vai trò URL hoặc trạng thái KLM_JUDGMENT_2024 court_judgment / adjudication_evidence Trang: n/a https://uitspraken.rechtspraak.nl/details?id=ECLI%3ANL%3ARBAMS%3A2024 KLM_ANNUAL_REPORT_2020 annual_report_and_financial_statements / corporate_context Trang: 7 https://img.static-kl.com/m/334e4d894aae270e/original/KLM-"}\n{"id": "B-A-59901c2a8533a4f4", "claim": "The judgment concerned specific past advertisements, not every KLM sustainability", "evidence": "Nguồn cần tải và kiểm tra Source ID Loại / vai trò URL hoặc trạng thái KLM_JUDGMENT_2024 court_judgment / adjudication_evidence Trang: n/a https://uitspraken.rechtspraak.nl/details?id=ECLI%3ANL%3ARBAMS%3A2024 KLM_ANNUAL_REPORT_2020 annual_report_and_financial_statements / corporate_context Trang: 7 https://img.static-kl.com/m/334e4d894aae270e/original/KLM-2020-Annual-Re Output JSON tối thiểu { \\"case_id\\": \\"KLM-2024\\", \\"claim_id\\": \\"KLM-CLAIM-001\\", \\"status\\": \\"CONTRADICTED\\", \\"risk_band\\": \\"CRITICAL\\", \\"evidence_source_ids\\": [ \\"KLM_JUDGMENT_2024\\", \\"KLM_ANNUAL_REPORT_2020\\" ], \\"requires_human_review\\": true, \\"legal_scope_note\\": \\"The judgment concerned specific past advertisements, not every KLM sustain"}\n{"id": "B-A-d0f64cc136a1101a", "claim": "Dạng case | vague sustainability, SAF and offset/reforestation claims", "evidence": "HỒ SƠ CASE TEST / DEMO KLM-2024 - KLM Royal Dutch Airlines Trường Giá trị Phạm vi Netherlands - Aviation Dạng case vague sustainability, SAF and offset/reforestation claims Trạng thái pháp lý DUTCH_COURT_FOUND_15_OF_19_STATEMENTS_MISLEADING Nhãn test kỳ vọng CONTRADICTED Độ ưu tiên rủi ro CRITICAL Split evaluation Điều kiện dùng train silver_until_exact_19_statements_archived Claim dùng để kiểm thử Join us in creating a more sustainable future. Nguồn claim: One of the advertisements summarized in reporting on the official court judgment (nguyên văn ngắn) Tóm tắt evidence The Amsterdam court found 15 of 19 assessed environmental statements misleading. It found that some were vague and general"}\n{"id": "B-A-ec683ce5397a529b", "claim": "\\"legal_scope_note\\": \\"The judgment concerned specific past advertisements, not every KLM sustainability statement.\\"", "evidence": "HỒ SƠ CASE TEST / DEMO KLM-2024 - KLM Royal Dutch Airlines Trường Giá trị Phạm vi Netherlands - Aviation Dạng case vague sustainability, SAF and offset/reforestation claims Trạng thái pháp lý DUTCH_COURT_FOUND_15_OF_19_STATEMENTS_MISLEADING Nhãn test kỳ vọng CONTRADICTED Độ ưu tiên rủi ro CRITICAL Split evaluation Điều kiện dùng train silver_until_exact_19_statements_archived Claim dùng để kiểm thử Join us in creating a more sustainable future. Nguồn claim: One of the advertisements summarized in reporting on the official court judgment (nguyên văn ngắn) Tóm tắt evidence The Amsterdam court found 15 of 19 assessed environmental statements misleading. It found that some were vague and general"}\n{"id": "B-A-1deef7830c2ca481", "claim": "130 Kiến tạo giá trị 159 Báo cáo lưu chuyển tiền tệ 160 Thuyết minh báo cáo tài chính 133 Trách nhiệm với môi trường 141 Phát triển xã hội bền vững", "evidence": "159 Báo cáo lưu chuyển tiền tệ 160 Thuyết minh báo cáo tài chính 125 Tầm quan trọng các lĩnh vực trọng yếu 130 Kiến tạo giá trị 133 Trách nhiệm với môi trường 141 Phát triển xã hội bền vững"}\n{"id": "B-A-e97c1534f36b5756", "claim": "Chúng tôi hướng tới việc tạo ra giá trị bền vững cho xã hội và thực hiện sứ mệnh: “Vì một cuộc sống khỏe đẹp hơn.” Với những kinh nghiệm và nền tảng tích luỹ được qua quá trình 50 năm hoạt động và phát triển, DHG Pharma luôn sẵn sàng đối diện với những cơ hội và thách thức, chủ động tìm kiếm những giải pháp trong mọi hoạt động sản xuất kinh doanh.", "evidence": "Hành trình hơn 5 thập kỷ hoạt động và kiên định với sứ mệnh “Vì một cuộc sống khỏe đẹp hơn”, DHG Pharma không chỉ dừng lại ở việc sản xuất và cung cấp những viên thuốc chất lượng cao. Hơn thế nữa, chúng tôi mang trong mình khát vọng cháy bỏng về một cộng đồng khỏe mạnh, đặc biệt là những mảnh đời còn khó khăn nơi vùng sâu, vùng xa. Với phương châm “Lấy lợi ích cộng đồng làm khởi sự cho mọi hoạt động”, DHG Pharma đã bền bỉ gieo những hạt mầm tử tế, kiến tạo nên những giá trị nhân văn sâu sắc cho xã hội. GRI Sáng Kiến Báo Cáo Toàn Cầu UBCL Ủy ban Chiến lược 50 năm đồng hành cùng sức khỏe cộng đồng, DHG Pharma tiếp tục giữ vững sứ mệnh “Vì một cuộc sống khỏe đẹp hơn”. Trên nền tảng vững chắc đã"}\n{"id": "B-A-1ad097554ebc9b8d", "claim": "Bên cạnh đó, DHG Pharma không ngừng cải tiến mô hình quản trị, nâng cao chất lượng sản phẩm và dịch vụ, luôn hướng đến mục tiêu gia tăng giá trị bền vững cho tất cả các bên liên quan.", "evidence": "2 KẾ HOẠCH KINH DOANH NĂM 2025 Cải tiến sản xuất và nâng cao chất lượng Cải thiện năng suất sản xuất thông qua tối ưu hóa chi phí, giảm hao hụt, quản lý ngân sách sử dụng của từng đơn vị một cách hiệu quả hướng tới quy trình sản xuất hiệu suất cao và phù hợp với môi trường kinh doanh đang thay đổi. Kế hoạch kinh doanh năm 2025 Liên tục nâng cao hệ thống quản lý chất lượng và sản xuất, đáp ứng các tiêu chuẩn quốc tế như EU-GMP, Japan-GMP, đồng thời duy trì và tuân thủ đầy đủ các tiêu chuẩn trong nước, nhằm nâng cao chất lượng sản phẩm và gia tăng năng lực cạnh tranh. BÁO CÁO CỦA BAN TỔNG GIÁM ĐỐC Khối Sản xuất Các dự án đang triển khai thực hiện Chất lượng sản phẩm luôn là yếu tố cốt lõi tron"}\n{"id": "B-A-cbba4ffcf12bf0be", "claim": "Với tầm nhìn dài hạn và niềm tin rằng “Đổi mới là chìa khóa phát triển bền vững,” DHG Pharma hướng tới việc chủ động tạo ra sự thay đổi và linh hoạt thích ứng để luôn kịp thời nắm bắt các cơ hội kinh doanh, hợp tác và phát triển.", "evidence": "Chúng tôi hướng tới việc tạo ra giá trị bền vững cho xã hội và thực hiện sứ mệnh: “Vì một cuộc sống khỏe đẹp hơn.” Với những kinh nghiệm và nền tảng tích luỹ được qua quá trình 50 năm hoạt động và phát triển, DHG Pharma luôn sẵn sàng đối diện với những cơ hội và thách thức, chủ động tìm kiếm những giải pháp trong mọi hoạt động sản xuất kinh doanh. Bên cạnh đó, DHG Pharma không ngừng cải tiến mô hình quản trị, nâng cao chất lượng sản phẩm và dịch vụ, luôn hướng đến mục tiêu gia tăng giá trị bền vững cho tất cả các bên liên quan. Thông qua những hoạt động trên, DHG Pharma tiếp tục khẳng định cam kết nâng cao hệ thống kiểm soát nội bộ, thích ứng linh hoạt với các thay đổi của môi trường kinh do"}\n{"id": "B-A-69c55bd34f990086", "claim": "Cải tiến về Chính sách, quản lý Tích hợp yếu tố phát triển bền vững (ESG) trong chính sách nội bộ, phản ánh cam kết của doanh nghiệp với xã hội và môi trường.", "evidence": "DHG Pharma cam kết tích hợp các yếu tố môi trường, xã hội và cộng đồng (ESG) vào hoạt động sản xuất – kinh doanh, nhằm phát triển bền vững và tạo ra giá trị lâu dài cho tất cả các bên liên quan. Chính sách lương thưởng và phúc lợi được duy trì cạnh tranh, tạo động lực và môi trường làm việc tích cực cho nhân viên. Năm 2024, Công ty tiếp tục được vinh danh trong “Top 10 môi trường làm việc tốt nhất ngành Dược/Thiết bị y tế” và “Top 100 nơi làm việc tốt nhất Việt Nam”. Công ty cũng thực hiện nghiêm túc chính sách cân bằng giới trong HĐQT, hiện có 01/07 thành viên không điều hành là nữ, phản ánh cam kết về bình đẳng giới trong quản trị doanh nghiệp. Công ty đã và đang tích hợp dần các yếu tố ph"}\n{"id": "B-A-c0b854dfdc6d5c45", "claim": "Với tầm nhìn dài hạn, Chính phủ đặt mục tiêu GDP năm 2025 tăng trưởng ít nhất 8%, vượt xa dự báo 6,1% - 6,7% từ các tổ chức quốc tế, thể hiện quyết tâm mạnh mẽ trong việc thúc đẩy phát triển kinh tế bền vững.", "evidence": "TABLE TỔNG QUAN VÀ Bước sang năm 2025, ngành dược Việt Nam đứng trước cơ hội phát triển mạnh mẽ nhờ bốn yếu tố chính: TRIỂN VỌNG NGÀNH DƯỢC 1. Mục tiêu tăng trưởng năm 2025 ở mức 8%, với GDP đầu người tăng từ 4.700 USD năm 2024 lên mức dự kiến 4.900 USD năm 2025 sẽ tăng khả năng chi tiêu cho sức khỏe Tăng trưởng Năm 2024, nền kinh tế Việt Nam duy trì đà phục hồi ấn tượng và đạt mức tăng trưởng đáng kể. Tổng sản phẩm quốc nội của người dân. Kết hợp với xu hướng chăm sóc sức khỏe chủ động của người dân kinh tế và nhu cầu (GDP) ước tính đạt 476,3 tỷ USD, tăng 7,09% so với năm 2023. Thu nhập bình quân đầu người ghi nhận mức 4.700 USD, ngày càng tăng. Theo nghiên cứu của Roland Berger năm 2023, 7"}\n{"id": "B-A-1374e564052a9269", "claim": "Trí tuệ nhân tạo và tự động hóa đang dần thay đổi quy trình nghiên cứu, sản xuất và phân phối dược phẩm.", "evidence": "Pháp, Mỹ, Đức, Ấn Độ và Bỉ. Điểm yếu của doanh nghiệp trong nước chính là thị trường manh mún, quy mô nhỏ lẻ và năng lực tài chính khá yếu. Khả năng tăng đầu tư vào nghiên cứu và trang bị công nghệ sản xuất bị hạn chế, khó khăn trong đầu tư dự án quy mô lớn, trong việc mua các bản quyền sở hữu trí tuệ về dược, thiếu đội ngũ nhân sự chất lượng cao phục vụ công tác nghiên cứu phát triển sản phẩm. Phần lớn các doanh nghiệp nội địa đều tập trung sản xuất các loại thuốc thông dụng, trong khi đó các loại thuốc chuyên khoa, đặc trị, đòi hỏi công nghệ bào chế hiện đại lại không được quan tâm. TABLE TỔNG QUAN VÀ Bước sang năm 2025, ngành dược Việt Nam đứng trước cơ hội phát triển mạnh mẽ nhờ bốn yếu "}\n{"id": "B-A-02efac449e11dbf2", "claim": "Các chiến lược và mục tiêu đề ra không chỉ hướng đến kết quả ngắn hạn, mà còn tập trung vào phát triển bền vững và thành công dài hạn, góp phần dẫn dắt sự tăng trưởng của toàn ngành Dược Việt Nam.", "evidence": "Hiện thực hóa chiến lược Phát triển bền vững với tầm nhìn trở thành doanh nghiệp dẫn đầu Việt Nam về mảng chăm sóc sức khỏe, đồng thời duy trì vững vị trí dẫn đầu ngành Công nghiệp dược Việt Nam, hướng đến mục tiêu “chuyển đổi xanh, tăng trưởng bền vững” với các giá trị nền tảng như: 17 Mục tiêu phát triển bền vững của Liên Hợp Quốc nhằm thúc đẩy sự phát triển bền vững toàn cầu. Nâng cao năng lực quản trị, đầu tư mạnh mẽ vào công nghệ, đẩy mạnh hoạt động sản xuất kinh doanh, nâng cao chất lượng sản phẩm. Tăng cường hợp tác với các Trung tâm, Viện, Trường, Nhà khoa học đầu ngành trong lĩnh vực Dược phẩm để phát triển danh mục sản phẩm và đảm bảo sự tiên phong trong phát triển Ngành Dược. Định"}\n{"id": "B-A-eebeb3ddbf46435d", "claim": "Đây không chỉ là sự kiện tri ân mà còn là hành trình gắn kết, sẻ chia, khẳng định mối quan hệ bền vững giữa DHG và khách hàng.", "evidence": "Đây cũng là sự khẳng định mạnh mẽ về cam kết của DHG Pharma trong hành trình hướng tới sự phát triển bền vững toàn diện, hài hòa giữa tăng trưởng kinh tế, trách nhiệm xã hội và bảo vệ môi trường. Đối với các vấn đề trọng yếu, DHG Pharma xác định phương pháp quản trị dựa trên bối cảnh ngành dược trong nước và thế giới, thông tin phản hồi và mối quan tâm giữa các bên liên quan. Đặc biệt, chương trình Hội nghị khách hàng “Dấu ấn phương Nam” đã trở thành một dấu mốc khó quên, thu hút sự tham gia của 6.000 khách hàng. Với DHG Pharma, đây là một hành trình gắn kết niềm tin, sẻ chia cảm xúc và cùng nhau kiến tạo tương lai. Không chỉ dừng lại ở chính sách tiền lương cạnh tranh,"}\n{"id": "B-A-4ece3006a0876fed", "claim": "DHG Pharma cam kết tích hợp các yếu tố môi trường, xã hội và cộng đồng (ESG) vào hoạt động sản xuất – kinh doanh, nhằm phát triển bền vững và tạo ra giá trị lâu dài cho tất cả các bên liên quan.", "evidence": "Quản trị hiệu quả, minh bạch gắn liền với trách nhiệm với môi trường và góp phần xây dựng cộng đồng, xã hội. Nội dung báo cáo là minh chứng cho sự cam kết mạnh mẽ của DHG Pharma trên con đường phát triển bền vững trước kỳ vọng của các bên liên quan. Bước 05 Nguyên tắc phát triển bền vững được tích hợp vào chiến lược tổng thể của DHG Pharma. Chúng tôi hướng tới việc tạo ra giá trị bền vững cho xã hội và thực hiện sứ mệnh: “Vì một cuộc sống khỏe đẹp hơn.” Với những kinh nghiệm và nền tảng tích luỹ được qua quá trình 50 năm hoạt động và phát triển, DHG Pharma luôn sẵn sàng đối diện với những cơ hội và thách thức, chủ động tìm kiếm những giải pháp trong mọi hoạt động sản xuất kinh doanh. Bên cạn"}\n{"id": "B-A-f77ea81a37c0f650", "claim": "Kiểm soát chặt chẽ việc sử dụng tài nguyên, năng lượng, nước và phát thải trong quá trình sản xuất.", "evidence": "Kiểm soát chặt chẽ khí thải từ sản xuất và nồi hơi. Nước, không khí, chất thải Quản lý xung đột lợi ích Quản lý rủi ro MÔI TRƯỜNG Năng lượng Nước, không khí, chất thải Đồng thời, tối ưu hóa danh mục sản phẩm, kiểm soát chặt chất lượng đầu vào - đầu ra và đảm bảo minh bạch thông tin trong toàn bộ quá trình cung ứng. Chuỗi cung ứng bền vững DHG Pharma tạo ra những sản phẩm chất lượng nhằm đảm bảo mỗi sản phẩm đến tay người dùng đều đạt chất lượng, hiệu quả và an toàn tuyệt đối. Tuân thủ nghiêm ngặt các tiêu chuẩn quốc tế trong sản xuất, kiểm soát chặt chẽ nguyên liệu đầu vào, theo dõi sát sao quá trình phân phối và sử dụng thuốc, hướng đến mục tiêu bảo vệ sức khỏe cộng đồng."}\n{"id": "B-A-0a9f5958986175a0", "claim": "MÔI TRƯỜNG Thực hiện các biện pháp giảm thiểu khí thải nhà kính, tái chế chất thải và đầu tư vào công nghệ sản xuất sạch hơn.", "evidence": "TỔNG QUAN VỀ NGUỒN NHÂN LỰC Số lượng nhân sự năm 2020 - 2024 3 Tăng yêu cầu pháp lý và chi phí tuân thủ môi trường Xây dựng và triển khai lộ trình giảm phát thải carbon, ứng dụng công nghệ sạch, công nghệ tái chế chất thải trong sản xuất. Kiểm soát chặt chẽ khí thải từ sản xuất và nồi hơi. Nước, không khí, chất thải Quản lý xung đột lợi ích Quản lý rủi ro MÔI TRƯỜNG Năng lượng Nước, không khí, chất thải"}\n{"id": "B-A-6ef45091a52eaf60", "claim": "Đồng hành cùng nỗ lực của Việt Nam hướng đến phát thải ròng bằng 0 vào năm 2025.", "evidence": "(*) Tại ngày 31 tháng 12 năm 2024, vay ngắn hạn ngân hàng thể hiện các khoản vay từ Ngân hàng Thương mại Cổ phần Ngoại thương Việt Nam - Chi nhánh Cần Thơ với hạn mức tối đa là 1.300.000.000.000 đồng. Khoản tín dụng này có thể rút bằng đồng Việt Nam và hết hiệu lực giải ngân vào ngày 15 tháng 8 năm 2025. Thời hạn cho vay của mỗi khoản nợ tối đa là 6 tháng. Trải qua hành trình 50 năm xây dựng và phát triển, DHG Pharma luôn hiểu rằng sự thành công không chỉ xuất phát từ nỗ lực của riêng mình, mà còn đến từ sự hợp tác, đồng hành đồng lợi cùng Quý khách hàng và đối tác. Sự gắn kết chặt chẽ này đã giúp chúng ta cùng nhau phát triển vững mạnh, tạo ra những giá trị thiết thực và lợi ích bền vững. C"}\n{"id": "B-A-38029ab626ada89f", "claim": "DHG Pharma đã và đang triển khai nhiều giải pháp giảm phát thải, khí thải như: Đầu tư hệ thống năng lượng mặt trời tại nhà máy sản xuất Dược phẩm tại khu công nghiệp Tân Phú Thạnh, triển khai các dự án tiết kiệm năng lượng,....", "evidence": "Nguồn năng lượng sạch từ hệ thống điện mặt trời hiện đang được sử dụng trực tiếp cho hoạt động sản xuất tại nhà máy, góp phần tối ưu chi phí vận hành, giảm đáng kể lượng khí nhà kính phát thải ra môi trường, đồng thời thể hiện cam kết của DHG Pharma trong việc thúc đẩy phát triển xanh và bền vững. Song song với các giải pháp tiết kiệm năng lượng được áp dụng nhiều năm qua như: truyền thông nội bộ để nâng cao ý thức sử dụng điện hiệu quả; Năm 2024, DHG Pharma đã đầu tư lắp đặt hệ thống điện mặt trời áp mái cho các khu vực nhà xưởng, góp phần giảm thiểu đáng kể tiêu hao năng lượng, hướng đến mục tiêu sản xuất xanh và nghiêm túc thực thi các chiến lược phát triển bền vững mạnh mẽ hơn. Hệ thống "}\n{"id": "B-A-f5effd2c56aa86c7", "claim": "Tỷ lệ nợ vay trên tổng tài sản vẫn ở mức an toàn (10,9%) và tỷ lệ đòn bẩy tài chính (D/E) tăng nhẹ từ 0,12 lên 0,16 lần – vẫn trong vùng kiểm soát tốt, phản ánh cơ cấu tài chính lành mạnh và chính sách sử dụng vốn thận trọng.", "evidence": "Lần 0,1 0,1 0,1 0,0 0,0 3.2 Chỉ tiêu về cơ cấu vốn 3.2.1 Tỷ số nợ trên tổng tài sản % 25,2% 18,6% 19,8% 17,9% 17,0% 3.2.2 Tỷ số nợ trên vốn chủ sở hữu TÀI CHÍNH TÀI CHÍNH LÀNH MẠNH LÀNH MẠNH NHÀ ĐẦU TƯ CỔ PHIẾU THU HÚT CỔ PHIẾU THU HÚT 4 5 VÀ LỢI THẾ CẠNH TRANH TIỀM NĂNG TĂNG TRƯỞNG LỚN Phát huy sức mạnh cộng hưởng từ Công ty mẹ"}\n{"id": "B-A-b9393384909f7033", "claim": "Việc tuân thủ và từng bước tiệm cận các nguyên tắc quản trị tiên tiến sẽ góp phần nâng cao chuẩn mực quản trị của DHG Pharma, tăng cường uy tín thương hiệu và tạo nền tảng vững chắc cho mục tiêu phát triển bền vững trong tương lai.", "evidence": "Những dự án này không chỉ thể hiện sự nỗ lực trong việc tiếp cận và ứng dụng các công nghệ tiên tiến mà còn góp phần quan trọng vào việc đa dạng hóa và mở rộng danh mục sản phẩm của DHG Pharma. Nhờ đó, tạo nền tảng vững chắc để thúc đẩy sự tăng trưởng doanh thu, nâng cao năng lực cạnh tranh và vị thế trên thị trường. Sản lượng sản xuất 5.181 TRIỆU ĐƠN VỊ SẢN PHẨM Tuân thủ tiêu chuẩn thực hành tốt Kiến tạo giá trị bền vững DHG Pharma 2 6 1 3 4 Phát triển bền vững 5. 130 131 VỮNG BƯỚC VƯƠN XA M NH MẼ CHUYỂN MÌNH"}\n{"id": "B-A-41f267256d8272d6", "claim": "Công ty sẽ xem xét bổ sung nội dung này trong thời gian tới nhằm hoàn thiện chính sách phát triển bền vững trong toàn bộ chuỗi giá trị.", "evidence": "Các chủ đề ESG có ảnh hưởng trọng yếu đến chiến lược phát triển của DHG Pharma đã được xác định và tích hợp trong Chương phát triển bền vững của Báo cáo thường niên 2024. Thông tin chi tiết được trình bày tại trang 118-149. Hiện DHG Pharma chưa có chính sách riêng về bảo vệ quyền lợi chủ nợ và sẽ xem xét bổ sung trong thời gian tới. BÁO CÁO QUẢN TRỊ RỦI RO Ngoài các bộ phận chuyên trách về tuân thủ và kiểm soát độc lập, DHG Pharma còn thường xuyên phối hợp chặt chẽ với Công ty mẹ Taisho nhằm rà soát, đánh giá, cải tiến và giám sát hệ thống kiểm soát nội bộ. Sự hợp tác này giúp DHG Pharma nâng cao tính hiệu quả, đồng bộ và toàn diện trong quản trị rủi ro, đảm bảo hệ thống kiểm soát nội bộ ngà"}\n{"id": "B-A-9535c42bbe547d18", "claim": "Khuôn khổ quản trị doanh nghiệp nên cho phép đối thoại giữa công ty, cổ đông và các bên liên quan để trao đổi quan điểm về các vấn đề bền vững B.2 Trong năm 2024, DHG Pharma đã tiếp nhận 172 ý tưởng, sáng kiến cải tiến từ người lao động thông qua Hội đồng", "evidence": "Khoa học Công nghệ, cho thấy tinh thần chủ động đóng góp và đổi mới không ngừng trong toàn thể CBNV. DHG Pharma thiết lập kênh đối thoại hiệu quả với cổ đông và các bên liên quan thông qua bộ phận Quan hệ nhà đầu tư (IR), nhằm tiếp nhận và phản hồi kịp thời các ý kiến, đề xuất liên quan đến hoạt động của Công ty, đặc biệt là các vấn đề về phát triển bền vững. Ngày 24/01/2025, DHG Pharma tổ chức Lễ tổng kết năm 2024 kết hợp Hội nghị Người Lao động năm 2025 nhằm ghi nhận kết quả, lắng nghe người lao động và tuyên dương các tập thể, cá nhân tiêu biểu. BÁO CÁO QUẢN TRỊ CÔNG TY Khuôn khổ quản trị doanh nghiệp cần công nhận quyền của các bên có quyền lợi liên quan được quy định theo pháp luật hoặc"}\n{"id": "B-A-cc3dce16cc156766", "claim": "Sự hợp tác này giúp DHG Pharma nâng cao tính hiệu quả, đồng bộ và toàn diện trong quản trị rủi ro, đảm bảo hệ thống kiểm soát nội bộ ngày càng hoàn thiện và thích ứng, đáp ứng yêu cầu phát triển bền vững trong bối cảnh thị trường ngày càng cạnh tranh.", "evidence": "Kiện toàn khung kiểm soát nội bộ Ngày càng hoàn thiện hệ thống quy định, chính sách theo các thông lệ, đảm bảo phù hợp với yêu cầu thực tiễn điều hành kinh doanh và các quy định pháp luật. Luôn luôn với tinh thần nâng cao chất lượng kiểm soát nội bộ, không chỉ để đáp ứng nhu cầu hiện tại mà còn sẵn sàng thích ứng với những thay đổi của các bối cảnh kinh doanh trong tương lai. Đặc biệt, từ năm 2025, DHG Pharma triển khai kế hoạch quản trị - hệ thống kiểm soát nội bộ với các nguyên tắc quản trị toàn diện, đơn giản và chuyên sâu với đề tài Simple Rules. Các nguyên tắc quản trị và kiểm soát nội bộ, đánh giá rủi ro được tiến hành toàn diện ở từng khía cạnh hoạt động của DHG Pharma với triết lý đơ"}\n{"id": "B-A-c531be52da345f26", "claim": "Báo cáo Phát triển bền vững năm 2024 được trình bày và công bố các thông tin chính thức liên quan đến định hướng chiến lược, mục tiêu và kết quả triển khai các chương trình hành động, sáng kiến phát triển bền vững của DHG Pharma đã và đang thực hiện trong năm 2024, làm cơ sở để đánh giá hiệu quả trong công tác thực hiện phát triển bền vững cho những năm tiếp theo.", "evidence": "TỔNG QUAN BÁO CÁO PHÁT TRIỂN BỀN VỮNG ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG NGUYÊN TẮC XÂY DỰNG BÁO CÁO MỤC TIÊU CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DHG PHARMA HƯỚNG TỚI Quy trình quản trị bền vững tại DHG Pharma được vận hành qua 5 bước: Bước 04 Mỗi CBNV là mắt xích quan trọng, trực tiếp thực hiện các nhiệm vụ cụ thể hàng ngày nhằm hiện thực hóa các mục tiêu phát triển bền vững của Công ty. CBNV cam kết thực hiện các mục tiêu phát triển bền vững đúng với định hướng được xây dựng và ban hành tại"}\n{"id": "B-A-d867b4b8b02864eb", "claim": "Với vai trò người đi đầu, dẫn dắt ngành Dược Việt Nam, mặc dù còn phải đối mặt với nhiều thách thức từ tình hình địa chính trị, kinh tế, xã hội chung, DHG Pharma vẫn đặt ra những mục tiêu cụ thể cho hành trình phát triển bền vững của mình.", "evidence": "M ột năm qua đi với sự chứng kiến những bước đi chậm lại của nền kinh tế toàn cầu, nhưng DHG Pharma đã cố gắng vượt qua khó khăn để duy trì và phát triển tích cực hoạt động kinh doanh của mình. Vị thế dẫn đầu mà DHG pharma đã đạt được thể hiện thuyết phục trên nhiều phương diện, từ chiến lược kinh doanh đến mở rộng quy mô phát triển, cả thành tích lẫn giá trị, không đơn thuần là những con số mang sức nóng tăng trưởng. Chúng tôi đáp lại sự tin tưởng của tất cả Quý vị bằng những kết quả thực tế, những giá trị hữu hình, vô hình và không ngừng gia tăng. TABLE TỔNG QUAN VÀ Bước sang năm 2025, ngành dược Việt Nam đứng trước cơ hội phát triển mạnh mẽ nhờ bốn yếu tố chính: TRIỂN VỌNG NGÀNH DƯỢC 1. M"}\n{"id": "B-A-9df553aa99e93d2e", "claim": "Quản trị nguồn nhân lực, quản trị chuỗi giá trị, quản trị hệ sinh thái, quản trị dựa trên nền tảng số, quản trị phát triển bền vững.", "evidence": "ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG Chiến lược phát triển bền vững XÃ HỘI Phát triển nguồn nhân lực QUẢN TRỊ An toàn sản phẩm Nâng cấp hệ thống chiếu sáng, HVAC và thiết bị sản xuất để tối ưu hiệu quả. Hỗ trợ cộng đồng địa phương, chăm sóc sức khỏe cộng đồng, giáo dục và phát triển nguồn nhân lực. Phát triển và triển khai hệ thống quản lý thông tin dữ liệu chuyên nghiệp để cải thiện quản lý và ra quyết định dựa trên dữ liệu. Tập trung vào các hoạt động kinh doanh có khả năng tạo ra giá trị;"}\n{"id": "B-A-db2b7d690e355ba2", "claim": "Đặc biệt, với phương châm “Không ngừng ước vọng”, DHG Pharma không ngừng vươn tới mục tiêu “Vì một cuộc sống khỏe đẹp hơn” và vì sự phát triển bền vững, thịnh vượng.", "evidence": "Hành trình hơn 5 thập kỷ hoạt động và kiên định với sứ mệnh “Vì một cuộc sống khỏe đẹp hơn”, DHG Pharma không chỉ dừng lại ở việc sản xuất và cung cấp những viên thuốc chất lượng cao. Hơn thế nữa, chúng tôi mang trong mình khát vọng cháy bỏng về một cộng đồng khỏe mạnh, đặc biệt là những mảnh đời còn khó khăn nơi vùng sâu, vùng xa. Với phương châm “Lấy lợi ích cộng đồng làm khởi sự cho mọi hoạt động”, DHG Pharma đã bền bỉ gieo những hạt mầm tử tế, kiến tạo nên những giá trị nhân văn sâu sắc cho xã hội. 50 năm không ngừng nỗ lực và vươn lên mạnh mẽ, những thế hệ DHG Pharma qua các thời kỳ đã không ngừng sáng tạo, song vẫn giữ cho mình bản sắc nhớ ơn, cống hiến và khát vọng kiên định: “Vì một "}\n{"id": "B-A-e6db8792be7bd63a", "claim": "đồng thời hướng tới tuân thủ các tiêu chuẩn và thực hành tốt theo hướng dẫn của Bộ Y tế và các tổ chức chuyên môn trong nước và quốc tế để đảm bảo sản xuất, phân phối và sử dụng thuốc an toàn, hiệu quả và bền vững, góp phần cải thiện đáng kể sức khỏe của cộng đồng.", "evidence": "Đồng thời, tối ưu hóa danh mục sản phẩm, kiểm soát chặt chất lượng đầu vào - đầu ra và đảm bảo minh bạch thông tin trong toàn bộ quá trình cung ứng. Chuỗi cung ứng bền vững DHG Pharma tạo ra những sản phẩm chất lượng nhằm đảm bảo mỗi sản phẩm đến tay người dùng đều đạt chất lượng, hiệu quả và an toàn tuyệt đối. Tuân thủ nghiêm ngặt các tiêu chuẩn quốc tế trong sản xuất, kiểm soát chặt chẽ nguyên liệu đầu vào, theo dõi sát sao quá trình phân phối và sử dụng thuốc, hướng đến mục tiêu bảo vệ sức khỏe cộng đồng. Công ty cam kết tuân thủ nghiêm ngặt các quy định pháp luật, các tiêu chuẩn an toàn và sức khỏe nghề nghiệp quốc tế và trong nước nhằm tạo ra môi trường làm việc an toàn, lành mạnh cho t"}\n{"id": "B-A-83cc8942f1114831", "claim": "Hiện thực hóa chiến lược Phát triển bền vững với tầm nhìn trở thành doanh nghiệp dẫn đầu Việt Nam về mảng chăm sóc sức khỏe, đồng thời duy trì vững vị trí dẫn đầu ngành Công nghiệp dược Việt Nam, hướng đến mục tiêu “chuyển đổi xanh, tăng trưởng bền vững” với các giá trị nền tảng như: 17 Mục tiêu phát triển bền vững của Liên Hợp Quốc nhằm thúc đẩy sự phát triển bền vững toàn cầu.", "evidence": "ĐỊNH HƯỚNG PHÁT TRIỂN 2024 - 2030 Tầm nhìn đến năm 2030 5. Duy trì vị trí dẫn đầu ngành Công nghiệp dược Việt Nam; Phấn đấu trở thành doanh nghiệp mảng chăm sóc sức khỏe dẫn đầu Việt Nam. TABLE TỔNG QUAN VÀ Bước sang năm 2025, ngành dược Việt Nam đứng trước cơ hội phát triển mạnh mẽ nhờ bốn yếu tố chính: TRIỂN VỌNG NGÀNH DƯỢC 1. Mục tiêu tăng trưởng năm 2025 ở mức 8%, với GDP đầu người tăng từ 4.700 USD năm 2024 lên mức dự kiến 4.900 USD năm 2025 sẽ tăng khả năng chi tiêu cho sức khỏe Tăng trưởng Năm 2024, nền kinh tế Việt Nam duy trì đà phục hồi ấn tượng và đạt mức tăng trưởng đáng kể. Tổng sản phẩm quốc nội của người dân. Kết hợp với xu hướng chăm sóc sức khỏe chủ động của người dân kinh t"}\n{"id": "B-A-44898e2219117b63", "claim": "Từng bước chuyển đổi sang sử dụng năng lượng tái tạo, đặc biệt là điện mặt trời áp mái tại các nhà máy, giảm phụ thuộc vào năng lượng hóa thạch.", "evidence": "Năm 2024, DHG Pharma đã đầu tư lắp đặt hệ thống điện mặt trời áp mái cho các khu vực nhà xưởng, góp phần giảm thiểu đáng kể tiêu hao năng lượng, hướng đến mục tiêu sản xuất xanh và nghiêm túc thực thi các chiến lược phát triển bền vững mạnh mẽ hơn. Hệ thống năng lượng Tiết kiệm điện năng (*) Cắt giảm lượng mặt trời đã lắp đặt khí thải CO2 (*) Nguồn năng lượng sạch từ hệ thống điện mặt trời hiện đang được sử dụng trực tiếp cho hoạt động sản xuất tại nhà máy, góp phần tối ưu chi phí vận hành, giảm đáng kể lượng khí nhà kính phát thải ra môi trường, đồng thời thể hiện cam kết của DHG Pharma trong việc thúc đẩy phát triển xanh và bền vững. Song song với các giải pháp tiết kiệm năng lượng được áp"}\n{"id": "B-A-ca8f30adbe1261c4", "claim": "Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt.", "evidence": "NƯỚC THẢI Đối với chất thải rắn thông thường, nhà máy thực hiện phân loại tại nguồn thành các nhóm như chất thải hữu cơ, chất thải tái chế và chất thải không tái chế. Các loại chất thải này được thu gom hằng ngày và lưu giữ tạm thời tại khu vực tập kết riêng biệt, bảo đảm vệ sinh và không gây ảnh hưởng đến môi trường xung quanh. Sau đó, chất thải được bàn giao định kỳ cho các đơn vị thu gom để tái chế, tái sử dụng hoặc xử lý theo quy định của pháp luật. Toàn bộ quy trình thu gom và xử lý chất thải rắn thông thường được kiểm soát chặt chẽ nhằm đảm bảo tuân thủ các tiêu chuẩn về bảo vệ môi trường. Đối với chất thải nguy hại, nhà máy phân loại, thu gom và lưu trữ riêng tại khu vực kho CTNH, tác"}\n{"id": "B-A-d0a1b64de3f79a22", "claim": "Nước, không khí, chất thải Quản lý xung đột lợi ích Quản lý rủi ro MÔI TRƯỜNG Năng lượng Nước, không khí, chất thải", "evidence": "Nước, không khí, quả và tiết kiệm trong 100% nước thải từ sản xuất và sinh hoạt được Tối đa hóa giá trị doanh phiếu (chi trả trong năm 2024) chất thải quá trình sản xuất và xử lý đạt chuẩn trước khi xả thải. nghiệp và gia tăng lợi ích hoạt động thường ngày. PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt."}\n{"id": "B-A-5664763b58653442", "claim": "Biến đổi khí hậu Cam kết ứng phó biến đổi khí hậu bằng lộ trình giảm phát thải khí nhà kính, hướng tới Net Zero quốc gia.", "evidence": "TRÁCH NHIỆM VỚI MÔI TRƯỜNG PHÁT TRIỂN XÃ HỘI BỀN VỮNG ĐÁNH GIÁ CƠ HỘI, THÁCH THỨC VÀ BIỆN PHÁP ỨNG PHÓ DO BIẾN ĐỔI KHÍ HẬU Phát triển nguồn nhân lực Biến đổi khí hậu là một trong những thách thức toàn cầu lớn nhất hiện nay, gây ra các tác động sâu rộng đến môi trường, nền kinh tế và hoạt động sản xuất, kinh doanh của các doanh nghiệp. Chuỗi cung ứng bền vững 6 6 6 Mức độ ảnh hưởng Tác động đến DHG Biến đổi khí hậu Chủ nợ 5 5 Báo chí truyền thông 4 4 3 3 2 2 1 1"}\n{"id": "B-A-016a0727a9aae543", "claim": "Ưu tiên công nghệ sạch, năng lượng tái tạo và chủ động thích ứng để đảm bảo hoạt động bền vững.", "evidence": "Từng bước chuyển đổi sang sử dụng năng lượng tái tạo, đặc biệt là điện mặt trời áp mái tại các nhà máy, giảm phụ thuộc vào năng lượng hóa thạch. Hiệu quả hoạt động Phúc lợi cho khách hàng Năng lượng Chuỗi cung ứng bền vững Phát triển cộng đồng Công ty, đảm bảo việc thực hiện mang lại hiệu quả. HĐQT đóng vai trò dẫn dắt trong việc xác lập các ưu tiên cốt lõi cho hoạt động bền vững của Công ty. HĐQT đưa ra các chỉ thị, ban hành định hướng chiến lược rõ ràng cho mọi vấn đề liên quan đến phát triển bền vững."}\n{"id": "B-A-33c0f0a08da08dec", "claim": "Phát triển nguồn nhân lực Xây dựng chuỗi cung ứng bền vững thông qua chiến lược R&D gắn với nhu cầu thị trường, đảm bảo nguyên liệu đạt chuẩn, ứng dụng công nghệ bào chế tiên tiến và quản trị chất lượng toàn diện theo tiêu chuẩn quốc tế.", "evidence": "Toàn bộ nguyên liệu, hoạt chất và tá dược đều trải qua quy trình kiểm nghiệm khắt khe theo các tiêu chuẩn đã được phê duyệt trước khi đưa vào sản xuất, tuyệt đối không sử dụng nguyên liệu không đạt chuẩn hoặc không rõ nguồn gốc. Cam kết Quản trị chất lượng Cung ứng Kiểm soát 100% chất lượng đầu vào nguồn toàn diện theo minh bạch và nguyên liệu chuẩn quốc tế tuân thủ pháp luật CHIẾN LƯỢC R&D ĐỊNH HƯỚNG THỊ TRƯỜNG NGHIÊN CỨU VÀ ỨNG DỤNG CÔNG NGHỆ TIÊN TIẾN Đồng thời, tối ưu hóa danh mục sản phẩm, kiểm soát chặt chất lượng đầu vào - đầu ra và đảm bảo minh bạch thông tin trong toàn bộ quá trình cung ứng. Chuỗi cung ứng bền vững DHG Pharma tạo ra những sản phẩm chất lượng nhằm đảm bảo mỗi sản phẩ"}\n{"id": "B-A-af20ec168232fd75", "claim": "Công ty chú trọng xây dựng mối quan hệ hợp tác bền vững nhằm mang đến trải nghiệm tối ưu, đồng hành cùng khách hàng trên hành trình chăm sóc sức khỏe toàn diện.", "evidence": "Đảm bảo được nguồn cung cấp hàng hóa ổn định về sản lượng & chất lượng, chú trọng công tác xây dựng quan hệ khách hàng bằng các chương trình chăm sóc, hỗ trợ kịp thời các hoạt động huấn luyện sản phẩm, các hoạt động POSM là một trong 3 chiến lược xuất khẩu mũi nhọn mà bộ phận xuất khẩu của DHG Pharma đã thực hiện. Đồng thời xây dựng và củng cố nhóm khách hàng truyền thống nhằm xây dựng mối quan hệ đối tác lâu dài, đồng hành, hỗ trợ & hướng đến sự phát triển bền vững. Hoạt động quan hệ Sản phẩm của DHG Pharma được phân phối đến khách hàng thông qua các kênh: Đồng thời xây dựng và củng cố nhóm khách hàng truyền thống nhằm xây dựng mối quan hệ đối tác lâu dài, đồng hành, hỗ trợ & hướng đến sự p"}\n{"id": "B-A-b931019126779fe1", "claim": "Quản lý rủi ro DHG Pharma 2 6 1 3 4 Phát triển bền vững", "evidence": "BÁO CÁO QUẢN TRỊ RỦI RO QUẢN LÝ CÁC RỦI RO QUAN TRỌNG Chuỗi cung ứng và nguồn nguyên liệu đóng vai trò then chốt trong sự phát triển bền vững của công ty. KẾ HOẠCH HOẠT ĐỘNG QUẢN LÝ RỦI RO TỪ NĂM 2025 Phân tích rủi ro Đánh giá xếp loại rủi ro Xử lý rủi ro Truyền thông và tham vấn 7 DHG Pharma 2 6 5 1 3 Quản trị Công ty 4. 112 113 VỮNG BƯỚC VƯƠN XA"}\n{"id": "B-A-047934113f78bf0c", "claim": "Quy trình quản trị bền vững tại DHG Pharma được vận hành qua 5 bước: Bước 04 Mỗi CBNV là mắt xích quan trọng, trực tiếp thực hiện các nhiệm vụ cụ thể hàng ngày nhằm hiện thực hóa các mục tiêu phát triển bền vững của Công ty.", "evidence": "Báo cáo Phát triển bền vững năm 2024 được trình bày và công bố các thông tin chính thức liên quan đến định hướng chiến lược, mục tiêu và kết quả triển khai các chương trình hành động, sáng kiến phát triển bền vững của DHG Pharma đã và đang thực hiện trong năm 2024, làm cơ sở để đánh giá hiệu quả trong công tác thực hiện phát triển bền vững cho những năm tiếp theo. Chương Báo cáo phát triển bền vững được xây dựng trên cơ sở tham chiếu Chuẩn mực GRI. Trong đó, Công ty công bố đầy đủ các mục tiêu phát triển bền vững kèm theo chỉ tiêu định lượng và kết quả thực hiện cụ thể trong kỳ. Hiện tại, DHG Pharma chưa công bố chính sách hoặc mô tả cụ thể các thực hành liên quan đến việc đảm bảo thực hiện "}\n{"id": "B-A-59238d675e3aedd1", "claim": "CBNV cam kết thực hiện các mục tiêu phát triển bền vững đúng với định hướng được xây dựng và ban hành tại", "evidence": "TỔNG QUAN BÁO CÁO PHÁT TRIỂN BỀN VỮNG ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG NGUYÊN TẮC XÂY DỰNG BÁO CÁO MỤC TIÊU CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DHG PHARMA HƯỚNG TỚI Báo cáo Phát triển bền vững năm 2024 được trình bày và công bố các thông tin chính thức liên quan đến định hướng chiến lược, mục tiêu và kết quả triển khai các chương trình hành động, sáng kiến phát triển bền vững của DHG Pharma đã và đang thực hiện trong năm 2024, làm cơ sở để đánh giá hiệu quả trong công tác thực hiện phát triển bền vững cho những năm tiếp theo."}\n{"id": "B-A-333e610e37b59692", "claim": "Nội dung báo cáo là minh chứng cho sự cam kết mạnh mẽ của DHG Pharma trên con đường phát triển bền vững trước kỳ vọng của các bên liên quan.", "evidence": "Đây cũng là sự khẳng định mạnh mẽ về cam kết của DHG Pharma trong hành trình hướng tới sự phát triển bền vững toàn diện, hài hòa giữa tăng trưởng kinh tế, trách nhiệm xã hội và bảo vệ môi trường. Đối với các vấn đề trọng yếu, DHG Pharma xác định phương pháp quản trị dựa trên bối cảnh ngành dược trong nước và thế giới, thông tin phản hồi và mối quan tâm giữa các bên liên quan. QUẢN TRỊ PHÁT TRIỂN BỀN VỮNG Mặc dù chưa chính thức thành lập Ủy ban quản trị riêng về Phát triển bền vững nhưng DHG Pharma luôn đảm bảo tốt các tiêu chí phát triển bền vững với sự phân cấp, quản lý chặt chẽ theo quy trình, đảm bảo mọi hoạt động hướng đến sự cân đối, hài hòa và gắn kết bền chặt với tất cả các bên liên q"}\n{"id": "B-A-dbebbef496287e61", "claim": "Ban Điều hành có trách nhiệm xây dựng, trình HĐQT phê duyệt các mục tiêu và kế hoạch chi tiết cho chiến lược phát triển bền vững.", "evidence": "Ông Hùng được ĐHĐCĐ phê chuẩn là thành viên độc lập HĐQT và chịu trách nhiệm phụ trách Ủy ban nhân sự kể từ ngày 23/04/2024. Với kiến thức chuyên sâu và kinh nghiệm quản trị, ông đã tham mưu cho HĐQT, Ban điều hành trong việc xây dựng và triển khai Dự án Nguồn nhân lực kế thừa, nhằm đảm bảo sự phát triển của đội ngũ nhân sự, đáp ứng mục tiêu chiến lược dài hạn của công ty. KẾT QUẢ HOẠT ĐỘNG GIÁM SÁT Dưới sự phụ trách của ông, Ủy ban Nhân sự giữ vai trò xem xét, đánh giá năng lực các vị trí chủ chốt trước khi trình TỔNG QUAN BÁO CÁO PHÁT TRIỂN BỀN VỮNG ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG NGUYÊN TẮC XÂY DỰNG BÁO CÁO MỤC TIÊU CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DHG PHARMA HƯỚNG TỚI"}\n{"id": "B-A-80103fd4596306cd", "claim": "Đây cũng là sự khẳng định mạnh mẽ về cam kết của DHG Pharma trong hành trình hướng tới sự phát triển bền vững toàn diện, hài hòa giữa tăng trưởng kinh tế, trách nhiệm xã hội và bảo vệ môi trường.", "evidence": "DHG Pharma xác định các lĩnh vực trọng yếu trong mối quan hệ giữa kinh tế, xã hội và môi trường với hoạt động sản xuất kinh doanh của Công ty, từ đó định hình chiến lược, đảm bảo phát triển bền vững trong dài hạn. Bên cạnh đó, chúng tôi cũng ý thức rõ việc xác định các lĩnh vực trọng yếu đóng vai trò then chốt trong chiến lược phát triển bền vững, giúp Công ty ưu tiên nguồn lực cho những vấn đề có ảnh hưởng lớn nhất đến hoạt động kinh doanh. Để đảm bảo tính khách quan và toàn diện, chúng tôi đã áp dụng phương pháp luận theo hướng dẫn của GRI. Quản trị hiệu quả, minh bạch gắn liền với trách nhiệm với môi trường và góp phần xây dựng cộng đồng, xã hội. Nội dung báo cáo là minh chứng cho sự cam "}\n{"id": "B-A-a1beb6850af66fd9", "claim": "Kết quả đánh giá trọng yếu cho thấy các vấn đề ưu tiên của Công ty có mối liên hệ chặt chẽ với 17 Mục tiêu Phát triển Bền vững (SDGs), góp phần thúc đẩy tăng trưởng dài hạn và tạo giá trị bền vững cho xã hội.", "evidence": "Hiện thực hóa chiến lược Phát triển bền vững với tầm nhìn trở thành doanh nghiệp dẫn đầu Việt Nam về mảng chăm sóc sức khỏe, đồng thời duy trì vững vị trí dẫn đầu ngành Công nghiệp dược Việt Nam, hướng đến mục tiêu “chuyển đổi xanh, tăng trưởng bền vững” với các giá trị nền tảng như: 17 Mục tiêu phát triển bền vững của Liên Hợp Quốc nhằm thúc đẩy sự phát triển bền vững toàn cầu. Nâng cao năng lực quản trị, đầu tư mạnh mẽ vào công nghệ, đẩy mạnh hoạt động sản xuất kinh doanh, nâng cao chất lượng sản phẩm. Quản trị hiệu quả, minh bạch gắn liền với trách nhiệm với môi trường và góp phần xây dựng cộng đồng, xã hội. Nội dung báo cáo là minh chứng cho sự cam kết mạnh mẽ của DHG Pharma trên con đườ"}\n{"id": "B-A-ab8894e6d6678b4a", "claim": "Phát triển cộng đồng Nước, không khí, chất thải Phát triển nguồn nhân lực Cổ đông, nhà đầu tư 7 7 Cộng đồng, xã hội 5 2 Nhà cung cấp, đối tác", "evidence": "Kiểm soát chặt chẽ khí thải từ sản xuất và nồi hơi. Nước, không khí, chất thải Quản lý xung đột lợi ích Quản lý rủi ro MÔI TRƯỜNG Năng lượng Nước, không khí, chất thải PHẠM VI BÁO CÁO Tập trung Phát triển nguồn nhân lực và Phát triển văn hóa doanh nghiệp bền vững: Đầu tư vào nguồn nhân lực chất lượng cao, đảm bảo môi trường làm việc an toàn và công bằng, thúc đẩy văn hóa doanh nghiệp phát triển kinh tế gắn liền với trách nhiệm cộng đồng, an sinh xã hội. Báo cáo được thực hiện tại Việt Nam và được lập cho Công ty Cổ phần Dược Hậu Giang."}\n{"id": "B-A-b715a30ecfc6dd54", "claim": "0 0 0 1 3 2 5 4 7 6 9 8 10 0 1 3 2 5 4 7 6 9 8 10 Mức độ quan tâm Quan tâm của các bên liên quan DHG Pharma 2 6 1 3 4 Phát triển bền vững", "evidence": "BỀN VỮNG PHÁT TRIỂN 5 118 Tổng quan báo cáo phát triển bền vững 119 Định hướng chiến lược phát triển bền vững 122 Quản trị phát triển bền vững 124 Gắn kết các bên liên quan TỔNG QUAN BÁO CÁO PHÁT TRIỂN BỀN VỮNG ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG NGUYÊN TẮC XÂY DỰNG BÁO CÁO MỤC TIÊU CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DHG PHARMA HƯỚNG TỚI"}\n{"id": "B-A-bf91000444a52e5f", "claim": "» Góp phần cắt giảm 1.907 tấn CO2 vật liệu, sử dụng máy móc, Nâng cao hiệu quả hoạt » Doanh thu: 4.885 tỷ đồng » Tiết kiệm chi phí khoảng 1.465 triệu đồng thiết bị công nghệ sạch.", "evidence": "Tiếp tục đầu tư lắp đặt hệ Sử dụng điện mặt trời góp phần giảm 100% thành phẩm được kiểm nghiệm tại phòng đạt chuẩn WHO-GMP và thống điện năng lượng 1.907 tấn CO2 HÀNH ĐỘNG ỨNG PHÓ VỚI BIẾN ĐỔI Công suất Giảm MWP TỶ ĐỒNG TẤN CO2 1,8 1,5 1.907 Phát triển xanh (*) Số liệu ước tính từ ngày 01/01/2024 - 31/12/2024 DHG Pharma 2 6 1 3 4 Phát triển bền vững"}\n{"id": "B-A-ca70b262ff70329a", "claim": "Nước, không khí, quả và tiết kiệm trong 100% nước thải từ sản xuất và sinh hoạt được Tối đa hóa giá trị doanh phiếu (chi trả trong năm 2024) chất thải quá trình sản xuất và xử lý đạt chuẩn trước khi xả thải.", "evidence": "Kiểm soát chặt chẽ khí thải từ sản xuất và nồi hơi. Nước, không khí, chất thải Quản lý xung đột lợi ích Quản lý rủi ro MÔI TRƯỜNG Năng lượng Nước, không khí, chất thải Song song với việc tối ưu hóa sử dụng nước, DHG Pharma đặc biệt chú trọng nâng cao năng lực kiểm soát và xử lý nước thải, nhằm giảm thiểu tối đa tác động tiêu cực tới môi trường, thể hiện rõ trách nhiệm với cộng đồng và đảm bảo tuân thủ các tiêu chuẩn nghiêm ngặt về bảo vệ môi trường. Tổng lưu lượng nước thải năm 2024 156.648M3 tương ứng chi phí xử lý 2,08 TỶ ĐỒNG Hệ thống xử lý nước thải của DHG Pharma được đảm bảo vận hành liên tục và được kiểm tra định kỳ, qua đó chất lượng nước thải đầu ra luôn đáp ứng tiêu chuẩn QCVN 40:2"}\n{"id": "B-A-7561ea075873346b", "claim": "chức năng tái chế, tái sử dụng hoặc xử lý theo sản phẩm chất lượng, an quy định.", "evidence": "PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt. NƯỚC THẢI Đối với chất thải rắn thông thường, nhà máy thực hiện phân loại tại nguồn thành các nhóm như chất thải hữu cơ, chất thải tái chế và chất thải không tái chế. Các loại chất thải này được thu gom hằng ngày và lưu giữ tạm thời tại khu vực tập kết riêng biệt, bảo đảm vệ sinh và không gây ảnh hưởng đến môi trường xung quanh. Sau đó, chất thải được bàn giao định kỳ cho các đơn "}\n{"id": "B-A-bee596768a68aaf0", "claim": "DHG Pharma 2 6 1 3 4 Phát triển bền vững 5.", "evidence": "DHG Pharma 2 6 1 3 4 Phát triển bền vững 5. 132 133 VỮNG BƯỚC VƯƠN XA M NH MẼ CHUYỂN MÌNH Báo cáo thường niên 2024 DHG Pharma 2 6 1 3 4 Phát triển bền vững 5. 136 137 VỮNG BƯỚC VƯƠN XA M NH MẼ CHUYỂN MÌNH Báo cáo thường niên 2024"}\n{"id": "B-A-d938315480542cd8", "claim": "DHG Pharma 2 6 1 3 4 Phát triển bền vững", "evidence": "TỔNG QUAN BÁO CÁO PHÁT TRIỂN BỀN VỮNG ĐỊNH HƯỚNG CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG NGUYÊN TẮC XÂY DỰNG BÁO CÁO MỤC TIÊU CHIẾN LƯỢC PHÁT TRIỂN BỀN VỮNG DHG PHARMA HƯỚNG TỚI Tuân thủ tiêu chuẩn thực hành tốt Kiến tạo giá trị bền vững DHG Pharma 2 6 1 3 4 Phát triển bền vững 5. 130 131 VỮNG BƯỚC VƯƠN XA M NH MẼ CHUYỂN MÌNH"}\n{"id": "B-A-ef2ac902552178bb", "claim": "Với tinh thần đoàn kết, quyết tâm cao của toàn thể CBNV và sự chỉ đạo sát sao từ Ban Lãnh đạo, Công ty đã vững vàng vượt qua khó khăn, không ngừng nỗ lực và tiếp tục phấn đấu vì những mục tiêu phát triển bền vững trong thời gian tới.", "evidence": "Các phân xưởng đã chủ động lựa chọn thiết bị, máy móc phù hợp đồng thời cũng chuẩn bị kỹ về nhân lực trong quá trình tổ chức sản xuất. Hiểu được sự khó khăn của CBNV, Ban Điều hành Công ty không chỉ liên tục đưa ra những chính sách kịp thời về chiến lược mà còn đưa ra những chính sách thể hiện sự quan tâm sâu sát cho từng CBNV và người thân của họ bằng sự tử tế và chân tình. Bước sang thập kỷ mới với sự kế thừa truyền thống vẻ vang và phát huy tinh thần tự chủ, sẵn sàng đối diện thách thức và vươn lên trong gian khó, tôi tin tưởng rằng thế hệ cán bộ lãnh đạo, nhân viên DHG Pharma hôm nay sẽ đoàn kết, đổi mới sáng tạo, phấn đấu hoàn thành xuất sắc kế hoạch năm và thực hiện tốt nhất vai trò nâ"}\n{"id": "B-A-8c10e41f296c1874", "claim": "Những kết quả và hành động này một lần nữa khẳng định cam kết mạnh mẽ của DHG Pharma trong việc kiến tạo giá trị bền vững và thịnh vượng chung cho cổ đông, người lao động và cộng đồng.", "evidence": "DHG Pharma cam kết tiếp tục tối ưu hóa các nguồn lực để tạo ra các giá trị dài hạn, đầu tư mạnh mẽ vào nghiên cứu và phát triển, ứng dụng công nghệ tiên tiến để tạo ra những sản phẩm chất lượng vượt trội, hướng tới phục vụ tốt nhất cho sức khỏe cộng đồng. Vượt lên trên cả sự phát triển bền vững của một doanh nghiệp, chúng tôi mong muốn trở thành biểu tượng của chất lượng, uy tín và sự tận tâm, mang lại giá trị lâu dài cho khách hàng, đối tác và toàn xã hội, góp phần nâng tầm ngành dược Việt Nam và khẳng định vị thế ngày càng vững mạnh trên trường quốc tế. Hành trình hơn 5 thập kỷ hoạt động và kiên định với sứ mệnh “Vì một cuộc sống khỏe đẹp hơn”, DHG Pharma không chỉ dừng lại ở việc sản xuất"}\n{"id": "B-A-72ec61447857736f", "claim": "Quy trình đánh giá nhà cung cấp toàn diện Chuỗi cung ứng bền vững Tỷ trọng mua sắm năm 2024", "evidence": "Mua sắm từ nhà cung cấp nước ngoài Mua sắm từ nhà cung cấp trong nước Chiến lược R&D Nghiên cứu và Tối ưu danh mục 78% 22% định hướng ứng dụng công nghệ và vòng đời thị trường tiên tiến sản phẩm 2.712 2.700 2.678 2.789 2.806 Tích hợp ESG vào chiến lược quản trị và phát triển bền vững của công ty. Đa dạng hóa nhà cung cấp, ưu tiên lựa chọn các nhà cung ứng có chiến lược thích ứng, có chứng nhận về quản lý môi trường và ứng phó BĐKH. Biến động trong chuỗi cung ứng do ảnh hưởng khí hậu tại nơi cung cấp nguyên liệu"}\n{"id": "B-A-f20719775e29dd60", "claim": "Tuân thủ tiêu chuẩn thực hành tốt Kiến tạo giá trị bền vững DHG Pharma 2 6 1 3 4 Phát triển bền vững 5.", "evidence": "Báo cáo Phát triển bền vững năm 2024 được trình bày và công bố các thông tin chính thức liên quan đến định hướng chiến lược, mục tiêu và kết quả triển khai các chương trình hành động, sáng kiến phát triển bền vững của DHG Pharma đã và đang thực hiện trong năm 2024, làm cơ sở để đánh giá hiệu quả trong công tác thực hiện phát triển bền vững cho những năm tiếp theo. Quy trình quản trị bền vững tại DHG Pharma được vận hành qua 5 bước: Bước 04 Mỗi CBNV là mắt xích quan trọng, trực tiếp thực hiện các nhiệm vụ cụ thể hàng ngày nhằm hiện thực hóa các mục tiêu phát triển bền vững của Công ty. CBNV cam kết thực hiện các mục tiêu phát triển bền vững đúng với định hướng được xây dựng và ban hành tại"}\n{"id": "B-A-86a43cc67b2d54eb", "claim": "Đây không chỉ là biện pháp tối ưu hóa hiệu quả vận hành mà còn đóng vai trò then chốt trong nỗ lực giảm thiểu phát Kiểm soát chất lượng từng công đoạn thải khí nhà kính, bảo vệ môi trường và thúc đẩy việc đạt được các mục tiêu phát triển bền vững trong dài hạn.", "evidence": "Kiểm soát chặt chẽ việc sử dụng tài nguyên, năng lượng, nước và phát thải trong quá trình sản xuất. MÔI TRƯỜNG Thực hiện các biện pháp giảm thiểu khí thải nhà kính, tái chế chất thải và đầu tư vào công nghệ sản xuất sạch hơn. Định kỳ đánh giá tác động môi trường và tuân thủ đầy đủ các quy định pháp luật hiện hành về bảo vệ môi trường. TRÁCH NHIỆM VỚI MÔI TRƯỜNG GIẢI PHÁP TIẾT KIỆM ĐIỆN NĂNG Tiết kiệm năng lượng đóng vai trò vô cùng quan trọng trong chiến lược phát triển bền vững của các doanh nghiệp vận hành các nhà máy sản xuất quy mô lớn. Đây không chỉ là giải pháp cấp thiết nhằm tối ưu hóa chi phí vận hành, nâng cao năng lực cạnh tranh mà còn thể hiện tầm nhìn dài hạn trong quản trị và sử"}\n{"id": "B-A-62d13cd269f20473", "claim": "Việc hướng tới và đầu tư xây dựng các “nhà máy xanh” là một cam kết mạnh mẽ, giúp doanh nghiệp khẳng định trách nhiệm với môi trường và xã hội, đồng thời duy trì và nâng cao uy tín, hình ảnh của doanh nghiệp trước cộng đồng và các bên liên quan.", "evidence": "Đây cũng là sự khẳng định mạnh mẽ về cam kết của DHG Pharma trong hành trình hướng tới sự phát triển bền vững toàn diện, hài hòa giữa tăng trưởng kinh tế, trách nhiệm xã hội và bảo vệ môi trường. Đối với các vấn đề trọng yếu, DHG Pharma xác định phương pháp quản trị dựa trên bối cảnh ngành dược trong nước và thế giới, thông tin phản hồi và mối quan tâm giữa các bên liên quan. Quản trị hiệu quả, minh bạch gắn liền với trách nhiệm với môi trường và góp phần xây dựng cộng đồng, xã hội. Nội dung báo cáo là minh chứng cho sự cam kết mạnh mẽ của DHG Pharma trên con đường phát triển bền vững trước kỳ vọng của các bên liên quan. Bước 05 Nguyên tắc phát triển bền vững được tích hợp vào chiến lược tổn"}\n{"id": "B-A-18bd89bd4fc7ab41", "claim": "Những giải pháp này vừa giúp tiết kiệm chi phí, vừa giảm phát thải khí nhà kính, phù hợp với định hướng phát triển xanh của Công ty.", "evidence": "Nguồn năng lượng sạch từ hệ thống điện mặt trời hiện đang được sử dụng trực tiếp cho hoạt động sản xuất tại nhà máy, góp phần tối ưu chi phí vận hành, giảm đáng kể lượng khí nhà kính phát thải ra môi trường, đồng thời thể hiện cam kết của DHG Pharma trong việc thúc đẩy phát triển xanh và bền vững. Song song với các giải pháp tiết kiệm năng lượng được áp dụng nhiều năm qua như: truyền thông nội bộ để nâng cao ý thức sử dụng điện hiệu quả; Trong năm 2024, DHG Pharma dành hơn 5 tỷ đồng triển khai nhiều chương trình hỗ trợ cộng đồng: tặng thiết bị y tế, hỗ trợ người nghèo, tài trợ phòng học thông minh tại tỉnh Hậu Giang... Đồng hành cùng nỗ lực của Việt Nam hướng đến phát thải ròng bằng 0 vào nă"}\n{"id": "B-A-f8e4fd515e577120", "claim": "Việc đưa vào vận hành hệ thống năng lượng mặt trời thể hiện bước khởi đầu thiết thực của DHG Pharma trong việc tăng cường sử dụng năng lượng tái tạo, vừa giúp tiết kiệm chi phí điện, vừa góp phần giảm phát thải CO2 trong hoạt động sản xuất.", "evidence": "Nguồn năng lượng sạch từ hệ thống điện mặt trời hiện đang được sử dụng trực tiếp cho hoạt động sản xuất tại nhà máy, góp phần tối ưu chi phí vận hành, giảm đáng kể lượng khí nhà kính phát thải ra môi trường, đồng thời thể hiện cam kết của DHG Pharma trong việc thúc đẩy phát triển xanh và bền vững. Song song với các giải pháp tiết kiệm năng lượng được áp dụng nhiều năm qua như: truyền thông nội bộ để nâng cao ý thức sử dụng điện hiệu quả; Năm 2024, DHG Pharma đã đầu tư lắp đặt hệ thống điện mặt trời áp mái cho các khu vực nhà xưởng, góp phần giảm thiểu đáng kể tiêu hao năng lượng, hướng đến mục tiêu sản xuất xanh và nghiêm túc thực thi các chiến lược phát triển bền vững mạnh mẽ hơn. Hệ thống "}\n{"id": "B-A-b3029f0acb130a82", "claim": "Nguồn năng lượng sạch từ hệ thống điện mặt trời hiện đang được sử dụng trực tiếp cho hoạt động sản xuất tại nhà máy, góp phần tối ưu chi phí vận hành, giảm đáng kể lượng khí nhà kính phát thải ra môi trường, đồng thời thể hiện cam kết của DHG Pharma trong việc thúc đẩy phát triển xanh và bền vững.", "evidence": "Song song đó, hệ thống giám sát tiêu thụ điện tự động cũng được triển khai tại nhà máy Betalactam để tăng cường kiểm soát năng lượng mà không làm gián đoạn quy trình sản xuất. Những giải pháp này vừa giúp tiết kiệm chi phí, vừa giảm phát thải khí nhà kính, phù hợp với định hướng phát triển xanh của Công ty. Việc đưa vào vận hành hệ thống năng lượng mặt trời thể hiện bước khởi đầu thiết thực của DHG Pharma trong việc tăng cường sử dụng năng lượng tái tạo, vừa giúp tiết kiệm chi phí điện, vừa góp phần giảm phát thải CO2 trong hoạt động sản xuất. Năm 2024, DHG Pharma đã đầu tư lắp đặt hệ thống điện mặt trời áp mái cho các khu vực nhà xưởng, góp phần giảm thiểu đáng kể tiêu hao năng lượng, hướng"}\n{"id": "B-A-b8067b91279e5d65", "claim": "Song song với các giải pháp tiết kiệm năng lượng được áp dụng nhiều năm qua như: truyền thông nội bộ để nâng cao ý thức sử dụng điện hiệu quả;", "evidence": "TRÁCH NHIỆM VỚI MÔI TRƯỜNG GIẢI PHÁP TIẾT KIỆM ĐIỆN NĂNG Tiết kiệm năng lượng đóng vai trò vô cùng quan trọng trong chiến lược phát triển bền vững của các doanh nghiệp vận hành các nhà máy sản xuất quy mô lớn. Đây không chỉ là giải pháp cấp thiết nhằm tối ưu hóa chi phí vận hành, nâng cao năng lực cạnh tranh mà còn thể hiện tầm nhìn dài hạn trong quản trị và sử dụng nguồn tài nguyên hiệu quả. Song song đó, hệ thống giám sát tiêu thụ điện tự động cũng được triển khai tại nhà máy Betalactam để tăng cường kiểm soát năng lượng mà không làm gián đoạn quy trình sản xuất. Những giải pháp này vừa giúp tiết kiệm chi phí, vừa giảm phát thải khí nhà kính, phù hợp với định hướng phát triển xanh của Công"}\n{"id": "B-A-d244b26576d7ee94", "claim": "Hệ thống năng lượng Tiết kiệm điện năng (*) Cắt giảm lượng mặt trời đã lắp đặt khí thải CO2 (*)", "evidence": "Tiếp tục đầu tư lắp đặt hệ Sử dụng điện mặt trời góp phần giảm 100% thành phẩm được kiểm nghiệm tại phòng đạt chuẩn WHO-GMP và thống điện năng lượng 1.907 tấn CO2 HÀNH ĐỘNG ỨNG PHÓ VỚI BIẾN ĐỔI Song song đó, hệ thống giám sát tiêu thụ điện tự động cũng được triển khai tại nhà máy Betalactam để tăng cường kiểm soát năng lượng mà không làm gián đoạn quy trình sản xuất. Những giải pháp này vừa giúp tiết kiệm chi phí, vừa giảm phát thải khí nhà kính, phù hợp với định hướng phát triển xanh của Công ty. Việc đưa vào vận hành hệ thống năng lượng mặt trời thể hiện bước khởi đầu thiết thực của DHG Pharma trong việc tăng cường sử dụng năng lượng tái tạo, vừa giúp tiết kiệm chi phí điện, vừa góp phần g"}\n{"id": "B-A-076ea026589bf6f7", "claim": "Để kiểm soát chất thải và hạn chế tác động đến môi trường, DHG Pharma triển khai nghiêm túc công tác quản lý chất thải theo đúng quy định.", "evidence": "Kiểm soát chặt chẽ khí thải từ sản xuất và nồi hơi. Nước, không khí, chất thải Quản lý xung đột lợi ích Quản lý rủi ro MÔI TRƯỜNG Năng lượng Nước, không khí, chất thải Kiểm soát chặt chẽ việc sử dụng tài nguyên, năng lượng, nước và phát thải trong quá trình sản xuất. MÔI TRƯỜNG Thực hiện các biện pháp giảm thiểu khí thải nhà kính, tái chế chất thải và đầu tư vào công nghệ sản xuất sạch hơn. Định kỳ đánh giá tác động môi trường và tuân thủ đầy đủ các quy định pháp luật hiện hành về bảo vệ môi trường."}\n{"id": "B-A-388dc645da0f33e8", "claim": "Hằng năm, Công ty ký hợp đồng với các đơn vị có chức năng thu gom và xử lý chất thải.", "evidence": "NƯỚC THẢI Đối với chất thải rắn thông thường, nhà máy thực hiện phân loại tại nguồn thành các nhóm như chất thải hữu cơ, chất thải tái chế và chất thải không tái chế. Các loại chất thải này được thu gom hằng ngày và lưu giữ tạm thời tại khu vực tập kết riêng biệt, bảo đảm vệ sinh và không gây ảnh hưởng đến môi trường xung quanh. Sau đó, chất thải được bàn giao định kỳ cho các đơn vị thu gom để tái chế, tái sử dụng hoặc xử lý theo quy định của pháp luật. Xây dựng công trình thu gom, xử lý nước thải, khí thải tại các nhà máy. HỢP TÁC VỚI NHÀ CUNG CẤP ĐỂ BẢO VỆ MÔI TRƯỜNG Trong năm 2024, DHG Pharma tiếp tục duy trì hợp đồng với các đơn vị cung cấp dịch vụ chuyên nghiệp về vệ sinh công nghiệp, t"}\n{"id": "B-A-4d35c80409fca849", "claim": "Toàn bộ chất thải từ hoạt động sản xuất và sinh hoạt được thu gom, phân loại và xử lý theo đúng quy định, đảm bảo tuân thủ các tiêu chuẩn môi trường hiện hành.", "evidence": "PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt. Toàn bộ quy trình thu gom và xử lý chất thải rắn thông thường được kiểm soát chặt chẽ nhằm đảm bảo tuân thủ các tiêu chuẩn về bảo vệ môi trường. Đối với chất thải nguy hại, nhà máy phân loại, thu gom và lưu trữ riêng tại khu vực kho CTNH, tách biệt hoàn toàn với khu vực sản xuất. Khu lưu trữ được gắn biển cảnh báo rõ ràng, bên trong bố trí các thùng chứa riêng theo từng loại CTNH,"}\n{"id": "B-A-02f0d9b73fbc73f6", "claim": "NƯỚC THẢI Đối với chất thải rắn thông thường, nhà máy thực hiện phân loại tại nguồn thành các nhóm như chất thải hữu cơ, chất thải tái chế và chất thải không tái chế.", "evidence": "PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt. Thường xuyên kiểm toán năng lượng nhằm phát hiện và loại bỏ các điểm tiêu hao năng lượng không cần thiết. Tiếp nhận nước thải đạt Cột B và xử lý nước thải từ cột B sang Cột A theo QCVN 40:2011/BTNMT, tiếp tục thải ra nguồn tiếp nhận. 6 Hợp đồng dịch vụ thoát nước và xử lý nước thải (áp dụng tại"}\n{"id": "B-A-7c9fbb71e8380729", "claim": "Các loại chất thải này được thu gom hằng ngày và lưu giữ tạm thời tại khu vực tập kết riêng biệt, bảo đảm vệ sinh và không gây ảnh hưởng đến môi trường xung quanh.", "evidence": "Toàn bộ quy trình thu gom và xử lý chất thải rắn thông thường được kiểm soát chặt chẽ nhằm đảm bảo tuân thủ các tiêu chuẩn về bảo vệ môi trường. Đối với chất thải nguy hại, nhà máy phân loại, thu gom và lưu trữ riêng tại khu vực kho CTNH, tách biệt hoàn toàn với khu vực sản xuất. Khu lưu trữ được gắn biển cảnh báo rõ ràng, bên trong bố trí các thùng chứa riêng theo từng loại CTNH, có dán mã số theo đúng quy định pháp luật. Để kiểm soát chất thải và hạn chế tác động đến môi trường, DHG Pharma triển khai nghiêm túc công tác quản lý chất thải theo đúng quy định. Hằng năm, Công ty ký hợp đồng với các đơn vị có chức năng thu gom và xử lý chất thải. Toàn bộ chất thải từ hoạt động sản xuất và sinh "}\n{"id": "B-A-829f2f80f8ee3bbc", "claim": "Sau đó, chất thải được bàn giao định kỳ cho các đơn vị thu gom để tái chế, tái sử dụng hoặc xử lý theo quy định của pháp luật.", "evidence": "PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt. Để kiểm soát chất thải và hạn chế tác động đến môi trường, DHG Pharma triển khai nghiêm túc công tác quản lý chất thải theo đúng quy định. Hằng năm, Công ty ký hợp đồng với các đơn vị có chức năng thu gom và xử lý chất thải. Toàn bộ chất thải từ hoạt động sản xuất và sinh hoạt được thu gom, phân loại và xử lý theo đúng quy định, đảm bảo tuân thủ các tiêu chuẩn môi trường hiện hành"}\n{"id": "B-A-1d2f788c70f34712", "claim": "Toàn bộ quy trình thu gom và xử lý chất thải rắn thông thường được kiểm soát chặt chẽ nhằm đảm bảo tuân thủ các tiêu chuẩn về bảo vệ môi trường.", "evidence": "Để kiểm soát chất thải và hạn chế tác động đến môi trường, DHG Pharma triển khai nghiêm túc công tác quản lý chất thải theo đúng quy định. Hằng năm, Công ty ký hợp đồng với các đơn vị có chức năng thu gom và xử lý chất thải. Toàn bộ chất thải từ hoạt động sản xuất và sinh hoạt được thu gom, phân loại và xử lý theo đúng quy định, đảm bảo tuân thủ các tiêu chuẩn môi trường hiện hành. PHÁT TRIỂN Đạo đức kinh doanh BỀN VỮNG MÔI TRƯỜNG XÃ HỘI Tuân thủ các quy định Vận hành hệ thống xử lý nước thải đảm bảo chất lượng nước sau xử lý đạt tiêu chuẩn trước khi thải ra môi trường hoặc tái sử dụng. Thực hiện phân loại, thu gom và xử lý chất thải (đặc biệt là chất thải nguy hại) theo quy định nghiêm ngặt"}\n{"id": "B-A-53bb5bd20a5f52e1", "claim": "Năm 2024, DHG Pharma tiếp tục triển khai và duy trì hiệu quả các biện pháp giám sát, kiểm soát chặt chẽ nguồn nước thải, thể hiện rõ nét cam kết của Công ty trong việc bảo vệ môi trường và phát triển bền vững.", "evidence": "TRÁCH NHIỆM VỚI MÔI TRƯỜNG Nước và nước thải Tại DHG Pharma, nước không chỉ là nguồn tài nguyên thiết yếu phục vụ sản xuất mà còn đóng vai trò then chốt trong hoạt động vận hành hàng ngày. Trước những thách thức từ nguồn cung hạn chế, ô nhiễm môi trường và biến đổi khí hậu ngày càng rõ nét, việc sử dụng nước một cách tiết kiệm, hiệu quả đã trở thành chiến lược sống còn để duy trì ổn định và phát triển bền vững. Kiểm soát chặt chẽ việc sử dụng tài nguyên, năng lượng, nước và phát thải trong quá trình sản xuất. MÔI TRƯỜNG Thực hiện các biện pháp giảm thiểu khí thải nhà kính, tái chế chất thải và đầu tư vào công nghệ sản xuất sạch hơn. Định kỳ đánh giá tác động môi trường và tuân thủ đầy đủ các"}\n{"id": "B-A-55a055ff101e4cec", "claim": "Tại Việt Nam, các hiện tượng lũ lụt, hạn hán và thời tiết bất thường xuất hiện ngày càng thường xuyên và khắc nghiệt hơn, đặt ra yêu cầu cấp thiết về khả năng thích ứng, chống chịu và hướng tới phát triển bền vững cho các doanh nghiệp.", "evidence": "BÁO CÁO QUẢN TRỊ RỦI RO Ngoài các bộ phận chuyên trách về tuân thủ và kiểm soát độc lập, DHG Pharma còn thường xuyên phối hợp chặt chẽ với Công ty mẹ Taisho nhằm rà soát, đánh giá, cải tiến và giám sát hệ thống kiểm soát nội bộ. Sự hợp tác này giúp DHG Pharma nâng cao tính hiệu quả, đồng bộ và toàn diện trong quản trị rủi ro, đảm bảo hệ thống kiểm soát nội bộ ngày càng hoàn thiện và thích ứng, đáp ứng yêu cầu phát triển bền vững trong bối cảnh thị trường ngày càng cạnh tranh. DHG Pharma cam kết tiếp tục tối ưu hóa các nguồn lực để tạo ra các giá trị dài hạn, đầu tư mạnh mẽ vào nghiên cứu và phát triển, ứng dụng công nghệ tiên tiến để tạo ra những sản phẩm chất lượng vượt trội, hướng tới phục"}\n{"id": "B-A-ef5f24c7a1198889", "claim": "Công ty đã và đang triển khai đồng bộ nhiều giải pháp thích ứng và ứng phó hiệu quả, nhằm giảm thiểu tối đa những tác động tiêu cực, đồng thời tận dụng cơ hội để nâng cao năng lực vận hành, hướng đến các mục tiêu phát triển bền vững trong tương lai.", "evidence": "Báo cáo Phát triển bền vững năm 2024 được trình bày và công bố các thông tin chính thức liên quan đến định hướng chiến lược, mục tiêu và kết quả triển khai các chương trình hành động, sáng kiến phát triển bền vững của DHG Pharma đã và đang thực hiện trong năm 2024, làm cơ sở để đánh giá hiệu quả trong công tác thực hiện phát triển bền vững cho những năm tiếp theo. Biến đổi khí hậu Cam kết ứng phó biến đổi khí hậu bằng lộ trình giảm phát thải khí nhà kính, hướng tới Net Zero quốc gia. Ưu tiên công nghệ sạch, năng lượng tái tạo và chủ động thích ứng để đảm bảo hoạt động bền vững. Biến đổi khí hậu Phát triển vững mạnh, duy trì vị thế dẫn đầu ngành dược Việt Nam, nâng cao vai trò quản trị, tối ư"}\n{"id": "B-A-bfbab78525545d40", "claim": "Phúc lợi cho khách hàng Với triết lý lấy khách hàng làm trung tâm, DHG Pharma không ngừng nâng cao chất lượng dịch vụ và xây dựng các chính sách phúc lợi thiết thực, đồng hành cùng các nhà thuốc, chuỗi nhà thuốc lớn trong hành trình phát triển bền vững.", "evidence": "Tập trung vào sự gắn kết của khách hàng bằng các chính sách bán hàng, chương trình khách hàng thân thiết để nâng cao thị phần số 1 tại kênh bán lẻ. Xây dựng hoàn thiện hệ thống quản trị rủi ro, không ngừng nâng cao năng lực xây dựng, triển khai, thực thi công tác quản trị Công ty trên cơ sở tuân thủ Điều lệ, quy định của Pháp luật; Quy chế Công ty; Gia tăng thị phần tại các chuỗi nhà thuốc THÔNG TIN KHÁI QUÁT VỀ DHG PHARMA Lấy con người làm trung tâm CỦA MỌI SỰ PHÁT TRIỂN 50 năm không ngừng phát triển, DHG Pharma không ngừng tiếp nối và nuôi dưỡng “môi trường làm việc hạnh phúc”, lấy chế độ lương thưởng làm đòn bẩy thúc đẩy sức sáng tạo, chế độ phúc lợi toàn diện"}\n{"id": "B-A-603628f0f3ca1d13", "claim": "Với những nỗ lực toàn diện này, DHG Pharma không chỉ giảm thiểu tối đa các rủi ro về tai nạn lao động, mà còn khẳng định trách nhiệm xã hội và cam kết lâu dài đối với người lao động, đóng góp tích cực vào mục tiêu phát triển bền vững chung.", "evidence": "Doanh nghiệp thường xuyên tổ chức các chương trình khám bệnh, cấp thuốc miễn phí, tài trợ cho các quỹ từ thiện và các chương trình hỗ trợ trẻ em, người già và hoàn cảnh khó khăn. TRÁCH NHIỆM VỚI CỘNG ĐỒNG ĐỊA PHƯƠNG Những hoạt động này không chỉ thể hiện trách nhiệm xã hội mà còn lan tỏa tinh thần nhân ái, góp phần nâng cao chất lượng cuộc sống và phát triển cộng đồng bền vững. (Vui lòng xem thêm nội dung đánh giá chi tiết về Người lao động, trách nhiệm của Doanh nghiệp và các chỉ tiêu môi trường tại Chương 5 Báo cáo phát triển bền vững của Công ty) Công ty đã và đang triển khai đồng bộ nhiều giải pháp thích ứng và ứng phó hiệu quả, nhằm giảm thiểu tối đa những tác động tiêu cực, đồng thời t"}\n{"id": "B-A-6c3ca1d1473dd509", "claim": "Bảo mật thông tin khách hàng DHG Pharma 2 6 1 3 4 Phát triển bền vững 5.", "evidence": "6.000 khách hàng Hệ thống chăm sóc khách hàng vận hành hiệu quả 24/7 Không ghi nhận trường hợp nào vi phạm liên quan đến Quyền bảo mật thông tin khách hàng. DHG Pharma 2 6 1 3 4 Phát triển bền vững DHG Pharma 2 6 1 3 4 Phát triển bền vững 5. 132 133 VỮNG BƯỚC VƯƠN XA M NH MẼ CHUYỂN MÌNH Báo cáo thường niên 2024"}\n{"id": "B-A-f6961dd6e70298da", "claim": "Dự án “Toa Thuốc Vì Cộng Đồng” vừa được vinh danh với hạng mục “Dự án bền bỉ” (giải thưởng cấp quốc gia), khẳng định sự đóng góp ý nghĩa và bền vững trong lĩnh vực chăm sóc sức khỏe cộng đồng Trước hoàn cảnh đó, DHG Pharma đã triển khai các hoạt động hỗ trợ khẩn cấp với tổng kinh phí lên đến hơn 2,5 tỷ đồng.", "evidence": "Năm 2024, đánh dấu cột mốc ý nghĩa khi dự án “Toa thuốc vì cộng đồng” của DHG Pharma được vinh danh ở hạng mục “Dự án Bền bỉ” của Giải thưởng Hành động vì cộng đồng - Human Act Prize 2024 với chủ đề “Cộng đồng kiến tạo”. Đây chính là sự ghi nhận xứng đáng cho hơn 2 thập kỷ kiên định mang sức khỏe đến với người nghèo, là minh chứng cho sự tận tâm và bền bỉ vì một cộng đồng khỏe mạnh, hạnh phúc. PHÁT TRIỂN XÃ HỘI BỀN VỮNG PHÁT TRIỂN CỘNG ĐỒNG “TOA THUỐC VÌ CỘNG ĐỒNG” – chuyến xe nghĩa tình và dấu ấn được vinh danh Trong số những dự án cộng đồng tâm huyết, “Toa thuốc vì cộng đồng” nổi lên như minh chứng sống động nhất cho cam kết bền bỉ của DHG Pharma."}\n{"id": "B-A-7371706c8c9bc64f", "claim": "Định hướng bền vững - vì một tương lai khỏe đẹp hơn Hơn 5 thập kỷ đã qua, DHG Pharma luôn kiên định với con đường phát triển bền vững và giá trị sẻ chia dành cho cộng đồng.", "evidence": "Chúng tôi hướng tới việc tạo ra giá trị bền vững cho xã hội và thực hiện sứ mệnh: “Vì một cuộc sống khỏe đẹp hơn.” Với những kinh nghiệm và nền tảng tích luỹ được qua quá trình 50 năm hoạt động và phát triển, DHG Pharma luôn sẵn sàng đối diện với những cơ hội và thách thức, chủ động tìm kiếm những giải pháp trong mọi hoạt động sản xuất kinh doanh. Bên cạnh đó, DHG Pharma không ngừng cải tiến mô hình quản trị, nâng cao chất lượng sản phẩm và dịch vụ, luôn hướng đến mục tiêu gia tăng giá trị bền vững cho tất cả các bên liên quan. Hành trình hơn 5 thập kỷ hoạt động và kiên định với sứ mệnh “Vì một cuộc sống khỏe đẹp hơn”, DHG Pharma không chỉ dừng lại ở việc sản xuất và cung cấp những viên thuố"}\n{"id": "B-A-70c5799902324144", "claim": "02 11 02 0,3 TĐ-HR Công ty CP CNMT Trái Đất Xanh qua sử dụng Tổng Chất thải rắn sinh hoạt 999,2 8 Bùn thải từ hệ thống 03 05 08 56,3 TĐ-HR Công ty CP CNMT Trái Đất Xanh", "evidence": "3 Các loại dầu 17 02 04 0,0 TĐ-HR Công ty CP CNMT Trái Đất Xanh động cơ thải 4 Giẻ lau bị nhiễm các Công ty CP CNMT Trái Đất Xanh + TỔNG LƯỢNG CHẤT THẢI RẮN KHÔNG NGUY HẠI 18 02 01 (triệu đồng) Tổ chức tiếp nhận và xử lý 13 Chất thải rắn có các 03 05 09 99,4 TĐ-HR Công ty CP CNMT Trái Đất Xanh thành phần nguy hại"}\n',
}
for name, text in FILES.items():
    open(name, 'w', encoding='utf-8').write(text)
    print('wrote', name, len(text), flush=True)


In [ ]:
print('[cell] run 1: INPUTS=gsi_layer_b3.jsonl python -u multirun.py jobs_32b.jso', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('INPUTS=gsi_layer_b3.jsonl python -u multirun.py jobs_32b.json issue_worker.py')


In [ ]:
print('[cell] run 2: ls -la out; cat out/*_summary.json', flush=True)
import subprocess, sys, os
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=dict(os.environ, PYTHONUNBUFFERED="1"))
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait():
        raise SystemExit(f"command failed (rc {p.returncode}): {cmd}")
sh('ls -la out; cat out/*_summary.json')


In [ ]:
print("[cell] done", flush=True)
print("ALL CELLS OK", flush=True)